# Klaus Tschira pre-proposal analysis: neurovascular screening in metastasis

This notebook is for **pre-draft discovery only**. It does not write proposal text.

Primary question:

> Which signals from lung-innervating neurons are received by endothelial/perivascular cells, change with vascular aging, and intersect with the endothelial response to early metastatic seeding?

The workflow is deliberately **screen-first**, not candidate-first:

1. Download and cache public GEO datasets directly in Google Drive.
2. Build a pulmonary sensory-neuron signalling catalogue.
3. Build a ligand–receptor network using OmniPath and human→mouse orthology.
4. Measure ligand expression across pulmonary sensory-neuron subtypes.
5. Measure receptor expression and age effects in young vs aged lung vascular cells.
6. Measure receptor/pathway changes during early B16F10 metastatic seeding.
7. Add pericyte and independent vascular datasets.
8. Rank ligand–receptor axes using convergent evidence.
9. Export only compact tables/figures for review.

Designed for **Google Colab**, but it also works in local Jupyter.

## Runtime recommendations

For Colab:

- Runtime → Change runtime type → **High-RAM** if available.
- Standard CPU is enough for the first-pass screen.
- Leave `DEEP_GSE253749 = False` initially.
- Large raw files remain in Google Drive; you do not need to upload them to ChatGPT.

Outputs are saved under:

`MyDrive/KT_Neurovascular_Screen/outputs/`

In [ ]:
%pip install -q \
    "pandas==2.2.3" \
    "scanpy==1.11.5" \
    "anndata==0.12.6" \
    omnipath \
    gprofiler-official \
    statsmodels \
    pyarrow \
    openpyxl \
    pyreadr \
    beautifulsoup4

In [ ]:
import sys
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
import scanpy as sc
import anndata as ad

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Scanpy:", sc.__version__)
print("AnnData:", ad.__version__)

print("\nALL IMPORTS OK")

In [ ]:
from pathlib import Path
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/KT_Neurovascular_Screen")
else:
    ROOT = Path.cwd() / "KT_Neurovascular_Screen"

RAW = ROOT / "raw"
EXTRACTED = ROOT / "extracted"
CACHE = ROOT / "cache"
OUT = ROOT / "outputs"
FIG = OUT / "figures"
for p in [RAW, EXTRACTED, CACHE, OUT, FIG]:
    p.mkdir(parents=True, exist_ok=True)

DOWNLOAD_GSE253749 = True
DEEP_GSE253749 = False
RUN_OPTIONAL_VALIDATION = False
FORCE_REDOWNLOAD = False

print("ROOT:", ROOT)
print("IN_COLAB:", IN_COLAB)

In [ ]:
import os, sys, re, io, gc, math, json, gzip, tarfile, shutil, hashlib, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy
from scipy import sparse
from scipy.io import mmread
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import scanpy as sc
import anndata as ad
import statsmodels.api as sm

np.random.seed(17)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

versions = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scanpy": sc.__version__,
    "anndata": ad.__version__,
}
version_series = pd.Series(versions, name="version")
display(version_series)

# Save only if project paths have already been initialized
if "OUT" in globals():
    version_series.to_csv(OUT / "software_versions.csv")

## Dataset registry

### Core discovery
- **GSE186180** — pulmonary sensory-neuron Smart-seq2 atlas.
- **GSE124872** — young (3 mo) vs aged (24 mo) whole-lung scRNA-seq.
- **GSE253749** — control, 6 h and 30 h B16F10 lung endothelial scRNA-seq.
- **GSE137363** — lung pericyte-like cell RNA-seq.
- **GSE210291** — Ashik/Augustin lung endothelial bulk RNA-seq.
- **GSE278974** — endothelial/fibroblast RNA-seq from B16F10 lung melanoma models.

The notebook also contains helpers for optional validation datasets.

In [ ]:
def geo_bucket(accession):
    m = re.fullmatch(r"(GSE)(\d+)", accession)
    if not m:
        raise ValueError(accession)
    digits = m.group(2)
    return "GSE" + digits[:-3] + "nnn"

def geo_suppl_url(accession, filename):
    return f"https://ftp.ncbi.nlm.nih.gov/geo/series/{geo_bucket(accession)}/{accession}/suppl/{filename}"

CORE_FILES = {
    "GSE186180": [
        "GSE186180_TPM_table_PSN.txt.gz",
        "GSE186180_Subtype_identity_PSN.txt.txt.gz",
    ],
    "GSE124872": [
        "GSE124872_Angelidis_2018_metadata.csv.gz",
        "GSE124872_raw_counts_single_cell.mtx.gz",
        "GSE124872_raw_counts_single_cell.RData.gz",
    ],
    "GSE253749": ["GSE253749_RAW.tar"],
    "GSE137363": ["GSE137363_processed_data_counts.txt.gz"],
    "GSE210291": ["GSE210291_Count_matrix.tsv.gz"],
    "GSE278974": ["GSE278974_B16F10_counts_matrix.csv.gz"],
}

OPTIONAL_ACCESSIONS = [
    "GSE235394", "GSE281219", "GSE233052",
    "GSE216592", "GSE275763", "GSE275770"
]

for acc, files in CORE_FILES.items():
    print(acc)
    for fn in files:
        print(" ", geo_suppl_url(acc, fn))

In [ ]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Mozilla/5.0 KT-neurovascular-screen/1.0"})

def download_file(url, dest, force=False, chunk_mb=4, retries=5):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0 and not force:
        print(f"SKIP  {dest.name}  ({dest.stat().st_size/1e6:.1f} MB)")
        return dest

    tmp = Path(str(dest) + ".part")
    if force and tmp.exists():
        tmp.unlink()

    for attempt in range(1, retries + 1):
        try:
            existing = tmp.stat().st_size if tmp.exists() else 0
            headers = {"Range": f"bytes={existing}-"} if existing else {}
            with SESSION.get(url, stream=True, timeout=(30, 300), headers=headers) as r:
                if existing and r.status_code == 200:
                    existing = 0
                    tmp.unlink(missing_ok=True)
                r.raise_for_status()
                total = r.headers.get("Content-Length")
                total = int(total) + existing if total else None
                mode = "ab" if existing else "wb"
                with open(tmp, mode) as fh, tqdm(
                    total=total, initial=existing, unit="B",
                    unit_scale=True, desc=dest.name
                ) as bar:
                    for chunk in r.iter_content(chunk_size=chunk_mb * 1024 * 1024):
                        if chunk:
                            fh.write(chunk)
                            bar.update(len(chunk))
            tmp.replace(dest)
            return dest
        except Exception as e:
            print(f"Attempt {attempt}/{retries} failed: {e}")
            if attempt == retries:
                raise
            time.sleep(2 ** attempt)

def md5sum(path, block=1024*1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(block)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def list_geo_supplementary(accession):
    base = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{geo_bucket(accession)}/{accession}/suppl/"
    r = SESSION.get(base, timeout=60)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    files = []
    for a in soup.find_all("a"):
        href = a.get("href", "")
        if href and href not in ("../", "/") and not href.startswith("?") and not href.endswith("/"):
            files.append(href)
    return base, sorted(set(files))

In [ ]:
manifest_rows = []
for acc, files in CORE_FILES.items():
    if acc == "GSE253749" and not DOWNLOAD_GSE253749:
        print("Skipping GSE253749 by configuration.")
        continue
    acc_dir = RAW / acc
    acc_dir.mkdir(parents=True, exist_ok=True)
    for fn in files:
        url = geo_suppl_url(acc, fn)
        dest = acc_dir / fn
        path = download_file(url, dest, force=FORCE_REDOWNLOAD)
        manifest_rows.append({
            "accession": acc,
            "filename": fn,
            "bytes": path.stat().st_size,
            "mb": path.stat().st_size / 1e6,
            "md5_local": md5sum(path),
            "url": url,
        })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUT / "download_manifest.csv", index=False)
display(manifest)

In [ ]:
def check_gzip(path):
    with gzip.open(path, "rb") as f:
        _ = f.read(1024)
    return True

audit = []
for _, row in manifest.iterrows():
    p = RAW / row["accession"] / row["filename"]
    status, note = "OK", ""
    try:
        if p.suffix == ".gz":
            check_gzip(p)
        elif p.suffix == ".tar":
            with tarfile.open(p, "r") as tf:
                note = f"{len(tf.getmembers())} tar members"
    except Exception as e:
        status, note = "FAIL", str(e)
    audit.append({"file": str(p), "status": status, "note": note})

audit = pd.DataFrame(audit)
audit.to_csv(OUT / "file_integrity_audit.csv", index=False)
display(audit)
assert (audit["status"] == "OK").all()

# Phase A — pulmonary sensory-neuron signalling screen

The data nominate signals. We do not pre-select CGRP, Substance P, adrenergic signalling, BDNF, etc.

In [ ]:
# ============================================================
# CELL 9 — CORRECT GSE186180 PSN SUBTYPE PARSING
# ============================================================

import csv
import gzip
import pandas as pd
import numpy as np


psn_tpm_path = (
    RAW
    / "GSE186180"
    / "GSE186180_TPM_table_PSN.txt.gz"
)

psn_sub_path = (
    RAW
    / "GSE186180"
    / "GSE186180_Subtype_identity_PSN.txt.txt.gz"
)


# ------------------------------------------------------------
# 1. Load TPM expression matrix
# ------------------------------------------------------------

psn = pd.read_csv(
    psn_tpm_path,
    sep="\t",
    compression="gzip",
    index_col=0
)

psn.index = psn.index.astype(str)
psn.columns = psn.columns.astype(str)

psn = (
    psn
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)

# Collapse duplicate gene symbols if present
psn = psn.groupby(psn.index).mean()


print(
    "Original PSN matrix:",
    psn.shape
)


# ------------------------------------------------------------
# 2. Read subtype file properly as QUOTED TSV
#
# Actual rows look like:
# "1"    "N5"    "PSN7"
#
# Header is:
# "V1"   "V2"
# ------------------------------------------------------------

rows = []

with gzip.open(
    psn_sub_path,
    "rt",
    encoding="utf-8-sig"
) as fh:

    reader = csv.reader(
        fh,
        delimiter="\t",
        quotechar='"'
    )

    for row in reader:

        if row:
            rows.append(row)


print(
    "Subtype file rows:",
    len(rows)
)

print(
    "Header:",
    rows[0]
)

print(
    "First 10 parsed rows:"
)

for r in rows[1:11]:
    print(r)


# ------------------------------------------------------------
# 3. Parse:
#
# column 0 = R row number
# column 1 = neuron/cell ID
# column 2 = subtype label
# ------------------------------------------------------------

records = []

for row in rows[1:]:

    if len(row) < 3:
        raise ValueError(
            f"Unexpected subtype row: {row}"
        )

    cell_id = str(
        row[1]
    ).strip()

    subtype = str(
        row[2]
    ).strip()

    records.append({
        "cell_id": cell_id,
        "subtype_raw": subtype
    })


sub_df = pd.DataFrame(
    records
)


# ------------------------------------------------------------
# 4. Validate exact alignment to TPM cells
# ------------------------------------------------------------

print(
    "\nSubtype records:",
    len(sub_df)
)

print(
    "TPM cells:",
    psn.shape[1]
)


if sub_df["cell_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate cell IDs in subtype file."
    )


tpm_cells = set(
    psn.columns
)

sub_cells = set(
    sub_df["cell_id"]
)


missing_annotation = (
    tpm_cells - sub_cells
)

extra_annotation = (
    sub_cells - tpm_cells
)


print(
    "Missing annotations:",
    len(missing_annotation)
)

print(
    "Extra annotations:",
    len(extra_annotation)
)


if missing_annotation:

    print(
        "Missing examples:",
        list(missing_annotation)[:20]
    )

    raise RuntimeError(
        "Some TPM cells have no subtype annotation."
    )


if extra_annotation:

    print(
        "Extra examples:",
        list(extra_annotation)[:20]
    )

    raise RuntimeError(
        "Subtype file contains cells absent from TPM matrix."
    )


# ------------------------------------------------------------
# 5. Mark the explicitly designated outliers
# ------------------------------------------------------------

sub_df["is_outlier"] = (
    sub_df["subtype_raw"]
    .str.lower()
    .str.startswith(
        "outlier",
        na=False
    )
)


print(
    "\nExplicit outlier-cross cells:",
    int(
        sub_df["is_outlier"].sum()
    )
)


print(
    "\nOUTLIERS:"
)

display(
    sub_df.loc[
        sub_df["is_outlier"],
        [
            "cell_id",
            "subtype_raw"
        ]
    ]
)


# ------------------------------------------------------------
# 6. Keep only bona fide PSN molecular subtypes
# ------------------------------------------------------------

valid_sub = (
    sub_df.loc[
        ~sub_df["is_outlier"]
    ]
    .copy()
)


expected_subtypes = {
    "PSN-H",
    "PSN1",
    "PSN2",
    "PSN3",
    "PSN4",
    "PSN5",
    "PSN6",
    "PSN7",
    "PSN8",
    "PSN9"
}


observed_subtypes = set(
    valid_sub[
        "subtype_raw"
    ]
)


print(
    "\nObserved valid subtypes:",
    sorted(observed_subtypes)
)


if observed_subtypes != expected_subtypes:

    raise RuntimeError(
        "Unexpected PSN subtype set.\n"
        f"Observed: {sorted(observed_subtypes)}\n"
        f"Expected: {sorted(expected_subtypes)}"
    )


# ------------------------------------------------------------
# 7. Build subtype Series
# ------------------------------------------------------------

subtype_map = (
    valid_sub
    .set_index(
        "cell_id"
    )["subtype_raw"]
)


valid_cells = [
    c
    for c in psn.columns
    if c in subtype_map.index
]


# IMPORTANT:
# remove the four explicit outlier cells from expression analysis
psn = psn.loc[
    :,
    valid_cells
].copy()


psn_subtype = (
    subtype_map
    .reindex(
        valid_cells
    )
)

psn_subtype.name = "subtype"


# ------------------------------------------------------------
# 8. Final validation
# ------------------------------------------------------------

assert psn.shape[1] == len(
    psn_subtype
)

assert psn_subtype.notna().all()

assert psn_subtype.nunique() == 10


print(
    "\n======================================"
)

print(
    "CORRECT PSN DATASET"
)

print(
    "======================================"
)

print(
    "Valid PSN cells:",
    psn.shape[1]
)

print(
    "Genes:",
    psn.shape[0]
)

print(
    "Number of molecular subtypes:",
    psn_subtype.nunique()
)


print(
    "\nSubtype counts:"
)

display(
    psn_subtype
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 9. Save transparent mapping
# ------------------------------------------------------------

sub_df.to_csv(
    OUT
    / "GSE186180_PSN_subtype_mapping_FULL_with_outliers.csv",
    index=False
)

psn_subtype.to_csv(
    OUT
    / "GSE186180_PSN_subtype_mapping_VALID.csv",
    header=True
)


print(
    "\nCELL 9 CORRECTED"
)

In [ ]:
expr = psn.copy()
psn_stats = pd.DataFrame(index=expr.index)
psn_stats["psn_mean"] = expr.mean(axis=1)
psn_stats["psn_median"] = expr.median(axis=1)
psn_stats["psn_max"] = expr.max(axis=1)
psn_stats["psn_pct_nonzero"] = (expr > 0).mean(axis=1)

subtype_means = {}
for st in sorted(psn_subtype.dropna().unique()):
    cols = psn_subtype.index[psn_subtype == st].intersection(expr.columns)
    if len(cols):
        subtype_means[st] = expr.loc[:, cols].mean(axis=1)

psn_by_subtype = pd.DataFrame(subtype_means)
psn_stats["psn_max_subtype_mean"] = psn_by_subtype.max(axis=1)
psn_stats["psn_top_subtype"] = psn_by_subtype.idxmax(axis=1)
psn_stats["psn_subtype_specificity"] = psn_stats["psn_max_subtype_mean"] / (psn_stats["psn_mean"] + 1e-9)

for col in ["psn_mean", "psn_pct_nonzero", "psn_max_subtype_mean", "psn_subtype_specificity"]:
    psn_stats[col + "_pctile"] = psn_stats[col].rank(pct=True)

psn_stats["psn_expression_score"] = (
    0.35 * psn_stats["psn_mean_pctile"] +
    0.25 * psn_stats["psn_pct_nonzero_pctile"] +
    0.25 * psn_stats["psn_max_subtype_mean_pctile"] +
    0.15 * psn_stats["psn_subtype_specificity_pctile"]
)

psn_stats = psn_stats.sort_values("psn_expression_score", ascending=False)
psn_stats.to_csv(OUT / "GSE186180_PSN_gene_expression_scores.csv")
psn_by_subtype.to_csv(OUT / "GSE186180_PSN_subtype_mean_expression.csv")
display(psn_stats.head(30))

# Phase B — ligand–receptor universe

Uses OmniPath ligand→receptor interactions, then maps human symbols to mouse orthologues using g:Profiler g:Orth.

In [ ]:
# ============================================================
# PHASE B — DEDICATED OMNIPATH LIGAND–RECEPTOR DATASET
# ============================================================

import omnipath as op
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Use LigRecExtra directly
#
# IMPORTANT:
# Do NOT construct LR pairs from the general signaling network.
# LigRecExtra is OmniPath's dedicated ligand–receptor dataset.
# ------------------------------------------------------------

ligrec_cache = CACHE / "omnipath_LigRecExtra_raw.parquet"

if ligrec_cache.exists():

    lr_raw = pd.read_parquet(ligrec_cache)

    print("Loaded cached LigRecExtra.")

else:

    print("Downloading OmniPath LigRecExtra...")

    lr_raw = op.interactions.LigRecExtra.get(
        genesymbols=True
    )

    lr_raw.to_parquet(
        ligrec_cache,
        index=False
    )


print(
    "Raw LigRecExtra rows:",
    len(lr_raw)
)

print(
    "\nColumns:"
)

print(
    list(lr_raw.columns)
)


# ------------------------------------------------------------
# 2. Identify ligand/receptor gene-symbol columns
# ------------------------------------------------------------

def first_existing(cols, candidates):

    for c in candidates:

        if c in cols:
            return c

    raise KeyError(
        f"Could not find any of: {candidates}"
    )


src_gene_col = first_existing(
    lr_raw.columns,
    [
        "source_genesymbol",
        "genesymbol_source",
        "source"
    ]
)

tgt_gene_col = first_existing(
    lr_raw.columns,
    [
        "target_genesymbol",
        "genesymbol_target",
        "target"
    ]
)


print(
    "\nLigand column:",
    src_gene_col
)

print(
    "Receptor column:",
    tgt_gene_col
)


# ------------------------------------------------------------
# 3. Build clean gene-symbol LR table
# ------------------------------------------------------------

lr = lr_raw.rename(
    columns={
        src_gene_col: "human_ligand",
        tgt_gene_col: "human_receptor"
    }
).copy()


lr["human_ligand"] = (
    lr["human_ligand"]
    .astype(str)
    .str.strip()
)

lr["human_receptor"] = (
    lr["human_receptor"]
    .astype(str)
    .str.strip()
)


bad = {
    "",
    "nan",
    "None",
    "NA",
    "N/A"
}


lr = lr[
    ~lr["human_ligand"].isin(bad)
    &
    ~lr["human_receptor"].isin(bad)
].copy()


# Remove self-interactions for this screen
lr = lr[
    lr["human_ligand"]
    !=
    lr["human_receptor"]
].copy()


# ------------------------------------------------------------
# 4. Preserve provenance/evidence columns
# ------------------------------------------------------------

evidence_cols = [
    c
    for c in [
        "sources",
        "references",
        "curation_effort",
        "n_references",
        "n_resources",
        "consensus_direction",
        "consensus_stimulation",
        "consensus_inhibition"
    ]
    if c in lr.columns
]


lr = lr[
    [
        "human_ligand",
        "human_receptor"
    ]
    +
    evidence_cols
].copy()


lr = lr.drop_duplicates()


# ------------------------------------------------------------
# 5. Pair-level summary
# ------------------------------------------------------------

pair_cols = [
    "human_ligand",
    "human_receptor"
]


print(
    "\n======================================"
)

print(
    "OMNIPATH LigRecExtra NETWORK"
)

print(
    "======================================"
)


print(
    "Rows:",
    len(lr)
)

print(
    "Unique ligands:",
    lr["human_ligand"].nunique()
)

print(
    "Unique receptors:",
    lr["human_receptor"].nunique()
)

print(
    "Unique LR pairs:",
    lr[pair_cols]
    .drop_duplicates()
    .shape[0]
)


# ------------------------------------------------------------
# 6. Canonical positive controls
# ------------------------------------------------------------

def pair_present(ligand, receptor):

    return (
        (
            lr["human_ligand"].str.upper()
            == ligand.upper()
        )
        &
        (
            lr["human_receptor"].str.upper()
            == receptor.upper()
        )
    ).any()


print(
    "\nCanonical checks:"
)

for lig, rec in [
    ("CALCA", "CALCRL"),
    ("TAC1", "TACR1"),
    ("BDNF", "NTRK2"),
    ("VEGFA", "KDR"),
    ("PDGFB", "PDGFRB")
]:

    print(
        f"{lig} -> {rec}:",
        pair_present(lig, rec)
    )


# ------------------------------------------------------------
# 7. Explicit contamination diagnostics
# ------------------------------------------------------------

bad_examples = [
    ("FGF13", "VEGFA"),
    ("S100B", "MAP3K5"),
    ("S100B", "CASP2"),
    ("S100B", "RIPK1"),
    ("FGF13", "MAPK1"),
    ("FGF13", "PIK3CA")
]


print(
    "\nPairs that should NOT survive as direct LR pairs:"
)

bad_found = []

for lig, rec in bad_examples:

    present = pair_present(
        lig,
        rec
    )

    print(
        f"{lig} -> {rec}:",
        present
    )

    if present:
        bad_found.append(
            (lig, rec)
        )


# ------------------------------------------------------------
# 8. Save
# ------------------------------------------------------------

lr.to_csv(
    OUT
    / "omnipath_human_ligand_receptor_pairs.csv",
    index=False
)


print(
    "\nBad diagnostic pairs remaining:",
    bad_found
)


if bad_found:

    print(
        "\nWARNING: unexpected pairs remain."
        " Stop and send this output."
    )

else:

    print(
        "\nPHASE B LIGREC NETWORK CLEAN"
    )

In [ ]:
# ============================================================
# PHASE B2 — HIGH-CONFIDENCE INTERCELL FILTER
# Robust pandas-compatible implementation
# ============================================================

import omnipath as op
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# Helper: robust conversion of annotation flags to boolean
# ------------------------------------------------------------

def as_true(x):
    if pd.api.types.is_bool_dtype(x):
        return x.fillna(False)

    return (
        x.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes", "t"])
    )


# ============================================================
# 1. REBUILD LigRecExtra CLEANLY FROM THE ORIGINAL CACHE
#
# IMPORTANT:
# Never use an older/mutated `lr` object here.
# ============================================================

ligrec_cache = CACHE / "omnipath_LigRecExtra_raw.parquet"

if ligrec_cache.exists():

    lr0 = pd.read_parquet(ligrec_cache)
    print("Loaded cached OmniPath LigRecExtra.")

else:

    print("Downloading OmniPath LigRecExtra...")

    lr0 = op.interactions.LigRecExtra.get(
        genesymbols=True
    )

    lr0.to_parquet(
        ligrec_cache,
        index=False
    )


def first_existing(cols, candidates):

    for c in candidates:
        if c in cols:
            return c

    raise KeyError(
        f"Could not identify column. Tried: {candidates}"
    )


src_gene_col = first_existing(
    lr0.columns,
    [
        "source_genesymbol",
        "genesymbol_source",
        "source"
    ]
)

tgt_gene_col = first_existing(
    lr0.columns,
    [
        "target_genesymbol",
        "genesymbol_target",
        "target"
    ]
)


lr_base = lr0.rename(
    columns={
        src_gene_col: "human_ligand",
        tgt_gene_col: "human_receptor"
    }
).copy()


lr_base["human_ligand"] = (
    lr_base["human_ligand"]
    .astype(str)
    .str.strip()
)

lr_base["human_receptor"] = (
    lr_base["human_receptor"]
    .astype(str)
    .str.strip()
)


bad_symbols = {
    "",
    "nan",
    "None",
    "NA",
    "N/A"
}


lr_base = lr_base[
    ~lr_base["human_ligand"].isin(bad_symbols)
    &
    ~lr_base["human_receptor"].isin(bad_symbols)
    &
    (
        lr_base["human_ligand"]
        !=
        lr_base["human_receptor"]
    )
].copy()


print(
    "Clean LigRecExtra pairs:",
    lr_base[
        ["human_ligand", "human_receptor"]
    ]
    .drop_duplicates()
    .shape[0]
)


# ============================================================
# 2. LOAD OMNIPATH INTERCELL ANNOTATIONS
# ============================================================

lig_cache = (
    CACHE
    / "omnipath_ligand_annotations_generic.parquet"
)

rec_cache = (
    CACHE
    / "omnipath_receptor_annotations_generic.parquet"
)


if lig_cache.exists():

    lig_ann = pd.read_parquet(lig_cache)

else:

    print("Downloading ligand annotations...")

    lig_ann = op.requests.Intercell.get(
        categories="ligand",
        scope="generic",
        genesymbols=True
    )

    lig_ann.to_parquet(
        lig_cache,
        index=False
    )


if rec_cache.exists():

    rec_ann = pd.read_parquet(rec_cache)

else:

    print("Downloading receptor annotations...")

    rec_ann = op.requests.Intercell.get(
        categories="receptor",
        scope="generic",
        genesymbols=True
    )

    rec_ann.to_parquet(
        rec_cache,
        index=False
    )


print(
    "Ligand annotation rows:",
    len(lig_ann)
)

print(
    "Receptor annotation rows:",
    len(rec_ann)
)


# ============================================================
# 3. SANITY-CHECK REQUIRED COLUMNS
# ============================================================

required_lig_cols = {
    "genesymbol",
    "entity_type",
    "source",
    "transmitter",
    "secreted",
    "database",
    "consensus_score"
}

required_rec_cols = {
    "genesymbol",
    "entity_type",
    "source",
    "receiver",
    "plasma_membrane_transmembrane",
    "database",
    "consensus_score"
}


missing_lig = required_lig_cols - set(lig_ann.columns)
missing_rec = required_rec_cols - set(rec_ann.columns)


if missing_lig:

    raise RuntimeError(
        f"Missing ligand annotation columns: {missing_lig}"
    )


if missing_rec:

    raise RuntimeError(
        f"Missing receptor annotation columns: {missing_rec}"
    )


# ============================================================
# 4. KEEP PROTEIN ENTITIES ONLY
# ============================================================

lig_ann = lig_ann[
    lig_ann["entity_type"]
    .astype(str)
    .str.lower()
    .eq("protein")
].copy()


rec_ann = rec_ann[
    rec_ann["entity_type"]
    .astype(str)
    .str.lower()
    .eq("protein")
].copy()


# Normalize grouping columns.
#
# This also avoids pandas categorical groupby producing
# unobserved gene/database combinations.
# ============================================================

for df in [lig_ann, rec_ann]:

    df["genesymbol"] = (
        df["genesymbol"]
        .astype(str)
        .str.strip()
    )

    df["database"] = (
        df["database"]
        .astype(str)
        .str.strip()
    )

    df["source"] = (
        df["source"]
        .astype(str)
        .str.strip()
        .str.lower()
    )


# ============================================================
# 5. GENERIC COMPOSITE CLASSIFICATION
#
# consensus_score = support across annotation resources.
# We retain classifications at or above the median support.
# ============================================================

lig_comp = lig_ann[
    lig_ann["source"].eq("composite")
].copy()


rec_comp = rec_ann[
    rec_ann["source"].eq("composite")
].copy()


lig_comp["consensus_score"] = pd.to_numeric(
    lig_comp["consensus_score"],
    errors="coerce"
)


rec_comp["consensus_score"] = pd.to_numeric(
    rec_comp["consensus_score"],
    errors="coerce"
)


lig_q50 = (
    lig_comp["consensus_score"]
    .dropna()
    .quantile(0.50)
)


rec_q50 = (
    rec_comp["consensus_score"]
    .dropna()
    .quantile(0.50)
)


print(
    "\nLigand consensus 50th percentile:",
    lig_q50
)

print(
    "Receptor consensus 50th percentile:",
    rec_q50
)


lig_good_comp = lig_comp[
    as_true(lig_comp["transmitter"])
    &
    (
        lig_comp["consensus_score"]
        >= lig_q50
    )
].copy()


rec_good_comp = rec_comp[
    as_true(rec_comp["receiver"])
    &
    (
        rec_comp["consensus_score"]
        >= rec_q50
    )
].copy()


# ============================================================
# 6. RESOURCE-SPECIFIC LOCALIZATION SUPPORT
#
# For each gene/database:
#   ligand: does the resource support secretion?
#   receptor: does it support plasma-membrane transmembrane?
#
# IMPORTANT:
# observed=True fixes the pandas categorical-groupby failure
# that stopped the original notebook.
# ============================================================

lig_rs = lig_ann[
    lig_ann["source"].eq("resource_specific")
].copy()


rec_rs = rec_ann[
    rec_ann["source"].eq("resource_specific")
].copy()


lig_rs["secreted_bool"] = as_true(
    lig_rs["secreted"]
)


rec_rs["pmtm_bool"] = as_true(
    rec_rs["plasma_membrane_transmembrane"]
)


lig_votes = (
    lig_rs[
        [
            "genesymbol",
            "database",
            "secreted_bool"
        ]
    ]
    .dropna(
        subset=[
            "genesymbol",
            "database"
        ]
    )
    .groupby(
        [
            "genesymbol",
            "database"
        ],
        observed=True,
        as_index=False
    )
    .agg(
        secreted=(
            "secreted_bool",
            "max"
        )
    )
)


rec_votes = (
    rec_rs[
        [
            "genesymbol",
            "database",
            "pmtm_bool"
        ]
    ]
    .dropna(
        subset=[
            "genesymbol",
            "database"
        ]
    )
    .groupby(
        [
            "genesymbol",
            "database"
        ],
        observed=True,
        as_index=False
    )
    .agg(
        pmtm=(
            "pmtm_bool",
            "max"
        )
    )
)


print(
    "\nObserved ligand gene/resource votes:",
    len(lig_votes)
)

print(
    "Observed receptor gene/resource votes:",
    len(rec_votes)
)


lig_localization = (
    lig_votes
    .groupby(
        "genesymbol",
        observed=True
    )
    .agg(
        secreted_resource_fraction=(
            "secreted",
            "mean"
        ),
        n_ligand_resources=(
            "database",
            "nunique"
        )
    )
)


rec_localization = (
    rec_votes
    .groupby(
        "genesymbol",
        observed=True
    )
    .agg(
        pmtm_resource_fraction=(
            "pmtm",
            "mean"
        ),
        n_receptor_resources=(
            "database",
            "nunique"
        )
    )
)


# ============================================================
# 7. JOIN GENERIC CLASSIFICATION WITH LOCALIZATION EVIDENCE
# ============================================================

lig_gene = (
    lig_good_comp[
        [
            "genesymbol",
            "consensus_score"
        ]
    ]
    .sort_values(
        "consensus_score",
        ascending=False
    )
    .drop_duplicates(
        "genesymbol"
    )
    .set_index(
        "genesymbol"
    )
    .join(
        lig_localization,
        how="left"
    )
)


rec_gene = (
    rec_good_comp[
        [
            "genesymbol",
            "consensus_score"
        ]
    ]
    .sort_values(
        "consensus_score",
        ascending=False
    )
    .drop_duplicates(
        "genesymbol"
    )
    .set_index(
        "genesymbol"
    )
    .join(
        rec_localization,
        how="left"
    )
)


# ============================================================
# 8. DEFINE HIGH-CONFIDENCE INTERCELL GENE SETS
#
# Require:
#   >= 50% localization-resource agreement
#   >= 2 independent resources
# ============================================================

secreted_ligands_hc = set(
    lig_gene.loc[
        (
            lig_gene[
                "secreted_resource_fraction"
            ]
            .fillna(0)
            >= 0.50
        )
        &
        (
            lig_gene[
                "n_ligand_resources"
            ]
            .fillna(0)
            >= 2
        )
    ].index
)


# ------------------------------------------------------------
# Biological veto: intracellular FGF homologous factors
#
# FGF11–FGF14 are FHFs / intracrine FGFs.
# They are not physiological secreted FGF ligands and should
# not enter a neuron -> vascular extracellular signalling screen.
# ------------------------------------------------------------

INTRACELLULAR_FGF_HOMOLOGS = {
    "FGF11",
    "FGF12",
    "FGF13",
    "FGF14",
}

secreted_ligands_hc = {
    gene
    for gene in secreted_ligands_hc
    if str(gene).upper()
    not in INTRACELLULAR_FGF_HOMOLOGS
}

print(
    "Excluded intracellular FGF homologs:",
    sorted(
        INTRACELLULAR_FGF_HOMOLOGS
        &
        {
            str(g).upper()
            for g in lig_gene.index
        }
    )
)


surface_receptors_hc = set(
    rec_gene.loc[
        (
            rec_gene[
                "pmtm_resource_fraction"
            ]
            .fillna(0)
            >= 0.50
        )
        &
        (
            rec_gene[
                "n_receptor_resources"
            ]
            .fillna(0)
            >= 2
        )
    ].index
)


print(
    "\nHigh-confidence secreted ligands:",
    len(secreted_ligands_hc)
)

print(
    "High-confidence surface receptors:",
    len(surface_receptors_hc)
)


if len(secreted_ligands_hc) == 0:

    raise RuntimeError(
        "No high-confidence secreted ligands survived."
    )


if len(surface_receptors_hc) == 0:

    raise RuntimeError(
        "No high-confidence surface receptors survived."
    )


# ============================================================
# 9. INTERACTION-LEVEL CURATION FILTER
# ============================================================

if "curation_effort" not in lr_base.columns:

    raise RuntimeError(
        "LigRecExtra lacks curation_effort; "
        "cannot apply interaction-confidence filter."
    )


lr_hc = lr_base.copy()


lr_hc["curation_effort"] = pd.to_numeric(
    lr_hc["curation_effort"],
    errors="coerce"
)


lr_hc = lr_hc[
    lr_hc["curation_effort"]
    .fillna(0)
    > 1
].copy()


print(
    "\nPairs after curation_effort > 1:",
    lr_hc[
        [
            "human_ligand",
            "human_receptor"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


# ============================================================
# 10. REQUIRE SECRETED LIGAND + SURFACE RECEPTOR
# ============================================================

lr_hc = lr_hc[
    lr_hc["human_ligand"]
    .isin(secreted_ligands_hc)
    &
    lr_hc["human_receptor"]
    .isin(surface_receptors_hc)
].copy()


lr_hc = (
    lr_hc
    .sort_values(
        "curation_effort",
        ascending=False
    )
    .drop_duplicates(
        [
            "human_ligand",
            "human_receptor"
        ]
    )
)


print(
    "\nFINAL HIGH-CONFIDENCE NETWORK"
)

print(
    "Pairs:",
    len(lr_hc)
)

print(
    "Ligands:",
    lr_hc["human_ligand"].nunique()
)

print(
    "Receptors:",
    lr_hc["human_receptor"].nunique()
)


if lr_hc.empty:

    raise RuntimeError(
        "High-confidence LR network is empty."
    )


# ============================================================
# 11. BIOLOGICAL SANITY CHECKS
# ============================================================

def pair_present(df, ligand, receptor):

    return (
        (
            df["human_ligand"]
            .str.upper()
            .eq(ligand.upper())
        )
        &
        (
            df["human_receptor"]
            .str.upper()
            .eq(receptor.upper())
        )
    ).any()


print(
    "\nCanonical LR controls:"
)

for ligand, receptor in [
    ("CALCA", "CALCRL"),
    ("TAC1", "TACR1"),
    ("BDNF", "NTRK2"),
    ("VEGFA", "KDR"),
    ("PDGFB", "PDGFRB")
]:

    print(
        f"{ligand} -> {receptor}:",
        pair_present(
            lr_hc,
            ligand,
            receptor
        )
    )


print(
    "\nNegative-control intracellular genes:"
)


for gene in [
    "FGF13",
    "GNAS"
]:

    present = (
        lr_hc["human_ligand"]
        .str.upper()
        .eq(gene)
        .any()
    )

    print(
        f"{gene} present as ligand:",
        present
    )


# These should not survive as diffusible neuronal ligands.
assert not (
    lr_hc["human_ligand"]
    .str.upper()
    .eq("FGF13")
    .any()
), "FGF13 incorrectly survived the ligand filter."


assert not (
    lr_hc["human_ligand"]
    .str.upper()
    .eq("GNAS")
    .any()
), "GNAS incorrectly survived the ligand filter."


# ============================================================
# 12. SAVE AUDIT TABLES
# ============================================================

lig_gene.to_csv(
    OUT
    / "omnipath_HC_ligand_annotation_audit.csv"
)


rec_gene.to_csv(
    OUT
    / "omnipath_HC_receptor_annotation_audit.csv"
)


lr_hc.to_csv(
    OUT
    / "omnipath_HC_secreted_ligand_surface_receptor_pairs.csv",
    index=False
)


# ============================================================
# 13. CRITICAL HANDOFF TO ALL DOWNSTREAM CELLS
#
# Downstream orthology MUST use this object.
# ============================================================

lr = lr_hc.copy()


print(
    "\nPHASE B2 HIGH-CONFIDENCE FILTER COMPLETE"
)

print(
    "Downstream `lr` object:",
    lr.shape
)


display(
    lr[
        [
            "human_ligand",
            "human_receptor",
            "curation_effort"
        ]
    ]
    .head(25)
)

In [ ]:
# ============================================================
# PHASE B3 — HUMAN -> MOUSE ORTHOLOGY
# Reciprocal one-to-one mapping for the primary screen
# ============================================================

from gprofiler import GProfiler
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import time

gp = GProfiler(return_dataframe=True)


# ============================================================
# 1. CURRENT HUMAN GENES FROM THE CLEAN HC NETWORK
# ============================================================

all_human_genes = sorted(
    set(lr["human_ligand"])
    |
    set(lr["human_receptor"])
)

print("Current human genes requiring orthology:", len(all_human_genes))


# ============================================================
# 2. HELPER — QUERY g:ORTH IN CHUNKS
# ============================================================

def query_orthology(genes, organism, target, label):

    if not genes:
        return pd.DataFrame()

    parts = []

    for i in tqdm(
        range(0, len(genes), 500),
        desc=label
    ):

        q = genes[i:i+500]

        x = gp.orth(
            organism=organism,
            query=q,
            target=target
        )

        if x is not None and len(x):
            parts.append(x)

        time.sleep(0.15)

    if not parts:
        return pd.DataFrame()

    return pd.concat(parts, ignore_index=True)


# ============================================================
# 3. HUMAN -> MOUSE
#
# Reuse cache, but explicitly query any genes missing from it.
# This prevents stale-cache problems.
# ============================================================

forward_cache = (
    CACHE
    / "gprofiler_human_to_mouse_orthologs.csv"
)


if forward_cache.exists():

    forward = pd.read_csv(
        forward_cache,
        keep_default_na=False
    )

    print("Loaded existing human->mouse orthology cache.")

else:

    forward = pd.DataFrame()


if len(forward) and "incoming" in forward.columns:

    cached_human = set(
        forward["incoming"]
        .astype(str)
        .str.strip()
    )

else:

    cached_human = set()


missing_human = sorted(
    set(all_human_genes)
    -
    cached_human
)


print(
    "Human genes missing from cache:",
    len(missing_human)
)


if missing_human:

    new_forward = query_orthology(
        missing_human,
        organism="hsapiens",
        target="mmusculus",
        label="Human -> mouse g:Orth"
    )

    if len(new_forward):

        forward = pd.concat(
            [
                forward,
                new_forward
            ],
            ignore_index=True
        )

        forward = forward.drop_duplicates()

        forward.to_csv(
            forward_cache,
            index=False
        )


if forward.empty:

    raise RuntimeError(
        "Human->mouse orthology query returned no data."
    )


# Restrict cache back to genes in CURRENT network.
forward = forward[
    forward["incoming"]
    .astype(str)
    .isin(all_human_genes)
].copy()


forward["incoming"] = (
    forward["incoming"]
    .astype(str)
    .str.strip()
)

forward["name"] = (
    forward["name"]
    .astype(str)
    .str.strip()
)


bad_orth = {
    "",
    "N/A",
    "NA",
    "nan",
    "None"
}


forward_valid = forward[
    ~forward["name"].isin(bad_orth)
].copy()


fwd_map = (
    forward_valid[
        [
            "incoming",
            "name"
        ]
    ]
    .rename(
        columns={
            "incoming": "human_gene",
            "name": "mouse_gene"
        }
    )
    .drop_duplicates()
)


print(
    "\nValid human->mouse mappings:",
    len(fwd_map)
)

print(
    "Human genes with >=1 mouse ortholog:",
    fwd_map["human_gene"].nunique()
)


# ============================================================
# 4. MOUSE -> HUMAN RECIPROCAL QUERY
# ============================================================

all_mouse_genes = sorted(
    fwd_map["mouse_gene"]
    .dropna()
    .unique()
)


reverse_cache = (
    CACHE
    / "gprofiler_mouse_to_human_orthologs.csv"
)


if reverse_cache.exists():

    reverse = pd.read_csv(
        reverse_cache,
        keep_default_na=False
    )

    print(
        "Loaded existing mouse->human orthology cache."
    )

else:

    reverse = pd.DataFrame()


if len(reverse) and "incoming" in reverse.columns:

    cached_mouse = set(
        reverse["incoming"]
        .astype(str)
        .str.strip()
    )

else:

    cached_mouse = set()


missing_mouse = sorted(
    set(all_mouse_genes)
    -
    cached_mouse
)


print(
    "Mouse genes missing from reverse cache:",
    len(missing_mouse)
)


if missing_mouse:

    new_reverse = query_orthology(
        missing_mouse,
        organism="mmusculus",
        target="hsapiens",
        label="Mouse -> human g:Orth"
    )

    if len(new_reverse):

        reverse = pd.concat(
            [
                reverse,
                new_reverse
            ],
            ignore_index=True
        )

        reverse = reverse.drop_duplicates()

        reverse.to_csv(
            reverse_cache,
            index=False
        )


if reverse.empty:

    raise RuntimeError(
        "Mouse->human orthology query returned no data."
    )


reverse = reverse[
    reverse["incoming"]
    .astype(str)
    .isin(all_mouse_genes)
].copy()


reverse["incoming"] = (
    reverse["incoming"]
    .astype(str)
    .str.strip()
)

reverse["name"] = (
    reverse["name"]
    .astype(str)
    .str.strip()
)


reverse_valid = reverse[
    ~reverse["name"].isin(bad_orth)
].copy()


rev_map = (
    reverse_valid[
        [
            "incoming",
            "name"
        ]
    ]
    .rename(
        columns={
            "incoming": "mouse_gene",
            "name": "reverse_human_gene"
        }
    )
    .drop_duplicates()
)


# ============================================================
# 5. ORTHOLOGY MULTIPLICITY
# ============================================================

forward_n = (
    fwd_map
    .groupby(
        "human_gene",
        observed=True
    )["mouse_gene"]
    .nunique()
)


reverse_n = (
    rev_map
    .groupby(
        "mouse_gene",
        observed=True
    )["reverse_human_gene"]
    .nunique()
)


fwd_map["forward_orthology_n"] = (
    fwd_map["human_gene"]
    .map(forward_n)
)


fwd_map["reverse_orthology_n"] = (
    fwd_map["mouse_gene"]
    .map(reverse_n)
)


# ============================================================
# 6. REQUIRE RECIPROCITY
# ============================================================

reciprocal = (
    fwd_map
    .merge(
        rev_map,
        on="mouse_gene",
        how="left"
    )
)


reciprocal["is_reciprocal"] = (
    reciprocal["human_gene"]
    ==
    reciprocal["reverse_human_gene"]
)


reciprocal_true = (
    reciprocal[
        reciprocal["is_reciprocal"]
    ]
    [
        [
            "human_gene",
            "mouse_gene",
            "forward_orthology_n",
            "reverse_orthology_n"
        ]
    ]
    .drop_duplicates()
)


print(
    "\nReciprocal human-mouse mappings:",
    len(reciprocal_true)
)


# ============================================================
# 7. PRIMARY SCREEN = RECIPROCAL ONE-TO-ONE ONLY
# ============================================================

strict_map = reciprocal_true[
    (
        reciprocal_true[
            "forward_orthology_n"
        ]
        == 1
    )
    &
    (
        reciprocal_true[
            "reverse_orthology_n"
        ]
        == 1
    )
].copy()


print(
    "Reciprocal one-to-one mappings:",
    len(strict_map)
)


if strict_map.empty:

    raise RuntimeError(
        "No reciprocal one-to-one orthologs survived."
    )


# ============================================================
# 8. COVERAGE OF CURRENT HC NETWORK
# ============================================================

human_ligands = set(
    lr["human_ligand"]
)

human_receptors = set(
    lr["human_receptor"]
)


mapped_human = set(
    strict_map["human_gene"]
)


mapped_ligands = (
    human_ligands
    &
    mapped_human
)

mapped_receptors = (
    human_receptors
    &
    mapped_human
)


print(
    "\nHuman HC ligands:",
    len(human_ligands)
)

print(
    "Ligands with strict mouse ortholog:",
    len(mapped_ligands),
    f"({len(mapped_ligands)/len(human_ligands):.1%})"
)

print(
    "Human HC receptors:",
    len(human_receptors)
)

print(
    "Receptors with strict mouse ortholog:",
    len(mapped_receptors),
    f"({len(mapped_receptors)/len(human_receptors):.1%})"
)


# ============================================================
# 9. BUILD LIGAND AND RECEPTOR MAPS
# ============================================================

lig_map = (
    strict_map[
        [
            "human_gene",
            "mouse_gene"
        ]
    ]
    .rename(
        columns={
            "human_gene": "human_ligand",
            "mouse_gene": "mouse_ligand"
        }
    )
)


rec_map = (
    strict_map[
        [
            "human_gene",
            "mouse_gene"
        ]
    ]
    .rename(
        columns={
            "human_gene": "human_receptor",
            "mouse_gene": "mouse_receptor"
        }
    )
)


# ============================================================
# 10. BUILD STRICT MOUSE LR NETWORK
# ============================================================

lr_mouse = (
    lr
    .merge(
        lig_map,
        on="human_ligand",
        how="inner"
    )
    .merge(
        rec_map,
        on="human_receptor",
        how="inner"
    )
)


lr_mouse = lr_mouse.drop_duplicates(
    [
        "human_ligand",
        "human_receptor",
        "mouse_ligand",
        "mouse_receptor"
    ]
).copy()


# These are now guaranteed one-to-one.
lr_mouse["ligand_orthology_n"] = 1
lr_mouse["receptor_orthology_n"] = 1
lr_mouse["orthology_class"] = "reciprocal_one_to_one"


print(
    "\n======================================"
)

print(
    "STRICT MOUSE LR NETWORK"
)

print(
    "======================================"
)

print(
    "Mouse LR pairs:",
    len(lr_mouse)
)

print(
    "Mouse ligands:",
    lr_mouse[
        "mouse_ligand"
    ].nunique()
)

print(
    "Mouse receptors:",
    lr_mouse[
        "mouse_receptor"
    ].nunique()
)


if lr_mouse.empty:

    raise RuntimeError(
        "Strict mouse LR network is empty."
    )


# ============================================================
# 11. SANITY CHECK IMPORTANT GENES
# ============================================================

print(
    "\nSelected orthology checks:"
)


for gene in [
    "CALCA",
    "CALCRL",
    "BDNF",
    "NTRK2",
    "VEGFA",
    "KDR",
    "PDGFB",
    "PDGFRB",
    "TAC1",
    "TACR1"
]:

    hit = strict_map[
        strict_map["human_gene"]
        .str.upper()
        .eq(gene)
    ]

    if len(hit):

        targets = sorted(
            hit["mouse_gene"]
            .astype(str)
            .unique()
        )

        print(
            f"{gene}:",
            targets
        )

    else:

        print(
            f"{gene}:",
            "NO STRICT 1:1 MAP"
        )


# ============================================================
# 12. SAVE AUDIT + PRIMARY NETWORK
# ============================================================

fwd_map.to_csv(
    OUT
    / "gprofiler_human_mouse_all_mappings_AUDIT.csv",
    index=False
)


strict_map.to_csv(
    OUT
    / "gprofiler_human_mouse_reciprocal_1to1.csv",
    index=False
)


lr_mouse.to_csv(
    OUT
    / "omnipath_mouse_ligand_receptor_pairs.csv",
    index=False
)


print(
    "\nPHASE B3 ORTHOLOGY COMPLETE"
)

print(
    "Downstream `lr_mouse` object:",
    lr_mouse.shape
)


display(
    lr_mouse[
        [
            "human_ligand",
            "human_receptor",
            "mouse_ligand",
            "mouse_receptor",
            "orthology_class"
        ]
    ]
    .head(30)
)

In [ ]:
# ============================================================
# PHASE B4 — PSN LIGAND EXPRESSION SUPPORT
#
# GSE186180 Smart-seq2 TPM
#
# Goal:
# Keep LR axes only when the ligand has credible expression
# in pulmonary sensory neurons, while preserving subtype
# specificity and avoiding "TPM > 0 = expressed" logic.
# ============================================================

import numpy as np
import pandas as pd
# ============================================================
# 0. RESTORE GSE186180 PSN DATA IF RUNTIME WAS RESTARTED
# ============================================================

import csv
import gzip


if "psn" not in globals() or "psn_subtype" not in globals():

    print("PSN objects missing — rebuilding GSE186180 from cached raw files...")

    psn_tpm_path = (
        RAW
        / "GSE186180"
        / "GSE186180_TPM_table_PSN.txt.gz"
    )

    psn_sub_path = (
        RAW
        / "GSE186180"
        / "GSE186180_Subtype_identity_PSN.txt.txt.gz"
    )


    # --------------------------------------------------------
    # Load TPM matrix
    # --------------------------------------------------------

    psn = pd.read_csv(
        psn_tpm_path,
        sep="\t",
        compression="gzip",
        index_col=0
    )

    psn.index = psn.index.astype(str)
    psn.columns = psn.columns.astype(str)

    psn = (
        psn
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
    )

    # Collapse duplicate gene symbols
    psn = (
        psn
        .groupby(psn.index)
        .mean()
    )


    # --------------------------------------------------------
    # Parse subtype file
    # --------------------------------------------------------

    rows = []

    with gzip.open(
        psn_sub_path,
        "rt",
        encoding="utf-8-sig"
    ) as fh:

        reader = csv.reader(
            fh,
            delimiter="\t",
            quotechar='"'
        )

        for row in reader:

            if row:
                rows.append(row)


    records = []

    for row in rows[1:]:

        if len(row) < 3:
            raise RuntimeError(
                f"Unexpected PSN subtype row: {row}"
            )

        records.append({
            "cell_id": str(row[1]).strip(),
            "subtype": str(row[2]).strip()
        })


    sub_df_runtime = pd.DataFrame(records)


    # Remove explicitly designated outlier-cross cells
    sub_df_runtime["is_outlier"] = (
        sub_df_runtime["subtype"]
        .str.lower()
        .str.startswith(
            "outlier",
            na=False
        )
    )


    valid_sub_runtime = (
        sub_df_runtime[
            ~sub_df_runtime["is_outlier"]
        ]
        .copy()
    )


    psn_subtype = (
        valid_sub_runtime
        .set_index("cell_id")["subtype"]
    )


    # Keep only valid annotated PSNs
    valid_cells_runtime = [
        c
        for c in psn.columns
        if c in psn_subtype.index
    ]


    psn = (
        psn.loc[
            :,
            valid_cells_runtime
        ]
        .copy()
    )


    psn_subtype = (
        psn_subtype
        .reindex(valid_cells_runtime)
    )


    # --------------------------------------------------------
    # Hard validation
    # --------------------------------------------------------

    assert psn.shape[1] == 119, (
        f"Expected 119 valid PSNs, got {psn.shape[1]}"
    )

    assert psn_subtype.nunique() == 10, (
        f"Expected 10 PSN subtypes, got "
        f"{psn_subtype.nunique()}"
    )

    assert psn_subtype.notna().all()


    print(
        "PSN restored:",
        psn.shape[0],
        "genes x",
        psn.shape[1],
        "cells"
    )

    print(
        "PSN subtypes:",
        psn_subtype.nunique()
    )

else:

    print(
        "Existing PSN objects found:",
        psn.shape
    )

# ============================================================
# 1. CANDIDATE LIGANDS FROM THE CLEAN MOUSE LR NETWORK
# ============================================================

candidate_ligands = sorted(
    set(lr_mouse["mouse_ligand"])
    &
    set(psn.index)
)

print(
    "Mouse LR ligands:",
    lr_mouse["mouse_ligand"].nunique()
)

print(
    "Ligands present in GSE186180:",
    len(candidate_ligands)
)


if len(candidate_ligands) == 0:
    raise RuntimeError(
        "No LR ligands were found in the PSN expression matrix."
    )


# Use only valid, non-outlier PSN cells
psn_cells = list(psn_subtype.index)

expr_lig = (
    psn.loc[
        candidate_ligands,
        psn_cells
    ]
    .copy()
)


# ============================================================
# 2. OVERALL CELL-LEVEL EXPRESSION METRICS
# ============================================================

lig_metrics = pd.DataFrame(
    index=expr_lig.index
)


lig_metrics["psn_mean_tpm"] = (
    expr_lig.mean(axis=1)
)

lig_metrics["psn_median_tpm"] = (
    expr_lig.median(axis=1)
)

lig_metrics["psn_max_tpm"] = (
    expr_lig.max(axis=1)
)


# Raw nonzero fraction retained for audit only
lig_metrics["psn_pct_nonzero"] = (
    (expr_lig > 0)
    .mean(axis=1)
)


# More meaningful detection thresholds
lig_metrics["psn_pct_tpm_ge_1"] = (
    (expr_lig >= 1.0)
    .mean(axis=1)
)

lig_metrics["psn_pct_tpm_ge_5"] = (
    (expr_lig >= 5.0)
    .mean(axis=1)
)

lig_metrics["psn_n_tpm_ge_1"] = (
    (expr_lig >= 1.0)
    .sum(axis=1)
)

lig_metrics["psn_n_tpm_ge_5"] = (
    (expr_lig >= 5.0)
    .sum(axis=1)
)


# ============================================================
# 3. SUBTYPE-LEVEL METRICS
# ============================================================

subtype_mean = {}
subtype_median = {}
subtype_pct_ge1 = {}
subtype_pct_ge5 = {}
subtype_n_ge1 = {}
subtype_n_cells = {}


for subtype in sorted(
    psn_subtype.dropna().unique()
):

    cells_st = (
        psn_subtype.index[
            psn_subtype == subtype
        ]
    )

    x = expr_lig.loc[:, cells_st]

    subtype_n_cells[subtype] = len(cells_st)

    subtype_mean[subtype] = (
        x.mean(axis=1)
    )

    subtype_median[subtype] = (
        x.median(axis=1)
    )

    subtype_pct_ge1[subtype] = (
        (x >= 1.0)
        .mean(axis=1)
    )

    subtype_pct_ge5[subtype] = (
        (x >= 5.0)
        .mean(axis=1)
    )

    subtype_n_ge1[subtype] = (
        (x >= 1.0)
        .sum(axis=1)
    )


subtype_mean = pd.DataFrame(
    subtype_mean
)

subtype_median = pd.DataFrame(
    subtype_median
)

subtype_pct_ge1 = pd.DataFrame(
    subtype_pct_ge1
)

subtype_pct_ge5 = pd.DataFrame(
    subtype_pct_ge5
)

subtype_n_ge1 = pd.DataFrame(
    subtype_n_ge1
)


# ============================================================
# 4. IDENTIFY THE BEST-SUPPORTED PSN SUBTYPE
# ============================================================

lig_metrics["psn_top_subtype"] = (
    subtype_mean.idxmax(axis=1)
)

lig_metrics["psn_max_subtype_mean"] = (
    subtype_mean.max(axis=1)
)

lig_metrics["psn_max_subtype_median"] = (
    subtype_median.max(axis=1)
)

lig_metrics["psn_max_subtype_pct_tpm_ge_1"] = (
    subtype_pct_ge1.max(axis=1)
)

lig_metrics["psn_max_subtype_pct_tpm_ge_5"] = (
    subtype_pct_ge5.max(axis=1)
)


# For each gene, pull number of TPM>=1 cells from
# the subtype with maximum mean expression.

lig_metrics["psn_top_subtype_n_tpm_ge_1"] = [
    subtype_n_ge1.loc[
        gene,
        lig_metrics.loc[
            gene,
            "psn_top_subtype"
        ]
    ]
    for gene in lig_metrics.index
]


lig_metrics["psn_top_subtype_n_cells"] = [
    subtype_n_cells[
        lig_metrics.loc[
            gene,
            "psn_top_subtype"
        ]
    ]
    for gene in lig_metrics.index
]


# ============================================================
# 5. SUBTYPE ENRICHMENT
#
# Log ratio is more stable than max_mean / overall_mean.
# ============================================================

lig_metrics["psn_subtype_enrichment_log2"] = (
    np.log2(
        (
            lig_metrics["psn_max_subtype_mean"]
            + 1.0
        )
        /
        (
            lig_metrics["psn_mean_tpm"]
            + 1.0
        )
    )
)


# ============================================================
# 6. DEFINE EXPRESSION-SUPPORT TIERS
#
# PRIMARY SUPPORT:
#   TPM >= 1 in at least 25% of one subtype
#   AND at least 2 cells in that subtype
#   AND subtype mean TPM >= 1
#
# STRONG:
#   Same prevalence requirement
#   AND subtype mean TPM >= 5
#
# This retains genuinely subtype-restricted signals while
# avoiding single-cell / trace-expression artifacts.
# ============================================================

supported = (
    (
        lig_metrics[
            "psn_max_subtype_pct_tpm_ge_1"
        ]
        >= 0.25
    )
    &
    (
        lig_metrics[
            "psn_top_subtype_n_tpm_ge_1"
        ]
        >= 2
    )
    &
    (
        lig_metrics[
            "psn_max_subtype_mean"
        ]
        >= 1.0
    )
)


strong = (
    supported
    &
    (
        lig_metrics[
            "psn_max_subtype_mean"
        ]
        >= 5.0
    )
)


lig_metrics["psn_support"] = supported
lig_metrics["psn_strong_support"] = strong


lig_metrics["psn_support_tier"] = np.select(
    [
        strong,
        supported
    ],
    [
        "strong",
        "supported"
    ],
    default="weak_or_absent"
)


# Flag evidence driven by extremely small subtype groups.
# PSN9 has only 2 cells, so we retain rather than delete it,
# but mark it for caution.

lig_metrics["psn_small_top_subtype_flag"] = (
    lig_metrics[
        "psn_top_subtype_n_cells"
    ]
    < 5
)


# ============================================================
# 7. EXPRESSION SCORE
#
# Rank only among candidate LR ligands.
#
# No weight is given to raw TPM > 0 prevalence.
# ============================================================

score_cols = [
    "psn_mean_tpm",
    "psn_max_subtype_mean",
    "psn_max_subtype_pct_tpm_ge_1",
    "psn_subtype_enrichment_log2"
]


for col in score_cols:

    lig_metrics[
        col + "_pctile"
    ] = (
        lig_metrics[col]
        .rank(
            pct=True,
            method="average"
        )
    )


lig_metrics["psn_expression_score"] = (
    0.25
    *
    lig_metrics[
        "psn_mean_tpm_pctile"
    ]
    +
    0.30
    *
    lig_metrics[
        "psn_max_subtype_mean_pctile"
    ]
    +
    0.30
    *
    lig_metrics[
        "psn_max_subtype_pct_tpm_ge_1_pctile"
    ]
    +
    0.15
    *
    lig_metrics[
        "psn_subtype_enrichment_log2_pctile"
    ]
)


# ============================================================
# 8. MERGE BACK INTO LR NETWORK
# ============================================================

lig_metrics.index.name = "mouse_ligand"


lr_psn = (
    lr_mouse
    .merge(
        lig_metrics.reset_index(),
        on="mouse_ligand",
        how="inner"
    )
)


# Save EVERYTHING for audit
lr_psn.to_csv(
    OUT
    / "PSN_all_detected_ligand_receptor_pairs.csv",
    index=False
)


# Primary downstream network
lr_psn_detected = (
    lr_psn[
        lr_psn["psn_support"]
    ]
    .copy()
    .sort_values(
        [
            "psn_expression_score",
            "psn_max_subtype_mean"
        ],
        ascending=False
    )
)


lr_psn_detected.to_csv(
    OUT
    / "PSN_detected_ligand_receptor_pairs.csv",
    index=False
)


# Save ligand-level evidence separately
lig_metrics.to_csv(
    OUT
    / "GSE186180_PSN_ligand_support_metrics.csv"
)


# ============================================================
# 9. SANITY CHECKS
# ============================================================

print(
    "\n======================================"
)

print(
    "PSN LIGAND SUPPORT"
)

print(
    "======================================"
)

print(
    "Candidate LR ligands tested:",
    len(lig_metrics)
)

print(
    "PSN-supported ligands:",
    int(
        lig_metrics[
            "psn_support"
        ].sum()
    )
)

print(
    "Strongly supported ligands:",
    int(
        lig_metrics[
            "psn_strong_support"
        ].sum()
    )
)

print(
    "Supported LR pairs:",
    len(lr_psn_detected)
)

print(
    "Candidate receptors retained:",
    lr_psn_detected[
        "mouse_receptor"
    ].nunique()
)


# The upstream exclusions must remain gone.

for bad_gene in [
    "Fgf11",
    "Fgf12",
    "Fgf13",
    "Fgf14",
    "Gnas"
]:

    present = (
        lr_psn_detected[
            "mouse_ligand"
        ]
        .str.lower()
        .eq(
            bad_gene.lower()
        )
        .any()
    )

    print(
        f"{bad_gene} present:",
        present
    )

    assert not present, (
        f"{bad_gene} unexpectedly survived "
        "the upstream ligand filter."
    )


# Useful known neuronal signalling genes.
print(
    "\nSelected PSN ligand checks:"
)


for gene in [
    "Calca",
    "Tac1",
    "Bdnf",
    "Vegfa",
    "Pdgfb",
    "Vip",
    "Gal",
    "Sst"
]:

    hit = lig_metrics[
        lig_metrics.index
        .astype(str)
        .str.lower()
        ==
        gene.lower()
    ]

    if len(hit):

        row = hit.iloc[0]

        print(
            gene,
            "| support:",
            bool(
                row[
                    "psn_support"
                ]
            ),
            "| tier:",
            row[
                "psn_support_tier"
            ],
            "| mean TPM:",
            round(
                float(
                    row[
                        "psn_mean_tpm"
                    ]
                ),
                3
            ),
            "| max subtype mean:",
            round(
                float(
                    row[
                        "psn_max_subtype_mean"
                    ]
                ),
                3
            ),
            "| top subtype:",
            row[
                "psn_top_subtype"
            ]
        )

    else:

        print(
            gene,
            "| absent from current LR ligand set"
        )


print(
    "\nPHASE B4 PSN FILTER COMPLETE"
)

print(
    "Downstream `lr_psn_detected` object:",
    lr_psn_detected.shape
)


display(
    lr_psn_detected[
        [
            "mouse_ligand",
            "mouse_receptor",
            "psn_support_tier",
            "psn_expression_score",
            "psn_mean_tpm",
            "psn_max_subtype_mean",
            "psn_max_subtype_pct_tpm_ge_1",
            "psn_top_subtype",
            "psn_top_subtype_n_cells",
            "psn_small_top_subtype_flag"
        ]
    ]
    .head(40)
)

# Phase C — young vs aged lung vascular receptor states (GSE124872)

The notebook uses the sparse Matrix Market counts and the authors' metadata. The RData object is used only to recover dimnames if needed.

In [ ]:
g124_dir = RAW / "GSE124872"
g124_mtx = g124_dir / "GSE124872_raw_counts_single_cell.mtx.gz"
g124_meta = g124_dir / "GSE124872_Angelidis_2018_metadata.csv.gz"
g124_rdata_gz = g124_dir / "GSE124872_raw_counts_single_cell.RData.gz"

g124_rdata = CACHE / "GSE124872_raw_counts_single_cell.RData"
g124_gene_file = CACHE / "GSE124872_genes.txt"
g124_cell_file = CACHE / "GSE124872_cells.txt"

if not g124_rdata.exists():
    with gzip.open(g124_rdata_gz, "rb") as src, open(g124_rdata, "wb") as dst:
        shutil.copyfileobj(src, dst)

genes = None
cells = None

if g124_gene_file.exists() and g124_cell_file.exists():
    genes = pd.read_csv(g124_gene_file, header=None)[0].astype(str).tolist()
    cells = pd.read_csv(g124_cell_file, header=None)[0].astype(str).tolist()

print("Cached dimnames:", None if genes is None else len(genes), None if cells is None else len(cells))

In [ ]:
import subprocess
import shutil
import pandas as pd


# ============================================================
# R fallback: extract row/column names from the RData object
# ============================================================

if genes is None or cells is None:

    if shutil.which("Rscript") is None:

        print("Installing minimal R + Matrix package...")

        subprocess.check_call([
            "apt-get",
            "update",
            "-qq"
        ])

        subprocess.check_call([
            "apt-get",
            "install",
            "-y",
            "-qq",
            "r-base-core",
            "r-cran-matrix"
        ])


    r_script = CACHE / "extract_gse124872_dimnames.R"


    r_script.write_text(
r'''
suppressPackageStartupMessages(library(Matrix))

args <- commandArgs(trailingOnly=TRUE)

rdata <- args[1]
gene_out <- args[2]
cell_out <- args[3]


e <- new.env()

load(
    rdata,
    envir=e
)

nms <- ls(e)


find_matrix <- function(x) {

    if (
        inherits(x, "Matrix") ||
        is.matrix(x)
    ) {
        return(x)
    }

    if (is.list(x)) {

        for (nm in names(x)) {

            y <- x[[nm]]

            if (
                inherits(y, "Matrix") ||
                is.matrix(y)
            ) {
                return(y)
            }
        }
    }

    return(NULL)
}


mat <- NULL


for (nm in nms) {

    x <- get(
        nm,
        envir=e
    )

    y <- find_matrix(x)

    if (!is.null(y)) {

        mat <- y

        cat(
            "Selected:",
            nm,
            "class:",
            paste(
                class(y),
                collapse=","
            ),
            "dim:",
            paste(
                dim(y),
                collapse="x"
            ),
            "\n"
        )

        break
    }
}


if (is.null(mat)) {
    stop(
        "No matrix-like object found."
    )
}


if (is.null(rownames(mat))) {
    stop(
        "No rownames."
    )
}


if (is.null(colnames(mat))) {
    stop(
        "No colnames."
    )
}


writeLines(
    rownames(mat),
    gene_out
)

writeLines(
    colnames(mat),
    cell_out
)
'''
    )


    print(
        "Extracting GSE124872 gene and cell names..."
    )


    subprocess.check_call([
        "Rscript",
        str(r_script),
        str(g124_rdata),
        str(g124_gene_file),
        str(g124_cell_file)
    ])


    genes = (
        pd.read_csv(
            g124_gene_file,
            header=None
        )[0]
        .astype(str)
        .tolist()
    )


    cells = (
        pd.read_csv(
            g124_cell_file,
            header=None
        )[0]
        .astype(str)
        .tolist()
    )


print(
    "\nGSE124872 labels:",
    len(genes),
    "genes;",
    len(cells),
    "cells"
)

print("\nCELL 17 COMPLETE")

In [ ]:
# ============================================================
# PHASE C1 — LOAD GSE124872 COUNT MATRIX ONLY
# ============================================================

import gzip
from scipy.io import mmread


with gzip.open(
    g124_mtx,
    "rb"
) as fh:

    X124 = mmread(fh).tocsr()


# Matrix is stored gene x cell; convert to cell x gene.
if X124.shape == (
    len(genes),
    len(cells)
):

    X124 = X124.T.tocsr()


elif X124.shape != (
    len(cells),
    len(genes)
):

    raise RuntimeError(
        f"GSE124872 matrix shape {X124.shape} "
        f"does not match "
        f"{len(cells)} cells x {len(genes)} genes."
    )


print(
    "GSE124872 count matrix:",
    X124.shape
)

print(
    "Nonzero entries:",
    X124.nnz
)

print(
    "PHASE C1 MATRIX LOAD COMPLETE"
)

In [ ]:
def find_column_by_keywords(df, keywords):
    scored = []
    for c in df.columns:
        name = c.lower()
        score = sum(k in name for k in keywords)
        if score:
            scored.append((score, c, df[c].nunique(dropna=True)))
    return sorted(scored, reverse=True)

print("Cell type candidates:", find_column_by_keywords(adata124.obs, ["celltype","cell_type","type","cluster"]))
print("Age candidates:", find_column_by_keywords(adata124.obs, ["age","young","old"]))
print("Mouse/sample candidates:", find_column_by_keywords(adata124.obs, ["mouse","sample","animal","individual"]))

CELLTYPE_COL_124 = None
AGE_COL_124 = None
MOUSE_COL_124 = None

for c in adata124.obs.columns:
    lc = c.lower()
    if CELLTYPE_COL_124 is None and ("celltype" in lc or "cell_type" in lc or "cell.type" in lc):
        CELLTYPE_COL_124 = c
    if AGE_COL_124 is None and "age" in lc:
        AGE_COL_124 = c
    if MOUSE_COL_124 is None and ("mouse" in lc or "animal" in lc or "individual" in lc):
        MOUSE_COL_124 = c

if AGE_COL_124 is None:
    for c in adata124.obs.columns:
        vals = set(adata124.obs[c].dropna().astype(str).str.lower().unique())
        if vals & {"young","old","aged","3m","24m","3","24"}:
            AGE_COL_124 = c
            break

print({
    "CELLTYPE_COL_124": CELLTYPE_COL_124,
    "AGE_COL_124": AGE_COL_124,
    "MOUSE_COL_124": MOUSE_COL_124,
})

In [ ]:
# ============================================================
# CELL 21 — ROBUST GSE124872 SAMPLE/BARCODE RECONCILIATION
# ============================================================

import pandas as pd
import numpy as np
import anndata as ad

from scipy.optimize import linear_sum_assignment


# ------------------------------------------------------------
# 1. Reload pristine metadata
# ------------------------------------------------------------

meta124_raw = pd.read_csv(
    g124_meta,
    compression="gzip"
)

print("Metadata rows:", len(meta124_raw))
print("Matrix cells:", len(cells))
print("Matrix genes:", len(genes))


# ------------------------------------------------------------
# 2. Parse MATRIX IDs
#
# Examples:
# old_1:TTCCGTGCCCCT
# young_1:XXXXXXXXXXXX
# ------------------------------------------------------------

matrix_info = pd.DataFrame({
    "matrix_cell_id": [str(x).strip() for x in cells]
})

matrix_info["matrix_sample"] = (
    matrix_info["matrix_cell_id"]
    .str.split(":", n=1)
    .str[0]
)

matrix_info["barcode"] = (
    matrix_info["matrix_cell_id"]
    .str.rsplit(":", n=1)
    .str[-1]
)


print("\nMatrix sample aliases:")

display(
    matrix_info["matrix_sample"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 3. Parse METADATA IDs
#
# Example:
# muc3838:muc3838:TTCCGTGCCCCT
# ------------------------------------------------------------

meta = meta124_raw.copy()

meta["metadata_cell_id"] = (
    meta["Unnamed: 0"]
    .astype(str)
    .str.strip()
)

meta["barcode"] = (
    meta["metadata_cell_id"]
    .str.rsplit(":", n=1)
    .str[-1]
)

meta["identifier"] = (
    meta["identifier"]
    .astype(str)
    .str.strip()
)

meta["grouping"] = (
    meta["grouping"]
    .astype(str)
    .str.strip()
)


print("\nMetadata mice:")

mouse_summary = (
    meta
    .groupby("identifier")
    .agg(
        age=("grouping", "first"),
        n_cells=("barcode", "size")
    )
    .sort_values(["age", "identifier"])
)

display(mouse_summary)


# ------------------------------------------------------------
# 4. Sanity checks
# ------------------------------------------------------------

print(
    "\nMatrix sample aliases:",
    matrix_info["matrix_sample"].nunique()
)

print(
    "Metadata mice:",
    meta["identifier"].nunique()
)


if matrix_info["matrix_sample"].nunique() != 15:
    raise RuntimeError(
        "Expected 15 matrix samples."
    )

if meta["identifier"].nunique() != 15:
    raise RuntimeError(
        "Expected 15 metadata mice."
    )


# Each mouse/barcode combination must be unique
dup = meta.duplicated(
    subset=["identifier", "barcode"]
)

if dup.any():

    raise RuntimeError(
        "Duplicate identifier+barcode combinations found "
        "in metadata."
    )


# ------------------------------------------------------------
# 5. Determine age of each MATRIX alias
#
# old_*   = 24m
# young_* = 3m
# ------------------------------------------------------------

def alias_age(alias):

    alias = str(alias).lower()

    if alias.startswith("old_"):
        return "24m"

    if alias.startswith("young_"):
        return "3m"

    raise ValueError(
        f"Unexpected matrix sample alias: {alias}"
    )


matrix_info["grouping"] = (
    matrix_info["matrix_sample"]
    .map(alias_age)
)


# ------------------------------------------------------------
# 6. Build barcode-overlap matrix between
#    matrix aliases and actual metadata mice
# ------------------------------------------------------------

sample_barcode_sets = (
    matrix_info
    .groupby("matrix_sample")["barcode"]
    .apply(set)
    .to_dict()
)

mouse_barcode_sets = (
    meta
    .groupby("identifier")["barcode"]
    .apply(set)
    .to_dict()
)

mouse_age = (
    meta
    .drop_duplicates("identifier")
    .set_index("identifier")["grouping"]
    .to_dict()
)


mapping_rows = []


for age in ["24m", "3m"]:

    matrix_samples = sorted(
        [
            s
            for s in sample_barcode_sets
            if alias_age(s) == age
        ],
        key=lambda x: int(x.split("_")[1])
    )

    metadata_mice = sorted(
        [
            m
            for m in mouse_barcode_sets
            if mouse_age[m] == age
        ]
    )


    print(
        f"\nAGE {age}:",
        len(matrix_samples),
        "matrix samples vs",
        len(metadata_mice),
        "metadata mice"
    )


    if len(matrix_samples) != len(metadata_mice):

        raise RuntimeError(
            f"Sample number mismatch for {age}"
        )


    # Rows = old_1/old_2/... or young_1/...
    # Columns = mucXXXX mice
    overlap = np.zeros(
        (
            len(matrix_samples),
            len(metadata_mice)
        ),
        dtype=int
    )


    for i, s in enumerate(matrix_samples):

        sbar = sample_barcode_sets[s]

        for j, m in enumerate(metadata_mice):

            mbar = mouse_barcode_sets[m]

            overlap[i, j] = len(
                sbar & mbar
            )


    overlap_df = pd.DataFrame(
        overlap,
        index=matrix_samples,
        columns=metadata_mice
    )


    print(
        f"\nBarcode-overlap matrix ({age}):"
    )

    display(overlap_df)


    # --------------------------------------------------------
    # Hungarian assignment:
    # maximize total barcode overlap while forcing a
    # one-to-one sample ↔ mouse mapping.
    # --------------------------------------------------------

    row_ind, col_ind = linear_sum_assignment(
        -overlap
    )


    for r, c in zip(
        row_ind,
        col_ind
    ):

        sample = matrix_samples[r]
        mouse = metadata_mice[c]

        n_overlap = int(
            overlap[r, c]
        )

        n_matrix = len(
            sample_barcode_sets[sample]
        )

        n_meta = len(
            mouse_barcode_sets[mouse]
        )


        mapping_rows.append({
            "matrix_sample": sample,
            "identifier": mouse,
            "grouping": age,
            "barcode_overlap": n_overlap,
            "matrix_cells": n_matrix,
            "metadata_cells": n_meta,
            "fraction_matrix_matched":
                n_overlap / n_matrix
                if n_matrix else np.nan,
            "fraction_metadata_matched":
                n_overlap / n_meta
                if n_meta else np.nan
        })


sample_mapping = pd.DataFrame(
    mapping_rows
)

sample_mapping = sample_mapping.sort_values(
    [
        "grouping",
        "matrix_sample"
    ]
)


print(
    "\n======================================"
)

print(
    "INFERRED SAMPLE → MOUSE MAPPING"
)

print(
    "======================================"
)

display(sample_mapping)


# ------------------------------------------------------------
# 7. Require extremely strong mapping
# ------------------------------------------------------------

if (
    sample_mapping[
        "fraction_matrix_matched"
    ].min()
    < 0.95
):

    raise RuntimeError(
        "At least one inferred sample mapping "
        "has <95% barcode agreement. "
        "Do not continue."
    )


sample_to_mouse = dict(
    zip(
        sample_mapping["matrix_sample"],
        sample_mapping["identifier"]
    )
)


# ------------------------------------------------------------
# 8. Assign actual mouse ID to every MATRIX cell
# ------------------------------------------------------------

matrix_info["identifier"] = (
    matrix_info["matrix_sample"]
    .map(sample_to_mouse)
)


if matrix_info["identifier"].isna().any():

    raise RuntimeError(
        "Some matrix cells could not be assigned "
        "to a metadata mouse."
    )


# ------------------------------------------------------------
# 9. Create exact (mouse, barcode) lookup in metadata
# ------------------------------------------------------------

meta_lookup = (
    meta
    .set_index(
        [
            "identifier",
            "barcode"
        ]
    )
)


# ------------------------------------------------------------
# 10. Reconstruct metadata row-by-row in MATRIX order
# ------------------------------------------------------------

aligned_rows = []

missing_pairs = []


for row in matrix_info.itertuples(
    index=False
):

    key = (
        row.identifier,
        row.barcode
    )


    if key not in meta_lookup.index:

        missing_pairs.append(
            (
                row.matrix_cell_id,
                row.identifier,
                row.barcode
            )
        )

        aligned_rows.append(
            None
        )

        continue


    x = meta_lookup.loc[key]


    # Should be exactly one row
    if isinstance(
        x,
        pd.DataFrame
    ):

        if len(x) != 1:

            raise RuntimeError(
                f"Multiple metadata rows for {key}"
            )

        x = x.iloc[0]


    aligned_rows.append(
        x.to_dict()
    )


print(
    "\nUnmatched matrix cells:",
    len(missing_pairs)
)


if missing_pairs:

    print(
        "\nFirst unmatched cells:"
    )

    print(
        missing_pairs[:20]
    )

    raise RuntimeError(
        "Not all matrix cells could be mapped "
        "to metadata."
    )


# ------------------------------------------------------------
# 11. Build aligned metadata dataframe
# ------------------------------------------------------------

meta124_fixed = pd.DataFrame(
    aligned_rows,
    index=pd.Index(
        matrix_info["matrix_cell_id"],
        name="matrix_cell_id"
    )
)


print(
    "\nAligned metadata shape:",
    meta124_fixed.shape
)
# ============================================================
# RESTORE COLUMNS DROPPED WHEN identifier/barcode BECAME INDEX
# ============================================================

meta124_fixed["identifier"] = (
    matrix_info["identifier"]
    .astype(str)
    .to_numpy()
)

meta124_fixed["matrix_sample"] = (
    matrix_info["matrix_sample"]
    .astype(str)
    .to_numpy()
)

# Two cells in the original metadata have no celltype annotation.
# Keep them explicit rather than inventing an identity.
print(
    "Original missing celltype:",
    meta124_fixed["celltype"].isna().sum()
)

meta124_fixed["celltype"] = (
    meta124_fixed["celltype"]
    .fillna("Unannotated")
)

# ------------------------------------------------------------
# 12. Validate biology
# ------------------------------------------------------------

print(
    "Non-null celltype:",
    meta124_fixed["celltype"]
    .notna()
    .sum(),
    "/",
    len(meta124_fixed)
)

print(
    "Non-null grouping:",
    meta124_fixed["grouping"]
    .notna()
    .sum(),
    "/",
    len(meta124_fixed)
)

print(
    "Non-null identifier:",
    meta124_fixed["identifier"]
    .notna()
    .sum(),
    "/",
    len(meta124_fixed)
)


assert (
    meta124_fixed["celltype"]
    .notna()
    .all()
)

assert (
    meta124_fixed["grouping"]
    .notna()
    .all()
)

assert (
    meta124_fixed["identifier"]
    .notna()
    .all()
)


# ------------------------------------------------------------
# 13. Rebuild AnnData
# ------------------------------------------------------------

adata124 = ad.AnnData(
    X=X124,
    obs=meta124_fixed.copy(),
    var=pd.DataFrame(
        index=pd.Index(
            genes,
            name="gene"
        )
    )
)

adata124.var_names_make_unique()


# ------------------------------------------------------------
# 14. Set metadata variables definitively
# ------------------------------------------------------------

CELLTYPE_COL_124 = "celltype"

AGE_COL_124 = "grouping"

MOUSE_COL_124 = "identifier"


# ------------------------------------------------------------
# 15. Final biological verification
# ------------------------------------------------------------

print(
    "\n======================================"
)

print(
    "CELL TYPES"
)

print(
    "======================================"
)

display(
    adata124.obs[
        "celltype"
    ]
    .value_counts()
)


print(
    "\n======================================"
)

print(
    "CELLS BY AGE"
)

print(
    "======================================"
)

display(
    adata124.obs[
        "grouping"
    ]
    .value_counts()
)


print(
    "\n======================================"
)

print(
    "MICE"
)

print(
    "======================================"
)

mouse_table = (
    adata124.obs
    .groupby("identifier")
    .agg(
        age=(
            "grouping",
            "first"
        ),
        n_cells=(
            "grouping",
            "size"
        )
    )
    .sort_values(
        [
            "age",
            "identifier"
        ]
    )
)

display(
    mouse_table
)


# ------------------------------------------------------------
# 16. Verify every mouse belongs to ONE age only
# ------------------------------------------------------------

age_per_mouse = (
    adata124.obs
    .groupby(
        "identifier"
    )["grouping"]
    .nunique()
)

assert (
    age_per_mouse == 1
).all()


mouse_age_counts = (
    adata124.obs[
        [
            "identifier",
            "grouping"
        ]
    ]
    .drop_duplicates()
    ["grouping"]
    .value_counts()
)


print(
    "\nMICE PER AGE:"
)

display(
    mouse_age_counts
)


print(
    "\nNumber of mice:",
    adata124.obs[
        "identifier"
    ]
    .nunique()
)


# ------------------------------------------------------------
# 17. Save mapping for reproducibility
# ------------------------------------------------------------

sample_mapping.to_csv(
    OUT
    / "GSE124872_matrix_sample_to_mouse_mapping.csv",
    index=False
)


print(
    "\nCorrected AnnData:"
)

print(
    adata124
)


print(
    "\nCELL 21 COMPLETE"
)

In [ ]:
# ============================================================
# PATCH AFTER GSE124872 MAPPING
# Fix dropped identifier + 2 unannotated cells
# ============================================================

import pandas as pd
import anndata as ad


# ------------------------------------------------------------
# 1. Restore mouse identifier
#
# The previous lookup used identifier as an index, so pandas
# dropped it from the reconstructed rows.
# matrix_info is already in EXACT matrix-cell order.
# ------------------------------------------------------------

meta124_fixed["identifier"] = (
    matrix_info["identifier"]
    .astype(str)
    .to_numpy()
)


# Also retain matrix sample alias for reproducibility
meta124_fixed["matrix_sample"] = (
    matrix_info["matrix_sample"]
    .astype(str)
    .to_numpy()
)


# ------------------------------------------------------------
# 2. Do NOT invent identities for the two cells without
#    cell-type annotation.
# ------------------------------------------------------------

n_missing_celltype = (
    meta124_fixed["celltype"]
    .isna()
    .sum()
)

print(
    "Cells without original celltype annotation:",
    n_missing_celltype
)

meta124_fixed["celltype"] = (
    meta124_fixed["celltype"]
    .fillna("Unannotated")
)


# ------------------------------------------------------------
# 3. Validate core metadata
# ------------------------------------------------------------

print(
    "Non-null celltype:",
    meta124_fixed["celltype"].notna().sum(),
    "/",
    len(meta124_fixed)
)

print(
    "Non-null grouping:",
    meta124_fixed["grouping"].notna().sum(),
    "/",
    len(meta124_fixed)
)

print(
    "Non-null identifier:",
    meta124_fixed["identifier"].notna().sum(),
    "/",
    len(meta124_fixed)
)


assert meta124_fixed["grouping"].notna().all()
assert meta124_fixed["identifier"].notna().all()


# ------------------------------------------------------------
# 4. Rebuild AnnData
# ------------------------------------------------------------

adata124 = ad.AnnData(
    X=X124,
    obs=meta124_fixed.copy(),
    var=pd.DataFrame(
        index=pd.Index(
            genes,
            name="gene"
        )
    )
)

adata124.var_names_make_unique()


# ------------------------------------------------------------
# 5. Set definitive metadata fields
# ------------------------------------------------------------

CELLTYPE_COL_124 = "celltype"
AGE_COL_124 = "grouping"
MOUSE_COL_124 = "identifier"


# ------------------------------------------------------------
# 6. Biological-replicate QC
# ------------------------------------------------------------

mouse_table = (
    adata124.obs
    .groupby("identifier")
    .agg(
        age=("grouping", "first"),
        n_cells=("grouping", "size")
    )
    .sort_values(
        ["age", "identifier"]
    )
)

print("\nMOUSE TABLE:")
display(mouse_table)


# Every mouse must belong to exactly one age
mouse_age_n = (
    adata124.obs
    .groupby("identifier")["grouping"]
    .nunique()
)

assert (mouse_age_n == 1).all()


mouse_age_counts = (
    adata124.obs[
        ["identifier", "grouping"]
    ]
    .drop_duplicates()
    ["grouping"]
    .value_counts()
)


print("\nMICE PER AGE:")
display(mouse_age_counts)


print(
    "\nNumber of mice:",
    adata124.obs["identifier"].nunique()
)


print("\nCELL TYPES:")
display(
    adata124.obs["celltype"]
    .value_counts()
    .head(50)
)


# ------------------------------------------------------------
# 7. Save the already-validated sample mapping
# ------------------------------------------------------------

sample_mapping.to_csv(
    OUT / "GSE124872_matrix_sample_to_mouse_mapping.csv",
    index=False
)


print("\nCorrected AnnData:")
print(adata124)

print("\nPHASE C METADATA ALIGNMENT COMPLETE")

In [ ]:
# ============================================================
# PHASE C2 — DEFINE VASCULAR COMPARTMENTS CLEANLY
#
# Primary aging analysis:
#   BLOOD ENDOTHELIAL CELLS ONLY
#
# Secondary compartments kept separately:
#   smooth muscle / mural-like
#   lymphatic endothelial
# ============================================================

import numpy as np
import pandas as pd
import scanpy as sc


# ============================================================
# 1. EXACT CELL-TYPE LABELS FROM AUTHORS' ANNOTATION
# ============================================================

ct = (
    adata124.obs[CELLTYPE_COL_124]
    .astype(str)
)


blood_ec_types = {
    "vascular_endothelial_cells",
    "Vcam1+_endothelial_cells",
    "Capillary_endothelial_cells",
}


mural_types = {
    "Smooth_muscle_cells",
}


lymphatic_types = {
    "lymphatic_endothelial_cells",
}


# ============================================================
# 2. BUILD SEPARATE COMPARTMENTS
# ============================================================

blood_ec124 = (
    adata124[
        ct.isin(blood_ec_types)
    ]
    .copy()
)


mural124 = (
    adata124[
        ct.isin(mural_types)
    ]
    .copy()
)


lymphatic124 = (
    adata124[
        ct.isin(lymphatic_types)
    ]
    .copy()
)


# For backward compatibility with downstream cells:
# vas124 now means PRIMARY BLOOD ENDOTHELIUM ONLY.
vas124 = blood_ec124


# ============================================================
# 3. AGE STANDARDIZATION
# ============================================================

def standardize_age(x):

    x = str(x).lower().strip()

    if x in {
        "young",
        "3m",
        "3 month",
        "3-month",
        "3"
    }:
        return "young"

    if x in {
        "old",
        "aged",
        "24m",
        "24 month",
        "24-month",
        "24"
    }:
        return "aged"

    return x


for obj in [
    blood_ec124,
    mural124,
    lymphatic124
]:

    obj.obs["age_std"] = (
        obj.obs[AGE_COL_124]
        .astype(str)
        .map(standardize_age)
    )


# ============================================================
# 4. RETAIN RAW COUNTS + NORMALIZED LOG EXPRESSION
#
# Raw counts are required later for pseudobulk.
# ============================================================

def prepare_expression(obj):

    obj.layers["counts"] = (
        obj.X.copy()
    )

    sc.pp.normalize_total(
        obj,
        target_sum=1e4
    )

    sc.pp.log1p(obj)


prepare_expression(blood_ec124)
prepare_expression(mural124)
prepare_expression(lymphatic124)


# Reset alias AFTER preprocessing
vas124 = blood_ec124


# ============================================================
# 5. HARD VALIDATION
# ============================================================

assert blood_ec124.n_obs == (
    631 + 217 + 210
), (
    "Unexpected blood endothelial cell count."
)

assert mural124.n_obs == 78, (
    "Unexpected smooth-muscle cell count."
)

assert lymphatic124.n_obs == 56, (
    "Unexpected lymphatic endothelial count."
)


assert set(
    blood_ec124.obs["age_std"].unique()
) == {
    "young",
    "aged"
}


# ============================================================
# 6. REPORT PRIMARY BLOOD-ENDOTHELIAL COHORT
# ============================================================

print(
    "\n======================================"
)

print(
    "PRIMARY BLOOD ENDOTHELIAL COMPARTMENT"
)

print(
    "======================================"
)

print(
    "Blood EC cells:",
    blood_ec124.n_obs
)

print(
    "\nBlood EC subtypes:"
)

display(
    blood_ec124.obs[
        CELLTYPE_COL_124
    ]
    .value_counts()
)


print(
    "\nBlood EC cells by age:"
)

display(
    blood_ec124.obs[
        "age_std"
    ]
    .value_counts()
)


blood_ec_mouse_coverage = (
    blood_ec124.obs[
        [
            MOUSE_COL_124,
            "age_std"
        ]
    ]
    .drop_duplicates()
    .groupby(
        "age_std",
        observed=True
    )[MOUSE_COL_124]
    .nunique()
)


print(
    "\nBlood EC biological replicates by age:"
)

display(
    blood_ec_mouse_coverage
)


# ============================================================
# 7. SECONDARY COMPARTMENTS
# ============================================================

print(
    "\n======================================"
)

print(
    "SECONDARY VASCULAR COMPARTMENTS"
)

print(
    "======================================"
)

print(
    "Smooth-muscle / mural-like cells:",
    mural124.n_obs
)

print(
    "Lymphatic endothelial cells:",
    lymphatic124.n_obs
)


print(
    "\nSmooth-muscle cells by age:"
)

display(
    mural124.obs[
        "age_std"
    ]
    .value_counts()
)


print(
    "\nSmooth-muscle biological replicates by age:"
)

display(
    mural124.obs[
        [
            MOUSE_COL_124,
            "age_std"
        ]
    ]
    .drop_duplicates()
    .groupby(
        "age_std",
        observed=True
    )[MOUSE_COL_124]
    .nunique()
)


print(
    "\nLymphatic EC cells by age:"
)

display(
    lymphatic124.obs[
        "age_std"
    ]
    .value_counts()
)


print(
    "\nPHASE C2 VASCULAR COMPARTMENTS COMPLETE"
)

print(
    "Primary downstream `vas124`:",
    vas124.shape
)

In [ ]:
# ============================================================
# PHASE C3 — BLOOD-EC RECEPTOR EXPRESSION AT MOUSE LEVEL
#
# Biological replicate = mouse.
#
# We retain the three author-defined blood EC subtypes separately:
#   vascular_endothelial_cells
#   Vcam1+_endothelial_cells
#   Capillary_endothelial_cells
#
# No age-effect testing yet.
# This cell establishes receptor expression support and
# mouse/subtype coverage for the next pseudobulk analysis.
# ============================================================

import numpy as np
import pandas as pd

from scipy import sparse


# ============================================================
# 1. CURRENT RECEPTOR UNIVERSE
# ============================================================

candidate_receptors = sorted(
    set(
        lr_psn_detected[
            "mouse_receptor"
        ]
    )
    &
    set(
        vas124.var_names
    )
)


print(
    "PSN-linked candidate receptors:",
    lr_psn_detected[
        "mouse_receptor"
    ].nunique()
)

print(
    "Candidate receptors present in GSE124872:",
    len(candidate_receptors)
)


if len(candidate_receptors) == 0:

    raise RuntimeError(
        "No candidate receptors found in GSE124872."
    )


receptor_idx = (
    vas124.var_names
    .get_indexer(
        candidate_receptors
    )
)


assert (
    receptor_idx >= 0
).all()


# ============================================================
# 2. RAW COUNT MATRIX
# ============================================================

if "counts" not in vas124.layers:

    raise RuntimeError(
        "Raw counts layer missing from vas124."
    )


raw_counts = (
    vas124.layers[
        "counts"
    ]
)


# ============================================================
# 3. BUILD MOUSE × EC-SUBTYPE RECEPTOR TABLE
# ============================================================

mouse_rows = []


for celltype in sorted(
    vas124.obs[
        CELLTYPE_COL_124
    ]
    .astype(str)
    .unique()
):

    ct_mask = (
        vas124.obs[
            CELLTYPE_COL_124
        ]
        .astype(str)
        ==
        celltype
    )

    ct_positions = np.where(
        ct_mask.to_numpy()
    )[0]


    if len(ct_positions) == 0:
        continue


    ct_obs = (
        vas124.obs
        .iloc[
            ct_positions
        ]
        .copy()
    )


    for mouse in sorted(
        ct_obs[
            MOUSE_COL_124
        ]
        .astype(str)
        .unique()
    ):

        mouse_mask_local = (
            ct_obs[
                MOUSE_COL_124
            ]
            .astype(str)
            ==
            mouse
        )

        local_positions = np.where(
            mouse_mask_local.to_numpy()
        )[0]

        rows = (
            ct_positions[
                local_positions
            ]
        )


        if len(rows) == 0:
            continue


        ages = (
            vas124.obs
            .iloc[
                rows
            ][
                "age_std"
            ]
            .astype(str)
            .unique()
        )


        if len(ages) != 1:

            raise RuntimeError(
                f"Mouse {mouse} has multiple ages "
                f"within {celltype}: {ages}"
            )


        age = ages[0]


        X_all = (
            raw_counts[
                rows,
                :
            ]
        )


        X_rec = (
            raw_counts[
                rows,
                :
            ][
                :,
                receptor_idx
            ]
        )


        # ----------------------------------------------------
        # Total pseudobulk library size for this mouse/subtype
        # ----------------------------------------------------

        total_library_counts = float(
            X_all.sum()
        )


        # ----------------------------------------------------
        # Receptor summed raw counts
        # ----------------------------------------------------

        receptor_sum = np.asarray(
            X_rec.sum(
                axis=0
            )
        ).ravel()


        # ----------------------------------------------------
        # Number / fraction of cells expressing each receptor
        # ----------------------------------------------------

        if sparse.issparse(
            X_rec
        ):

            receptor_detected_cells = np.asarray(
                (
                    X_rec > 0
                ).sum(
                    axis=0
                )
            ).ravel()

        else:

            receptor_detected_cells = (
                X_rec > 0
            ).sum(
                axis=0
            )


        pct_expressing = (
            receptor_detected_cells
            /
            len(rows)
        )


        # ----------------------------------------------------
        # CPM using the full mouse/subtype library
        # ----------------------------------------------------

        if total_library_counts > 0:

            receptor_cpm = (
                1e6
                *
                receptor_sum
                /
                total_library_counts
            )

        else:

            receptor_cpm = np.zeros(
                len(
                    candidate_receptors
                ),
                dtype=float
            )


        receptor_log2cpm = np.log2(
            receptor_cpm
            + 1.0
        )


        for (
            gene,
            raw_sum,
            cpm,
            log2cpm,
            n_detected,
            pct
        ) in zip(

            candidate_receptors,
            receptor_sum,
            receptor_cpm,
            receptor_log2cpm,
            receptor_detected_cells,
            pct_expressing
        ):

            mouse_rows.append({

                "mouse":
                    mouse,

                "age_std":
                    age,

                "celltype":
                    celltype,

                "gene":
                    gene,

                "n_cells":
                    int(
                        len(rows)
                    ),

                "library_counts":
                    total_library_counts,

                "raw_sum":
                    float(
                        raw_sum
                    ),

                "cpm":
                    float(
                        cpm
                    ),

                "log2cpm":
                    float(
                        log2cpm
                    ),

                "n_expressing_cells":
                    int(
                        n_detected
                    ),

                "pct_expressing":
                    float(
                        pct
                    )
            })


receptor_mouse = pd.DataFrame(
    mouse_rows
)


if receptor_mouse.empty:

    raise RuntimeError(
        "Mouse-level receptor table is empty."
    )


# ============================================================
# 4. MOUSE/SUBTYPE COVERAGE AUDIT
# ============================================================

coverage = (
    receptor_mouse[
        [
            "mouse",
            "age_std",
            "celltype",
            "n_cells"
        ]
    ]
    .drop_duplicates()
)


print(
    "\n======================================"
)

print(
    "MOUSE × EC-SUBTYPE COVERAGE"
)

print(
    "======================================"
)


coverage_summary = (
    coverage
    .groupby(
        [
            "celltype",
            "age_std"
        ],
        observed=True
    )
    .agg(

        n_mice=(
            "mouse",
            "nunique"
        ),

        total_cells=(
            "n_cells",
            "sum"
        ),

        median_cells_per_mouse=(
            "n_cells",
            "median"
        ),

        min_cells_per_mouse=(
            "n_cells",
            "min"
        ),

        max_cells_per_mouse=(
            "n_cells",
            "max"
        )
    )
    .reset_index()
)


display(
    coverage_summary
)


# ============================================================
# 5. EXPRESSION SUPPORT BY AGE + EC SUBTYPE
#
# Descriptive only.
# No p-values / age claims in this cell.
# ============================================================

receptor_age_summary = (
    receptor_mouse
    .groupby(
        [
            "celltype",
            "age_std",
            "gene"
        ],
        observed=True
    )
    .agg(

        n_mice=(
            "mouse",
            "nunique"
        ),

        mean_log2cpm=(
            "log2cpm",
            "mean"
        ),

        median_log2cpm=(
            "log2cpm",
            "median"
        ),

        mean_pct_expressing=(
            "pct_expressing",
            "mean"
        ),

        median_pct_expressing=(
            "pct_expressing",
            "median"
        ),

        n_mice_raw_detected=(
            "raw_sum",
            lambda x: int(
                (
                    x > 0
                ).sum()
            )
        ),

        n_mice_pct_ge_05=(
            "pct_expressing",
            lambda x: int(
                (
                    x >= 0.05
                ).sum()
            )
        )

    )
    .reset_index()
)


# ============================================================
# 6. OVERALL RECEPTOR SUPPORT ACROSS BLOOD ECs
#
# This is only a support annotation.
# We do NOT use it for age significance.
# ============================================================

overall_support = (
    receptor_mouse
    .groupby(
        "gene",
        observed=True
    )
    .agg(

        total_mouse_subtype_groups=(
            "mouse",
            "size"
        ),

        n_groups_raw_detected=(
            "raw_sum",
            lambda x: int(
                (
                    x > 0
                ).sum()
            )
        ),

        n_groups_pct_ge_05=(
            "pct_expressing",
            lambda x: int(
                (
                    x >= 0.05
                ).sum()
            )
        ),

        median_log2cpm=(
            "log2cpm",
            "median"
        ),

        max_log2cpm=(
            "log2cpm",
            "max"
        ),

        median_pct_expressing=(
            "pct_expressing",
            "median"
        ),

        max_pct_expressing=(
            "pct_expressing",
            "max"
        )
    )
    .reset_index()
)


# ============================================================
# 7. SAVE
# ============================================================

receptor_mouse.to_csv(

    OUT
    / "GSE124872_bloodEC_receptor_mouse_level.csv",

    index=False
)


receptor_age_summary.to_csv(

    OUT
    / "GSE124872_bloodEC_receptor_expression_by_age_and_subtype.csv",

    index=False
)


overall_support.to_csv(

    OUT
    / "GSE124872_bloodEC_receptor_expression_support.csv",

    index=False
)


# Keep compatibility object name,
# but this is now descriptive mouse-level summary.

age_receptor_summary = (
    receptor_age_summary.copy()
)


# ============================================================
# 8. SELECTED RECEPTOR CHECKS
# ============================================================

print(
    "\n======================================"
)

print(
    "SELECTED RECEPTOR CHECKS"
)

print(
    "======================================"
)


selected_receptors = [

    "Calcrl",
    "Ramp1",
    "Ramp2",
    "Ramp3",

    "Ntrk2",

    "Kdr",
    "Nrp1",

    "Pdgfrb",

    "Vipr1",
    "Vipr2",

    "Galr1",
    "Galr2",

    "Sstr1",
    "Sstr2",

    "Tacr2"
]


for gene in selected_receptors:

    hit = (
        overall_support[
            overall_support[
                "gene"
            ]
            ==
            gene
        ]
    )

    if len(hit):

        row = hit.iloc[0]

        print(

            gene,

            "| groups detected:",
            int(
                row[
                    "n_groups_raw_detected"
                ]
            ),
            "/",
            int(
                row[
                    "total_mouse_subtype_groups"
                ]
            ),

            "| groups >=5% cells:",
            int(
                row[
                    "n_groups_pct_ge_05"
                ]
            ),

            "| max % expressing:",
            round(
                float(
                    row[
                        "max_pct_expressing"
                    ]
                ),
                3
            ),

            "| max log2CPM:",
            round(
                float(
                    row[
                        "max_log2cpm"
                    ]
                ),
                3
            )
        )

    else:

        print(
            gene,
            "| absent from current candidate receptor set"
        )


# ============================================================
# 9. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE C3 RECEPTOR EXPRESSION COMPLETE"
)

print(
    "Mouse-level rows:",
    len(
        receptor_mouse
    )
)

print(
    "Candidate receptors:",
    receptor_mouse[
        "gene"
    ].nunique()
)

print(
    "Blood EC subtypes:",
    receptor_mouse[
        "celltype"
    ].nunique()
)

print(
    "Mice represented:",
    receptor_mouse[
        "mouse"
    ].nunique()
)

In [ ]:
# ============================================================
# PHASE C4 — AGING DESIGN / CONFOUNDING AUDIT
#
# Determine whether batch or other available metadata can
# legitimately enter the mouse-level age model.
#
# NO AGE DIFFERENTIAL TESTING YET.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. AVAILABLE DESIGN VARIABLES
# ============================================================

possible_covariates = [
    "batch",
    "orig.ident"
]

available_covariates = [
    c
    for c in possible_covariates
    if c in vas124.obs.columns
]


print(
    "Available possible covariates:",
    available_covariates
)


# ============================================================
# 2. BUILD ONE-MOUSE DESIGN TABLE
# ============================================================

design_cols = [
    MOUSE_COL_124,
    "age_std"
] + available_covariates


cell_design = (
    vas124.obs[
        design_cols
    ]
    .copy()
)


cell_design[MOUSE_COL_124] = (
    cell_design[MOUSE_COL_124]
    .astype(str)
)

cell_design["age_std"] = (
    cell_design["age_std"]
    .astype(str)
)


# ------------------------------------------------------------
# Every mouse MUST belong to one age group.
# ------------------------------------------------------------

age_n_per_mouse = (
    cell_design
    .groupby(
        MOUSE_COL_124,
        observed=True
    )["age_std"]
    .nunique()
)


assert (
    age_n_per_mouse == 1
).all(), (
    "At least one mouse has multiple age labels."
)


# ============================================================
# 3. CHECK WHETHER COVARIATES ARE CONSTANT WITHIN EACH MOUSE
# ============================================================

covariate_constant = {}


for cov in available_covariates:

    n_per_mouse = (
        cell_design
        .groupby(
            MOUSE_COL_124,
            observed=True
        )[cov]
        .nunique(
            dropna=False
        )
    )

    covariate_constant[cov] = bool(
        (
            n_per_mouse == 1
        ).all()
    )


    print(
        f"{cov} constant within each mouse:",
        covariate_constant[cov]
    )


# ============================================================
# 4. COLLAPSE TO ONE ROW PER BIOLOGICAL REPLICATE
# ============================================================

agg_dict = {
    "age_std": "first"
}


for cov in available_covariates:

    if covariate_constant[cov]:

        agg_dict[cov] = "first"


mouse_design = (
    cell_design
    .groupby(
        MOUSE_COL_124,
        observed=True
    )
    .agg(
        agg_dict
    )
    .reset_index()
)


print(
    "\n======================================"
)

print(
    "MOUSE-LEVEL DESIGN"
)

print(
    "======================================"
)

display(
    mouse_design
    .sort_values(
        [
            "age_std",
            MOUSE_COL_124
        ]
    )
)


print(
    "\nMice by age:"
)

display(
    mouse_design[
        "age_std"
    ]
    .value_counts()
)


assert (
    mouse_design[
        MOUSE_COL_124
    ]
    .nunique()
    ==
    15
), (
    "Expected 15 biological replicates."
)


# ============================================================
# 5. AGE × COVARIATE CROSS-TABS
# ============================================================

for cov in available_covariates:

    if not covariate_constant[cov]:

        print(
            f"\n{cov}: cannot use as mouse-level covariate "
            "because it varies within mouse."
        )

        continue


    print(
        f"\n======================================"
    )

    print(
        f"AGE × {cov}"
    )

    print(
        "======================================"
    )


    tab = pd.crosstab(
        mouse_design["age_std"],
        mouse_design[cov],
        dropna=False
    )


    display(
        tab
    )


    print(
        f"Unique {cov} values:",
        mouse_design[cov].nunique(
            dropna=False
        )
    )


# ============================================================
# 6. CELL-COVERAGE TABLE FOR THE THREE BLOOD-EC SUBTYPES
# ============================================================

coverage_mouse = (
    receptor_mouse[
        [
            "mouse",
            "age_std",
            "celltype",
            "n_cells"
        ]
    ]
    .drop_duplicates()
    .copy()
)


coverage_wide = (
    coverage_mouse
    .pivot(
        index=[
            "mouse",
            "age_std"
        ],
        columns="celltype",
        values="n_cells"
    )
    .fillna(0)
    .reset_index()
)


print(
    "\n======================================"
)

print(
    "EC CELLS PER MOUSE × SUBTYPE"
)

print(
    "======================================"
)

display(
    coverage_wide
    .sort_values(
        [
            "age_std",
            "mouse"
        ]
    )
)


# ============================================================
# 7. ELIGIBILITY UNDER MINIMUM CELL THRESHOLDS
#
# We inspect several thresholds BEFORE choosing one.
# ============================================================

threshold_rows = []


for min_cells in [
    1,
    3,
    5,
    10
]:

    tmp = (
        coverage_mouse[
            coverage_mouse[
                "n_cells"
            ]
            >=
            min_cells
        ]
    )


    counts = (
        tmp
        .groupby(
            [
                "celltype",
                "age_std"
            ],
            observed=True
        )["mouse"]
        .nunique()
    )


    all_celltypes = sorted(
        coverage_mouse[
            "celltype"
        ]
        .unique()
    )


    for ct in all_celltypes:

        threshold_rows.append({

            "min_cells_per_mouse":
                min_cells,

            "celltype":
                ct,

            "young_mice":
                int(
                    counts.get(
                        (
                            ct,
                            "young"
                        ),
                        0
                    )
                ),

            "aged_mice":
                int(
                    counts.get(
                        (
                            ct,
                            "aged"
                        ),
                        0
                    )
                )
        })


threshold_audit = pd.DataFrame(
    threshold_rows
)


print(
    "\n======================================"
)

print(
    "REPLICATE COVERAGE BY MINIMUM CELL THRESHOLD"
)

print(
    "======================================"
)

display(
    threshold_audit
)


# ============================================================
# 8. DESIGN-MATRIX RANK CHECK
#
# Test age-only design and, where possible, age + covariate.
# ============================================================

def matrix_rank_report(df, covariates):

    X_parts = [
        pd.Series(
            1.0,
            index=df.index,
            name="Intercept"
        ),

        (
            df["age_std"]
            ==
            "aged"
        )
        .astype(float)
        .rename("aged")
    ]


    for cov in covariates:

        dummies = pd.get_dummies(
            df[cov].astype(str),
            prefix=cov,
            drop_first=True,
            dtype=float
        )

        for c in dummies.columns:

            X_parts.append(
                dummies[c]
            )


    X = pd.concat(
        X_parts,
        axis=1
    )


    rank = np.linalg.matrix_rank(
        X.to_numpy(
            dtype=float
        )
    )


    print(
        "\nDesign:",
        "age"
        if not covariates
        else
        "age + " + " + ".join(covariates)
    )

    print(
        "Shape:",
        X.shape
    )

    print(
        "Rank:",
        rank
    )

    print(
        "Full rank:",
        rank == X.shape[1]
    )

    return X


X_age_only = matrix_rank_report(
    mouse_design,
    []
)


usable_covariates = []


for cov in available_covariates:

    if not covariate_constant[cov]:
        continue

    # A covariate with one unique value adds nothing.
    if (
        mouse_design[cov]
        .nunique(
            dropna=False
        )
        <=
        1
    ):
        continue


    X_test = matrix_rank_report(
        mouse_design,
        [
            cov
        ]
    )


    if (
        np.linalg.matrix_rank(
            X_test.to_numpy(
                dtype=float
            )
        )
        ==
        X_test.shape[1]
    ):

        usable_covariates.append(
            cov
        )


print(
    "\nPotential usable covariates:",
    usable_covariates
)


# ============================================================
# 9. SAVE AUDIT
# ============================================================

mouse_design.to_csv(
    OUT
    / "GSE124872_bloodEC_mouse_design_audit.csv",
    index=False
)


threshold_audit.to_csv(
    OUT
    / "GSE124872_bloodEC_subtype_replicate_threshold_audit.csv",
    index=False
)


print(
    "\nPHASE C4 DESIGN AUDIT COMPLETE"
)

### Replicate discipline

Cells are not biological replicates. The next cell attempts mouse-level pseudobulk when the metadata exposes an animal identifier.

In [ ]:
# ============================================================
# PHASE C5 — REPLICATE-AWARE BLOOD-EC AGING TEST
#
# Biological replicate = mouse.
#
# Primary/global compartment:
#   bloodEC_all
#
# Sensitivity / localization:
#   vascular_endothelial_cells
#   Capillary_endothelial_cells
#   Vcam1+_endothelial_cells
#
# Subtype mouse samples require >=3 cells.
#
# Statistics:
#   - mouse-level pseudobulk log2(CPM + 1)
#   - aged minus young effect
#   - Welch two-sample t-test
#   - 95% Welch CI
#   - BH FDR within each endothelial compartment
#   - Hedges' g effect size
#
# This is a targeted receptor-screening analysis, not
# genome-wide differential-expression inference.
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import ttest_ind, t
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. LOCK ANALYSIS PARAMETERS
# ============================================================

MIN_CELLS_PER_MOUSE_SUBTYPE = 3

MIN_MICE_PER_AGE = 5

MIN_DETECTED_MICE_IN_EITHER_AGE = 2


print(
    "Minimum cells per mouse/subtype:",
    MIN_CELLS_PER_MOUSE_SUBTYPE
)

print(
    "Minimum mice per age group:",
    MIN_MICE_PER_AGE
)


# ============================================================
# 2. BUILD POOLED BLOOD-EC PSEUDOBULK PER MOUSE
#
# Sum the three blood endothelial annotations within each mouse.
#
# This gives a global blood-endothelial state while the
# subtype-specific analyses below tell us whether an effect
# localizes to a particular endothelial population.
# ============================================================

pooled_mouse = (
    receptor_mouse
    .groupby(
        [
            "mouse",
            "age_std",
            "gene"
        ],
        observed=True,
        as_index=False
    )
    .agg(

        n_cells=(
            "n_cells",
            "sum"
        ),

        library_counts=(
            "library_counts",
            "sum"
        ),

        raw_sum=(
            "raw_sum",
            "sum"
        ),

        n_expressing_cells=(
            "n_expressing_cells",
            "sum"
        )
    )
)


pooled_mouse["pct_expressing"] = (
    pooled_mouse[
        "n_expressing_cells"
    ]
    /
    pooled_mouse[
        "n_cells"
    ]
)


pooled_mouse["cpm"] = np.where(

    pooled_mouse[
        "library_counts"
    ]
    > 0,

    1e6
    *
    pooled_mouse[
        "raw_sum"
    ]
    /
    pooled_mouse[
        "library_counts"
    ],

    0.0
)


pooled_mouse["log2cpm"] = np.log2(
    pooled_mouse[
        "cpm"
    ]
    + 1.0
)


pooled_mouse["celltype"] = (
    "bloodEC_all"
)


# ============================================================
# 3. COMBINE GLOBAL + SUBTYPE-SPECIFIC MOUSE TABLES
# ============================================================

analysis_mouse = pd.concat(
    [
        pooled_mouse[
            receptor_mouse.columns
        ],
        receptor_mouse
    ],
    ignore_index=True
)


# Global blood EC samples all have ample cells.
# Subtype samples require >=3 cells per mouse.

is_global = (
    analysis_mouse[
        "celltype"
    ]
    ==
    "bloodEC_all"
)


analysis_mouse = analysis_mouse[
    is_global
    |
    (
        analysis_mouse[
            "n_cells"
        ]
        >=
        MIN_CELLS_PER_MOUSE_SUBTYPE
    )
].copy()


print(
    "\nCompartments entering age analysis:"
)

display(
    analysis_mouse[
        [
            "mouse",
            "age_std",
            "celltype",
            "n_cells"
        ]
    ]
    .drop_duplicates()
    .groupby(
        [
            "celltype",
            "age_std"
        ],
        observed=True
    )
    .agg(
        n_mice=(
            "mouse",
            "nunique"
        ),
        total_cells=(
            "n_cells",
            "sum"
        ),
        median_cells=(
            "n_cells",
            "median"
        )
    )
    .reset_index()
)


# ============================================================
# 4. HELPER — WELCH CI FOR DIFFERENCE OF MEANS
# ============================================================

def welch_ci(
    aged,
    young,
    alpha=0.05
):

    aged = np.asarray(
        aged,
        dtype=float
    )

    young = np.asarray(
        young,
        dtype=float
    )


    na = len(aged)
    ny = len(young)


    delta = (
        aged.mean()
        -
        young.mean()
    )


    va = np.var(
        aged,
        ddof=1
    )

    vy = np.var(
        young,
        ddof=1
    )


    se2 = (
        va / na
        +
        vy / ny
    )


    if (
        not np.isfinite(se2)
        or
        se2 <= 0
    ):

        return (
            float(delta),
            np.nan,
            np.nan,
            np.nan
        )


    se = np.sqrt(
        se2
    )


    numerator = (
        se2 ** 2
    )


    denominator = (
        (
            va / na
        ) ** 2
        /
        (
            na - 1
        )
        +
        (
            vy / ny
        ) ** 2
        /
        (
            ny - 1
        )
    )


    if denominator <= 0:

        df = np.nan

    else:

        df = (
            numerator
            /
            denominator
        )


    if np.isfinite(df):

        critical = t.ppf(
            1
            -
            alpha / 2,
            df
        )

        low = (
            delta
            -
            critical * se
        )

        high = (
            delta
            +
            critical * se
        )

    else:

        low = np.nan
        high = np.nan


    return (
        float(delta),
        float(low)
        if np.isfinite(low)
        else np.nan,
        float(high)
        if np.isfinite(high)
        else np.nan,
        float(df)
        if np.isfinite(df)
        else np.nan
    )


# ============================================================
# 5. HELPER — HEDGES' g
# ============================================================

def hedges_g(
    aged,
    young
):

    aged = np.asarray(
        aged,
        dtype=float
    )

    young = np.asarray(
        young,
        dtype=float
    )


    na = len(aged)
    ny = len(young)


    if (
        na < 2
        or
        ny < 2
    ):

        return np.nan


    va = np.var(
        aged,
        ddof=1
    )

    vy = np.var(
        young,
        ddof=1
    )


    pooled_var = (
        (
            na - 1
        )
        *
        va
        +
        (
            ny - 1
        )
        *
        vy
    ) / (
        na
        +
        ny
        -
        2
    )


    if (
        not np.isfinite(
            pooled_var
        )
        or
        pooled_var <= 0
    ):

        return np.nan


    pooled_sd = np.sqrt(
        pooled_var
    )


    d = (
        aged.mean()
        -
        young.mean()
    ) / pooled_sd


    df = (
        na
        +
        ny
        -
        2
    )


    correction = (
        1
        -
        3
        /
        (
            4 * df
            -
            1
        )
    )


    return float(
        correction
        *
        d
    )


# ============================================================
# 6. TEST EVERY CANDIDATE RECEPTOR WITHIN EACH COMPARTMENT
# ============================================================

effect_rows = []


compartments = [
    "bloodEC_all",
    "vascular_endothelial_cells",
    "Capillary_endothelial_cells",
    "Vcam1+_endothelial_cells"
]


for compartment in compartments:

    df_ct = (
        analysis_mouse[
            analysis_mouse[
                "celltype"
            ]
            ==
            compartment
        ]
        .copy()
    )


    for gene in candidate_receptors:

        g = (
            df_ct[
                df_ct[
                    "gene"
                ]
                ==
                gene
            ]
            .copy()
        )


        young_df = (
            g[
                g[
                    "age_std"
                ]
                ==
                "young"
            ]
        )


        aged_df = (
            g[
                g[
                    "age_std"
                ]
                ==
                "aged"
            ]
        )


        young = (
            young_df[
                "log2cpm"
            ]
            .to_numpy(
                dtype=float
            )
        )


        aged = (
            aged_df[
                "log2cpm"
            ]
            .to_numpy(
                dtype=float
            )
        )


        n_young = len(
            young
        )

        n_aged = len(
            aged
        )


        n_young_detected = int(
            (
                young_df[
                    "raw_sum"
                ]
                > 0
            )
            .sum()
        )


        n_aged_detected = int(
            (
                aged_df[
                    "raw_sum"
                ]
                > 0
            )
            .sum()
        )


        testable = (

            n_young
            >=
            MIN_MICE_PER_AGE

            and

            n_aged
            >=
            MIN_MICE_PER_AGE

            and

            max(
                n_young_detected,
                n_aged_detected
            )
            >=
            MIN_DETECTED_MICE_IN_EITHER_AGE
        )


        if (
            n_young > 0
            and
            n_aged > 0
        ):

            young_mean = float(
                np.mean(
                    young
                )
            )

            aged_mean = float(
                np.mean(
                    aged
                )
            )


            young_median = float(
                np.median(
                    young
                )
            )

            aged_median = float(
                np.median(
                    aged
                )
            )


            delta = (
                aged_mean
                -
                young_mean
            )


            young_pct = float(
                young_df[
                    "pct_expressing"
                ]
                .mean()
            )


            aged_pct = float(
                aged_df[
                    "pct_expressing"
                ]
                .mean()
            )


            delta_pct = (
                aged_pct
                -
                young_pct
            )

        else:

            young_mean = np.nan
            aged_mean = np.nan

            young_median = np.nan
            aged_median = np.nan

            delta = np.nan

            young_pct = np.nan
            aged_pct = np.nan
            delta_pct = np.nan


        if testable:

            test = ttest_ind(
                aged,
                young,
                equal_var=False,
                nan_policy="omit"
            )


            p_value = float(
                test.pvalue
            )


            (
                delta_ci,
                ci_low,
                ci_high,
                welch_df
            ) = welch_ci(
                aged,
                young
            )


            effect_g = hedges_g(
                aged,
                young
            )


            # Ensure the independently calculated
            # difference agrees exactly.

            if not np.isclose(
                delta,
                delta_ci,
                atol=1e-10,
                equal_nan=True
            ):

                raise RuntimeError(
                    f"Effect-size mismatch for "
                    f"{compartment} / {gene}"
                )


        else:

            p_value = np.nan
            ci_low = np.nan
            ci_high = np.nan
            welch_df = np.nan
            effect_g = np.nan


        effect_rows.append({

            "celltype":
                compartment,

            "gene":
                gene,

            "n_young_animals":
                n_young,

            "n_aged_animals":
                n_aged,

            "n_young_detected":
                n_young_detected,

            "n_aged_detected":
                n_aged_detected,

            "young_mean_log2cpm":
                young_mean,

            "aged_mean_log2cpm":
                aged_mean,

            "young_median_log2cpm":
                young_median,

            "aged_median_log2cpm":
                aged_median,

            "age_delta_log2cpm":
                delta,

            "ci95_low":
                ci_low,

            "ci95_high":
                ci_high,

            "young_mean_pct_expressing":
                young_pct,

            "aged_mean_pct_expressing":
                aged_pct,

            "age_delta_pct_expressing":
                delta_pct,

            "hedges_g":
                effect_g,

            "welch_df":
                welch_df,

            "p_value":
                p_value,

            "testable":
                bool(
                    testable
                )
        })


age_effects = pd.DataFrame(
    effect_rows
)


# ============================================================
# 7. BH MULTIPLE-TESTING CORRECTION
#
# Correction is performed independently within each
# endothelial compartment.
# ============================================================

age_effects[
    "fdr_bh"
] = np.nan


for compartment in compartments:

    mask = (

        age_effects[
            "celltype"
        ]
        ==
        compartment

    ) & (

        age_effects[
            "testable"
        ]

    ) & (

        age_effects[
            "p_value"
        ]
        .notna()
    )


    if mask.sum():

        qvals = multipletests(

            age_effects.loc[
                mask,
                "p_value"
            ]
            .to_numpy(),

            method="fdr_bh"
        )[1]


        age_effects.loc[
            mask,
            "fdr_bh"
        ] = qvals


# ============================================================
# 8. ADD INTERPRETIVE FLAGS
#
# Do not filter on them yet.
# ============================================================

age_effects[
    "abs_age_delta_log2cpm"
] = (
    age_effects[
        "age_delta_log2cpm"
    ]
    .abs()
)


age_effects[
    "fdr_lt_005"
] = (
    age_effects[
        "fdr_bh"
    ]
    <
    0.05
)


age_effects[
    "fdr_lt_010"
] = (
    age_effects[
        "fdr_bh"
    ]
    <
    0.10
)


age_effects[
    "ci_excludes_zero"
] = (

    (
        age_effects[
            "ci95_low"
        ]
        >
        0
    )

    |

    (
        age_effects[
            "ci95_high"
        ]
        <
        0
    )
)


# ============================================================
# 9. SAVE + BACKWARD-COMPATIBILITY OBJECTS
# ============================================================

age_effect_overall = (
    age_effects[
        age_effects[
            "celltype"
        ]
        ==
        "bloodEC_all"
    ]
    .copy()
)


age_effect_by_celltype = (
    age_effects[
        age_effects[
            "celltype"
        ]
        !=
        "bloodEC_all"
    ]
    .copy()
)


pseudobulk_age = (
    age_effects.copy()
)


age_effect_overall.to_csv(

    OUT
    / "GSE124872_vascular_receptor_age_effect_overall.csv",

    index=False
)


age_effect_by_celltype.to_csv(

    OUT
    / "GSE124872_vascular_receptor_age_effect_by_celltype.csv",

    index=False
)


pseudobulk_age.to_csv(

    OUT
    / "GSE124872_pseudobulk_receptor_age_effects.csv",

    index=False
)


age_effects.to_csv(

    OUT
    / "GSE124872_bloodEC_mouselevel_age_statistics.csv",

    index=False
)


# ============================================================
# 10. STATISTICAL SUMMARY
# ============================================================

print(
    "\n======================================"
)

print(
    "REPLICATE-AWARE AGE TEST SUMMARY"
)

print(
    "======================================"
)


summary_rows = []


for compartment in compartments:

    z = age_effects[
        age_effects[
            "celltype"
        ]
        ==
        compartment
    ]


    summary_rows.append({

        "celltype":
            compartment,

        "candidate_receptors":
            len(z),

        "testable_receptors":
            int(
                z[
                    "testable"
                ]
                .sum()
            ),

        "FDR<0.05":
            int(
                (
                    z[
                        "fdr_bh"
                    ]
                    <
                    0.05
                )
                .sum()
            ),

        "FDR<0.10":
            int(
                (
                    z[
                        "fdr_bh"
                    ]
                    <
                    0.10
                )
                .sum()
            )
    })


age_test_summary = pd.DataFrame(
    summary_rows
)


display(
    age_test_summary
)


# ============================================================
# 11. TOP GLOBAL BLOOD-EC AGE EFFECTS
# ============================================================

print(
    "\n======================================"
)

print(
    "TOP GLOBAL BLOOD-EC AGE EFFECTS"
)

print(
    "======================================"
)


top_global = (
    age_effect_overall[
        age_effect_overall[
            "testable"
        ]
    ]
    .sort_values(
        [
            "fdr_bh",
            "abs_age_delta_log2cpm"
        ],
        ascending=[
            True,
            False
        ],
        na_position="last"
    )
)


display(
    top_global[
        [
            "gene",
            "n_young_animals",
            "n_aged_animals",
            "n_young_detected",
            "n_aged_detected",
            "young_mean_log2cpm",
            "aged_mean_log2cpm",
            "age_delta_log2cpm",
            "ci95_low",
            "ci95_high",
            "hedges_g",
            "p_value",
            "fdr_bh"
        ]
    ]
    .head(40)
)


# ============================================================
# 12. SELECTED BIOLOGICAL RECEPTOR CHECKS
# ============================================================

selected_receptors_age = [

    "Calcrl",
    "Ramp1",
    "Ramp2",
    "Ramp3",

    "Ntrk2",

    "Kdr",
    "Nrp1",

    "Pdgfrb",

    "Vipr1",
    "Vipr2",

    "Galr1",
    "Galr2",

    "Sstr1",
    "Sstr2",

    "Tacr2"
]


selected_age = (
    age_effects[
        age_effects[
            "gene"
        ]
        .isin(
            selected_receptors_age
        )
    ]
    [
        [
            "celltype",
            "gene",
            "n_young_animals",
            "n_aged_animals",
            "n_young_detected",
            "n_aged_detected",
            "age_delta_log2cpm",
            "ci95_low",
            "ci95_high",
            "hedges_g",
            "p_value",
            "fdr_bh",
            "testable"
        ]
    ]
    .sort_values(
        [
            "gene",
            "celltype"
        ]
    )
)


print(
    "\n======================================"
)

print(
    "SELECTED RECEPTOR AGE CHECKS"
)

print(
    "======================================"
)


display(
    selected_age
)


# ============================================================
# 13. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE C5 REPLICATE-AWARE AGING TEST COMPLETE"
)

print(
    "Total statistical rows:",
    len(
        age_effects
    )
)

print(
    "Global blood-EC receptors:",
    len(
        age_effect_overall
    )
)

print(
    "Subtype receptor rows:",
    len(
        age_effect_by_celltype
    )
)

In [ ]:
# ============================================================
# PHASE C6 — ENDOTHELIAL COMPOSITION SENSITIVITY ANALYSIS
#
# Question:
# Do pooled blood-EC age effects remain after accounting for
# differences in the relative abundance of the three annotated
# endothelial populations?
#
# This is a SENSITIVITY analysis.
#
# Adjustment can remove a real biological effect if aging
# genuinely changes endothelial composition. Therefore:
#
#   unadjusted age model = total blood-EC aging signal
#   composition-adjusted model = composition-independent signal
#
# Neither replaces the subtype-specific results.
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm

from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. BUILD MOUSE-LEVEL ENDOTHELIAL COMPOSITION TABLE
# ============================================================

composition_counts = (
    vas124.obs
    .groupby(
        [
            MOUSE_COL_124,
            CELLTYPE_COL_124
        ],
        observed=True
    )
    .size()
    .unstack(
        fill_value=0
    )
)


required_types = [
    "Capillary_endothelial_cells",
    "Vcam1+_endothelial_cells",
    "vascular_endothelial_cells"
]


for ct in required_types:

    if ct not in composition_counts.columns:

        composition_counts[ct] = 0


composition_counts = (
    composition_counts[
        required_types
    ]
    .copy()
)


composition_counts[
    "total_bloodEC"
] = (
    composition_counts[
        required_types
    ]
    .sum(
        axis=1
    )
)


composition_counts[
    "frac_capillary"
] = (
    composition_counts[
        "Capillary_endothelial_cells"
    ]
    /
    composition_counts[
        "total_bloodEC"
    ]
)


composition_counts[
    "frac_vcam1"
] = (
    composition_counts[
        "Vcam1+_endothelial_cells"
    ]
    /
    composition_counts[
        "total_bloodEC"
    ]
)


composition_counts[
    "frac_vascular"
] = (
    composition_counts[
        "vascular_endothelial_cells"
    ]
    /
    composition_counts[
        "total_bloodEC"
    ]
)


age_map = (
    vas124.obs
    .groupby(
        MOUSE_COL_124,
        observed=True
    )[
        "age_std"
    ]
    .first()
)


composition_counts[
    "age_std"
] = (
    age_map
    .reindex(
        composition_counts.index
    )
)


composition_counts.index.name = "mouse"


composition_table = (
    composition_counts
    .reset_index()
)


print(
    "\n======================================"
)

print(
    "MOUSE-LEVEL ENDOTHELIAL COMPOSITION"
)

print(
    "======================================"
)


display(
    composition_table[
        [
            "mouse",
            "age_std",
            "Capillary_endothelial_cells",
            "Vcam1+_endothelial_cells",
            "vascular_endothelial_cells",
            "total_bloodEC",
            "frac_capillary",
            "frac_vcam1",
            "frac_vascular"
        ]
    ]
    .sort_values(
        [
            "age_std",
            "mouse"
        ]
    )
)


# ============================================================
# 2. TEST WHETHER ENDOTHELIAL COMPOSITION ITSELF CHANGES WITH AGE
# ============================================================

composition_age_rows = []


for frac_col in [
    "frac_capillary",
    "frac_vcam1",
    "frac_vascular"
]:

    young = (
        composition_table.loc[
            composition_table[
                "age_std"
            ]
            ==
            "young",
            frac_col
        ]
        .to_numpy(
            dtype=float
        )
    )


    aged = (
        composition_table.loc[
            composition_table[
                "age_std"
            ]
            ==
            "aged",
            frac_col
        ]
        .to_numpy(
            dtype=float
        )
    )


    result = ttest_ind(
        aged,
        young,
        equal_var=False
    )


    composition_age_rows.append({

        "fraction":
            frac_col,

        "young_mean":
            float(
                young.mean()
            ),

        "aged_mean":
            float(
                aged.mean()
            ),

        "aged_minus_young":
            float(
                aged.mean()
                -
                young.mean()
            ),

        "p_value":
            float(
                result.pvalue
            )
    })


composition_age = pd.DataFrame(
    composition_age_rows
)


composition_age[
    "fdr_bh"
] = multipletests(
    composition_age[
        "p_value"
    ],
    method="fdr_bh"
)[1]


print(
    "\n======================================"
)

print(
    "AGE EFFECT ON ENDOTHELIAL COMPOSITION"
)

print(
    "======================================"
)


display(
    composition_age
)


# ============================================================
# 3. CHECK COMPOSITION-ADJUSTED DESIGN MATRIX
#
# vascular_endothelial_cells is the reference population.
#
# Because:
#
# capillary_fraction
# + VCAM1_fraction
# + vascular_fraction
# = 1
#
# we include only TWO fractions to avoid exact collinearity.
# ============================================================

design = (
    composition_table[
        [
            "mouse",
            "age_std",
            "frac_capillary",
            "frac_vcam1"
        ]
    ]
    .copy()
)


design[
    "aged"
] = (
    design[
        "age_std"
    ]
    ==
    "aged"
).astype(float)


X_check = pd.DataFrame({

    "Intercept":
        1.0,

    "aged":
        design[
            "aged"
        ],

    "frac_capillary":
        design[
            "frac_capillary"
        ],

    "frac_vcam1":
        design[
            "frac_vcam1"
        ]
})


rank = np.linalg.matrix_rank(
    X_check.to_numpy(
        dtype=float
    )
)


print(
    "\nComposition-adjusted design shape:",
    X_check.shape
)

print(
    "Composition-adjusted design rank:",
    rank
)

print(
    "Full rank:",
    rank == X_check.shape[1]
)


if rank != X_check.shape[1]:

    raise RuntimeError(
        "Composition-adjusted design is not full rank."
    )


# ============================================================
# 4. GET POOLED BLOOD-EC MOUSE EXPRESSION
# ============================================================

pooled_expression = (
    analysis_mouse[
        analysis_mouse[
            "celltype"
        ]
        ==
        "bloodEC_all"
    ]
    [
        [
            "mouse",
            "age_std",
            "gene",
            "log2cpm",
            "raw_sum",
            "pct_expressing"
        ]
    ]
    .copy()
)


# ============================================================
# 5. COMPOSITION-ADJUSTED AGE MODEL
#
# OLS:
#
# log2CPM ~ age + capillary_fraction + VCAM1_fraction
#
# HC3 heteroskedasticity-robust standard errors are used
# because n = 15 mice is small.
# ============================================================

adjusted_rows = []


global_testable_genes = set(

    age_effect_overall.loc[
        age_effect_overall[
            "testable"
        ],
        "gene"
    ]
)


for gene in sorted(
    global_testable_genes
):

    g = (
        pooled_expression[
            pooled_expression[
                "gene"
            ]
            ==
            gene
        ]
        .merge(
            design,
            on=[
                "mouse",
                "age_std"
            ],
            how="inner"
        )
    )


    if len(g) != 15:

        raise RuntimeError(
            f"{gene}: expected 15 mice, "
            f"found {len(g)}."
        )


    y = (
        g[
            "log2cpm"
        ]
        .to_numpy(
            dtype=float
        )
    )


    # Skip pathological constant outcomes.
    if np.nanstd(
        y
    ) == 0:

        continue


    X = pd.DataFrame({

        "Intercept":
            np.ones(
                len(g)
            ),

        "aged":
            g[
                "aged"
            ]
            .to_numpy(
                dtype=float
            ),

        "frac_capillary":
            g[
                "frac_capillary"
            ]
            .to_numpy(
                dtype=float
            ),

        "frac_vcam1":
            g[
                "frac_vcam1"
            ]
            .to_numpy(
                dtype=float
            )
    })


    model = sm.OLS(
        y,
        X
    )


    fit = model.fit(
        cov_type="HC3"
    )


    ci = fit.conf_int(
        alpha=0.05
    )


    adjusted_rows.append({

        "gene":
            gene,

        "composition_adjusted_age_coef":
            float(
                fit.params[
                    "aged"
                ]
            ),

        "composition_adjusted_age_se_HC3":
            float(
                fit.bse[
                    "aged"
                ]
            ),

        "composition_adjusted_ci95_low":
            float(
                ci.loc[
                    "aged",
                    0
                ]
            ),

        "composition_adjusted_ci95_high":
            float(
                ci.loc[
                    "aged",
                    1
                ]
            ),

        "composition_adjusted_p_value":
            float(
                fit.pvalues[
                    "aged"
                ]
            ),

        "model_r2":
            float(
                fit.rsquared
            ),

        "n_mice":
            int(
                len(g)
            )
    })


composition_adjusted = pd.DataFrame(
    adjusted_rows
)


# ============================================================
# 6. BH FDR ACROSS TESTABLE RECEPTORS
# ============================================================

composition_adjusted[
    "composition_adjusted_fdr_bh"
] = multipletests(

    composition_adjusted[
        "composition_adjusted_p_value"
    ],

    method="fdr_bh"
)[1]


# ============================================================
# 7. JOIN TO ORIGINAL UNADJUSTED AGE RESULTS
# ============================================================

comparison = (
    age_effect_overall[
        [
            "gene",
            "n_young_animals",
            "n_aged_animals",
            "n_young_detected",
            "n_aged_detected",
            "age_delta_log2cpm",
            "ci95_low",
            "ci95_high",
            "hedges_g",
            "p_value",
            "fdr_bh"
        ]
    ]
    .merge(
        composition_adjusted,
        on="gene",
        how="left"
    )
)


comparison[
    "same_direction"
] = (

    np.sign(
        comparison[
            "age_delta_log2cpm"
        ]
    )

    ==

    np.sign(
        comparison[
            "composition_adjusted_age_coef"
        ]
    )
)


# ============================================================
# 8. CLASSIFY GLOBAL AGE SIGNALS
#
# Important:
# "composition-sensitive" does NOT mean false.
# It means the pooled age association is not independent of
# endothelial population composition.
# ============================================================

def classify_age_signal(row):

    unadjusted_sig = (
        pd.notna(
            row[
                "fdr_bh"
            ]
        )
        and
        row[
            "fdr_bh"
        ]
        <
        0.05
    )


    adjusted_sig = (
        pd.notna(
            row[
                "composition_adjusted_fdr_bh"
            ]
        )
        and
        row[
            "composition_adjusted_fdr_bh"
        ]
        <
        0.05
    )


    if (
        unadjusted_sig
        and
        adjusted_sig
        and
        row[
            "same_direction"
        ]
    ):

        return "composition_robust"


    if unadjusted_sig:

        return "composition_sensitive"


    return "not_global_FDR05"


comparison[
    "age_signal_class"
] = comparison.apply(
    classify_age_signal,
    axis=1
)


# ============================================================
# 9. REPORT ORIGINAL FDR<0.10 SIGNALS
# ============================================================

print(
    "\n======================================"
)

print(
    "GLOBAL AGE SIGNALS WITH COMPOSITION SENSITIVITY"
)

print(
    "======================================"
)


interesting = (
    comparison[
        comparison[
            "fdr_bh"
        ]
        <
        0.10
    ]
    .sort_values(
        [
            "fdr_bh",
            "composition_adjusted_fdr_bh"
        ]
    )
)


display(
    interesting[
        [
            "gene",

            "n_young_detected",
            "n_aged_detected",

            "age_delta_log2cpm",
            "fdr_bh",

            "composition_adjusted_age_coef",
            "composition_adjusted_ci95_low",
            "composition_adjusted_ci95_high",
            "composition_adjusted_p_value",
            "composition_adjusted_fdr_bh",

            "same_direction",
            "age_signal_class"
        ]
    ]
)


# ============================================================
# 10. TOP COMPOSITION-ADJUSTED SIGNALS
# ============================================================

print(
    "\n======================================"
)

print(
    "TOP COMPOSITION-ADJUSTED AGE EFFECTS"
)

print(
    "======================================"
)


display(
    comparison
    .sort_values(
        [
            "composition_adjusted_fdr_bh",
            "composition_adjusted_p_value"
        ]
    )
    [
        [
            "gene",
            "age_delta_log2cpm",
            "fdr_bh",
            "composition_adjusted_age_coef",
            "composition_adjusted_ci95_low",
            "composition_adjusted_ci95_high",
            "composition_adjusted_p_value",
            "composition_adjusted_fdr_bh",
            "same_direction"
        ]
    ]
    .head(
        30
    )
)


# ============================================================
# 11. SAVE
# ============================================================

composition_table.to_csv(

    OUT
    / "GSE124872_bloodEC_mouse_composition.csv",

    index=False
)


composition_age.to_csv(

    OUT
    / "GSE124872_bloodEC_composition_age_effect.csv",

    index=False
)


comparison.to_csv(

    OUT
    / "GSE124872_bloodEC_composition_adjusted_age_sensitivity.csv",

    index=False
)


# ============================================================
# 12. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE C6 COMPOSITION SENSITIVITY COMPLETE"
)


print(
    "Unadjusted global FDR<0.05:",
    int(
        (
            comparison[
                "fdr_bh"
            ]
            <
            0.05
        )
        .sum()
    )
)


print(
    "Composition-adjusted FDR<0.05:",
    int(
        (
            comparison[
                "composition_adjusted_fdr_bh"
            ]
            <
            0.05
        )
        .sum()
    )
)


print(
    "\nAge signal classes:"
)


display(
    comparison[
        "age_signal_class"
    ]
    .value_counts()
)

# Phase D — early B16F10 metastatic seeding in lung endothelium (GSE253749)

Default mode is memory-efficient: one sparse sample at a time, only candidate receptor summaries retained.

In [ ]:
# ============================================================
# PHASE C7 — MURAL / SMOOTH-MUSCLE RECEPTOR COVERAGE AUDIT
#
# Goal:
# Evaluate whether PSN-linked receptors are expressed in the
# secondary perivascular / smooth-muscle compartment.
#
# IMPORTANT:
# Only 78 cells exist in this annotation.
#
# Therefore this cell does NOT make age-regulation claims.
# First we establish:
#   1. mouse coverage
#   2. receptor detection
#   3. whether replicate-aware age testing is defensible
# ============================================================

import numpy as np
import pandas as pd

from scipy import sparse


# ============================================================
# 1. BASIC COMPARTMENT CHECK
# ============================================================

print(
    "\n======================================"
)

print(
    "MURAL / SMOOTH-MUSCLE COMPARTMENT"
)

print(
    "======================================"
)

print(
    "Total cells:",
    mural124.n_obs
)

print(
    "Genes:",
    mural124.n_vars
)


# ============================================================
# 2. MOUSE-LEVEL CELL COVERAGE
# ============================================================

mural_coverage = (
    mural124.obs
    .groupby(
        [
            MOUSE_COL_124,
            "age_std"
        ],
        observed=True
    )
    .size()
    .reset_index(
        name="n_cells"
    )
)


print(
    "\nMouse-level mural coverage:"
)

display(
    mural_coverage
    .sort_values(
        [
            "age_std",
            MOUSE_COL_124
        ]
    )
)


print(
    "\nMice represented by age:"
)

display(
    mural_coverage
    .groupby(
        "age_std",
        observed=True
    )[MOUSE_COL_124]
    .nunique()
)


# ============================================================
# 3. REPLICATE RETENTION UNDER CELL-COUNT THRESHOLDS
# ============================================================

threshold_rows = []


for min_cells in [
    1,
    2,
    3,
    5
]:

    tmp = mural_coverage[
        mural_coverage[
            "n_cells"
        ]
        >=
        min_cells
    ]


    counts = (
        tmp
        .groupby(
            "age_std",
            observed=True
        )[MOUSE_COL_124]
        .nunique()
    )


    threshold_rows.append({

        "min_cells_per_mouse":
            min_cells,

        "young_mice":
            int(
                counts.get(
                    "young",
                    0
                )
            ),

        "aged_mice":
            int(
                counts.get(
                    "aged",
                    0
                )
            )
    })


mural_threshold_audit = pd.DataFrame(
    threshold_rows
)


print(
    "\n======================================"
)

print(
    "MURAL REPLICATE THRESHOLD AUDIT"
)

print(
    "======================================"
)

display(
    mural_threshold_audit
)


# ============================================================
# 4. CANDIDATE RECEPTORS PRESENT IN MATRIX
# ============================================================

mural_candidate_receptors = sorted(

    set(
        lr_psn_detected[
            "mouse_receptor"
        ]
    )

    &

    set(
        mural124.var_names
    )
)


print(
    "\nPSN-linked candidate receptors:",
    lr_psn_detected[
        "mouse_receptor"
    ].nunique()
)

print(
    "Candidate receptors present in mural matrix:",
    len(
        mural_candidate_receptors
    )
)


if len(
    mural_candidate_receptors
) == 0:

    raise RuntimeError(
        "No candidate receptors present in mural compartment."
    )


receptor_idx = (
    mural124.var_names
    .get_indexer(
        mural_candidate_receptors
    )
)


assert (
    receptor_idx >= 0
).all()


# ============================================================
# 5. RAW COUNTS REQUIRED
# ============================================================

if "counts" not in mural124.layers:

    raise RuntimeError(
        "Raw counts layer missing from mural124."
    )


mural_counts = (
    mural124.layers[
        "counts"
    ]
)


# ============================================================
# 6. BUILD MOUSE-LEVEL RECEPTOR TABLE
# ============================================================

mural_mouse_rows = []


for _, cov_row in mural_coverage.iterrows():

    mouse = str(
        cov_row[
            MOUSE_COL_124
        ]
    )

    age = str(
        cov_row[
            "age_std"
        ]
    )


    mask = (

        mural124.obs[
            MOUSE_COL_124
        ]
        .astype(str)
        ==
        mouse

    ) & (

        mural124.obs[
            "age_std"
        ]
        .astype(str)
        ==
        age
    )


    rows = np.where(
        mask.to_numpy()
    )[0]


    if len(rows) == 0:
        continue


    X_all = (
        mural_counts[
            rows,
            :
        ]
    )


    X_rec = (
        mural_counts[
            rows,
            :
        ][
            :,
            receptor_idx
        ]
    )


    library_counts = float(
        X_all.sum()
    )


    receptor_sum = np.asarray(
        X_rec.sum(
            axis=0
        )
    ).ravel()


    if sparse.issparse(
        X_rec
    ):

        n_detected = np.asarray(
            (
                X_rec > 0
            )
            .sum(
                axis=0
            )
        ).ravel()

    else:

        n_detected = (
            X_rec > 0
        ).sum(
            axis=0
        )


    pct_expressing = (
        n_detected
        /
        len(rows)
    )


    if library_counts > 0:

        cpm = (
            1e6
            *
            receptor_sum
            /
            library_counts
        )

    else:

        cpm = np.zeros(
            len(
                mural_candidate_receptors
            )
        )


    log2cpm = np.log2(
        cpm
        + 1.0
    )


    for (
        gene,
        raw_sum,
        gene_cpm,
        gene_log2cpm,
        detected,
        pct
    ) in zip(

        mural_candidate_receptors,
        receptor_sum,
        cpm,
        log2cpm,
        n_detected,
        pct_expressing
    ):

        mural_mouse_rows.append({

            "mouse":
                mouse,

            "age_std":
                age,

            "gene":
                gene,

            "n_cells":
                int(
                    len(rows)
                ),

            "library_counts":
                library_counts,

            "raw_sum":
                float(
                    raw_sum
                ),

            "cpm":
                float(
                    gene_cpm
                ),

            "log2cpm":
                float(
                    gene_log2cpm
                ),

            "n_expressing_cells":
                int(
                    detected
                ),

            "pct_expressing":
                float(
                    pct
                )
        })


mural_receptor_mouse = pd.DataFrame(
    mural_mouse_rows
)


# ============================================================
# 7. OVERALL MURAL RECEPTOR SUPPORT
# ============================================================

mural_support = (
    mural_receptor_mouse
    .groupby(
        "gene",
        observed=True
    )
    .agg(

        n_mice_total=(
            "mouse",
            "nunique"
        ),

        n_mice_detected=(
            "raw_sum",
            lambda x: int(
                (
                    x > 0
                ).sum()
            )
        ),

        n_mice_pct_ge_10=(
            "pct_expressing",
            lambda x: int(
                (
                    x >= 0.10
                ).sum()
            )
        ),

        median_log2cpm=(
            "log2cpm",
            "median"
        ),

        max_log2cpm=(
            "log2cpm",
            "max"
        ),

        median_pct_expressing=(
            "pct_expressing",
            "median"
        ),

        max_pct_expressing=(
            "pct_expressing",
            "max"
        )
    )
    .reset_index()
)


# ============================================================
# 8. SELECTED PERIVASCULAR / NEURAL AXIS CHECKS
# ============================================================

selected_mural_receptors = [

    "Pdgfrb",

    "Calcrl",
    "Ramp1",
    "Ramp2",
    "Ramp3",

    "Ntrk2",

    "Kdr",
    "Nrp1",

    "Vipr1",
    "Vipr2",

    "Galr1",
    "Galr2",

    "Sstr1",
    "Sstr2",

    "Tacr2"
]


print(
    "\n======================================"
)

print(
    "SELECTED MURAL RECEPTOR CHECKS"
)

print(
    "======================================"
)


for gene in selected_mural_receptors:

    hit = (
        mural_support[
            mural_support[
                "gene"
            ]
            ==
            gene
        ]
    )


    if len(hit):

        row = hit.iloc[0]


        print(

            gene,

            "| mice detected:",
            int(
                row[
                    "n_mice_detected"
                ]
            ),
            "/",
            int(
                row[
                    "n_mice_total"
                ]
            ),

            "| mice >=10% cells:",
            int(
                row[
                    "n_mice_pct_ge_10"
                ]
            ),

            "| max % expressing:",
            round(
                float(
                    row[
                        "max_pct_expressing"
                    ]
                ),
                3
            ),

            "| max log2CPM:",
            round(
                float(
                    row[
                        "max_log2cpm"
                    ]
                ),
                3
            )
        )

    else:

        print(
            gene,
            "| absent from candidate receptor set"
        )


# ============================================================
# 9. SAVE
# ============================================================

mural_coverage.to_csv(

    OUT
    / "GSE124872_mural_mouse_cell_coverage.csv",

    index=False
)


mural_threshold_audit.to_csv(

    OUT
    / "GSE124872_mural_replicate_threshold_audit.csv",

    index=False
)


mural_receptor_mouse.to_csv(

    OUT
    / "GSE124872_mural_receptor_mouse_level.csv",

    index=False
)


mural_support.to_csv(

    OUT
    / "GSE124872_mural_receptor_support.csv",

    index=False
)


# ============================================================
# 10. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE C7 MURAL RECEPTOR AUDIT COMPLETE"
)

print(
    "Mural cells:",
    mural124.n_obs
)

print(
    "Mice represented:",
    mural_receptor_mouse[
        "mouse"
    ].nunique()
)

print(
    "Candidate receptors:",
    mural_receptor_mouse[
        "gene"
    ].nunique()
)

In [ ]:
# ============================================================
# PHASE C8 — EXPLORATORY MURAL AGING ANALYSIS
#
# Biological replicate = mouse.
#
# Inclusion:
#   >=2 mural/smooth-muscle cells per mouse
#
# Retained design:
#   young = 6 mice
#   aged  = 5 mice
#
# Because n is small, age significance is evaluated with an
# exact permutation test of the mouse-level mean difference.
#
# This is exploratory evidence only.
# ============================================================

import itertools
import numpy as np
import pandas as pd

from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. LOCK MOUSE-INCLUSION THRESHOLD
# ============================================================

MIN_MURAL_CELLS_PER_MOUSE = 2
MIN_DETECTED_MICE_EITHER_AGE = 2


mural_analysis = (
    mural_receptor_mouse[
        mural_receptor_mouse[
            "n_cells"
        ]
        >=
        MIN_MURAL_CELLS_PER_MOUSE
    ]
    .copy()
)


eligible_mice = (
    mural_analysis[
        [
            "mouse",
            "age_std",
            "n_cells"
        ]
    ]
    .drop_duplicates()
)


print(
    "\n======================================"
)

print(
    "MURAL AGING DESIGN"
)

print(
    "======================================"
)


display(
    eligible_mice
    .sort_values(
        [
            "age_std",
            "mouse"
        ]
    )
)


design_counts = (
    eligible_mice
    .groupby(
        "age_std",
        observed=True
    )["mouse"]
    .nunique()
)


print(
    "\nEligible mice by age:"
)

display(
    design_counts
)


n_young_expected = int(
    design_counts.get(
        "young",
        0
    )
)

n_aged_expected = int(
    design_counts.get(
        "aged",
        0
    )
)


assert n_young_expected == 6
assert n_aged_expected == 5


# ============================================================
# 2. EXACT PERMUTATION TEST
#
# With 11 mice and 5 aged labels:
#
# C(11, 5) = 462 possible assignments.
#
# We enumerate ALL of them.
# ============================================================

def exact_permutation_pvalue(
    values,
    labels
):

    values = np.asarray(
        values,
        dtype=float
    )

    labels = np.asarray(
        labels
    )


    aged_mask = (
        labels
        ==
        "aged"
    )


    n_total = len(
        values
    )

    n_aged = int(
        aged_mask.sum()
    )


    observed = (
        values[
            aged_mask
        ]
        .mean()
        -
        values[
            ~aged_mask
        ]
        .mean()
    )


    permuted = []


    for aged_idx in itertools.combinations(
        range(
            n_total
        ),
        n_aged
    ):

        mask = np.zeros(
            n_total,
            dtype=bool
        )

        mask[
            list(
                aged_idx
            )
        ] = True


        delta = (
            values[
                mask
            ]
            .mean()
            -
            values[
                ~mask
            ]
            .mean()
        )


        permuted.append(
            delta
        )


    permuted = np.asarray(
        permuted,
        dtype=float
    )


    p_two_sided = float(
        (
            np.abs(
                permuted
            )
            >=
            (
                abs(
                    observed
                )
                -
                1e-12
            )
        )
        .mean()
    )


    return (
        float(
            observed
        ),
        p_two_sided,
        int(
            len(
                permuted
            )
        )
    )


# ============================================================
# 3. HEDGES' g
# ============================================================

def mural_hedges_g(
    aged,
    young
):

    aged = np.asarray(
        aged,
        dtype=float
    )

    young = np.asarray(
        young,
        dtype=float
    )


    na = len(
        aged
    )

    ny = len(
        young
    )


    if (
        na < 2
        or
        ny < 2
    ):

        return np.nan


    va = np.var(
        aged,
        ddof=1
    )

    vy = np.var(
        young,
        ddof=1
    )


    pooled_var = (

        (
            na - 1
        )
        *
        va

        +

        (
            ny - 1
        )
        *
        vy

    ) / (

        na
        +
        ny
        -
        2

    )


    if (
        not np.isfinite(
            pooled_var
        )
        or
        pooled_var
        <=
        0
    ):

        return np.nan


    d = (

        aged.mean()
        -
        young.mean()

    ) / np.sqrt(
        pooled_var
    )


    df = (
        na
        +
        ny
        -
        2
    )


    correction = (
        1
        -
        3
        /
        (
            4 * df
            -
            1
        )
    )


    return float(
        correction
        *
        d
    )


# ============================================================
# 4. TEST PSN-LINKED MURAL RECEPTORS
# ============================================================

mural_age_rows = []


for gene in sorted(
    mural_analysis[
        "gene"
    ]
    .unique()
):

    g = (
        mural_analysis[
            mural_analysis[
                "gene"
            ]
            ==
            gene
        ]
        .copy()
    )


    young_df = (
        g[
            g[
                "age_std"
            ]
            ==
            "young"
        ]
    )


    aged_df = (
        g[
            g[
                "age_std"
            ]
            ==
            "aged"
        ]
    )


    young = (
        young_df[
            "log2cpm"
        ]
        .to_numpy(
            dtype=float
        )
    )


    aged = (
        aged_df[
            "log2cpm"
        ]
        .to_numpy(
            dtype=float
        )
    )


    n_young = len(
        young
    )

    n_aged = len(
        aged
    )


    n_young_detected = int(
        (
            young_df[
                "raw_sum"
            ]
            >
            0
        )
        .sum()
    )


    n_aged_detected = int(
        (
            aged_df[
                "raw_sum"
            ]
            >
            0
        )
        .sum()
    )


    testable = (

        n_young
        ==
        n_young_expected

        and

        n_aged
        ==
        n_aged_expected

        and

        max(
            n_young_detected,
            n_aged_detected
        )
        >=
        MIN_DETECTED_MICE_EITHER_AGE
    )


    young_mean = float(
        young.mean()
    )

    aged_mean = float(
        aged.mean()
    )


    young_pct = float(
        young_df[
            "pct_expressing"
        ]
        .mean()
    )

    aged_pct = float(
        aged_df[
            "pct_expressing"
        ]
        .mean()
    )


    if testable:

        values = (
            g[
                "log2cpm"
            ]
            .to_numpy(
                dtype=float
            )
        )


        labels = (
            g[
                "age_std"
            ]
            .astype(str)
            .to_numpy()
        )


        (
            delta,
            permutation_p,
            n_permutations
        ) = exact_permutation_pvalue(
            values,
            labels
        )


        effect_g = mural_hedges_g(
            aged,
            young
        )


    else:

        delta = (
            aged_mean
            -
            young_mean
        )

        permutation_p = np.nan
        n_permutations = 0
        effect_g = np.nan


    mural_age_rows.append({

        "gene":
            gene,

        "n_young_animals":
            n_young,

        "n_aged_animals":
            n_aged,

        "n_young_detected":
            n_young_detected,

        "n_aged_detected":
            n_aged_detected,

        "young_mean_log2cpm":
            young_mean,

        "aged_mean_log2cpm":
            aged_mean,

        "age_delta_log2cpm":
            float(
                delta
            ),

        "young_mean_pct_expressing":
            young_pct,

        "aged_mean_pct_expressing":
            aged_pct,

        "age_delta_pct_expressing":
            float(
                aged_pct
                -
                young_pct
            ),

        "hedges_g":
            effect_g,

        "permutation_p_value":
            permutation_p,

        "n_exact_permutations":
            n_permutations,

        "testable":
            bool(
                testable
            )
    })


mural_age = pd.DataFrame(
    mural_age_rows
)


# ============================================================
# 5. BH FDR
# ============================================================

mural_age[
    "fdr_bh"
] = np.nan


test_mask = (

    mural_age[
        "testable"
    ]

    &

    mural_age[
        "permutation_p_value"
    ]
    .notna()
)


if test_mask.sum():

    mural_age.loc[
        test_mask,
        "fdr_bh"
    ] = multipletests(

        mural_age.loc[
            test_mask,
            "permutation_p_value"
        ],

        method="fdr_bh"
    )[1]


mural_age[
    "abs_age_delta_log2cpm"
] = (
    mural_age[
        "age_delta_log2cpm"
    ]
    .abs()
)


# ============================================================
# 6. SUMMARY
# ============================================================

print(
    "\n======================================"
)

print(
    "EXPLORATORY MURAL AGE TEST SUMMARY"
)

print(
    "======================================"
)


print(
    "Candidate receptors:",
    len(
        mural_age
    )
)

print(
    "Testable receptors:",
    int(
        mural_age[
            "testable"
        ]
        .sum()
    )
)

print(
    "Permutation FDR < 0.05:",
    int(
        (
            mural_age[
                "fdr_bh"
            ]
            <
            0.05
        )
        .sum()
    )
)

print(
    "Permutation FDR < 0.10:",
    int(
        (
            mural_age[
                "fdr_bh"
            ]
            <
            0.10
        )
        .sum()
    )
)


# ============================================================
# 7. TOP EXPLORATORY AGE EFFECTS
# ============================================================

print(
    "\n======================================"
)

print(
    "TOP MURAL AGE EFFECTS"
)

print(
    "======================================"
)


display(
    mural_age[
        mural_age[
            "testable"
        ]
    ]
    .sort_values(
        [
            "fdr_bh",
            "permutation_p_value",
            "abs_age_delta_log2cpm"
        ],
        ascending=[
            True,
            True,
            False
        ],
        na_position="last"
    )
    [
        [
            "gene",
            "n_young_detected",
            "n_aged_detected",
            "young_mean_log2cpm",
            "aged_mean_log2cpm",
            "age_delta_log2cpm",
            "young_mean_pct_expressing",
            "aged_mean_pct_expressing",
            "age_delta_pct_expressing",
            "hedges_g",
            "permutation_p_value",
            "fdr_bh"
        ]
    ]
    .head(
        40
    )
)


# ============================================================
# 8. SELECTED NEUROVASCULAR RECEPTOR CHECKS
# ============================================================

selected_mural_age_genes = [

    "Pdgfrb",

    "Calcrl",
    "Ramp1",
    "Ramp2",
    "Ramp3",

    "Ntrk2",

    "Nrp1",
    "Kdr"
]


print(
    "\n======================================"
)

print(
    "SELECTED MURAL AGE CHECKS"
)

print(
    "======================================"
)


display(
    mural_age[
        mural_age[
            "gene"
        ]
        .isin(
            selected_mural_age_genes
        )
    ]
    [
        [
            "gene",
            "n_young_animals",
            "n_aged_animals",
            "n_young_detected",
            "n_aged_detected",
            "young_mean_log2cpm",
            "aged_mean_log2cpm",
            "age_delta_log2cpm",
            "young_mean_pct_expressing",
            "aged_mean_pct_expressing",
            "hedges_g",
            "permutation_p_value",
            "fdr_bh",
            "testable"
        ]
    ]
    .sort_values(
        "gene"
    )
)


# ============================================================
# 9. SAVE
# ============================================================

mural_age.to_csv(

    OUT
    / "GSE124872_mural_mouselevel_exploratory_age_statistics.csv",

    index=False
)


# ============================================================
# 10. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE C8 EXPLORATORY MURAL AGING COMPLETE"
)

print(
    "Eligible young mice:",
    n_young_expected
)

print(
    "Eligible aged mice:",
    n_aged_expected
)

print(
    "Exact assignments per receptor:",
    int(
        np.math.comb(
            n_young_expected
            +
            n_aged_expected,
            n_aged_expected
        )
    )
    if hasattr(
        np,
        "math"
    )
    else
    462
)

In [ ]:
g253_tar = RAW / "GSE253749" / "GSE253749_RAW.tar"
g253_extract = EXTRACTED / "GSE253749"
g253_extract.mkdir(parents=True, exist_ok=True)

if DOWNLOAD_GSE253749:
    marker = g253_extract / ".extracted_ok"
    if not marker.exists():
        with tarfile.open(g253_tar, "r") as tf:
            tf.extractall(g253_extract)
        marker.write_text("ok")
    files253 = sorted([p.name for p in g253_extract.rglob("*") if p.is_file() and p.name != ".extracted_ok"])
    print("Extracted files:", len(files253))
    print("\n".join(files253[:30]))

In [ ]:
def group_10x_files(folder):
    folder = Path(folder)
    groups = {}
    for mtx in folder.rglob("*matrix.mtx.gz"):
        prefix = mtx.name[:-len("matrix.mtx.gz")]
        features = mtx.with_name(prefix + "features.tsv.gz")
        barcodes = mtx.with_name(prefix + "barcodes.tsv.gz")
        if features.exists() and barcodes.exists():
            groups[prefix] = {"matrix":mtx, "features":features, "barcodes":barcodes}
    return groups

groups253 = group_10x_files(g253_extract) if DOWNLOAD_GSE253749 else {}
COND_253 = {}
for prefix in groups253:
    if "GSM8027388" in prefix:
        COND_253[prefix] = "control"
    elif "GSM8027389" in prefix:
        COND_253[prefix] = "6h"
    elif "GSM8027390" in prefix:
        COND_253[prefix] = "30h"
    else:
        COND_253[prefix] = "unknown"

print(COND_253)

In [ ]:
# ============================================================
# PHASE D1 — GSE253749 RAW-DROPLET QC / CELL-CALLING AUDIT
#
# IMPORTANT:
# These GEO matrices contain the RAW 10x barcode universe.
# Raw barcodes are NOT cells.
#
# We therefore evaluate every barcode using the documented QC:
#
#   detected genes >= 1000
#   detected genes <= 5000
#   mitochondrial counts < 5%
#   ribosomal counts < 20%
#
# One sample is processed at a time to control memory.
#
# NO receptor expression or metastatic effect is calculated
# in this cell.
# ============================================================

import gc
import gzip
import numpy as np
import pandas as pd

from scipy.io import mmread


# ============================================================
# 1. QC PARAMETERS
# ============================================================

MIN_GENES_253 = 1000
MAX_GENES_253 = 5000
MAX_MT_PCT_253 = 5.0
MAX_RIBO_PCT_253 = 20.0


print(
    "GSE253749 QC thresholds:"
)

print(
    "Detected genes:",
    MIN_GENES_253,
    "to",
    MAX_GENES_253
)

print(
    "Mitochondrial counts <",
    MAX_MT_PCT_253,
    "%"
)

print(
    "Ribosomal counts <",
    MAX_RIBO_PCT_253,
    "%"
)


# ============================================================
# 2. OBJECTS RETAINED FOR THE NEXT CELL
# ============================================================

g253_qc_rows = []

g253_qc_pass_indices = {}

g253_qc_pass_barcodes = {}


# ============================================================
# 3. PROCESS EACH RAW 10x SAMPLE INDEPENDENTLY
# ============================================================

for prefix in sorted(
    groups253
):

    fg = groups253[
        prefix
    ]

    condition = COND_253[
        prefix
    ]


    print(
        "\n======================================"
    )

    print(
        prefix,
        "|",
        condition
    )

    print(
        "======================================"
    )


    # --------------------------------------------------------
    # Features
    # --------------------------------------------------------

    features = pd.read_csv(

        fg[
            "features"
        ],

        sep="\t",
        header=None,
        compression="gzip"
    )


    if features.shape[1] >= 2:

        feature_symbols = (
            features.iloc[
                :,
                1
            ]
            .astype(str)
        )

    else:

        feature_symbols = (
            features.iloc[
                :,
                0
            ]
            .astype(str)
        )


    # --------------------------------------------------------
    # Barcodes
    # --------------------------------------------------------

    barcodes = (
        pd.read_csv(

            fg[
                "barcodes"
            ],

            sep="\t",
            header=None,
            compression="gzip"
        )
        .iloc[
            :,
            0
        ]
        .astype(str)
        .to_numpy()
    )


    # --------------------------------------------------------
    # Raw sparse matrix
    #
    # Expected orientation:
    # features x raw barcodes
    # --------------------------------------------------------

    with gzip.open(
        fg[
            "matrix"
        ],
        "rb"
    ) as fh:

        M = (
            mmread(
                fh
            )
            .tocsr()
        )


    if (
        M.shape[0]
        !=
        len(
            feature_symbols
        )
        and
        M.shape[1]
        ==
        len(
            feature_symbols
        )
    ):

        M = (
            M.T
            .tocsr()
        )


    if M.shape[0] != len(
        feature_symbols
    ):

        raise RuntimeError(

            f"{prefix}: feature count mismatch. "
            f"Matrix={M.shape}, "
            f"features={len(feature_symbols)}"
        )


    if M.shape[1] != len(
        barcodes
    ):

        raise RuntimeError(

            f"{prefix}: barcode count mismatch. "
            f"Matrix={M.shape}, "
            f"barcodes={len(barcodes)}"
        )


    print(
        "Raw matrix:",
        M.shape
    )

    print(
        "Raw barcode universe:",
        len(
            barcodes
        )
    )


    # ========================================================
    # 4. PER-BARCODE BASIC QC
    # ========================================================

    total_counts = np.asarray(
        M.sum(
            axis=0
        )
    ).ravel()


    n_genes = np.asarray(
        M.getnnz(
            axis=0
        )
    ).ravel()


    # --------------------------------------------------------
    # Mitochondrial feature mask
    # --------------------------------------------------------

    symbols_lower = (
        feature_symbols
        .str.lower()
    )


    mt_mask = (
        symbols_lower
        .str.startswith(
            "mt-"
        )
        .to_numpy()
    )


    # --------------------------------------------------------
    # Ribosomal protein genes
    # --------------------------------------------------------

    ribo_mask = (
        symbols_lower
        .str.startswith(
            (
                "rpl",
                "rps"
            )
        )
        .to_numpy()
    )


    print(
        "Mitochondrial features:",
        int(
            mt_mask.sum()
        )
    )

    print(
        "Ribosomal features:",
        int(
            ribo_mask.sum()
        )
    )


    if mt_mask.any():

        mt_counts = np.asarray(

            M[
                mt_mask,
                :
            ]
            .sum(
                axis=0
            )

        ).ravel()

    else:

        mt_counts = np.zeros_like(
            total_counts,
            dtype=float
        )


    if ribo_mask.any():

        ribo_counts = np.asarray(

            M[
                ribo_mask,
                :
            ]
            .sum(
                axis=0
            )

        ).ravel()

    else:

        ribo_counts = np.zeros_like(
            total_counts,
            dtype=float
        )


    # --------------------------------------------------------
    # Percentages
    #
    # Zero-count droplets get 0 here, but can never pass
    # because they fail the >=1000-gene criterion.
    # --------------------------------------------------------

    pct_mt = np.divide(

        100.0
        *
        mt_counts,

        total_counts,

        out=np.zeros(
            len(
                total_counts
            ),
            dtype=float
        ),

        where=(
            total_counts
            >
            0
        )
    )


    pct_ribo = np.divide(

        100.0
        *
        ribo_counts,

        total_counts,

        out=np.zeros(
            len(
                total_counts
            ),
            dtype=float
        ),

        where=(
            total_counts
            >
            0
        )
    )


    # ========================================================
    # 5. SEQUENTIAL QC MASKS
    # ========================================================

    nonzero = (
        total_counts
        >
        0
    )


    genes_ge_100 = (
        n_genes
        >=
        100
    )


    genes_ge_500 = (
        n_genes
        >=
        500
    )


    genes_ge_1000 = (
        n_genes
        >=
        MIN_GENES_253
    )


    gene_range_pass = (

        (
            n_genes
            >=
            MIN_GENES_253
        )

        &

        (
            n_genes
            <=
            MAX_GENES_253
        )
    )


    gene_mt_pass = (

        gene_range_pass

        &

        (
            pct_mt
            <
            MAX_MT_PCT_253
        )
    )


    full_qc_pass = (

        gene_range_pass

        &

        (
            pct_mt
            <
            MAX_MT_PCT_253
        )

        &

        (
            pct_ribo
            <
            MAX_RIBO_PCT_253
        )
    )


    pass_idx = np.flatnonzero(
        full_qc_pass
    )


    pass_barcodes = (
        barcodes[
            pass_idx
        ]
    )


    # ========================================================
    # 6. SANITY CHECKS
    # ========================================================

    if len(
        pass_idx
    ) == 0:

        raise RuntimeError(

            f"{prefix}: zero barcodes passed QC. "
            "Inspect feature naming/QC metrics before continuing."
        )


    if not np.all(
        n_genes[
            pass_idx
        ]
        >=
        MIN_GENES_253
    ):

        raise AssertionError(
            "Minimum-gene QC failure."
        )


    if not np.all(
        n_genes[
            pass_idx
        ]
        <=
        MAX_GENES_253
    ):

        raise AssertionError(
            "Maximum-gene QC failure."
        )


    if not np.all(
        pct_mt[
            pass_idx
        ]
        <
        MAX_MT_PCT_253
    ):

        raise AssertionError(
            "Mitochondrial QC failure."
        )


    if not np.all(
        pct_ribo[
            pass_idx
        ]
        <
        MAX_RIBO_PCT_253
    ):

        raise AssertionError(
            "Ribosomal QC failure."
        )


    # ========================================================
    # 7. SAVE ACCEPTED BARCODE INDICES FOR NEXT PHASE
    # ========================================================

    g253_qc_pass_indices[
        prefix
    ] = (
        pass_idx.copy()
    )


    g253_qc_pass_barcodes[
        prefix
    ] = (
        pass_barcodes.copy()
    )


    np.save(

        CACHE
        /
        f"{prefix}_QC_pass_column_indices.npy",

        pass_idx
    )


    # ========================================================
    # 8. SAVE PASSED-BARCODE QC METADATA
    # ========================================================

    passed_qc = pd.DataFrame({

        "sample_prefix":
            prefix,

        "condition":
            condition,

        "raw_column_index":
            pass_idx,

        "barcode":
            pass_barcodes,

        "total_counts":
            total_counts[
                pass_idx
            ],

        "n_genes":
            n_genes[
                pass_idx
            ],

        "pct_counts_mt":
            pct_mt[
                pass_idx
            ],

        "pct_counts_ribo":
            pct_ribo[
                pass_idx
            ]
    })


    passed_qc.to_csv(

        CACHE
        /
        f"{prefix}_QC_pass_barcodes.csv.gz",

        index=False,
        compression="gzip"
    )


    # ========================================================
    # 9. SAMPLE-LEVEL AUDIT ROW
    # ========================================================

    row = {

        "sample_prefix":
            prefix,

        "condition":
            condition,

        "raw_barcodes":
            int(
                len(
                    barcodes
                )
            ),

        "nonzero_barcodes":
            int(
                nonzero.sum()
            ),

        "barcodes_ge_100_genes":
            int(
                genes_ge_100.sum()
            ),

        "barcodes_ge_500_genes":
            int(
                genes_ge_500.sum()
            ),

        "barcodes_ge_1000_genes":
            int(
                genes_ge_1000.sum()
            ),

        "pass_1000_to_5000_genes":
            int(
                gene_range_pass.sum()
            ),

        "pass_gene_range_and_mt":
            int(
                gene_mt_pass.sum()
            ),

        "pass_full_qc":
            int(
                full_qc_pass.sum()
            ),

        "median_genes_pass":
            float(
                np.median(
                    n_genes[
                        pass_idx
                    ]
                )
            ),

        "median_counts_pass":
            float(
                np.median(
                    total_counts[
                        pass_idx
                    ]
                )
            ),

        "median_mt_pct_pass":
            float(
                np.median(
                    pct_mt[
                        pass_idx
                    ]
                )
            ),

        "median_ribo_pct_pass":
            float(
                np.median(
                    pct_ribo[
                        pass_idx
                    ]
                )
            )
    }


    g253_qc_rows.append(
        row
    )


    print(
        "\nQC progression:"
    )

    print(
        "Nonzero barcodes:",
        row[
            "nonzero_barcodes"
        ]
    )

    print(
        ">=100 genes:",
        row[
            "barcodes_ge_100_genes"
        ]
    )

    print(
        ">=500 genes:",
        row[
            "barcodes_ge_500_genes"
        ]
    )

    print(
        ">=1000 genes:",
        row[
            "barcodes_ge_1000_genes"
        ]
    )

    print(
        "1000-5000 genes:",
        row[
            "pass_1000_to_5000_genes"
        ]
    )

    print(
        "+ mt <5%:",
        row[
            "pass_gene_range_and_mt"
        ]
    )

    print(
        "+ ribo <20%:",
        row[
            "pass_full_qc"
        ]
    )

    print(
        "\nPassed-barcode medians:"
    )

    print(
        "Genes:",
        round(
            row[
                "median_genes_pass"
            ],
            1
        )
    )

    print(
        "UMIs:",
        round(
            row[
                "median_counts_pass"
            ],
            1
        )
    )

    print(
        "MT %:",
        round(
            row[
                "median_mt_pct_pass"
            ],
            3
        )
    )

    print(
        "Ribo %:",
        round(
            row[
                "median_ribo_pct_pass"
            ],
            3
        )
    )


    # ========================================================
    # 10. AGGRESSIVE MEMORY CLEANUP BEFORE NEXT SAMPLE
    # ========================================================

    del M
    del features
    del feature_symbols
    del symbols_lower
    del barcodes

    del total_counts
    del n_genes

    del mt_counts
    del ribo_counts
    del pct_mt
    del pct_ribo

    del nonzero
    del genes_ge_100
    del genes_ge_500
    del genes_ge_1000
    del gene_range_pass
    del gene_mt_pass
    del full_qc_pass

    del mt_mask
    del ribo_mask

    del passed_qc

    gc.collect()


# ============================================================
# 11. FINAL QC SUMMARY
# ============================================================

g253_qc_summary = pd.DataFrame(
    g253_qc_rows
)


g253_qc_summary.to_csv(

    OUT
    /
    "GSE253749_raw_barcode_QC_audit.csv",

    index=False
)


print(
    "\n======================================"
)

print(
    "GSE253749 QC SUMMARY"
)

print(
    "======================================"
)


display(
    g253_qc_summary
)


print(
    "\nTotal raw barcodes:",
    int(
        g253_qc_summary[
            "raw_barcodes"
        ]
        .sum()
    )
)


print(
    "Total barcodes passing full QC:",
    int(
        g253_qc_summary[
            "pass_full_qc"
        ]
        .sum()
    )
)


print(
    "\nPHASE D1 RAW-DROPLET QC COMPLETE"
)

In [ ]:
# ============================================================
# PHASE D2 — GSE253749 ENDOTHELIAL IDENTITY AUDIT
#
# QC-passing barcodes are NOT automatically endothelial cells.
#
# We now ask whether they express endothelial markers and
# whether there is evidence of:
#
#   immune contamination
#   epithelial contamination
#   mural/fibroblast contamination
#   B16F10 / melanocytic contamination
#
# This cell is DESCRIPTIVE.
# It does NOT yet filter cells or calculate receptor effects.
# ============================================================

import gc
import gzip
import numpy as np
import pandas as pd

from collections import defaultdict
from scipy.io import mmread


# ============================================================
# 1. MARKER PANELS
# ============================================================

marker_panels = {

    "endothelial": [
        "Cdh5",
        "Pecam1",
        "Ptprb",
        "Esam",
        "Eng",
        "Kdr",
        "Tek",
        "Emcn",
        "Ramp2"
    ],

    "immune": [
        "Ptprc",
        "Lst1",
        "Tyrobp",
        "Fcerg1",
        "Csf1r",
        "Lyz2",
        "Cd3d",
        "Cd79a",
        "Ms4a1",
        "Nkg7"
    ],

    "epithelial": [
        "Epcam",
        "Krt8",
        "Krt18",
        "Krt19",
        "Scgb1a1",
        "Sftpc",
        "Ager",
        "Hopx"
    ],

    "mural_fibroblast": [
        "Pdgfrb",
        "Rgs5",
        "Cspg4",
        "Acta2",
        "Tagln",
        "Pdgfra",
        "Col1a1",
        "Col1a2",
        "Dcn",
        "Lum"
    ],

    "melanoma": [
        "Pmel",
        "Mlana",
        "Tyr",
        "Dct",
        "Sox10",
        "Mitf"
    ]
}


all_markers = sorted(
    {
        gene
        for panel in marker_panels.values()
        for gene in panel
    }
)


print(
    "Marker genes audited:",
    len(all_markers)
)


# ============================================================
# 2. OUTPUT OBJECTS
# ============================================================

identity_summary_rows = []

marker_summary_rows = []

g253_identity_tables = {}


# ============================================================
# 3. PROCESS ONE SAMPLE AT A TIME
# ============================================================

for prefix in sorted(
    groups253
):

    fg = groups253[prefix]

    condition = COND_253[prefix]


    print(
        "\n======================================"
    )

    print(
        prefix,
        "|",
        condition
    )

    print(
        "======================================"
    )


    # --------------------------------------------------------
    # Restore QC-passing raw column indices
    # --------------------------------------------------------

    if prefix in g253_qc_pass_indices:

        pass_idx = np.asarray(
            g253_qc_pass_indices[prefix],
            dtype=int
        )

    else:

        pass_idx = np.load(
            CACHE
            /
            f"{prefix}_QC_pass_column_indices.npy"
        )


    passed_qc_path = (
        CACHE
        /
        f"{prefix}_QC_pass_barcodes.csv.gz"
    )


    passed_qc = pd.read_csv(
        passed_qc_path,
        compression="gzip"
    )


    assert len(passed_qc) == len(pass_idx)


    # ========================================================
    # 4. FEATURES
    # ========================================================

    features = pd.read_csv(

        fg["features"],

        sep="\t",
        header=None,
        compression="gzip"
    )


    if features.shape[1] >= 2:

        feature_symbols = (
            features.iloc[:, 1]
            .astype(str)
            .tolist()
        )

    else:

        feature_symbols = (
            features.iloc[:, 0]
            .astype(str)
            .tolist()
        )


    # Case-insensitive lookup, retaining duplicate rows.
    symbol_rows = defaultdict(list)


    for i, symbol in enumerate(
        feature_symbols
    ):

        symbol_rows[
            symbol.lower()
        ].append(i)


    # ========================================================
    # 5. RAW MATRIX
    # ========================================================

    with gzip.open(
        fg["matrix"],
        "rb"
    ) as fh:

        M = (
            mmread(fh)
            .tocsr()
        )


    if (
        M.shape[0]
        != len(feature_symbols)
        and
        M.shape[1]
        ==
        len(feature_symbols)
    ):

        M = M.T.tocsr()


    if M.shape[0] != len(feature_symbols):

        raise RuntimeError(
            f"{prefix}: feature mismatch."
        )


    # ========================================================
    # 6. EXTRACT ONLY MARKER GENES
    #
    # We intentionally do NOT slice the complete 32k x 10k
    # QC-passing matrix here. Only ~40-50 marker genes are
    # extracted, keeping memory usage low.
    # ========================================================

    marker_raw = {}

    marker_detected = {}

    missing_markers = []


    for gene in all_markers:

        rows = symbol_rows.get(
            gene.lower(),
            []
        )


        if len(rows) == 0:

            x = np.zeros(
                len(pass_idx),
                dtype=float
            )

            missing_markers.append(
                gene
            )

        else:

            x = np.asarray(

                M[
                    rows,
                    :
                ][
                    :,
                    pass_idx
                ]
                .sum(
                    axis=0
                )

            ).ravel()


        marker_raw[gene] = x

        marker_detected[gene] = (
            x > 0
        )


        marker_summary_rows.append({

            "sample_prefix":
                prefix,

            "condition":
                condition,

            "panel":
                next(
                    panel_name
                    for panel_name, genes
                    in marker_panels.items()
                    if gene in genes
                ),

            "gene":
                gene,

            "n_qc_barcodes":
                len(pass_idx),

            "n_detected":
                int(
                    (
                        x > 0
                    ).sum()
                ),

            "pct_detected":
                float(
                    (
                        x > 0
                    ).mean()
                ),

            "mean_raw_count":
                float(
                    x.mean()
                ),

            "mean_raw_count_if_detected":
                float(
                    x[
                        x > 0
                    ].mean()
                )
                if (
                    x > 0
                ).any()
                else 0.0
        })


    print(
        "QC-passing barcodes:",
        len(pass_idx)
    )


    if missing_markers:

        print(
            "Markers absent from feature table:",
            missing_markers
        )


    # ========================================================
    # 7. PANEL HIT COUNTS PER QC-PASSING BARCODE
    # ========================================================

    panel_hits = {}


    for panel_name, genes in marker_panels.items():

        hit_matrix = np.vstack(
            [
                marker_detected[g]
                for g in genes
            ]
        )


        panel_hits[
            panel_name
        ] = (
            hit_matrix.sum(
                axis=0
            )
        )


    ec_hits = panel_hits[
        "endothelial"
    ]

    immune_hits = panel_hits[
        "immune"
    ]

    epi_hits = panel_hits[
        "epithelial"
    ]

    mural_hits = panel_hits[
        "mural_fibroblast"
    ]

    melanoma_hits = panel_hits[
        "melanoma"
    ]


    # ========================================================
    # 8. DESCRIPTIVE ENDOTHELIAL SUPPORT FLAGS
    #
    # These are NOT final cell calls yet.
    # ========================================================

    ec_ge1 = (
        ec_hits >= 1
    )

    ec_ge2 = (
        ec_hits >= 2
    )

    ec_ge3 = (
        ec_hits >= 3
    )


    any_immune = (
        immune_hits >= 1
    )

    any_epithelial = (
        epi_hits >= 1
    )

    any_mural = (
        mural_hits >= 1
    )

    any_melanoma = (
        melanoma_hits >= 1
    )


    clean_ec_ge2 = (

        ec_ge2

        &

        (
            immune_hits == 0
        )

        &

        (
            epi_hits == 0
        )

        &

        (
            melanoma_hits == 0
        )
    )


    # ========================================================
    # 9. PER-BARCODE IDENTITY AUDIT TABLE
    # ========================================================

    identity = passed_qc.copy()


    identity[
        "endothelial_marker_hits"
    ] = ec_hits


    identity[
        "immune_marker_hits"
    ] = immune_hits


    identity[
        "epithelial_marker_hits"
    ] = epi_hits


    identity[
        "mural_fibroblast_marker_hits"
    ] = mural_hits


    identity[
        "melanoma_marker_hits"
    ] = melanoma_hits


    identity[
        "endothelial_ge1"
    ] = ec_ge1


    identity[
        "endothelial_ge2"
    ] = ec_ge2


    identity[
        "endothelial_ge3"
    ] = ec_ge3


    identity[
        "any_immune_marker"
    ] = any_immune


    identity[
        "any_epithelial_marker"
    ] = any_epithelial


    identity[
        "any_mural_fibroblast_marker"
    ] = any_mural


    identity[
        "any_melanoma_marker"
    ] = any_melanoma


    identity[
        "clean_endothelial_ge2"
    ] = clean_ec_ge2


    g253_identity_tables[
        prefix
    ] = identity


    identity.to_csv(

        CACHE
        /
        f"{prefix}_identity_marker_audit.csv.gz",

        index=False,
        compression="gzip"
    )


    # ========================================================
    # 10. SAMPLE SUMMARY
    # ========================================================

    summary_row = {

        "sample_prefix":
            prefix,

        "condition":
            condition,

        "qc_barcodes":
            len(identity),

        "ec_ge1_n":
            int(
                ec_ge1.sum()
            ),

        "ec_ge1_pct":
            float(
                ec_ge1.mean()
            ),

        "ec_ge2_n":
            int(
                ec_ge2.sum()
            ),

        "ec_ge2_pct":
            float(
                ec_ge2.mean()
            ),

        "ec_ge3_n":
            int(
                ec_ge3.sum()
            ),

        "ec_ge3_pct":
            float(
                ec_ge3.mean()
            ),

        "clean_ec_ge2_n":
            int(
                clean_ec_ge2.sum()
            ),

        "clean_ec_ge2_pct":
            float(
                clean_ec_ge2.mean()
            ),

        "any_immune_n":
            int(
                any_immune.sum()
            ),

        "any_immune_pct":
            float(
                any_immune.mean()
            ),

        "any_epithelial_n":
            int(
                any_epithelial.sum()
            ),

        "any_epithelial_pct":
            float(
                any_epithelial.mean()
            ),

        "any_mural_n":
            int(
                any_mural.sum()
            ),

        "any_mural_pct":
            float(
                any_mural.mean()
            ),

        "any_melanoma_n":
            int(
                any_melanoma.sum()
            ),

        "any_melanoma_pct":
            float(
                any_melanoma.mean()
            )
    }


    identity_summary_rows.append(
        summary_row
    )


    print(
        "\nEndothelial marker support:"
    )

    print(
        ">=1 EC marker:",
        summary_row["ec_ge1_n"],
        f"({100*summary_row['ec_ge1_pct']:.1f}%)"
    )

    print(
        ">=2 EC markers:",
        summary_row["ec_ge2_n"],
        f"({100*summary_row['ec_ge2_pct']:.1f}%)"
    )

    print(
        ">=3 EC markers:",
        summary_row["ec_ge3_n"],
        f"({100*summary_row['ec_ge3_pct']:.1f}%)"
    )

    print(
        ">=2 EC markers, no immune/epithelial/melanoma markers:",
        summary_row["clean_ec_ge2_n"],
        f"({100*summary_row['clean_ec_ge2_pct']:.1f}%)"
    )


    print(
        "\nPotential non-EC marker support:"
    )

    print(
        "Any immune marker:",
        summary_row["any_immune_n"],
        f"({100*summary_row['any_immune_pct']:.1f}%)"
    )

    print(
        "Any epithelial marker:",
        summary_row["any_epithelial_n"],
        f"({100*summary_row['any_epithelial_pct']:.1f}%)"
    )

    print(
        "Any mural/fibroblast marker:",
        summary_row["any_mural_n"],
        f"({100*summary_row['any_mural_pct']:.1f}%)"
    )

    print(
        "Any melanoma marker:",
        summary_row["any_melanoma_n"],
        f"({100*summary_row['any_melanoma_pct']:.1f}%)"
    )


    # ========================================================
    # 11. SELECTED MARKER DETECTION
    # ========================================================

    print(
        "\nSelected markers (% QC-passing barcodes):"
    )


    selected_markers = [

        "Cdh5",
        "Pecam1",
        "Ptprb",
        "Kdr",
        "Esam",
        "Eng",
        "Emcn",
        "Tek",

        "Ptprc",
        "Lyz2",

        "Epcam",
        "Krt8",

        "Pdgfrb",
        "Col1a1",

        "Pmel",
        "Mlana",
        "Tyr"
    ]


    for gene in selected_markers:

        detected_pct = (
            marker_detected[
                gene
            ]
            .mean()
        )


        print(
            gene,
            ":",
            round(
                100
                *
                float(
                    detected_pct
                ),
                2
            ),
            "%"
        )


    # ========================================================
    # 12. CLEANUP
    # ========================================================

    del M
    del features
    del feature_symbols
    del symbol_rows

    del marker_raw
    del marker_detected
    del panel_hits

    del ec_hits
    del immune_hits
    del epi_hits
    del mural_hits
    del melanoma_hits

    del ec_ge1
    del ec_ge2
    del ec_ge3
    del clean_ec_ge2

    del any_immune
    del any_epithelial
    del any_mural
    del any_melanoma

    del passed_qc
    del identity

    gc.collect()


# ============================================================
# 13. FINAL SUMMARIES
# ============================================================

g253_identity_summary = pd.DataFrame(
    identity_summary_rows
)


g253_marker_summary = pd.DataFrame(
    marker_summary_rows
)


g253_identity_summary.to_csv(

    OUT
    /
    "GSE253749_endothelial_identity_audit.csv",

    index=False
)


g253_marker_summary.to_csv(

    OUT
    /
    "GSE253749_marker_detection_audit.csv",

    index=False
)


print(
    "\n======================================"
)

print(
    "GSE253749 ENDOTHELIAL IDENTITY SUMMARY"
)

print(
    "======================================"
)


display(
    g253_identity_summary
)


print(
    "\nPHASE D2 ENDOTHELIAL IDENTITY AUDIT COMPLETE"
)

In [ ]:
# ============================================================
# PHASE D3 — CONTAMINATION-PANEL DRIVER AUDIT
#
# D2 confirmed strong endothelial identity.
#
# Now determine WHICH individual genes are driving the
# apparently high immune / epithelial / mural / melanoma flags.
#
# No filtering is performed here.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. PER-GENE DETECTION BY PANEL AND SAMPLE
# ============================================================

print(
    "\n======================================"
)

print(
    "PER-GENE MARKER DETECTION"
)

print(
    "======================================"
)


for prefix in sorted(
    g253_marker_summary[
        "sample_prefix"
    ]
    .unique()
):

    condition = COND_253[
        prefix
    ]


    print(
        "\n--------------------------------------"
    )

    print(
        prefix,
        "|",
        condition
    )

    print(
        "--------------------------------------"
    )


    sample_markers = (
        g253_marker_summary[
            g253_marker_summary[
                "sample_prefix"
            ]
            ==
            prefix
        ]
        .copy()
    )


    for panel in [
        "endothelial",
        "immune",
        "epithelial",
        "mural_fibroblast",
        "melanoma"
    ]:

        print(
            f"\n{panel.upper()}"
        )


        display(
            sample_markers[
                sample_markers[
                    "panel"
                ]
                ==
                panel
            ]
            [
                [
                    "gene",
                    "n_detected",
                    "pct_detected",
                    "mean_raw_count",
                    "mean_raw_count_if_detected"
                ]
            ]
            .sort_values(
                "pct_detected",
                ascending=False
            )
        )


# ============================================================
# 2. PANEL-HIT COUNT DISTRIBUTIONS PER CELL
#
# This tells us whether the high contamination flags are mostly
# caused by ONE isolated marker (consistent with ambient RNA /
# non-specific expression) or by multiple lineage markers.
# ============================================================

panel_hit_columns = {

    "endothelial":
        "endothelial_marker_hits",

    "immune":
        "immune_marker_hits",

    "epithelial":
        "epithelial_marker_hits",

    "mural_fibroblast":
        "mural_fibroblast_marker_hits",

    "melanoma":
        "melanoma_marker_hits"
}


hit_distribution_rows = []


for prefix in sorted(
    g253_identity_tables
):

    identity = (
        g253_identity_tables[
            prefix
        ]
        .copy()
    )


    condition = COND_253[
        prefix
    ]


    for panel, col in panel_hit_columns.items():

        x = (
            identity[
                col
            ]
            .to_numpy(
                dtype=int
            )
        )


        hit_distribution_rows.append({

            "sample_prefix":
                prefix,

            "condition":
                condition,

            "panel":
                panel,

            "n_cells":
                len(x),

            "zero_hits_n":
                int(
                    (
                        x == 0
                    )
                    .sum()
                ),

            "zero_hits_pct":
                float(
                    (
                        x == 0
                    )
                    .mean()
                ),

            "one_hit_n":
                int(
                    (
                        x == 1
                    )
                    .sum()
                ),

            "one_hit_pct":
                float(
                    (
                        x == 1
                    )
                    .mean()
                ),

            "two_hits_n":
                int(
                    (
                        x == 2
                    )
                    .sum()
                ),

            "two_hits_pct":
                float(
                    (
                        x == 2
                    )
                    .mean()
                ),

            "three_plus_hits_n":
                int(
                    (
                        x >= 3
                    )
                    .sum()
                ),

            "three_plus_hits_pct":
                float(
                    (
                        x >= 3
                    )
                    .mean()
                ),

            "median_hits":
                float(
                    np.median(
                        x
                    )
                ),

            "max_hits":
                int(
                    x.max()
                )
        })


panel_hit_distribution = pd.DataFrame(
    hit_distribution_rows
)


print(
    "\n======================================"
)

print(
    "PANEL HIT-COUNT DISTRIBUTIONS"
)

print(
    "======================================"
)


display(
    panel_hit_distribution
    .sort_values(
        [
            "sample_prefix",
            "panel"
        ]
    )
)


# ============================================================
# 3. MULTI-MARKER NON-ENDOTHELIAL FLAGS
#
# We are NOT declaring contaminants yet.
#
# These are only more stringent descriptive flags than
# "any single marker".
# ============================================================

strict_flag_rows = []


for prefix in sorted(
    g253_identity_tables
):

    identity = (
        g253_identity_tables[
            prefix
        ]
    )


    condition = COND_253[
        prefix
    ]


    n = len(
        identity
    )


    ec3 = (
        identity[
            "endothelial_marker_hits"
        ]
        >=
        3
    )


    immune2 = (
        identity[
            "immune_marker_hits"
        ]
        >=
        2
    )


    epithelial2 = (
        identity[
            "epithelial_marker_hits"
        ]
        >=
        2
    )


    mural2 = (
        identity[
            "mural_fibroblast_marker_hits"
        ]
        >=
        2
    )


    melanoma2 = (
        identity[
            "melanoma_marker_hits"
        ]
        >=
        2
    )


    strict_flag_rows.append({

        "sample_prefix":
            prefix,

        "condition":
            condition,

        "qc_barcodes":
            n,

        "ec_ge3_n":
            int(
                ec3.sum()
            ),

        "ec_ge3_pct":
            float(
                ec3.mean()
            ),

        "immune_ge2_n":
            int(
                immune2.sum()
            ),

        "immune_ge2_pct":
            float(
                immune2.mean()
            ),

        "epithelial_ge2_n":
            int(
                epithelial2.sum()
            ),

        "epithelial_ge2_pct":
            float(
                epithelial2.mean()
            ),

        "mural_ge2_n":
            int(
                mural2.sum()
            ),

        "mural_ge2_pct":
            float(
                mural2.mean()
            ),

        "melanoma_ge2_n":
            int(
                melanoma2.sum()
            ),

        "melanoma_ge2_pct":
            float(
                melanoma2.mean()
            ),

        "ec_ge3_and_no_multilineage_n":
            int(
                (
                    ec3
                    &
                    ~immune2
                    &
                    ~epithelial2
                    &
                    ~mural2
                    &
                    ~melanoma2
                )
                .sum()
            ),

        "ec_ge3_and_no_multilineage_pct":
            float(
                (
                    ec3
                    &
                    ~immune2
                    &
                    ~epithelial2
                    &
                    ~mural2
                    &
                    ~melanoma2
                )
                .mean()
            )
    })


strict_flag_summary = pd.DataFrame(
    strict_flag_rows
)


print(
    "\n======================================"
)

print(
    "MULTI-MARKER LINEAGE FLAGS"
)

print(
    "======================================"
)


display(
    strict_flag_summary
)


# ============================================================
# 4. TOP SUSPECTED PANEL DRIVERS ACROSS ALL CONDITIONS
# ============================================================

driver_summary = (
    g253_marker_summary
    .groupby(
        [
            "panel",
            "gene"
        ],
        observed=True
    )
    .agg(

        mean_pct_detected=(
            "pct_detected",
            "mean"
        ),

        max_pct_detected=(
            "pct_detected",
            "max"
        ),

        mean_raw_count=(
            "mean_raw_count",
            "mean"
        )
    )
    .reset_index()
)


print(
    "\n======================================"
)

print(
    "TOP PANEL DRIVERS ACROSS CONDITIONS"
)

print(
    "======================================"
)


for panel in [
    "immune",
    "epithelial",
    "mural_fibroblast",
    "melanoma"
]:

    print(
        f"\n{panel.upper()}"
    )


    display(
        driver_summary[
            driver_summary[
                "panel"
            ]
            ==
            panel
        ]
        .sort_values(
            "mean_pct_detected",
            ascending=False
        )
    )


# ============================================================
# 5. SAVE
# ============================================================

panel_hit_distribution.to_csv(

    OUT
    /
    "GSE253749_marker_panel_hit_distributions.csv",

    index=False
)


strict_flag_summary.to_csv(

    OUT
    /
    "GSE253749_multimarker_lineage_flag_audit.csv",

    index=False
)


driver_summary.to_csv(

    OUT
    /
    "GSE253749_marker_panel_driver_summary.csv",

    index=False
)


# ============================================================
# 6. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE D3 CONTAMINATION-PANEL AUDIT COMPLETE"
)

In [ ]:
# ============================================================
# PHASE D4 — FINAL CONSERVATIVE ENDOTHELIAL CELL SET
#
# D3 showed that single-marker exclusion is inappropriate:
#
#   Lyz2 alone drives much of the immune signal
#   Hopx / Sftpc / Scgb1a1 behave like ambient lung RNA
#   Mitf alone drives much of the melanoma signal
#
# Final operational EC definition:
#
# REQUIRED:
#   >=3 endothelial markers
#
# EXCLUDE coherent contamination:
#
# Immune:
#   Ptprc detected
#       OR
#   >=2 immune-panel markers
#
# Epithelial:
#   Epcam detected
#       OR
#   >=2 keratin markers among Krt8/Krt18/Krt19
#
# Mural/fibroblast:
#   >=2 mural/fibroblast markers
#
# Melanoma:
#   >=2 specific melanocytic markers among
#   Pmel/Mlana/Tyr/Dct/Sox10
#
# Mitf alone is NOT sufficient for melanoma exclusion.
#
# This is a conservative purity filter, not a claim that every
# excluded cell belongs to the indicated lineage.
# ============================================================

import gc
import gzip
import numpy as np
import pandas as pd

from collections import defaultdict
from scipy.io import mmread


# ============================================================
# 1. OUTPUT OBJECTS
# ============================================================

g253_final_ec_indices = {}

g253_final_ec_barcodes = {}

g253_final_identity_tables = {}

final_ec_summary_rows = []


# ============================================================
# 2. SPECIFIC MARKERS NEEDED FOR FINAL FILTER
# ============================================================

filter_genes = [

    # specific immune marker
    "Ptprc",

    # epithelial-specific / keratin support
    "Epcam",
    "Krt8",
    "Krt18",
    "Krt19",

    # melanoma-specific panel
    "Pmel",
    "Mlana",
    "Tyr",
    "Dct",
    "Sox10"
]


epithelial_keratins = [
    "Krt8",
    "Krt18",
    "Krt19"
]


melanoma_specific = [
    "Pmel",
    "Mlana",
    "Tyr",
    "Dct",
    "Sox10"
]


# ============================================================
# 3. PROCESS EACH SAMPLE
# ============================================================

for prefix in sorted(
    groups253
):

    condition = COND_253[
        prefix
    ]

    fg = groups253[
        prefix
    ]


    print(
        "\n======================================"
    )

    print(
        prefix,
        "|",
        condition
    )

    print(
        "======================================"
    )


    # ========================================================
    # 4. RESTORE D2 IDENTITY TABLE
    # ========================================================

    if (
        "g253_identity_tables" in globals()
        and
        prefix in g253_identity_tables
    ):

        identity = (
            g253_identity_tables[
                prefix
            ]
            .copy()
        )

    else:

        identity = pd.read_csv(

            CACHE
            /
            f"{prefix}_identity_marker_audit.csv.gz",

            compression="gzip"
        )


    raw_idx = (
        identity[
            "raw_column_index"
        ]
        .to_numpy(
            dtype=int
        )
    )


    # ========================================================
    # 5. FEATURE LOOKUP
    # ========================================================

    features = pd.read_csv(

        fg[
            "features"
        ],

        sep="\t",
        header=None,
        compression="gzip"
    )


    if features.shape[1] >= 2:

        symbols = (
            features.iloc[
                :,
                1
            ]
            .astype(str)
            .tolist()
        )

    else:

        symbols = (
            features.iloc[
                :,
                0
            ]
            .astype(str)
            .tolist()
        )


    symbol_rows = defaultdict(
        list
    )


    for i, symbol in enumerate(
        symbols
    ):

        symbol_rows[
            symbol.lower()
        ].append(
            i
        )


    # ========================================================
    # 6. RAW MATRIX
    # ========================================================

    with gzip.open(
        fg[
            "matrix"
        ],
        "rb"
    ) as fh:

        M = (
            mmread(
                fh
            )
            .tocsr()
        )


    if (
        M.shape[0]
        !=
        len(
            symbols
        )
        and
        M.shape[1]
        ==
        len(
            symbols
        )
    ):

        M = (
            M.T
            .tocsr()
        )


    if M.shape[0] != len(
        symbols
    ):

        raise RuntimeError(
            f"{prefix}: feature mismatch."
        )


    # ========================================================
    # 7. EXTRACT SPECIFIC FILTER MARKERS
    # ========================================================

    filter_counts = {}


    for gene in filter_genes:

        rows = symbol_rows.get(
            gene.lower(),
            []
        )


        if len(
            rows
        ) == 0:

            x = np.zeros(
                len(
                    raw_idx
                ),
                dtype=float
            )

        else:

            x = np.asarray(

                M[
                    rows,
                    :
                ][
                    :,
                    raw_idx
                ]
                .sum(
                    axis=0
                )

            ).ravel()


        filter_counts[
            gene
        ] = x


    # ========================================================
    # 8. REQUIRED ENDOTHELIAL SUPPORT
    # ========================================================

    ec_ge3 = (

        identity[
            "endothelial_marker_hits"
        ]
        .to_numpy(
            dtype=int
        )

        >=
        3
    )


    # ========================================================
    # 9. IMMUNE EXCLUSION
    #
    # Lyz2 alone is NOT sufficient.
    # ========================================================

    immune_hits = (
        identity[
            "immune_marker_hits"
        ]
        .to_numpy(
            dtype=int
        )
    )


    ptprc_positive = (
        filter_counts[
            "Ptprc"
        ]
        >
        0
    )


    immune_exclude = (

        ptprc_positive

        |

        (
            immune_hits
            >=
            2
        )
    )


    # ========================================================
    # 10. EPITHELIAL EXCLUSION
    #
    # Hopx / Sftpc / Scgb1a1 are intentionally NOT used.
    # ========================================================

    epcam_positive = (
        filter_counts[
            "Epcam"
        ]
        >
        0
    )


    keratin_hits = np.zeros(
        len(
            identity
        ),
        dtype=int
    )


    for gene in epithelial_keratins:

        keratin_hits += (

            filter_counts[
                gene
            ]
            >
            0

        ).astype(
            int
        )


    epithelial_exclude = (

        epcam_positive

        |

        (
            keratin_hits
            >=
            2
        )
    )


    # ========================================================
    # 11. MURAL / FIBROBLAST EXCLUSION
    # ========================================================

    mural_hits = (
        identity[
            "mural_fibroblast_marker_hits"
        ]
        .to_numpy(
            dtype=int
        )
    )


    mural_exclude = (
        mural_hits
        >=
        2
    )


    # ========================================================
    # 12. MELANOMA EXCLUSION
    #
    # Mitf alone is intentionally excluded from this rule.
    # ========================================================

    melanoma_specific_hits = np.zeros(
        len(
            identity
        ),
        dtype=int
    )


    for gene in melanoma_specific:

        melanoma_specific_hits += (

            filter_counts[
                gene
            ]
            >
            0

        ).astype(
            int
        )


    melanoma_exclude = (
        melanoma_specific_hits
        >=
        2
    )


    # ========================================================
    # 13. FINAL OPERATIONAL EC CALL
    # ========================================================

    final_ec = (

        ec_ge3

        &

        ~immune_exclude

        &

        ~epithelial_exclude

        &

        ~mural_exclude

        &

        ~melanoma_exclude
    )


    # ========================================================
    # 14. ADD FILTER FLAGS TO TABLE
    # ========================================================

    identity[
        "final_ec_required_ec_ge3"
    ] = ec_ge3


    identity[
        "immune_exclude"
    ] = immune_exclude


    identity[
        "epithelial_specific_hits"
    ] = (

        keratin_hits

        +

        epcam_positive.astype(
            int
        )
    )


    identity[
        "epithelial_exclude"
    ] = epithelial_exclude


    identity[
        "mural_exclude"
    ] = mural_exclude


    identity[
        "melanoma_specific_hits"
    ] = melanoma_specific_hits


    identity[
        "melanoma_exclude"
    ] = melanoma_exclude


    identity[
        "final_endothelial"
    ] = final_ec


    # ========================================================
    # 15. FINAL RAW COLUMN INDICES / BARCODES
    # ========================================================

    final_raw_idx = (
        raw_idx[
            final_ec
        ]
    )


    final_barcodes = (
        identity.loc[
            final_ec,
            "barcode"
        ]
        .astype(str)
        .to_numpy()
    )


    if len(
        final_raw_idx
    ) == 0:

        raise RuntimeError(
            f"{prefix}: final EC filter retained zero cells."
        )


    g253_final_ec_indices[
        prefix
    ] = final_raw_idx.copy()


    g253_final_ec_barcodes[
        prefix
    ] = final_barcodes.copy()


    g253_final_identity_tables[
        prefix
    ] = identity.copy()


    np.save(

        CACHE
        /
        f"{prefix}_final_endothelial_column_indices.npy",

        final_raw_idx
    )


    identity.to_csv(

        CACHE
        /
        f"{prefix}_final_endothelial_identity_table.csv.gz",

        index=False,
        compression="gzip"
    )


    # ========================================================
    # 16. FILTER AUDIT
    #
    # Counts overlap; they are NOT intended to sum.
    # ========================================================

    summary = {

        "sample_prefix":
            prefix,

        "condition":
            condition,

        "qc_barcodes":
            int(
                len(
                    identity
                )
            ),

        "fail_ec_ge3":
            int(
                (
                    ~ec_ge3
                )
                .sum()
            ),

        "immune_exclude":
            int(
                immune_exclude.sum()
            ),

        "epithelial_exclude":
            int(
                epithelial_exclude.sum()
            ),

        "mural_exclude":
            int(
                mural_exclude.sum()
            ),

        "melanoma_exclude":
            int(
                melanoma_exclude.sum()
            ),

        "final_endothelial":
            int(
                final_ec.sum()
            ),

        "final_endothelial_pct":
            float(
                final_ec.mean()
            )
    }


    final_ec_summary_rows.append(
        summary
    )


    print(
        "QC barcodes:",
        summary[
            "qc_barcodes"
        ]
    )

    print(
        "Fail <3 EC markers:",
        summary[
            "fail_ec_ge3"
        ]
    )

    print(
        "Immune exclusion:",
        summary[
            "immune_exclude"
        ]
    )

    print(
        "Specific epithelial exclusion:",
        summary[
            "epithelial_exclude"
        ]
    )

    print(
        "Mural/fibroblast exclusion:",
        summary[
            "mural_exclude"
        ]
    )

    print(
        "Specific melanoma exclusion:",
        summary[
            "melanoma_exclude"
        ]
    )

    print(
        "\nFINAL ENDOTHELIAL:",
        summary[
            "final_endothelial"
        ],
        f"({100*summary['final_endothelial_pct']:.1f}%)"
    )


    # ========================================================
    # 17. FINAL ENDOTHELIAL-SUPPORT AUDIT
    # ========================================================

    final_hits = (
        identity.loc[
            final_ec,
            "endothelial_marker_hits"
        ]
        .to_numpy(
            dtype=int
        )
    )


    print(
        "Median endothelial markers retained:",
        round(
            float(
                np.median(
                    final_hits
                )
            ),
            1
        )
    )


    print(
        "Retained cells with >=5 EC markers:",
        int(
            (
                final_hits
                >=
                5
            )
            .sum()
        ),
        "/",
        len(
            final_hits
        )
    )


    # ========================================================
    # 18. CLEANUP
    # ========================================================

    del M
    del features
    del symbols
    del symbol_rows

    del filter_counts

    del ec_ge3

    del immune_hits
    del ptprc_positive
    del immune_exclude

    del epcam_positive
    del keratin_hits
    del epithelial_exclude

    del mural_hits
    del mural_exclude

    del melanoma_specific_hits
    del melanoma_exclude

    del final_ec
    del final_hits

    del identity

    gc.collect()


# ============================================================
# 19. FINAL SUMMARY
# ============================================================

g253_final_ec_summary = pd.DataFrame(
    final_ec_summary_rows
)


g253_final_ec_summary.to_csv(

    OUT
    /
    "GSE253749_final_endothelial_filter_summary.csv",

    index=False
)


print(
    "\n======================================"
)

print(
    "GSE253749 FINAL ENDOTHELIAL SUMMARY"
)

print(
    "======================================"
)


display(
    g253_final_ec_summary
)


print(
    "\nTotal QC-passing barcodes:",
    int(
        g253_final_ec_summary[
            "qc_barcodes"
        ]
        .sum()
    )
)


print(
    "Total final endothelial cells:",
    int(
        g253_final_ec_summary[
            "final_endothelial"
        ]
        .sum()
    )
)


print(
    "\nPHASE D4 FINAL ENDOTHELIAL FILTER COMPLETE"
)

In [ ]:
# ============================================================
# PHASE D5 — PSN-LINKED RECEPTOR RESPONSE IN FINAL LUNG ECs
#
# IMPORTANT:
#
# Each GSE253749 condition is represented here by ONE pooled
# sample. Therefore:
#
#   NO cell-level p-values
#   NO FDR
#   NO claim of biological replication
#
# This cell provides DESCRIPTIVE evidence only:
#
#   - receptor expression in purified lung ECs
#   - pseudobulk CPM
#   - cell-level prevalence
#   - normalized expression
#   - 6 h vs control directional change
#   - 30 h vs control directional change
#
# Candidate receptor universe is rebuilt DIRECTLY from
# lr_psn_detected rather than inherited from GSE124872.
# ============================================================

import gc
import gzip
import numpy as np
import pandas as pd

from collections import defaultdict
from scipy.io import mmread


# ============================================================
# 1. PSN-LINKED RECEPTOR UNIVERSE
# ============================================================

psn_receptor_universe = sorted(

    set(
        lr_psn_detected[
            "mouse_receptor"
        ]
        .dropna()
        .astype(str)
    )
)


print(
    "PSN-linked receptors from B4:",
    len(
        psn_receptor_universe
    )
)


# ============================================================
# 2. AUDIT FEATURE AVAILABILITY IN ALL THREE GSE253749 SAMPLES
# ============================================================

feature_sets_253 = {}


for prefix in sorted(
    groups253
):

    features = pd.read_csv(

        groups253[
            prefix
        ][
            "features"
        ],

        sep="\t",
        header=None,
        compression="gzip"
    )


    if features.shape[1] >= 2:

        symbols = (
            features.iloc[
                :,
                1
            ]
            .astype(str)
        )

    else:

        symbols = (
            features.iloc[
                :,
                0
            ]
            .astype(str)
        )


    feature_sets_253[
        prefix
    ] = set(
        symbols
    )


    print(
        prefix,
        "| features:",
        len(
            symbols
        ),
        "| PSN receptors present:",
        len(
            set(
                psn_receptor_universe
            )
            &
            feature_sets_253[
                prefix
            ]
        )
    )


# ============================================================
# 3. REQUIRE RECEPTORS TO BE REPRESENTED IN ALL CONDITIONS
# ============================================================

common_features_253 = set.intersection(
    *feature_sets_253.values()
)


g253_candidate_receptors = sorted(

    set(
        psn_receptor_universe
    )

    &

    common_features_253
)


missing_g253_receptors = sorted(

    set(
        psn_receptor_universe
    )

    -

    set(
        g253_candidate_receptors
    )
)


print(
    "\nComparable PSN-linked receptors in all three samples:",
    len(
        g253_candidate_receptors
    )
)


print(
    "PSN-linked receptors absent from GSE253749 feature space:",
    len(
        missing_g253_receptors
    )
)


if missing_g253_receptors:

    print(
        missing_g253_receptors
    )


if len(
    g253_candidate_receptors
) == 0:

    raise RuntimeError(
        "No PSN-linked receptors available in GSE253749."
    )


# ============================================================
# 4. PROCESS ONE CONDITION AT A TIME
# ============================================================

g253_receptor_rows = []


for prefix in sorted(
    groups253
):

    fg = groups253[
        prefix
    ]

    condition = COND_253[
        prefix
    ]


    print(
        "\n======================================"
    )

    print(
        prefix,
        "|",
        condition
    )

    print(
        "======================================"
    )


    # ========================================================
    # 5. RESTORE FINAL ENDOTHELIAL CELLS
    # ========================================================

    if (
        "g253_final_identity_tables" in globals()
        and
        prefix in g253_final_identity_tables
    ):

        identity = (
            g253_final_identity_tables[
                prefix
            ]
            .copy()
        )

    else:

        identity = pd.read_csv(

            CACHE
            /
            f"{prefix}_final_endothelial_identity_table.csv.gz",

            compression="gzip"
        )


    final_identity = (
        identity[
            identity[
                "final_endothelial"
            ]
            .astype(bool)
        ]
        .copy()
    )


    final_raw_idx = (
        final_identity[
            "raw_column_index"
        ]
        .to_numpy(
            dtype=int
        )
    )


    cell_total_umis = (
        final_identity[
            "total_counts"
        ]
        .to_numpy(
            dtype=float
        )
    )


    if len(
        final_raw_idx
    ) == 0:

        raise RuntimeError(
            f"{prefix}: no final endothelial cells."
        )


    if np.any(
        cell_total_umis <= 0
    ):

        raise RuntimeError(
            f"{prefix}: final EC with zero UMI total."
        )


    print(
        "Final endothelial cells:",
        len(
            final_raw_idx
        )
    )


    # ========================================================
    # 6. FEATURE LOOKUP
    # ========================================================

    features = pd.read_csv(

        fg[
            "features"
        ],

        sep="\t",
        header=None,
        compression="gzip"
    )


    if features.shape[1] >= 2:

        feature_symbols = (
            features.iloc[
                :,
                1
            ]
            .astype(str)
            .tolist()
        )

    else:

        feature_symbols = (
            features.iloc[
                :,
                0
            ]
            .astype(str)
            .tolist()
        )


    symbol_rows = defaultdict(
        list
    )


    for row_i, symbol in enumerate(
        feature_symbols
    ):

        if symbol in g253_candidate_receptors:

            symbol_rows[
                symbol
            ].append(
                row_i
            )


    # Every common receptor should have >=1 row.
    missing_here = [

        gene
        for gene in g253_candidate_receptors
        if gene not in symbol_rows

    ]


    if missing_here:

        raise RuntimeError(

            f"{prefix}: receptors unexpectedly missing: "
            f"{missing_here[:20]}"
        )


    # ========================================================
    # 7. RAW MATRIX
    # ========================================================

    with gzip.open(
        fg[
            "matrix"
        ],
        "rb"
    ) as fh:

        M = (
            mmread(
                fh
            )
            .tocsr()
        )


    if (
        M.shape[0]
        !=
        len(
            feature_symbols
        )
        and
        M.shape[1]
        ==
        len(
            feature_symbols
        )
    ):

        M = (
            M.T
            .tocsr()
        )


    if M.shape[0] != len(
        feature_symbols
    ):

        raise RuntimeError(
            f"{prefix}: feature/matrix mismatch."
        )


    # ========================================================
    # 8. EXTRACT ALL RECEPTOR ROWS IN ONE SMALL MATRIX
    #
    # This avoids repeatedly slicing ~1.8-1.9 million raw
    # barcode columns receptor-by-receptor.
    # ========================================================

    selected_rows = []

    selected_row_gene = []


    for gene in g253_candidate_receptors:

        for row_i in symbol_rows[
            gene
        ]:

            selected_rows.append(
                row_i
            )

            selected_row_gene.append(
                gene
            )


    R = (

        M[
            selected_rows,
            :
        ][
            :,
            final_raw_idx
        ]
        .tocsr()
    )


    selected_gene_positions = defaultdict(
        list
    )


    for local_i, gene in enumerate(
        selected_row_gene
    ):

        selected_gene_positions[
            gene
        ].append(
            local_i
        )


    # ========================================================
    # 9. SAMPLE-WIDE ENDOTHELIAL LIBRARY SIZE
    # ========================================================

    total_endothelial_umis = float(
        cell_total_umis.sum()
    )


    print(
        "Final endothelial UMIs:",
        int(
            total_endothelial_umis
        )
    )


    # ========================================================
    # 10. RECEPTOR METRICS
    # ========================================================

    for gene in g253_candidate_receptors:

        local_rows = (
            selected_gene_positions[
                gene
            ]
        )


        x = np.asarray(

            R[
                local_rows,
                :
            ]
            .sum(
                axis=0
            )

        ).ravel().astype(
            float
        )


        n_expressing = int(
            (
                x > 0
            )
            .sum()
        )


        pct_expressing = float(
            n_expressing
            /
            len(
                x
            )
        )


        # ----------------------------------------------------
        # Per-cell normalization: log1p(count * 1e4 / library)
        # ----------------------------------------------------

        normalized_1e4 = (

            x
            *
            1e4
            /
            cell_total_umis
        )


        log1p_norm = np.log1p(
            normalized_1e4
        )


        # ----------------------------------------------------
        # Pseudobulk CPM over the final endothelial compartment
        # ----------------------------------------------------

        raw_sum = float(
            x.sum()
        )


        pseudobulk_cpm = float(

            1e6
            *
            raw_sum
            /
            total_endothelial_umis

        )


        pseudobulk_log2cpm1 = float(
            np.log2(
                pseudobulk_cpm
                +
                1.0
            )
        )


        g253_receptor_rows.append({

            "sample_prefix":
                prefix,

            "condition":
                condition,

            "gene":
                gene,

            "n_endothelial_cells":
                int(
                    len(
                        x
                    )
                ),

            "total_endothelial_umis":
                total_endothelial_umis,

            "raw_sum":
                raw_sum,

            "n_expressing":
                n_expressing,

            "pct_expressing":
                pct_expressing,

            "mean_raw_count":
                float(
                    x.mean()
                ),

            "mean_log1p_norm":
                float(
                    log1p_norm.mean()
                ),

            "median_log1p_norm":
                float(
                    np.median(
                        log1p_norm
                    )
                ),

            "pseudobulk_cpm":
                pseudobulk_cpm,

            "pseudobulk_log2cpm1":
                pseudobulk_log2cpm1
        })


    # ========================================================
    # 11. CLEANUP
    # ========================================================

    del M
    del R

    del identity
    del final_identity

    del final_raw_idx
    del cell_total_umis

    del features
    del feature_symbols
    del symbol_rows

    del selected_rows
    del selected_row_gene
    del selected_gene_positions

    gc.collect()


# ============================================================
# 12. RECEPTOR EXPRESSION SUMMARY
# ============================================================

g253_receptor_summary = pd.DataFrame(
    g253_receptor_rows
)


# Backward-compatible object name, now corrected.
g253_summary = (
    g253_receptor_summary.copy()
)


g253_receptor_summary.to_csv(

    OUT
    /
    "GSE253749_finalEC_receptor_expression_by_condition.csv",

    index=False
)


print(
    "\n======================================"
)

print(
    "FINAL EC RECEPTOR EXPRESSION SUMMARY"
)

print(
    "======================================"
)


sample_summary = (

    g253_receptor_summary[
        [
            "sample_prefix",
            "condition",
            "n_endothelial_cells",
            "total_endothelial_umis"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "condition"
    )
)


display(
    sample_summary
)


# ============================================================
# 13. BUILD DESCRIPTIVE CONDITION-EFFECT TABLE
# ============================================================

metrics_to_pivot = [

    "pseudobulk_cpm",
    "pseudobulk_log2cpm1",
    "pct_expressing",
    "mean_log1p_norm",
    "raw_sum"
]


wide_parts = []


for metric in metrics_to_pivot:

    w = (
        g253_receptor_summary
        .pivot(
            index="gene",
            columns="condition",
            values=metric
        )
        .copy()
    )


    w.columns = [

        f"{condition}_{metric}"

        for condition in w.columns
    ]


    wide_parts.append(
        w
    )


g253_effect = pd.concat(
    wide_parts,
    axis=1
).reset_index()


# ============================================================
# 14. DESCRIPTIVE 6 h AND 30 h CHANGES
# ============================================================

for condition in [
    "6h",
    "30h"
]:

    # --------------------------------------------------------
    # Difference in log2(CPM + 1)
    # Stable even for low-expression receptors.
    # --------------------------------------------------------

    g253_effect[
        f"{condition}_delta_log2cpm1"
    ] = (

        g253_effect[
            f"{condition}_pseudobulk_log2cpm1"
        ]

        -

        g253_effect[
            "control_pseudobulk_log2cpm1"
        ]
    )


    # --------------------------------------------------------
    # Change in fraction of ECs expressing receptor
    # --------------------------------------------------------

    g253_effect[
        f"{condition}_delta_pct_expressing"
    ] = (

        g253_effect[
            f"{condition}_pct_expressing"
        ]

        -

        g253_effect[
            "control_pct_expressing"
        ]
    )


    # --------------------------------------------------------
    # Difference in average cell-normalized expression
    # --------------------------------------------------------

    g253_effect[
        f"{condition}_delta_mean_log1p_norm"
    ] = (

        g253_effect[
            f"{condition}_mean_log1p_norm"
        ]

        -

        g253_effect[
            "control_mean_log1p_norm"
        ]
    )


# ============================================================
# 15. EVIDENCE / EXPRESSION FLOOR ANNOTATIONS
#
# These are annotations only.
# We do NOT filter or rank yet.
# ============================================================

g253_effect[
    "max_pct_expressing"
] = (

    g253_effect[
        [
            "control_pct_expressing",
            "6h_pct_expressing",
            "30h_pct_expressing"
        ]
    ]
    .max(
        axis=1
    )
)


g253_effect[
    "min_pct_expressing"
] = (

    g253_effect[
        [
            "control_pct_expressing",
            "6h_pct_expressing",
            "30h_pct_expressing"
        ]
    ]
    .min(
        axis=1
    )
)


g253_effect[
    "max_pseudobulk_cpm"
] = (

    g253_effect[
        [
            "control_pseudobulk_cpm",
            "6h_pseudobulk_cpm",
            "30h_pseudobulk_cpm"
        ]
    ]
    .max(
        axis=1
    )
)


g253_effect[
    "detected_ge_1pct_any_condition"
] = (

    g253_effect[
        "max_pct_expressing"
    ]
    >=
    0.01
)


g253_effect[
    "detected_ge_5pct_any_condition"
] = (

    g253_effect[
        "max_pct_expressing"
    ]
    >=
    0.05
)


g253_effect[
    "max_abs_metastatic_delta"
] = (

    g253_effect[
        [
            "6h_delta_log2cpm1",
            "30h_delta_log2cpm1"
        ]
    ]
    .abs()
    .max(
        axis=1
    )
)


# ============================================================
# 16. SAVE DESCRIPTIVE EFFECT TABLE
# ============================================================

g253_effect.to_csv(

    OUT
    /
    "GSE253749_finalEC_receptor_descriptive_metastatic_effects.csv",

    index=False
)


# ============================================================
# 17. TOP DESCRIPTIVE EFFECTS
#
# Do NOT interpret these as statistically significant hits.
# ============================================================

print(
    "\n======================================"
)

print(
    "TOP DESCRIPTIVE METASTATIC RECEPTOR EFFECTS"
)

print(
    "======================================"
)


display(

    g253_effect[
        g253_effect[
            "detected_ge_1pct_any_condition"
        ]
    ]
    .sort_values(
        "max_abs_metastatic_delta",
        ascending=False
    )
    [
        [
            "gene",

            "control_pseudobulk_cpm",
            "6h_pseudobulk_cpm",
            "30h_pseudobulk_cpm",

            "control_pct_expressing",
            "6h_pct_expressing",
            "30h_pct_expressing",

            "6h_delta_log2cpm1",
            "30h_delta_log2cpm1",

            "6h_delta_pct_expressing",
            "30h_delta_pct_expressing",

            "max_abs_metastatic_delta"
        ]
    ]
    .head(
        40
    )
)


# ============================================================
# 18. SELECTED NEUROVASCULAR RECEPTOR CHECKS
# ============================================================

selected_receptors_253 = [

    "Calcrl",
    "Ramp1",
    "Ramp2",
    "Ramp3",

    "Ntrk2",

    "Kdr",
    "Nrp1",

    "Pdgfrb",

    "Vipr1",
    "Vipr2",

    "Galr1",
    "Galr2",

    "Sstr1",
    "Sstr2",

    "Tacr2"
]


print(
    "\n======================================"
)

print(
    "SELECTED NEUROVASCULAR RECEPTOR CHECKS"
)

print(
    "======================================"
)


selected_253 = (

    g253_effect[
        g253_effect[
            "gene"
        ]
        .isin(
            selected_receptors_253
        )
    ]
    [
        [
            "gene",

            "control_pseudobulk_cpm",
            "6h_pseudobulk_cpm",
            "30h_pseudobulk_cpm",

            "control_pct_expressing",
            "6h_pct_expressing",
            "30h_pct_expressing",

            "6h_delta_log2cpm1",
            "30h_delta_log2cpm1",

            "6h_delta_pct_expressing",
            "30h_delta_pct_expressing",

            "detected_ge_1pct_any_condition",
            "detected_ge_5pct_any_condition"
        ]
    ]
    .sort_values(
        "gene"
    )
)


display(
    selected_253
)


# ============================================================
# 19. SANITY CHECKS
# ============================================================

assert (
    g253_receptor_summary[
        "condition"
    ]
    .nunique()
    ==
    3
)


assert (
    g253_effect[
        "gene"
    ]
    .nunique()
    ==
    len(
        g253_candidate_receptors
    )
)


assert not g253_effect[
    "gene"
].duplicated().any()


# ============================================================
# 20. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE D5 FINAL-EC RECEPTOR RESPONSE COMPLETE"
)


print(
    "PSN receptors from B4:",
    len(
        psn_receptor_universe
    )
)


print(
    "Comparable receptors in GSE253749:",
    len(
        g253_candidate_receptors
    )
)


print(
    "Receptors >=1% ECs in any condition:",
    int(
        g253_effect[
            "detected_ge_1pct_any_condition"
        ]
        .sum()
    )
)


print(
    "Receptors >=5% ECs in any condition:",
    int(
        g253_effect[
            "detected_ge_5pct_any_condition"
        ]
        .sum()
    )
)


print(
    "\nIMPORTANT:"
)

print(
    "GSE253749 effects are DESCRIPTIVE because there is "
    "one pooled sample per condition in this analysis."
)

In [ ]:
# ============================================================
# PHASE D6A — DOWNLOAD / EXTRACT INDEPENDENT 6 h REPLICATION
#
# Three independent early-metastatic EC experiments:
#
#   set1 = GSE235394
#   set2 = GSE253749   <-- already processed
#   set3 = GSE253751
#
# SELF-CONTAINED:
# Does not depend on geo_suppl_url() or download_file()
# from earlier notebook cells.
#
# This cell ONLY downloads and extracts set1 + set3.
# No QC or receptor analysis yet.
# ============================================================

import re
import tarfile
from pathlib import Path

import requests


# ============================================================
# 1. SELF-CONTAINED GEO HELPERS
# ============================================================

def geo_bucket_local(accession):

    m = re.fullmatch(
        r"(GSE)(\d+)",
        accession
    )

    if not m:

        raise ValueError(
            f"Invalid GEO accession: {accession}"
        )


    digits = m.group(2)


    return (
        "GSE"
        +
        digits[:-3]
        +
        "nnn"
    )


def geo_suppl_url_local(
    accession,
    filename
):

    return (

        "https://ftp.ncbi.nlm.nih.gov/geo/series/"
        f"{geo_bucket_local(accession)}/"
        f"{accession}/suppl/"
        f"{filename}"

    )


def download_file_local(
    url,
    destination
):

    destination = Path(
        destination
    )


    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    temporary = destination.with_suffix(
        destination.suffix
        +
        ".part"
    )


    print(
        "Downloading:",
        url
    )


    with requests.get(
        url,
        stream=True,
        timeout=120
    ) as response:

        response.raise_for_status()


        total_bytes = int(
            response.headers.get(
                "content-length",
                0
            )
        )


        downloaded = 0


        with open(
            temporary,
            "wb"
        ) as fh:

            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):

                if not chunk:
                    continue


                fh.write(
                    chunk
                )


                downloaded += len(
                    chunk
                )


                if (
                    total_bytes > 0
                    and
                    (
                        downloaded
                        //
                        (
                            100
                            *
                            1024
                            *
                            1024
                        )
                    )
                    !=
                    (
                        (
                            downloaded
                            -
                            len(
                                chunk
                            )
                        )
                        //
                        (
                            100
                            *
                            1024
                            *
                            1024
                        )
                    )
                ):

                    print(
                        "Downloaded MB:",
                        round(
                            downloaded
                            /
                            1e6,
                            1
                        ),
                        "/",
                        round(
                            total_bytes
                            /
                            1e6,
                            1
                        )
                    )


    temporary.replace(
        destination
    )


    if not destination.exists():

        raise RuntimeError(
            f"Download failed: {destination}"
        )


    return destination


# ============================================================
# 2. REPLICATION DATASETS
# ============================================================

replication_archives = {

    "GSE235394":
        "GSE235394_RAW.tar",

    "GSE253751":
        "GSE253751_RAW.tar"
}


replication_extract_dirs = {}


# ============================================================
# 3. DOWNLOAD + EXTRACT
# ============================================================

for accession, filename in replication_archives.items():

    print(
        "\n======================================"
    )

    print(
        accession
    )

    print(
        "======================================"
    )


    # --------------------------------------------------------
    # Raw destination
    # --------------------------------------------------------

    raw_dir = (
        RAW
        /
        accession
    )


    raw_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    archive_path = (
        raw_dir
        /
        filename
    )


    # --------------------------------------------------------
    # Download only if absent
    # --------------------------------------------------------

    if not archive_path.exists():

        url = geo_suppl_url_local(
            accession,
            filename
        )


        download_file_local(
            url,
            archive_path
        )

    else:

        print(
            "Archive already present:",
            archive_path
        )


    if not archive_path.exists():

        raise RuntimeError(
            f"{accession}: archive download failed."
        )


    print(
        "Archive size MB:",
        round(
            archive_path.stat().st_size
            /
            1e6,
            1
        )
    )


    # --------------------------------------------------------
    # Verify tar archive before extraction
    # --------------------------------------------------------

    if not tarfile.is_tarfile(
        archive_path
    ):

        raise RuntimeError(
            f"{accession}: downloaded file is not a valid tar archive."
        )


    # --------------------------------------------------------
    # Extract
    # --------------------------------------------------------

    extract_dir = (
        EXTRACTED
        /
        accession
    )


    extract_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    replication_extract_dirs[
        accession
    ] = extract_dir


    marker = (
        extract_dir
        /
        ".extracted_ok"
    )


    if not marker.exists():

        print(
            "Extracting..."
        )


        with tarfile.open(
            archive_path,
            "r"
        ) as tf:

            tf.extractall(
                extract_dir
            )


        marker.write_text(
            "ok"
        )

    else:

        print(
            "Already extracted."
        )


    # --------------------------------------------------------
    # Audit extracted files
    # --------------------------------------------------------

    extracted_files = sorted(

        [
            p.name

            for p in extract_dir.rglob(
                "*"
            )

            if (
                p.is_file()
                and
                p.name
                !=
                ".extracted_ok"
            )
        ]
    )


    print(
        "Extracted files:",
        len(
            extracted_files
        )
    )


    for name in extracted_files:

        print(
            name
        )


    if len(
        extracted_files
    ) != 6:

        print(
            "WARNING:",
            accession,
            "has",
            len(
                extracted_files
            ),
            "files instead of the expected 6."
        )


# ============================================================
# 4. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE D6A REPLICATION ARCHIVES READY"
)

In [ ]:
# ============================================================
# PHASE D6B — QC OF INDEPENDENT 6 h REPLICATION SETS
#
# set1 = GSE235394
#   GSM7501730_063 = control
#   GSM7501731_064 = B16F10 6 h
#
# set3 = GSE253751
#   GSM8027478_058 = control
#   GSM8027479_059 = B16F10 6 h
#
# Apply SAME documented QC used for GSE253749/set2:
#
#   1000 <= detected genes <= 5000
#   mitochondrial UMIs < 5%
#   ribosomal UMIs < 20%
#
# Raw 10x barcode universes are NOT treated as cells.
# ============================================================

import gc
import gzip
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.io import mmread


# ============================================================
# 1. QC THRESHOLDS — LOCK TO SET2
# ============================================================

MIN_GENES_REP = 1000
MAX_GENES_REP = 5000
MAX_MT_PCT_REP = 5.0
MAX_RIBO_PCT_REP = 20.0


# ============================================================
# 2. SELF-CONTAINED 10x FILE GROUPER
# ============================================================

def group_replication_10x_files(folder):

    folder = Path(folder)

    groups = {}


    for mtx in folder.rglob(
        "*matrix.mtx.gz"
    ):

        prefix = mtx.name[
            :-
            len(
                "matrix.mtx.gz"
            )
        ]


        features = mtx.with_name(
            prefix
            +
            "features.tsv.gz"
        )


        barcodes = mtx.with_name(
            prefix
            +
            "barcodes.tsv.gz"
        )


        if (
            features.exists()
            and
            barcodes.exists()
        ):

            groups[
                prefix
            ] = {

                "matrix":
                    mtx,

                "features":
                    features,

                "barcodes":
                    barcodes
            }


    return groups


# ============================================================
# 3. GROUP BOTH INDEPENDENT DATASETS
# ============================================================

groups235394 = group_replication_10x_files(

    EXTRACTED
    /
    "GSE235394"
)


groups253751 = group_replication_10x_files(

    EXTRACTED
    /
    "GSE253751"
)


print(
    "GSE235394 groups:",
    sorted(
        groups235394
    )
)


print(
    "GSE253751 groups:",
    sorted(
        groups253751
    )
)


if len(
    groups235394
) != 2:

    raise RuntimeError(

        "Expected exactly 2 10x samples "
        f"for GSE235394, found {len(groups235394)}."
    )


if len(
    groups253751
) != 2:

    raise RuntimeError(

        "Expected exactly 2 10x samples "
        f"for GSE253751, found {len(groups253751)}."
    )


# ============================================================
# 4. VERIFIED CONDITION MAP
# ============================================================

COND_REPLICATION = {

    # set1
    "GSM7501730_063":
        "control",

    "GSM7501731_064":
        "6h",

    # set3
    "GSM8027478_058":
        "control",

    "GSM8027479_059":
        "6h"
}


DATASET_REPLICATION = {

    "GSM7501730_063":
        "set1_GSE235394",

    "GSM7501731_064":
        "set1_GSE235394",

    "GSM8027478_058":
        "set3_GSE253751",

    "GSM8027479_059":
        "set3_GSE253751"
}


# ============================================================
# 5. COMBINE GROUP DICTIONARIES
# ============================================================

groups_replication = {}


groups_replication.update(
    groups235394
)


groups_replication.update(
    groups253751
)


expected_prefixes = set(
    COND_REPLICATION
)


observed_prefixes = set(
    groups_replication
)


if observed_prefixes != expected_prefixes:

    raise RuntimeError(

        "Replication sample mismatch.\n"
        f"Expected: {sorted(expected_prefixes)}\n"
        f"Observed: {sorted(observed_prefixes)}"
    )


print(
    "\nVerified replication condition map:"
)


for prefix in sorted(
    groups_replication
):

    print(

        DATASET_REPLICATION[
            prefix
        ],

        "|",
        prefix,

        "|",
        COND_REPLICATION[
            prefix
        ]
    )


# ============================================================
# 6. OUTPUT OBJECTS
# ============================================================

rep_qc_rows = []

rep_qc_pass_indices = {}

rep_qc_pass_barcodes = {}


# ============================================================
# 7. PROCESS EACH RAW MATRIX
# ============================================================

for prefix in sorted(
    groups_replication
):

    fg = groups_replication[
        prefix
    ]


    condition = COND_REPLICATION[
        prefix
    ]


    dataset = DATASET_REPLICATION[
        prefix
    ]


    print(
        "\n======================================"
    )

    print(
        dataset,
        "|",
        prefix,
        "|",
        condition
    )

    print(
        "======================================"
    )


    # --------------------------------------------------------
    # Feature symbols
    # --------------------------------------------------------

    features = pd.read_csv(

        fg[
            "features"
        ],

        sep="\t",
        header=None,
        compression="gzip"
    )


    if features.shape[1] >= 2:

        feature_symbols = (

            features.iloc[
                :,
                1
            ]
            .astype(str)

        )

    else:

        feature_symbols = (

            features.iloc[
                :,
                0
            ]
            .astype(str)

        )


    # --------------------------------------------------------
    # Barcodes
    # --------------------------------------------------------

    barcodes = (

        pd.read_csv(

            fg[
                "barcodes"
            ],

            sep="\t",
            header=None,
            compression="gzip"

        )
        .iloc[
            :,
            0
        ]
        .astype(str)
        .to_numpy()

    )


    # --------------------------------------------------------
    # Sparse raw matrix
    # --------------------------------------------------------

    with gzip.open(

        fg[
            "matrix"
        ],

        "rb"

    ) as fh:

        M = (

            mmread(
                fh
            )
            .tocsr()

        )


    if (

        M.shape[0]
        !=
        len(
            feature_symbols
        )

        and

        M.shape[1]
        ==
        len(
            feature_symbols
        )

    ):

        M = (

            M.T
            .tocsr()

        )


    if M.shape[0] != len(
        feature_symbols
    ):

        raise RuntimeError(

            f"{prefix}: feature mismatch: "
            f"{M.shape} vs {len(feature_symbols)}"
        )


    if M.shape[1] != len(
        barcodes
    ):

        raise RuntimeError(

            f"{prefix}: barcode mismatch: "
            f"{M.shape} vs {len(barcodes)}"
        )


    print(
        "Raw matrix:",
        M.shape
    )


    print(
        "Raw barcode universe:",
        len(
            barcodes
        )
    )


    # ========================================================
    # 8. BASIC PER-BARCODE QC
    # ========================================================

    total_counts = np.asarray(

        M.sum(
            axis=0
        )

    ).ravel()


    n_genes = np.asarray(

        M.getnnz(
            axis=0
        )

    ).ravel()


    symbol_lower = (

        feature_symbols
        .str.lower()

    )


    mt_mask = (

        symbol_lower
        .str.startswith(
            "mt-"
        )
        .to_numpy()

    )


    ribo_mask = (

        symbol_lower
        .str.startswith(
            (
                "rpl",
                "rps"
            )
        )
        .to_numpy()

    )


    print(
        "Mitochondrial features:",
        int(
            mt_mask.sum()
        )
    )


    print(
        "Ribosomal features:",
        int(
            ribo_mask.sum()
        )
    )


    if mt_mask.any():

        mt_counts = np.asarray(

            M[
                mt_mask,
                :
            ]
            .sum(
                axis=0
            )

        ).ravel()

    else:

        mt_counts = np.zeros_like(
            total_counts,
            dtype=float
        )


    if ribo_mask.any():

        ribo_counts = np.asarray(

            M[
                ribo_mask,
                :
            ]
            .sum(
                axis=0
            )

        ).ravel()

    else:

        ribo_counts = np.zeros_like(
            total_counts,
            dtype=float
        )


    pct_mt = np.divide(

        100.0
        *
        mt_counts,

        total_counts,

        out=np.zeros(
            len(
                total_counts
            ),
            dtype=float
        ),

        where=(
            total_counts
            >
            0
        )
    )


    pct_ribo = np.divide(

        100.0
        *
        ribo_counts,

        total_counts,

        out=np.zeros(
            len(
                total_counts
            ),
            dtype=float
        ),

        where=(
            total_counts
            >
            0
        )
    )


    # ========================================================
    # 9. SEQUENTIAL QC
    # ========================================================

    nonzero = (
        total_counts
        >
        0
    )


    ge100 = (
        n_genes
        >=
        100
    )


    ge500 = (
        n_genes
        >=
        500
    )


    ge1000 = (
        n_genes
        >=
        MIN_GENES_REP
    )


    gene_range = (

        (
            n_genes
            >=
            MIN_GENES_REP
        )

        &

        (
            n_genes
            <=
            MAX_GENES_REP
        )
    )


    gene_mt = (

        gene_range

        &

        (
            pct_mt
            <
            MAX_MT_PCT_REP
        )
    )


    full_pass = (

        gene_range

        &

        (
            pct_mt
            <
            MAX_MT_PCT_REP
        )

        &

        (
            pct_ribo
            <
            MAX_RIBO_PCT_REP
        )
    )


    pass_idx = np.flatnonzero(
        full_pass
    )


    pass_barcodes = (
        barcodes[
            pass_idx
        ]
    )


    if len(
        pass_idx
    ) == 0:

        raise RuntimeError(

            f"{prefix}: zero QC-passing barcodes."
        )


    # ========================================================
    # 10. SAVE INDICES AND QC METADATA
    # ========================================================

    rep_qc_pass_indices[
        prefix
    ] = pass_idx.copy()


    rep_qc_pass_barcodes[
        prefix
    ] = pass_barcodes.copy()


    np.save(

        CACHE
        /
        f"{prefix}_replication_QC_pass_column_indices.npy",

        pass_idx
    )


    passed_qc = pd.DataFrame({

        "dataset":
            dataset,

        "sample_prefix":
            prefix,

        "condition":
            condition,

        "raw_column_index":
            pass_idx,

        "barcode":
            pass_barcodes,

        "total_counts":
            total_counts[
                pass_idx
            ],

        "n_genes":
            n_genes[
                pass_idx
            ],

        "pct_counts_mt":
            pct_mt[
                pass_idx
            ],

        "pct_counts_ribo":
            pct_ribo[
                pass_idx
            ]
    })


    passed_qc.to_csv(

        CACHE
        /
        f"{prefix}_replication_QC_pass_barcodes.csv.gz",

        index=False,
        compression="gzip"
    )


    # ========================================================
    # 11. SAMPLE AUDIT
    # ========================================================

    row = {

        "dataset":
            dataset,

        "sample_prefix":
            prefix,

        "condition":
            condition,

        "raw_barcodes":
            int(
                len(
                    barcodes
                )
            ),

        "nonzero_barcodes":
            int(
                nonzero.sum()
            ),

        "barcodes_ge_100_genes":
            int(
                ge100.sum()
            ),

        "barcodes_ge_500_genes":
            int(
                ge500.sum()
            ),

        "barcodes_ge_1000_genes":
            int(
                ge1000.sum()
            ),

        "pass_1000_to_5000_genes":
            int(
                gene_range.sum()
            ),

        "pass_gene_range_and_mt":
            int(
                gene_mt.sum()
            ),

        "pass_full_qc":
            int(
                full_pass.sum()
            ),

        "median_genes_pass":
            float(
                np.median(
                    n_genes[
                        pass_idx
                    ]
                )
            ),

        "median_counts_pass":
            float(
                np.median(
                    total_counts[
                        pass_idx
                    ]
                )
            ),

        "median_mt_pct_pass":
            float(
                np.median(
                    pct_mt[
                        pass_idx
                    ]
                )
            ),

        "median_ribo_pct_pass":
            float(
                np.median(
                    pct_ribo[
                        pass_idx
                    ]
                )
            )
    }


    rep_qc_rows.append(
        row
    )


    print(
        "\nQC progression:"
    )

    print(
        "Nonzero:",
        row[
            "nonzero_barcodes"
        ]
    )

    print(
        ">=100 genes:",
        row[
            "barcodes_ge_100_genes"
        ]
    )

    print(
        ">=500 genes:",
        row[
            "barcodes_ge_500_genes"
        ]
    )

    print(
        ">=1000 genes:",
        row[
            "barcodes_ge_1000_genes"
        ]
    )

    print(
        "1000-5000 genes:",
        row[
            "pass_1000_to_5000_genes"
        ]
    )

    print(
        "+ mt <5%:",
        row[
            "pass_gene_range_and_mt"
        ]
    )

    print(
        "+ ribo <20%:",
        row[
            "pass_full_qc"
        ]
    )


    print(
        "\nPassed medians:"
    )

    print(
        "Genes:",
        round(
            row[
                "median_genes_pass"
            ],
            1
        )
    )

    print(
        "UMIs:",
        round(
            row[
                "median_counts_pass"
            ],
            1
        )
    )

    print(
        "MT %:",
        round(
            row[
                "median_mt_pct_pass"
            ],
            3
        )
    )

    print(
        "Ribo %:",
        round(
            row[
                "median_ribo_pct_pass"
            ],
            3
        )
    )


    # ========================================================
    # 12. CLEANUP
    # ========================================================

    del M
    del features
    del feature_symbols
    del symbol_lower
    del barcodes

    del total_counts
    del n_genes

    del mt_counts
    del ribo_counts

    del pct_mt
    del pct_ribo

    del mt_mask
    del ribo_mask

    del nonzero
    del ge100
    del ge500
    del ge1000
    del gene_range
    del gene_mt
    del full_pass

    del passed_qc

    gc.collect()


# ============================================================
# 13. FINAL SUMMARY
# ============================================================

rep_qc_summary = pd.DataFrame(
    rep_qc_rows
)


rep_qc_summary.to_csv(

    OUT
    /
    "GSE235394_GSE253751_replication_raw_QC_audit.csv",

    index=False
)


print(
    "\n======================================"
)

print(
    "REPLICATION QC SUMMARY"
)

print(
    "======================================"
)


display(
    rep_qc_summary
)


print(
    "\nTotal raw barcodes:",
    int(
        rep_qc_summary[
            "raw_barcodes"
        ]
        .sum()
    )
)


print(
    "Total QC-passing barcodes:",
    int(
        rep_qc_summary[
            "pass_full_qc"
        ]
        .sum()
    )
)


print(
    "\nPHASE D6B REPLICATION RAW-QC COMPLETE"
)

In [ ]:
# ============================================================
# PHASE D6C — REPLICATION ENDOTHELIAL IDENTITY AUDIT
#
# Applies the lessons from GSE253749/set2, with one correction:
#
#   CORRECT immune gene = Fcer1g
#   NOT Fcerg1
#
# This cell audits whether QC-passing barcodes in set1/set3
# are strongly endothelial and estimates how many would pass
# the same conservative lineage-exclusion rules.
#
# NO receptor-response analysis yet.
# ============================================================

import gc
import gzip
import numpy as np
import pandas as pd

from collections import defaultdict
from scipy.io import mmread


# ============================================================
# 1. MARKER PANELS
# ============================================================

EC_MARKERS = [
    "Cdh5",
    "Pecam1",
    "Ptprb",
    "Esam",
    "Eng",
    "Kdr",
    "Tek",
    "Emcn",
    "Ramp2"
]


IMMUNE_MARKERS = [
    "Ptprc",
    "Lst1",
    "Tyrobp",
    "Fcer1g",   # corrected spelling
    "Csf1r",
    "Lyz2",
    "Cd3d",
    "Cd79a",
    "Ms4a1",
    "Nkg7"
]


# Stronger epithelial-specific evidence used for exclusion.
EPITHELIAL_KERATINS = [
    "Krt8",
    "Krt18",
    "Krt19"
]


EPITHELIAL_SPECIFIC = [
    "Epcam",
    "Krt8",
    "Krt18",
    "Krt19"
]


# Lung epithelial/ambient-prone markers:
# report them, but DO NOT use them for exclusion.
LUNG_AMBIENT_MARKERS = [
    "Hopx",
    "Scgb1a1",
    "Sftpc",
    "Ager"
]


MURAL_MARKERS = [
    "Pdgfrb",
    "Rgs5",
    "Cspg4",
    "Acta2",
    "Tagln",
    "Pdgfra",
    "Col1a1",
    "Col1a2",
    "Dcn",
    "Lum"
]


# Mitf alone is not sufficient for melanoma exclusion.
MELANOMA_SPECIFIC = [
    "Pmel",
    "Mlana",
    "Tyr",
    "Dct",
    "Sox10"
]


MELANOMA_REPORT_ONLY = [
    "Mitf"
]


ALL_AUDIT_MARKERS = sorted(
    set(
        EC_MARKERS
        +
        IMMUNE_MARKERS
        +
        EPITHELIAL_SPECIFIC
        +
        LUNG_AMBIENT_MARKERS
        +
        MURAL_MARKERS
        +
        MELANOMA_SPECIFIC
        +
        MELANOMA_REPORT_ONLY
    )
)


print(
    "Replication identity markers:",
    len(ALL_AUDIT_MARKERS)
)


# ============================================================
# 2. OUTPUT OBJECTS
# ============================================================

rep_identity_tables = {}

rep_identity_summary_rows = []

rep_marker_rows = []


# ============================================================
# 3. PROCESS EACH REPLICATION SAMPLE
# ============================================================

for prefix in sorted(
    groups_replication
):

    fg = groups_replication[
        prefix
    ]

    dataset = DATASET_REPLICATION[
        prefix
    ]

    condition = COND_REPLICATION[
        prefix
    ]


    print(
        "\n======================================"
    )

    print(
        dataset,
        "|",
        prefix,
        "|",
        condition
    )

    print(
        "======================================"
    )


    # ========================================================
    # 4. RESTORE QC-PASSING BARCODE METADATA
    # ========================================================

    qc_path = (
        CACHE
        /
        f"{prefix}_replication_QC_pass_barcodes.csv.gz"
    )


    qc = pd.read_csv(
        qc_path,
        compression="gzip"
    )


    pass_idx = (
        qc[
            "raw_column_index"
        ]
        .to_numpy(
            dtype=int
        )
    )


    if len(qc) == 0:

        raise RuntimeError(
            f"{prefix}: zero QC-passing barcodes."
        )


    print(
        "QC-passing barcodes:",
        len(qc)
    )


    # ========================================================
    # 5. FEATURE LOOKUP
    # ========================================================

    features = pd.read_csv(

        fg[
            "features"
        ],

        sep="\t",
        header=None,
        compression="gzip"
    )


    if features.shape[1] >= 2:

        symbols = (
            features.iloc[
                :,
                1
            ]
            .astype(str)
            .tolist()
        )

    else:

        symbols = (
            features.iloc[
                :,
                0
            ]
            .astype(str)
            .tolist()
        )


    symbol_rows = defaultdict(
        list
    )


    for i, symbol in enumerate(
        symbols
    ):

        symbol_rows[
            symbol.lower()
        ].append(
            i
        )


    # ========================================================
    # 6. LOAD RAW MATRIX
    # ========================================================

    with gzip.open(
        fg[
            "matrix"
        ],
        "rb"
    ) as fh:

        M = (
            mmread(
                fh
            )
            .tocsr()
        )


    if (
        M.shape[0]
        !=
        len(
            symbols
        )
        and
        M.shape[1]
        ==
        len(
            symbols
        )
    ):

        M = (
            M.T
            .tocsr()
        )


    if M.shape[0] != len(
        symbols
    ):

        raise RuntimeError(
            f"{prefix}: feature mismatch."
        )


    # ========================================================
    # 7. EXTRACT ONLY AUDIT MARKERS
    # ========================================================

    detected = {}

    raw_counts = {}

    missing_markers = []


    for gene in ALL_AUDIT_MARKERS:

        rows = symbol_rows.get(
            gene.lower(),
            []
        )


        if len(rows) == 0:

            x = np.zeros(
                len(pass_idx),
                dtype=float
            )

            missing_markers.append(
                gene
            )

        else:

            x = np.asarray(

                M[
                    rows,
                    :
                ][
                    :,
                    pass_idx
                ]
                .sum(
                    axis=0
                )

            ).ravel().astype(
                float
            )


        raw_counts[
            gene
        ] = x


        detected[
            gene
        ] = (
            x > 0
        )


        rep_marker_rows.append({

            "dataset":
                dataset,

            "sample_prefix":
                prefix,

            "condition":
                condition,

            "gene":
                gene,

            "n_qc_barcodes":
                len(pass_idx),

            "n_detected":
                int(
                    (
                        x > 0
                    ).sum()
                ),

            "pct_detected":
                float(
                    (
                        x > 0
                    ).mean()
                ),

            "mean_raw_count":
                float(
                    x.mean()
                ),

            "mean_if_detected":
                float(
                    x[
                        x > 0
                    ].mean()
                )
                if (
                    x > 0
                ).any()
                else 0.0
        })


    if missing_markers:

        print(
            "Markers absent from feature table:",
            missing_markers
        )


    # ========================================================
    # 8. PANEL HIT COUNTS
    # ========================================================

    def count_hits(
        genes
    ):

        return np.vstack(
            [
                detected[
                    gene
                ]
                for gene in genes
            ]
        ).sum(
            axis=0
        )


    ec_hits = count_hits(
        EC_MARKERS
    )


    immune_hits = count_hits(
        IMMUNE_MARKERS
    )


    epithelial_specific_hits = count_hits(
        EPITHELIAL_SPECIFIC
    )


    keratin_hits = count_hits(
        EPITHELIAL_KERATINS
    )


    ambient_lung_hits = count_hits(
        LUNG_AMBIENT_MARKERS
    )


    mural_hits = count_hits(
        MURAL_MARKERS
    )


    melanoma_specific_hits = count_hits(
        MELANOMA_SPECIFIC
    )


    # ========================================================
    # 9. CONSERVATIVE FILTER AUDIT
    #
    # Same operational logic as set2, now with Fcer1g fixed.
    # ========================================================

    ec_ge3 = (
        ec_hits
        >=
        3
    )


    ptprc_positive = (
        detected[
            "Ptprc"
        ]
    )


    immune_exclude = (

        ptprc_positive

        |

        (
            immune_hits
            >=
            2
        )
    )


    epcam_positive = (
        detected[
            "Epcam"
        ]
    )


    epithelial_exclude = (

        epcam_positive

        |

        (
            keratin_hits
            >=
            2
        )
    )


    mural_exclude = (
        mural_hits
        >=
        2
    )


    melanoma_exclude = (
        melanoma_specific_hits
        >=
        2
    )


    proposed_final_ec = (

        ec_ge3

        &

        ~immune_exclude

        &

        ~epithelial_exclude

        &

        ~mural_exclude

        &

        ~melanoma_exclude
    )


    # ========================================================
    # 10. PER-BARCODE TABLE
    # ========================================================

    identity = (
        qc.copy()
    )


    identity[
        "endothelial_marker_hits"
    ] = ec_hits


    identity[
        "immune_marker_hits"
    ] = immune_hits


    identity[
        "epithelial_specific_hits"
    ] = epithelial_specific_hits


    identity[
        "ambient_lung_marker_hits"
    ] = ambient_lung_hits


    identity[
        "mural_marker_hits"
    ] = mural_hits


    identity[
        "melanoma_specific_hits"
    ] = melanoma_specific_hits


    identity[
        "ec_ge3"
    ] = ec_ge3


    identity[
        "immune_exclude"
    ] = immune_exclude


    identity[
        "epithelial_exclude"
    ] = epithelial_exclude


    identity[
        "mural_exclude"
    ] = mural_exclude


    identity[
        "melanoma_exclude"
    ] = melanoma_exclude


    identity[
        "proposed_final_endothelial"
    ] = proposed_final_ec


    rep_identity_tables[
        prefix
    ] = identity.copy()


    identity.to_csv(

        CACHE
        /
        f"{prefix}_replication_identity_audit.csv.gz",

        index=False,
        compression="gzip"
    )


    # ========================================================
    # 11. SUMMARY
    # ========================================================

    summary = {

        "dataset":
            dataset,

        "sample_prefix":
            prefix,

        "condition":
            condition,

        "qc_barcodes":
            len(identity),

        "ec_ge3_n":
            int(
                ec_ge3.sum()
            ),

        "ec_ge3_pct":
            float(
                ec_ge3.mean()
            ),

        "immune_exclude_n":
            int(
                immune_exclude.sum()
            ),

        "immune_exclude_pct":
            float(
                immune_exclude.mean()
            ),

        "epithelial_exclude_n":
            int(
                epithelial_exclude.sum()
            ),

        "epithelial_exclude_pct":
            float(
                epithelial_exclude.mean()
            ),

        "mural_exclude_n":
            int(
                mural_exclude.sum()
            ),

        "mural_exclude_pct":
            float(
                mural_exclude.mean()
            ),

        "melanoma_exclude_n":
            int(
                melanoma_exclude.sum()
            ),

        "melanoma_exclude_pct":
            float(
                melanoma_exclude.mean()
            ),

        "proposed_final_ec_n":
            int(
                proposed_final_ec.sum()
            ),

        "proposed_final_ec_pct":
            float(
                proposed_final_ec.mean()
            ),

        "median_ec_hits":
            float(
                np.median(
                    ec_hits
                )
            )
    }


    rep_identity_summary_rows.append(
        summary
    )


    # ========================================================
    # 12. REPORT
    # ========================================================

    print(
        "\nEndothelial support:"
    )

    print(
        ">=3 EC markers:",
        summary[
            "ec_ge3_n"
        ],
        f"({100*summary['ec_ge3_pct']:.1f}%)"
    )

    print(
        "Median EC-marker hits:",
        round(
            summary[
                "median_ec_hits"
            ],
            1
        )
    )


    print(
        "\nConservative exclusion flags:"
    )

    print(
        "Immune:",
        summary[
            "immune_exclude_n"
        ],
        f"({100*summary['immune_exclude_pct']:.1f}%)"
    )

    print(
        "Epithelial-specific:",
        summary[
            "epithelial_exclude_n"
        ],
        f"({100*summary['epithelial_exclude_pct']:.1f}%)"
    )

    print(
        "Mural/fibroblast:",
        summary[
            "mural_exclude_n"
        ],
        f"({100*summary['mural_exclude_pct']:.1f}%)"
    )

    print(
        "Melanoma-specific:",
        summary[
            "melanoma_exclude_n"
        ],
        f"({100*summary['melanoma_exclude_pct']:.1f}%)"
    )


    print(
        "\nPROPOSED FINAL EC:",
        summary[
            "proposed_final_ec_n"
        ],
        f"({100*summary['proposed_final_ec_pct']:.1f}%)"
    )


    # ========================================================
    # 13. SELECTED MARKER DETECTION
    # ========================================================

    print(
        "\nSelected marker detection:"
    )


    selected_report = [

        "Cdh5",
        "Pecam1",
        "Ptprb",
        "Kdr",
        "Ramp2",

        "Ptprc",
        "Fcer1g",
        "Lyz2",

        "Epcam",
        "Krt8",
        "Krt18",
        "Krt19",

        "Hopx",
        "Scgb1a1",
        "Sftpc",

        "Pdgfrb",
        "Rgs5",
        "Col1a1",

        "Pmel",
        "Mlana",
        "Dct",
        "Sox10",
        "Mitf"
    ]


    for gene in selected_report:

        pct = float(
            detected[
                gene
            ].mean()
        )


        print(
            gene,
            ":",
            round(
                100
                *
                pct,
                2
            ),
            "%"
        )


    # ========================================================
    # 14. CLEANUP
    # ========================================================

    del M
    del features
    del symbols
    del symbol_rows

    del raw_counts
    del detected

    del ec_hits
    del immune_hits
    del epithelial_specific_hits
    del keratin_hits
    del ambient_lung_hits
    del mural_hits
    del melanoma_specific_hits

    del ec_ge3
    del ptprc_positive
    del immune_exclude
    del epcam_positive
    del epithelial_exclude
    del mural_exclude
    del melanoma_exclude
    del proposed_final_ec

    del qc
    del identity

    gc.collect()


# ============================================================
# 15. FINAL SUMMARY
# ============================================================

rep_identity_summary = pd.DataFrame(
    rep_identity_summary_rows
)


rep_marker_summary = pd.DataFrame(
    rep_marker_rows
)


rep_identity_summary.to_csv(

    OUT
    /
    "GSE235394_GSE253751_replication_identity_audit.csv",

    index=False
)


rep_marker_summary.to_csv(

    OUT
    /
    "GSE235394_GSE253751_replication_marker_detection.csv",

    index=False
)


print(
    "\n======================================"
)

print(
    "REPLICATION ENDOTHELIAL IDENTITY SUMMARY"
)

print(
    "======================================"
)


display(
    rep_identity_summary
)


print(
    "\nPHASE D6C REPLICATION IDENTITY AUDIT COMPLETE"
)

In [ ]:
# ============================================================
# PHASE D6D — CORRECT Fcer1g AND REBUILD SET2 ENDOTHELIAL SET
#
# Earlier GSE253749 identity filtering accidentally used:
#
#     Fcerg1   <-- wrong
#
# Correct mouse gene:
#
#     Fcer1g
#
# We therefore rebuild the set2 identity/filter directly from
# raw counts using EXACTLY the same marker logic now used for
# set1 and set3.
#
# D5 receptor-response results remain PROVISIONAL until they
# are recomputed from these corrected set2 EC indices.
# ============================================================

import gc
import gzip
import numpy as np
import pandas as pd

from collections import defaultdict
from scipy.io import mmread


# ============================================================
# 1. IDENTICAL MARKER DEFINITIONS USED IN D6C
# ============================================================

SET2_EC_MARKERS = [
    "Cdh5",
    "Pecam1",
    "Ptprb",
    "Esam",
    "Eng",
    "Kdr",
    "Tek",
    "Emcn",
    "Ramp2"
]


SET2_IMMUNE_MARKERS = [
    "Ptprc",
    "Lst1",
    "Tyrobp",
    "Fcer1g",   # CORRECTED
    "Csf1r",
    "Lyz2",
    "Cd3d",
    "Cd79a",
    "Ms4a1",
    "Nkg7"
]


SET2_EPITHELIAL_KERATINS = [
    "Krt8",
    "Krt18",
    "Krt19"
]


SET2_MURAL_MARKERS = [
    "Pdgfrb",
    "Rgs5",
    "Cspg4",
    "Acta2",
    "Tagln",
    "Pdgfra",
    "Col1a1",
    "Col1a2",
    "Dcn",
    "Lum"
]


SET2_MELANOMA_SPECIFIC = [
    "Pmel",
    "Mlana",
    "Tyr",
    "Dct",
    "Sox10"
]


SET2_FILTER_GENES = sorted(
    set(
        SET2_EC_MARKERS
        +
        SET2_IMMUNE_MARKERS
        +
        ["Epcam"]
        +
        SET2_EPITHELIAL_KERATINS
        +
        SET2_MURAL_MARKERS
        +
        SET2_MELANOMA_SPECIFIC
    )
)


print(
    "Set2 filter genes:",
    len(SET2_FILTER_GENES)
)


# ============================================================
# 2. OUTPUT OBJECTS
# ============================================================

g253_corrected_final_ec_indices = {}

g253_corrected_final_ec_barcodes = {}

g253_corrected_identity_tables = {}

set2_correction_rows = []


# ============================================================
# 3. PROCESS EACH SET2 SAMPLE
# ============================================================

for prefix in sorted(
    groups253
):

    fg = groups253[
        prefix
    ]

    condition = COND_253[
        prefix
    ]


    print(
        "\n======================================"
    )

    print(
        prefix,
        "|",
        condition
    )

    print(
        "======================================"
    )


    # ========================================================
    # 4. RESTORE ORIGINAL D1 QC-PASSING BARCODE TABLE
    # ============================================================

    qc = pd.read_csv(

        CACHE
        /
        f"{prefix}_QC_pass_barcodes.csv.gz",

        compression="gzip"
    )


    raw_idx = (
        qc[
            "raw_column_index"
        ]
        .to_numpy(
            dtype=int
        )
    )


    print(
        "QC-passing barcodes:",
        len(qc)
    )


    # ========================================================
    # 5. LOAD OLD D4 IDENTITY TABLE FOR COMPARISON ONLY
    # ============================================================

    old_path = (

        CACHE
        /
        f"{prefix}_final_endothelial_identity_table.csv.gz"
    )


    old_identity = pd.read_csv(
        old_path,
        compression="gzip"
    )


    old_final = (
        old_identity[
            "final_endothelial"
        ]
        .astype(bool)
        .to_numpy()
    )


    old_immune_exclude = (
        old_identity[
            "immune_exclude"
        ]
        .astype(bool)
        .to_numpy()
    )


    if len(old_identity) != len(qc):

        raise RuntimeError(
            f"{prefix}: old identity/QC row mismatch."
        )


    # ========================================================
    # 6. FEATURE LOOKUP
    # ============================================================

    features = pd.read_csv(

        fg[
            "features"
        ],

        sep="\t",
        header=None,
        compression="gzip"
    )


    if features.shape[1] >= 2:

        symbols = (
            features.iloc[
                :,
                1
            ]
            .astype(str)
            .tolist()
        )

    else:

        symbols = (
            features.iloc[
                :,
                0
            ]
            .astype(str)
            .tolist()
        )


    symbol_rows = defaultdict(
        list
    )


    for i, symbol in enumerate(
        symbols
    ):

        symbol_rows[
            symbol.lower()
        ].append(
            i
        )


    # Explicitly verify corrected gene exists
    fcer1g_rows = symbol_rows.get(
        "fcer1g",
        []
    )


    print(
        "Fcer1g feature rows:",
        len(fcer1g_rows)
    )


    if len(fcer1g_rows) == 0:

        raise RuntimeError(
            f"{prefix}: Fcer1g absent from feature table."
        )


    # ========================================================
    # 7. LOAD RAW MATRIX
    # ============================================================

    with gzip.open(
        fg[
            "matrix"
        ],
        "rb"
    ) as fh:

        M = (
            mmread(
                fh
            )
            .tocsr()
        )


    if (
        M.shape[0]
        !=
        len(symbols)
        and
        M.shape[1]
        ==
        len(symbols)
    ):

        M = (
            M.T
            .tocsr()
        )


    if M.shape[0] != len(
        symbols
    ):

        raise RuntimeError(
            f"{prefix}: matrix/feature mismatch."
        )


    # ========================================================
    # 8. EXTRACT FILTER MARKERS
    # ============================================================

    detected = {}

    raw_counts = {}


    for gene in SET2_FILTER_GENES:

        rows = symbol_rows.get(
            gene.lower(),
            []
        )


        if len(rows) == 0:

            x = np.zeros(
                len(raw_idx),
                dtype=float
            )

        else:

            x = np.asarray(

                M[
                    rows,
                    :
                ][
                    :,
                    raw_idx
                ]
                .sum(
                    axis=0
                )

            ).ravel().astype(
                float
            )


        raw_counts[
            gene
        ] = x


        detected[
            gene
        ] = (
            x > 0
        )


    # ========================================================
    # 9. PANEL HIT COUNTS
    # ============================================================

    def set2_count_hits(
        genes
    ):

        return np.vstack(
            [
                detected[
                    gene
                ]
                for gene in genes
            ]
        ).sum(
            axis=0
        )


    ec_hits = set2_count_hits(
        SET2_EC_MARKERS
    )


    immune_hits = set2_count_hits(
        SET2_IMMUNE_MARKERS
    )


    keratin_hits = set2_count_hits(
        SET2_EPITHELIAL_KERATINS
    )


    mural_hits = set2_count_hits(
        SET2_MURAL_MARKERS
    )


    melanoma_hits = set2_count_hits(
        SET2_MELANOMA_SPECIFIC
    )


    # ========================================================
    # 10. IDENTICAL CONSERVATIVE FILTER
    # ============================================================

    ec_ge3 = (
        ec_hits
        >=
        3
    )


    immune_exclude = (

        detected[
            "Ptprc"
        ]

        |

        (
            immune_hits
            >=
            2
        )
    )


    epithelial_exclude = (

        detected[
            "Epcam"
        ]

        |

        (
            keratin_hits
            >=
            2
        )
    )


    mural_exclude = (
        mural_hits
        >=
        2
    )


    melanoma_exclude = (
        melanoma_hits
        >=
        2
    )


    corrected_final = (

        ec_ge3

        &

        ~immune_exclude

        &

        ~epithelial_exclude

        &

        ~mural_exclude

        &

        ~melanoma_exclude
    )


    # ========================================================
    # 11. Fcer1g AUDIT
    # ============================================================

    fcer1g_positive = (
        detected[
            "Fcer1g"
        ]
    )


    print(
        "Fcer1g detected:",
        int(
            fcer1g_positive.sum()
        ),
        "/",
        len(fcer1g_positive),
        f"({100*fcer1g_positive.mean():.2f}%)"
    )


    newly_immune_excluded = (

        immune_exclude

        &

        ~old_immune_exclude
    )


    print(
        "New immune exclusions after Fcer1g correction:",
        int(
            newly_immune_excluded.sum()
        )
    )


    # ========================================================
    # 12. COMPARE OLD VS CORRECTED FINAL SET
    # ============================================================

    old_n = int(
        old_final.sum()
    )


    corrected_n = int(
        corrected_final.sum()
    )


    removed_from_old = (

        old_final

        &

        ~corrected_final
    )


    added_vs_old = (

        corrected_final

        &

        ~old_final
    )


    print(
        "\nOld D4 final EC:",
        old_n
    )


    print(
        "Corrected final EC:",
        corrected_n
    )


    print(
        "Removed from old final EC:",
        int(
            removed_from_old.sum()
        )
    )


    print(
        "Added relative to old final EC:",
        int(
            added_vs_old.sum()
        )
    )


    print(
        "Corrected final EC %:",
        round(
            100
            *
            float(
                corrected_final.mean()
            ),
            2
        )
    )


    # ========================================================
    # 13. BUILD CORRECTED IDENTITY TABLE
    # ============================================================

    corrected_identity = (
        qc.copy()
    )


    corrected_identity[
        "endothelial_marker_hits"
    ] = ec_hits


    corrected_identity[
        "immune_marker_hits_corrected"
    ] = immune_hits


    corrected_identity[
        "Fcer1g_detected"
    ] = fcer1g_positive


    corrected_identity[
        "keratin_hits"
    ] = keratin_hits


    corrected_identity[
        "mural_marker_hits"
    ] = mural_hits


    corrected_identity[
        "melanoma_specific_hits"
    ] = melanoma_hits


    corrected_identity[
        "ec_ge3"
    ] = ec_ge3


    corrected_identity[
        "immune_exclude"
    ] = immune_exclude


    corrected_identity[
        "epithelial_exclude"
    ] = epithelial_exclude


    corrected_identity[
        "mural_exclude"
    ] = mural_exclude


    corrected_identity[
        "melanoma_exclude"
    ] = melanoma_exclude


    corrected_identity[
        "final_endothelial"
    ] = corrected_final


    # ========================================================
    # 14. SAVE CORRECTED FINAL INDICES
    # ============================================================

    corrected_raw_idx = (
        raw_idx[
            corrected_final
        ]
    )


    corrected_barcodes = (

        corrected_identity.loc[
            corrected_final,
            "barcode"
        ]
        .astype(str)
        .to_numpy()
    )


    g253_corrected_final_ec_indices[
        prefix
    ] = (
        corrected_raw_idx.copy()
    )


    g253_corrected_final_ec_barcodes[
        prefix
    ] = (
        corrected_barcodes.copy()
    )


    g253_corrected_identity_tables[
        prefix
    ] = (
        corrected_identity.copy()
    )


    np.save(

        CACHE
        /
        f"{prefix}_final_endothelial_column_indices_Fcer1g_corrected.npy",

        corrected_raw_idx
    )


    corrected_identity.to_csv(

        CACHE
        /
        f"{prefix}_final_endothelial_identity_Fcer1g_corrected.csv.gz",

        index=False,
        compression="gzip"
    )


    # ========================================================
    # 15. SUMMARY ROW
    # ============================================================

    set2_correction_rows.append({

        "sample_prefix":
            prefix,

        "condition":
            condition,

        "qc_barcodes":
            len(qc),

        "Fcer1g_n":
            int(
                fcer1g_positive.sum()
            ),

        "Fcer1g_pct":
            float(
                fcer1g_positive.mean()
            ),

        "old_immune_exclude_n":
            int(
                old_immune_exclude.sum()
            ),

        "corrected_immune_exclude_n":
            int(
                immune_exclude.sum()
            ),

        "new_immune_exclusions_n":
            int(
                newly_immune_excluded.sum()
            ),

        "old_final_ec_n":
            old_n,

        "corrected_final_ec_n":
            corrected_n,

        "removed_from_old_final_n":
            int(
                removed_from_old.sum()
            ),

        "added_vs_old_final_n":
            int(
                added_vs_old.sum()
            ),

        "corrected_final_ec_pct":
            float(
                corrected_final.mean()
            ),

        "median_ec_marker_hits_corrected":
            float(
                np.median(
                    ec_hits[
                        corrected_final
                    ]
                )
            )
    })


    # ========================================================
    # 16. CLEANUP
    # ============================================================

    del M
    del features
    del symbols
    del symbol_rows

    del raw_counts
    del detected

    del ec_hits
    del immune_hits
    del keratin_hits
    del mural_hits
    del melanoma_hits

    del ec_ge3
    del immune_exclude
    del epithelial_exclude
    del mural_exclude
    del melanoma_exclude

    del fcer1g_positive
    del newly_immune_excluded

    del old_final
    del old_immune_exclude
    del corrected_final
    del removed_from_old
    del added_vs_old

    del qc
    del old_identity
    del corrected_identity

    gc.collect()


# ============================================================
# 17. FINAL CORRECTION SUMMARY
# ============================================================

set2_Fcer1g_correction_summary = pd.DataFrame(
    set2_correction_rows
)


set2_Fcer1g_correction_summary.to_csv(

    OUT
    /
    "GSE253749_Fcer1g_corrected_endothelial_filter_summary.csv",

    index=False
)


print(
    "\n======================================"
)

print(
    "SET2 Fcer1g CORRECTION SUMMARY"
)

print(
    "======================================"
)


display(
    set2_Fcer1g_correction_summary
)


print(
    "\nOld total final EC:",
    int(
        set2_Fcer1g_correction_summary[
            "old_final_ec_n"
        ]
        .sum()
    )
)


print(
    "Corrected total final EC:",
    int(
        set2_Fcer1g_correction_summary[
            "corrected_final_ec_n"
        ]
        .sum()
    )
)


print(
    "Total cells removed by corrected filter:",
    int(
        set2_Fcer1g_correction_summary[
            "removed_from_old_final_n"
        ]
        .sum()
    )
)


print(
    "\nPHASE D6D SET2 Fcer1g CORRECTION COMPLETE"
)

In [ ]:
# ============================================================
# PHASE D6E — RECOMPUTE SET2 RECEPTOR RESPONSE AFTER
#             Fcer1g-CORRECTED ENDOTHELIAL FILTER
#
# Recalculates all 247 PSN-linked receptor metrics using the
# corrected GSE253749 EC sets from D6D.
#
# GSE253749 remains DESCRIPTIVE:
# one pooled sample per condition.
#
# Outputs from this cell supersede the provisional D5 tables.
# ============================================================

import gc
import gzip
import numpy as np
import pandas as pd

from collections import defaultdict
from scipy.io import mmread


# ============================================================
# 1. PRESERVE PROVISIONAL D5 EFFECT TABLE FOR COMPARISON
# ============================================================

if (
    "g253_effect" in globals()
    and isinstance(g253_effect, pd.DataFrame)
    and len(g253_effect) > 0
):

    g253_effect_before_Fcer1g = (
        g253_effect.copy()
    )

else:

    g253_effect_before_Fcer1g = (
        pd.DataFrame()
    )


# ============================================================
# 2. REBUILD PSN-LINKED RECEPTOR UNIVERSE DIRECTLY FROM B4
# ============================================================

psn_receptor_universe = sorted(

    set(
        lr_psn_detected[
            "mouse_receptor"
        ]
        .dropna()
        .astype(str)
    )
)


print(
    "PSN-linked receptors from B4:",
    len(psn_receptor_universe)
)


# ============================================================
# 3. VERIFY FEATURE SPACE
# ============================================================

feature_sets = {}


for prefix in sorted(
    groups253
):

    features = pd.read_csv(

        groups253[
            prefix
        ][
            "features"
        ],

        sep="\t",
        header=None,
        compression="gzip"
    )


    if features.shape[1] >= 2:

        symbols = (
            features.iloc[
                :,
                1
            ]
            .astype(str)
        )

    else:

        symbols = (
            features.iloc[
                :,
                0
            ]
            .astype(str)
        )


    feature_sets[
        prefix
    ] = set(
        symbols
    )


common_features = set.intersection(
    *feature_sets.values()
)


g253_candidate_receptors = sorted(

    set(
        psn_receptor_universe
    )

    &

    common_features
)


print(
    "Comparable receptors:",
    len(g253_candidate_receptors)
)


if len(g253_candidate_receptors) != 247:

    print(
        "WARNING: expected 247 receptors based on D5."
    )


# ============================================================
# 4. PROCESS CORRECTED SET2 ECs
# ============================================================

corrected_receptor_rows = []


for prefix in sorted(
    groups253
):

    fg = groups253[
        prefix
    ]

    condition = COND_253[
        prefix
    ]


    print(
        "\n======================================"
    )

    print(
        prefix,
        "|",
        condition
    )

    print(
        "======================================"
    )


    # --------------------------------------------------------
    # Restore corrected identity table
    # --------------------------------------------------------

    if (
        "g253_corrected_identity_tables" in globals()
        and
        prefix in g253_corrected_identity_tables
    ):

        identity = (
            g253_corrected_identity_tables[
                prefix
            ]
            .copy()
        )

    else:

        identity = pd.read_csv(

            CACHE
            /
            f"{prefix}_final_endothelial_identity_Fcer1g_corrected.csv.gz",

            compression="gzip"
        )


    final_identity = (

        identity[
            identity[
                "final_endothelial"
            ]
            .astype(bool)
        ]
        .copy()
    )


    final_raw_idx = (
        final_identity[
            "raw_column_index"
        ]
        .to_numpy(
            dtype=int
        )
    )


    cell_total_umis = (
        final_identity[
            "total_counts"
        ]
        .to_numpy(
            dtype=float
        )
    )


    print(
        "Corrected final EC:",
        len(final_raw_idx)
    )


    # ========================================================
    # 5. FEATURES
    # ========================================================

    features = pd.read_csv(

        fg[
            "features"
        ],

        sep="\t",
        header=None,
        compression="gzip"
    )


    if features.shape[1] >= 2:

        feature_symbols = (
            features.iloc[
                :,
                1
            ]
            .astype(str)
            .tolist()
        )

    else:

        feature_symbols = (
            features.iloc[
                :,
                0
            ]
            .astype(str)
            .tolist()
        )


    symbol_rows = defaultdict(
        list
    )


    for i, gene in enumerate(
        feature_symbols
    ):

        if gene in g253_candidate_receptors:

            symbol_rows[
                gene
            ].append(
                i
            )


    missing = [

        gene
        for gene in g253_candidate_receptors
        if gene not in symbol_rows

    ]


    if missing:

        raise RuntimeError(
            f"{prefix}: missing receptors: {missing[:20]}"
        )


    # ========================================================
    # 6. RAW MATRIX
    # ========================================================

    with gzip.open(
        fg[
            "matrix"
        ],
        "rb"
    ) as fh:

        M = (
            mmread(
                fh
            )
            .tocsr()
        )


    if (
        M.shape[0]
        != len(feature_symbols)
        and
        M.shape[1]
        ==
        len(feature_symbols)
    ):

        M = (
            M.T
            .tocsr()
        )


    # ========================================================
    # 7. EXTRACT ALL RECEPTOR ROWS
    # ========================================================

    selected_rows = []

    selected_gene_for_row = []


    for gene in g253_candidate_receptors:

        for row_i in symbol_rows[
            gene
        ]:

            selected_rows.append(
                row_i
            )

            selected_gene_for_row.append(
                gene
            )


    R = (

        M[
            selected_rows,
            :
        ][
            :,
            final_raw_idx
        ]
        .tocsr()
    )


    gene_positions = defaultdict(
        list
    )


    for local_i, gene in enumerate(
        selected_gene_for_row
    ):

        gene_positions[
            gene
        ].append(
            local_i
        )


    total_ec_umis = float(
        cell_total_umis.sum()
    )


    print(
        "Corrected endothelial UMIs:",
        int(total_ec_umis)
    )


    # ========================================================
    # 8. RECEPTOR METRICS
    # ========================================================

    for gene in g253_candidate_receptors:

        x = np.asarray(

            R[
                gene_positions[
                    gene
                ],
                :
            ]
            .sum(
                axis=0
            )

        ).ravel().astype(
            float
        )


        pct_expressing = float(
            (
                x > 0
            )
            .mean()
        )


        lognorm = np.log1p(

            x
            *
            1e4
            /
            cell_total_umis
        )


        raw_sum = float(
            x.sum()
        )


        pseudobulk_cpm = float(

            1e6
            *
            raw_sum
            /
            total_ec_umis
        )


        corrected_receptor_rows.append({

            "sample_prefix":
                prefix,

            "condition":
                condition,

            "gene":
                gene,

            "n_endothelial_cells":
                int(
                    len(x)
                ),

            "total_endothelial_umis":
                total_ec_umis,

            "raw_sum":
                raw_sum,

            "n_expressing":
                int(
                    (
                        x > 0
                    )
                    .sum()
                ),

            "pct_expressing":
                pct_expressing,

            "mean_raw_count":
                float(
                    x.mean()
                ),

            "mean_log1p_norm":
                float(
                    lognorm.mean()
                ),

            "median_log1p_norm":
                float(
                    np.median(
                        lognorm
                    )
                ),

            "pseudobulk_cpm":
                pseudobulk_cpm,

            "pseudobulk_log2cpm1":
                float(
                    np.log2(
                        pseudobulk_cpm
                        +
                        1.0
                    )
                )
        })


    # ========================================================
    # 9. CLEANUP
    # ========================================================

    del M
    del R

    del identity
    del final_identity

    del final_raw_idx
    del cell_total_umis

    del features
    del feature_symbols
    del symbol_rows

    del selected_rows
    del selected_gene_for_row
    del gene_positions

    gc.collect()


# ============================================================
# 10. CORRECTED EXPRESSION TABLE
# ============================================================

g253_receptor_summary_corrected = pd.DataFrame(
    corrected_receptor_rows
)


g253_receptor_summary_corrected.to_csv(

    OUT
    /
    "GSE253749_Fcer1g_corrected_finalEC_receptor_expression.csv",

    index=False
)


# Canonical object from now onward
g253_receptor_summary = (
    g253_receptor_summary_corrected.copy()
)

g253_summary = (
    g253_receptor_summary_corrected.copy()
)


print(
    "\n======================================"
)

print(
    "CORRECTED SET2 SAMPLE SUMMARY"
)

print(
    "======================================"
)


display(

    g253_receptor_summary[
        [
            "sample_prefix",
            "condition",
            "n_endothelial_cells",
            "total_endothelial_umis"
        ]
    ]
    .drop_duplicates()
)


# ============================================================
# 11. BUILD CORRECTED EFFECT TABLE
# ============================================================

metrics = [

    "pseudobulk_cpm",
    "pseudobulk_log2cpm1",
    "pct_expressing",
    "mean_log1p_norm",
    "raw_sum"
]


wide_parts = []


for metric in metrics:

    tmp = (
        g253_receptor_summary
        .pivot(
            index="gene",
            columns="condition",
            values=metric
        )
    )


    tmp.columns = [

        f"{condition}_{metric}"
        for condition in tmp.columns

    ]


    wide_parts.append(
        tmp
    )


g253_effect_corrected = pd.concat(
    wide_parts,
    axis=1
).reset_index()


for condition in [
    "6h",
    "30h"
]:

    g253_effect_corrected[
        f"{condition}_delta_log2cpm1"
    ] = (

        g253_effect_corrected[
            f"{condition}_pseudobulk_log2cpm1"
        ]

        -

        g253_effect_corrected[
            "control_pseudobulk_log2cpm1"
        ]
    )


    g253_effect_corrected[
        f"{condition}_delta_pct_expressing"
    ] = (

        g253_effect_corrected[
            f"{condition}_pct_expressing"
        ]

        -

        g253_effect_corrected[
            "control_pct_expressing"
        ]
    )


    g253_effect_corrected[
        f"{condition}_delta_mean_log1p_norm"
    ] = (

        g253_effect_corrected[
            f"{condition}_mean_log1p_norm"
        ]

        -

        g253_effect_corrected[
            "control_mean_log1p_norm"
        ]
    )


g253_effect_corrected[
    "max_pct_expressing"
] = (

    g253_effect_corrected[
        [
            "control_pct_expressing",
            "6h_pct_expressing",
            "30h_pct_expressing"
        ]
    ]
    .max(
        axis=1
    )
)


g253_effect_corrected[
    "detected_ge_1pct_any_condition"
] = (

    g253_effect_corrected[
        "max_pct_expressing"
    ]
    >=
    0.01
)


g253_effect_corrected[
    "detected_ge_5pct_any_condition"
] = (

    g253_effect_corrected[
        "max_pct_expressing"
    ]
    >=
    0.05
)


g253_effect_corrected[
    "max_abs_metastatic_delta"
] = (

    g253_effect_corrected[
        [
            "6h_delta_log2cpm1",
            "30h_delta_log2cpm1"
        ]
    ]
    .abs()
    .max(
        axis=1
    )
)


# Canonical effect table from now onward
g253_effect = (
    g253_effect_corrected.copy()
)


g253_effect.to_csv(

    OUT
    /
    "GSE253749_Fcer1g_corrected_receptor_descriptive_effects.csv",

    index=False
)


# ============================================================
# 12. TOP CORRECTED DESCRIPTIVE EFFECTS
# ============================================================

print(
    "\n======================================"
)

print(
    "TOP CORRECTED SET2 RECEPTOR EFFECTS"
)

print(
    "======================================"
)


display(

    g253_effect[
        g253_effect[
            "detected_ge_1pct_any_condition"
        ]
    ]
    .sort_values(
        "max_abs_metastatic_delta",
        ascending=False
    )
    [
        [
            "gene",

            "control_pseudobulk_cpm",
            "6h_pseudobulk_cpm",
            "30h_pseudobulk_cpm",

            "control_pct_expressing",
            "6h_pct_expressing",
            "30h_pct_expressing",

            "6h_delta_log2cpm1",
            "30h_delta_log2cpm1",

            "6h_delta_pct_expressing",
            "30h_delta_pct_expressing",

            "max_abs_metastatic_delta"
        ]
    ]
    .head(
        40
    )
)


# ============================================================
# 13. SELECTED NEUROVASCULAR RECEPTORS
# ============================================================

selected_receptors = [

    "Calcrl",
    "Ramp1",
    "Ramp2",
    "Ramp3",

    "Ntrk2",

    "Kdr",
    "Nrp1",

    "Pdgfrb",

    "Vipr1",
    "Vipr2",

    "Galr1",
    "Galr2",

    "Sstr1",
    "Sstr2",

    "Tacr2"
]


selected_corrected = (

    g253_effect[
        g253_effect[
            "gene"
        ]
        .isin(
            selected_receptors
        )
    ]
    [
        [
            "gene",

            "control_pseudobulk_cpm",
            "6h_pseudobulk_cpm",
            "30h_pseudobulk_cpm",

            "control_pct_expressing",
            "6h_pct_expressing",
            "30h_pct_expressing",

            "6h_delta_log2cpm1",
            "30h_delta_log2cpm1",

            "6h_delta_pct_expressing",
            "30h_delta_pct_expressing"
        ]
    ]
    .sort_values(
        "gene"
    )
)


print(
    "\n======================================"
)

print(
    "SELECTED RECEPTORS — CORRECTED SET2"
)

print(
    "======================================"
)


display(
    selected_corrected
)


# ============================================================
# 14. OLD D5 VS CORRECTED D6E COMPARISON
# ============================================================

if len(
    g253_effect_before_Fcer1g
):

    old_compare = (

        g253_effect_before_Fcer1g[
            [
                "gene",
                "6h_delta_log2cpm1",
                "30h_delta_log2cpm1"
            ]
        ]
        .rename(
            columns={

                "6h_delta_log2cpm1":
                    "old_6h_delta",

                "30h_delta_log2cpm1":
                    "old_30h_delta"
            }
        )
    )


    new_compare = (

        g253_effect[
            [
                "gene",
                "6h_delta_log2cpm1",
                "30h_delta_log2cpm1"
            ]
        ]
        .rename(
            columns={

                "6h_delta_log2cpm1":
                    "corrected_6h_delta",

                "30h_delta_log2cpm1":
                    "corrected_30h_delta"
            }
        )
    )


    correction_compare = (
        old_compare.merge(
            new_compare,
            on="gene",
            how="inner"
        )
    )


    correction_compare[
        "change_in_6h_delta"
    ] = (

        correction_compare[
            "corrected_6h_delta"
        ]

        -

        correction_compare[
            "old_6h_delta"
        ]
    )


    correction_compare[
        "change_in_30h_delta"
    ] = (

        correction_compare[
            "corrected_30h_delta"
        ]

        -

        correction_compare[
            "old_30h_delta"
        ]
    )


    correction_compare[
        "direction_flip_6h"
    ] = (

        np.sign(
            correction_compare[
                "old_6h_delta"
            ]
        )

        !=

        np.sign(
            correction_compare[
                "corrected_6h_delta"
            ]
        )
    )


    correction_compare[
        "direction_flip_30h"
    ] = (

        np.sign(
            correction_compare[
                "old_30h_delta"
            ]
        )

        !=

        np.sign(
            correction_compare[
                "corrected_30h_delta"
            ]
        )
    )


    print(
        "\n======================================"
    )

    print(
        "Fcer1g CORRECTION IMPACT"
    )

    print(
        "======================================"
    )


    print(
        "6h direction flips:",
        int(
            correction_compare[
                "direction_flip_6h"
            ]
            .sum()
        )
    )


    print(
        "30h direction flips:",
        int(
            correction_compare[
                "direction_flip_30h"
            ]
            .sum()
        )
    )


    print(
        "Median absolute change in 6h delta:",
        round(
            float(
                correction_compare[
                    "change_in_6h_delta"
                ]
                .abs()
                .median()
            ),
            5
        )
    )


    print(
        "Median absolute change in 30h delta:",
        round(
            float(
                correction_compare[
                    "change_in_30h_delta"
                ]
                .abs()
                .median()
            ),
            5
        )
    )


    print(
        "\nSelected receptor correction impact:"
    )


    display(

        correction_compare[
            correction_compare[
                "gene"
            ]
            .isin(
                selected_receptors
            )
        ]
        .sort_values(
            "gene"
        )
    )


# ============================================================
# 15. FINAL CHECKS
# ============================================================

assert (
    g253_effect[
        "gene"
    ]
    .nunique()
    ==
    len(
        g253_candidate_receptors
    )
)


assert not g253_effect[
    "gene"
].duplicated().any()


print(
    "\n======================================"
)

print(
    "PHASE D6E CORRECTED SET2 RESPONSE COMPLETE"
)

print(
    "======================================"
)


print(
    "Corrected set2 ECs:"
)


for prefix in sorted(
    groups253
):

    n = (

        g253_receptor_summary[
            g253_receptor_summary[
                "sample_prefix"
            ]
            ==
            prefix
        ][
            "n_endothelial_cells"
        ]
        .iloc[
            0
        ]
    )


    print(
        prefix,
        "|",
        COND_253[
            prefix
        ],
        "|",
        int(
            n
        )
    )


print(
    "\nReceptors >=1% ECs:",
    int(
        g253_effect[
            "detected_ge_1pct_any_condition"
        ]
        .sum()
    )
)


print(
    "Receptors >=5% ECs:",
    int(
        g253_effect[
            "detected_ge_5pct_any_condition"
        ]
        .sum()
    )
)


print(
    "\nGSE253749 remains descriptive: "
    "one pooled sample per condition."
)

In [ ]:
# ============================================================
# PHASE D6E — RECOMPUTE SET2 RECEPTOR RESPONSE AFTER
#             Fcer1g-CORRECTED ENDOTHELIAL FILTER
#
# Recalculates all 247 PSN-linked receptor metrics using the
# corrected GSE253749 EC sets from D6D.
#
# GSE253749 remains DESCRIPTIVE:
# one pooled sample per condition.
#
# Outputs from this cell supersede the provisional D5 tables.
# ============================================================

import gc
import gzip
import numpy as np
import pandas as pd

from collections import defaultdict
from scipy.io import mmread


# ============================================================
# 1. PRESERVE PROVISIONAL D5 EFFECT TABLE FOR COMPARISON
# ============================================================

if (
    "g253_effect" in globals()
    and isinstance(g253_effect, pd.DataFrame)
    and len(g253_effect) > 0
):

    g253_effect_before_Fcer1g = (
        g253_effect.copy()
    )

else:

    g253_effect_before_Fcer1g = (
        pd.DataFrame()
    )


# ============================================================
# 2. REBUILD PSN-LINKED RECEPTOR UNIVERSE DIRECTLY FROM B4
# ============================================================

psn_receptor_universe = sorted(

    set(
        lr_psn_detected[
            "mouse_receptor"
        ]
        .dropna()
        .astype(str)
    )
)


print(
    "PSN-linked receptors from B4:",
    len(psn_receptor_universe)
)


# ============================================================
# 3. VERIFY FEATURE SPACE
# ============================================================

feature_sets = {}


for prefix in sorted(
    groups253
):

    features = pd.read_csv(

        groups253[
            prefix
        ][
            "features"
        ],

        sep="\t",
        header=None,
        compression="gzip"
    )


    if features.shape[1] >= 2:

        symbols = (
            features.iloc[
                :,
                1
            ]
            .astype(str)
        )

    else:

        symbols = (
            features.iloc[
                :,
                0
            ]
            .astype(str)
        )


    feature_sets[
        prefix
    ] = set(
        symbols
    )


common_features = set.intersection(
    *feature_sets.values()
)


g253_candidate_receptors = sorted(

    set(
        psn_receptor_universe
    )

    &

    common_features
)


print(
    "Comparable receptors:",
    len(g253_candidate_receptors)
)


if len(g253_candidate_receptors) != 247:

    print(
        "WARNING: expected 247 receptors based on D5."
    )


# ============================================================
# 4. PROCESS CORRECTED SET2 ECs
# ============================================================

corrected_receptor_rows = []


for prefix in sorted(
    groups253
):

    fg = groups253[
        prefix
    ]

    condition = COND_253[
        prefix
    ]


    print(
        "\n======================================"
    )

    print(
        prefix,
        "|",
        condition
    )

    print(
        "======================================"
    )


    # --------------------------------------------------------
    # Restore corrected identity table
    # --------------------------------------------------------

    if (
        "g253_corrected_identity_tables" in globals()
        and
        prefix in g253_corrected_identity_tables
    ):

        identity = (
            g253_corrected_identity_tables[
                prefix
            ]
            .copy()
        )

    else:

        identity = pd.read_csv(

            CACHE
            /
            f"{prefix}_final_endothelial_identity_Fcer1g_corrected.csv.gz",

            compression="gzip"
        )


    final_identity = (

        identity[
            identity[
                "final_endothelial"
            ]
            .astype(bool)
        ]
        .copy()
    )


    final_raw_idx = (
        final_identity[
            "raw_column_index"
        ]
        .to_numpy(
            dtype=int
        )
    )


    cell_total_umis = (
        final_identity[
            "total_counts"
        ]
        .to_numpy(
            dtype=float
        )
    )


    print(
        "Corrected final EC:",
        len(final_raw_idx)
    )


    # ========================================================
    # 5. FEATURES
    # ========================================================

    features = pd.read_csv(

        fg[
            "features"
        ],

        sep="\t",
        header=None,
        compression="gzip"
    )


    if features.shape[1] >= 2:

        feature_symbols = (
            features.iloc[
                :,
                1
            ]
            .astype(str)
            .tolist()
        )

    else:

        feature_symbols = (
            features.iloc[
                :,
                0
            ]
            .astype(str)
            .tolist()
        )


    symbol_rows = defaultdict(
        list
    )


    for i, gene in enumerate(
        feature_symbols
    ):

        if gene in g253_candidate_receptors:

            symbol_rows[
                gene
            ].append(
                i
            )


    missing = [

        gene
        for gene in g253_candidate_receptors
        if gene not in symbol_rows

    ]


    if missing:

        raise RuntimeError(
            f"{prefix}: missing receptors: {missing[:20]}"
        )


    # ========================================================
    # 6. RAW MATRIX
    # ========================================================

    with gzip.open(
        fg[
            "matrix"
        ],
        "rb"
    ) as fh:

        M = (
            mmread(
                fh
            )
            .tocsr()
        )


    if (
        M.shape[0]
        != len(feature_symbols)
        and
        M.shape[1]
        ==
        len(feature_symbols)
    ):

        M = (
            M.T
            .tocsr()
        )


    # ========================================================
    # 7. EXTRACT ALL RECEPTOR ROWS
    # ========================================================

    selected_rows = []

    selected_gene_for_row = []


    for gene in g253_candidate_receptors:

        for row_i in symbol_rows[
            gene
        ]:

            selected_rows.append(
                row_i
            )

            selected_gene_for_row.append(
                gene
            )


    R = (

        M[
            selected_rows,
            :
        ][
            :,
            final_raw_idx
        ]
        .tocsr()
    )


    gene_positions = defaultdict(
        list
    )


    for local_i, gene in enumerate(
        selected_gene_for_row
    ):

        gene_positions[
            gene
        ].append(
            local_i
        )


    total_ec_umis = float(
        cell_total_umis.sum()
    )


    print(
        "Corrected endothelial UMIs:",
        int(total_ec_umis)
    )


    # ========================================================
    # 8. RECEPTOR METRICS
    # ========================================================

    for gene in g253_candidate_receptors:

        x = np.asarray(

            R[
                gene_positions[
                    gene
                ],
                :
            ]
            .sum(
                axis=0
            )

        ).ravel().astype(
            float
        )


        pct_expressing = float(
            (
                x > 0
            )
            .mean()
        )


        lognorm = np.log1p(

            x
            *
            1e4
            /
            cell_total_umis
        )


        raw_sum = float(
            x.sum()
        )


        pseudobulk_cpm = float(

            1e6
            *
            raw_sum
            /
            total_ec_umis
        )


        corrected_receptor_rows.append({

            "sample_prefix":
                prefix,

            "condition":
                condition,

            "gene":
                gene,

            "n_endothelial_cells":
                int(
                    len(x)
                ),

            "total_endothelial_umis":
                total_ec_umis,

            "raw_sum":
                raw_sum,

            "n_expressing":
                int(
                    (
                        x > 0
                    )
                    .sum()
                ),

            "pct_expressing":
                pct_expressing,

            "mean_raw_count":
                float(
                    x.mean()
                ),

            "mean_log1p_norm":
                float(
                    lognorm.mean()
                ),

            "median_log1p_norm":
                float(
                    np.median(
                        lognorm
                    )
                ),

            "pseudobulk_cpm":
                pseudobulk_cpm,

            "pseudobulk_log2cpm1":
                float(
                    np.log2(
                        pseudobulk_cpm
                        +
                        1.0
                    )
                )
        })


    # ========================================================
    # 9. CLEANUP
    # ========================================================

    del M
    del R

    del identity
    del final_identity

    del final_raw_idx
    del cell_total_umis

    del features
    del feature_symbols
    del symbol_rows

    del selected_rows
    del selected_gene_for_row
    del gene_positions

    gc.collect()


# ============================================================
# 10. CORRECTED EXPRESSION TABLE
# ============================================================

g253_receptor_summary_corrected = pd.DataFrame(
    corrected_receptor_rows
)


g253_receptor_summary_corrected.to_csv(

    OUT
    /
    "GSE253749_Fcer1g_corrected_finalEC_receptor_expression.csv",

    index=False
)


# Canonical object from now onward
g253_receptor_summary = (
    g253_receptor_summary_corrected.copy()
)

g253_summary = (
    g253_receptor_summary_corrected.copy()
)


print(
    "\n======================================"
)

print(
    "CORRECTED SET2 SAMPLE SUMMARY"
)

print(
    "======================================"
)


display(

    g253_receptor_summary[
        [
            "sample_prefix",
            "condition",
            "n_endothelial_cells",
            "total_endothelial_umis"
        ]
    ]
    .drop_duplicates()
)


# ============================================================
# 11. BUILD CORRECTED EFFECT TABLE
# ============================================================

metrics = [

    "pseudobulk_cpm",
    "pseudobulk_log2cpm1",
    "pct_expressing",
    "mean_log1p_norm",
    "raw_sum"
]


wide_parts = []


for metric in metrics:

    tmp = (
        g253_receptor_summary
        .pivot(
            index="gene",
            columns="condition",
            values=metric
        )
    )


    tmp.columns = [

        f"{condition}_{metric}"
        for condition in tmp.columns

    ]


    wide_parts.append(
        tmp
    )


g253_effect_corrected = pd.concat(
    wide_parts,
    axis=1
).reset_index()


for condition in [
    "6h",
    "30h"
]:

    g253_effect_corrected[
        f"{condition}_delta_log2cpm1"
    ] = (

        g253_effect_corrected[
            f"{condition}_pseudobulk_log2cpm1"
        ]

        -

        g253_effect_corrected[
            "control_pseudobulk_log2cpm1"
        ]
    )


    g253_effect_corrected[
        f"{condition}_delta_pct_expressing"
    ] = (

        g253_effect_corrected[
            f"{condition}_pct_expressing"
        ]

        -

        g253_effect_corrected[
            "control_pct_expressing"
        ]
    )


    g253_effect_corrected[
        f"{condition}_delta_mean_log1p_norm"
    ] = (

        g253_effect_corrected[
            f"{condition}_mean_log1p_norm"
        ]

        -

        g253_effect_corrected[
            "control_mean_log1p_norm"
        ]
    )


g253_effect_corrected[
    "max_pct_expressing"
] = (

    g253_effect_corrected[
        [
            "control_pct_expressing",
            "6h_pct_expressing",
            "30h_pct_expressing"
        ]
    ]
    .max(
        axis=1
    )
)


g253_effect_corrected[
    "detected_ge_1pct_any_condition"
] = (

    g253_effect_corrected[
        "max_pct_expressing"
    ]
    >=
    0.01
)


g253_effect_corrected[
    "detected_ge_5pct_any_condition"
] = (

    g253_effect_corrected[
        "max_pct_expressing"
    ]
    >=
    0.05
)


g253_effect_corrected[
    "max_abs_metastatic_delta"
] = (

    g253_effect_corrected[
        [
            "6h_delta_log2cpm1",
            "30h_delta_log2cpm1"
        ]
    ]
    .abs()
    .max(
        axis=1
    )
)


# Canonical effect table from now onward
g253_effect = (
    g253_effect_corrected.copy()
)


g253_effect.to_csv(

    OUT
    /
    "GSE253749_Fcer1g_corrected_receptor_descriptive_effects.csv",

    index=False
)


# ============================================================
# 12. TOP CORRECTED DESCRIPTIVE EFFECTS
# ============================================================

print(
    "\n======================================"
)

print(
    "TOP CORRECTED SET2 RECEPTOR EFFECTS"
)

print(
    "======================================"
)


display(

    g253_effect[
        g253_effect[
            "detected_ge_1pct_any_condition"
        ]
    ]
    .sort_values(
        "max_abs_metastatic_delta",
        ascending=False
    )
    [
        [
            "gene",

            "control_pseudobulk_cpm",
            "6h_pseudobulk_cpm",
            "30h_pseudobulk_cpm",

            "control_pct_expressing",
            "6h_pct_expressing",
            "30h_pct_expressing",

            "6h_delta_log2cpm1",
            "30h_delta_log2cpm1",

            "6h_delta_pct_expressing",
            "30h_delta_pct_expressing",

            "max_abs_metastatic_delta"
        ]
    ]
    .head(
        40
    )
)


# ============================================================
# 13. SELECTED NEUROVASCULAR RECEPTORS
# ============================================================

selected_receptors = [

    "Calcrl",
    "Ramp1",
    "Ramp2",
    "Ramp3",

    "Ntrk2",

    "Kdr",
    "Nrp1",

    "Pdgfrb",

    "Vipr1",
    "Vipr2",

    "Galr1",
    "Galr2",

    "Sstr1",
    "Sstr2",

    "Tacr2"
]


selected_corrected = (

    g253_effect[
        g253_effect[
            "gene"
        ]
        .isin(
            selected_receptors
        )
    ]
    [
        [
            "gene",

            "control_pseudobulk_cpm",
            "6h_pseudobulk_cpm",
            "30h_pseudobulk_cpm",

            "control_pct_expressing",
            "6h_pct_expressing",
            "30h_pct_expressing",

            "6h_delta_log2cpm1",
            "30h_delta_log2cpm1",

            "6h_delta_pct_expressing",
            "30h_delta_pct_expressing"
        ]
    ]
    .sort_values(
        "gene"
    )
)


print(
    "\n======================================"
)

print(
    "SELECTED RECEPTORS — CORRECTED SET2"
)

print(
    "======================================"
)


display(
    selected_corrected
)


# ============================================================
# 14. OLD D5 VS CORRECTED D6E COMPARISON
# ============================================================

if len(
    g253_effect_before_Fcer1g
):

    old_compare = (

        g253_effect_before_Fcer1g[
            [
                "gene",
                "6h_delta_log2cpm1",
                "30h_delta_log2cpm1"
            ]
        ]
        .rename(
            columns={

                "6h_delta_log2cpm1":
                    "old_6h_delta",

                "30h_delta_log2cpm1":
                    "old_30h_delta"
            }
        )
    )


    new_compare = (

        g253_effect[
            [
                "gene",
                "6h_delta_log2cpm1",
                "30h_delta_log2cpm1"
            ]
        ]
        .rename(
            columns={

                "6h_delta_log2cpm1":
                    "corrected_6h_delta",

                "30h_delta_log2cpm1":
                    "corrected_30h_delta"
            }
        )
    )


    correction_compare = (
        old_compare.merge(
            new_compare,
            on="gene",
            how="inner"
        )
    )


    correction_compare[
        "change_in_6h_delta"
    ] = (

        correction_compare[
            "corrected_6h_delta"
        ]

        -

        correction_compare[
            "old_6h_delta"
        ]
    )


    correction_compare[
        "change_in_30h_delta"
    ] = (

        correction_compare[
            "corrected_30h_delta"
        ]

        -

        correction_compare[
            "old_30h_delta"
        ]
    )


    correction_compare[
        "direction_flip_6h"
    ] = (

        np.sign(
            correction_compare[
                "old_6h_delta"
            ]
        )

        !=

        np.sign(
            correction_compare[
                "corrected_6h_delta"
            ]
        )
    )


    correction_compare[
        "direction_flip_30h"
    ] = (

        np.sign(
            correction_compare[
                "old_30h_delta"
            ]
        )

        !=

        np.sign(
            correction_compare[
                "corrected_30h_delta"
            ]
        )
    )


    print(
        "\n======================================"
    )

    print(
        "Fcer1g CORRECTION IMPACT"
    )

    print(
        "======================================"
    )


    print(
        "6h direction flips:",
        int(
            correction_compare[
                "direction_flip_6h"
            ]
            .sum()
        )
    )


    print(
        "30h direction flips:",
        int(
            correction_compare[
                "direction_flip_30h"
            ]
            .sum()
        )
    )


    print(
        "Median absolute change in 6h delta:",
        round(
            float(
                correction_compare[
                    "change_in_6h_delta"
                ]
                .abs()
                .median()
            ),
            5
        )
    )


    print(
        "Median absolute change in 30h delta:",
        round(
            float(
                correction_compare[
                    "change_in_30h_delta"
                ]
                .abs()
                .median()
            ),
            5
        )
    )


    print(
        "\nSelected receptor correction impact:"
    )


    display(

        correction_compare[
            correction_compare[
                "gene"
            ]
            .isin(
                selected_receptors
            )
        ]
        .sort_values(
            "gene"
        )
    )


# ============================================================
# 15. FINAL CHECKS
# ============================================================

assert (
    g253_effect[
        "gene"
    ]
    .nunique()
    ==
    len(
        g253_candidate_receptors
    )
)


assert not g253_effect[
    "gene"
].duplicated().any()


print(
    "\n======================================"
)

print(
    "PHASE D6E CORRECTED SET2 RESPONSE COMPLETE"
)

print(
    "======================================"
)


print(
    "Corrected set2 ECs:"
)


for prefix in sorted(
    groups253
):

    n = (

        g253_receptor_summary[
            g253_receptor_summary[
                "sample_prefix"
            ]
            ==
            prefix
        ][
            "n_endothelial_cells"
        ]
        .iloc[
            0
        ]
    )


    print(
        prefix,
        "|",
        COND_253[
            prefix
        ],
        "|",
        int(
            n
        )
    )


print(
    "\nReceptors >=1% ECs:",
    int(
        g253_effect[
            "detected_ge_1pct_any_condition"
        ]
        .sum()
    )
)


print(
    "Receptors >=5% ECs:",
    int(
        g253_effect[
            "detected_ge_5pct_any_condition"
        ]
        .sum()
    )
)


print(
    "\nGSE253749 remains descriptive: "
    "one pooled sample per condition."
)

In [ ]:
# ============================================================
# PHASE D6F — THREE-EXPERIMENT 6 h RECEPTOR REPLICATION
#
# Independent experiments:
#
#   set1 = GSE235394
#   set2 = GSE253749  [Fcer1g-corrected]
#   set3 = GSE253751
#
# For each experiment:
#
#   final EC pseudobulk CPM
#   prevalence
#   6 h - control delta in log2(CPM + 1)
#   6 h - control prevalence delta
#
# Then determine:
#
#   - receptor present in all experiments
#   - >=1% expression in each experiment
#   - direction concordance across all 3 experiments
#   - median effect across experiments
#
# NO cell-level statistical testing.
# NO claim of significance from n=3 experiments.
# ============================================================

import gc
import gzip
import numpy as np
import pandas as pd

from collections import defaultdict
from scipy.io import mmread


# ============================================================
# 1. PSN-LINKED RECEPTOR UNIVERSE
# ============================================================

psn_receptors = sorted(

    set(
        lr_psn_detected[
            "mouse_receptor"
        ]
        .dropna()
        .astype(str)
    )
)


print(
    "PSN-linked receptor universe:",
    len(psn_receptors)
)


# ============================================================
# 2. CHECK RECEPTOR FEATURE AVAILABILITY IN SET1 + SET3
# ============================================================

rep_feature_sets = {}


for prefix in sorted(
    groups_replication
):

    features = pd.read_csv(

        groups_replication[
            prefix
        ][
            "features"
        ],

        sep="\t",
        header=None,
        compression="gzip"
    )


    if features.shape[1] >= 2:

        symbols = (
            features.iloc[
                :,
                1
            ]
            .astype(str)
        )

    else:

        symbols = (
            features.iloc[
                :,
                0
            ]
            .astype(str)
        )


    rep_feature_sets[
        prefix
    ] = set(
        symbols
    )


    print(
        prefix,
        "| PSN receptors present:",
        len(
            set(
                psn_receptors
            )
            &
            rep_feature_sets[
                prefix
            ]
        )
    )


common_rep_features = set.intersection(
    *rep_feature_sets.values()
)


rep_candidate_receptors = sorted(

    set(
        psn_receptors
    )

    &

    common_rep_features
)


print(
    "\nPSN receptors comparable in all set1/set3 samples:",
    len(rep_candidate_receptors)
)


# ============================================================
# 3. PROCESS SET1 + SET3 FINAL ECs
# ============================================================

rep_receptor_rows = []


for prefix in sorted(
    groups_replication
):

    fg = groups_replication[
        prefix
    ]

    dataset = DATASET_REPLICATION[
        prefix
    ]

    condition = COND_REPLICATION[
        prefix
    ]


    print(
        "\n======================================"
    )

    print(
        dataset,
        "|",
        prefix,
        "|",
        condition
    )

    print(
        "======================================"
    )


    # --------------------------------------------------------
    # Restore identity/filter table
    # --------------------------------------------------------

    if (
        "rep_identity_tables" in globals()
        and
        prefix in rep_identity_tables
    ):

        identity = (
            rep_identity_tables[
                prefix
            ]
            .copy()
        )

    else:

        identity = pd.read_csv(

            CACHE
            /
            f"{prefix}_replication_identity_audit.csv.gz",

            compression="gzip"
        )


    final_identity = (

        identity[
            identity[
                "proposed_final_endothelial"
            ]
            .astype(bool)
        ]
        .copy()
    )


    final_raw_idx = (
        final_identity[
            "raw_column_index"
        ]
        .to_numpy(
            dtype=int
        )
    )


    cell_total_umis = (
        final_identity[
            "total_counts"
        ]
        .to_numpy(
            dtype=float
        )
    )


    print(
        "Final EC:",
        len(final_identity)
    )


    if len(
        final_identity
    ) == 0:

        raise RuntimeError(
            f"{prefix}: zero final endothelial cells."
        )


    # ========================================================
    # 4. FEATURES
    # ========================================================

    features = pd.read_csv(

        fg[
            "features"
        ],

        sep="\t",
        header=None,
        compression="gzip"
    )


    if features.shape[1] >= 2:

        feature_symbols = (
            features.iloc[
                :,
                1
            ]
            .astype(str)
            .tolist()
        )

    else:

        feature_symbols = (
            features.iloc[
                :,
                0
            ]
            .astype(str)
            .tolist()
        )


    symbol_rows = defaultdict(
        list
    )


    for row_i, gene in enumerate(
        feature_symbols
    ):

        if gene in rep_candidate_receptors:

            symbol_rows[
                gene
            ].append(
                row_i
            )


    missing = [

        gene
        for gene in rep_candidate_receptors
        if gene not in symbol_rows

    ]


    if missing:

        raise RuntimeError(
            f"{prefix}: missing receptor rows: {missing[:20]}"
        )


    # ========================================================
    # 5. RAW MATRIX
    # ========================================================

    with gzip.open(
        fg[
            "matrix"
        ],
        "rb"
    ) as fh:

        M = (
            mmread(
                fh
            )
            .tocsr()
        )


    if (
        M.shape[0]
        !=
        len(
            feature_symbols
        )
        and
        M.shape[1]
        ==
        len(
            feature_symbols
        )
    ):

        M = (
            M.T
            .tocsr()
        )


    if M.shape[0] != len(
        feature_symbols
    ):

        raise RuntimeError(
            f"{prefix}: matrix/feature mismatch."
        )


    # ========================================================
    # 6. EXTRACT ALL RECEPTOR ROWS AT ONCE
    # ========================================================

    selected_rows = []

    selected_row_genes = []


    for gene in rep_candidate_receptors:

        for row_i in symbol_rows[
            gene
        ]:

            selected_rows.append(
                row_i
            )

            selected_row_genes.append(
                gene
            )


    R = (

        M[
            selected_rows,
            :
        ][
            :,
            final_raw_idx
        ]
        .tocsr()
    )


    local_positions = defaultdict(
        list
    )


    for local_i, gene in enumerate(
        selected_row_genes
    ):

        local_positions[
            gene
        ].append(
            local_i
        )


    total_ec_umis = float(
        cell_total_umis.sum()
    )


    print(
        "Final EC UMIs:",
        int(total_ec_umis)
    )


    # ========================================================
    # 7. RECEPTOR METRICS
    # ========================================================

    for gene in rep_candidate_receptors:

        x = np.asarray(

            R[
                local_positions[
                    gene
                ],
                :
            ]
            .sum(
                axis=0
            )

        ).ravel().astype(
            float
        )


        raw_sum = float(
            x.sum()
        )


        pct_expressing = float(
            (
                x > 0
            )
            .mean()
        )


        pseudobulk_cpm = float(

            1e6
            *
            raw_sum
            /
            total_ec_umis
        )


        rep_receptor_rows.append({

            "dataset":
                dataset,

            "sample_prefix":
                prefix,

            "condition":
                condition,

            "gene":
                gene,

            "n_endothelial_cells":
                int(
                    len(x)
                ),

            "total_endothelial_umis":
                total_ec_umis,

            "raw_sum":
                raw_sum,

            "pct_expressing":
                pct_expressing,

            "pseudobulk_cpm":
                pseudobulk_cpm,

            "pseudobulk_log2cpm1":
                float(
                    np.log2(
                        pseudobulk_cpm
                        +
                        1.0
                    )
                )
        })


    # ========================================================
    # 8. CLEANUP SAMPLE
    # ========================================================

    del M
    del R

    del identity
    del final_identity
    del final_raw_idx
    del cell_total_umis

    del features
    del feature_symbols
    del symbol_rows

    del selected_rows
    del selected_row_genes
    del local_positions

    gc.collect()


# ============================================================
# 9. SET1 + SET3 RECEPTOR SUMMARY
# ============================================================

rep_receptor_summary = pd.DataFrame(
    rep_receptor_rows
)


rep_receptor_summary.to_csv(

    OUT
    /
    "GSE235394_GSE253751_finalEC_receptor_expression.csv",

    index=False
)


print(
    "\n======================================"
)

print(
    "SET1 / SET3 FINAL EC SAMPLE SUMMARY"
)

print(
    "======================================"
)


display(

    rep_receptor_summary[
        [
            "dataset",
            "sample_prefix",
            "condition",
            "n_endothelial_cells",
            "total_endothelial_umis"
        ]
    ]
    .drop_duplicates()
)


# ============================================================
# 10. BUILD SET1 + SET3 6 h EFFECT TABLES
# ============================================================

rep_effect_parts = []


for dataset in sorted(
    rep_receptor_summary[
        "dataset"
    ]
    .unique()
):

    sub = (

        rep_receptor_summary[
            rep_receptor_summary[
                "dataset"
            ]
            ==
            dataset
        ]
        .copy()
    )


    cpm = sub.pivot(
        index="gene",
        columns="condition",
        values="pseudobulk_cpm"
    )


    logcpm = sub.pivot(
        index="gene",
        columns="condition",
        values="pseudobulk_log2cpm1"
    )


    pct = sub.pivot(
        index="gene",
        columns="condition",
        values="pct_expressing"
    )


    if not {
        "control",
        "6h"
    }.issubset(
        cpm.columns
    ):

        raise RuntimeError(
            f"{dataset}: control/6h pair incomplete."
        )


    effect = pd.DataFrame({

        "gene":
            cpm.index,

        "dataset":
            dataset,

        "control_cpm":
            cpm[
                "control"
            ].values,

        "6h_cpm":
            cpm[
                "6h"
            ].values,

        "control_pct":
            pct[
                "control"
            ].values,

        "6h_pct":
            pct[
                "6h"
            ].values,

        "delta_log2cpm1":
            (
                logcpm[
                    "6h"
                ]
                -
                logcpm[
                    "control"
                ]
            ).values,

        "delta_pct":
            (
                pct[
                    "6h"
                ]
                -
                pct[
                    "control"
                ]
            ).values
    })


    effect[
        "max_pair_pct"
    ] = (

        effect[
            [
                "control_pct",
                "6h_pct"
            ]
        ]
        .max(
            axis=1
        )
    )


    effect[
        "detected_ge_1pct"
    ] = (

        effect[
            "max_pair_pct"
        ]
        >=
        0.01
    )


    effect[
        "detected_ge_5pct"
    ] = (

        effect[
            "max_pair_pct"
        ]
        >=
        0.05
    )


    rep_effect_parts.append(
        effect
    )


rep_6h_effects = pd.concat(
    rep_effect_parts,
    ignore_index=True
)


# ============================================================
# 11. ADD CORRECTED SET2 6 h EFFECTS
# ============================================================

set2_effect = pd.DataFrame({

    "gene":
        g253_effect[
            "gene"
        ],

    "dataset":
        "set2_GSE253749",

    "control_cpm":
        g253_effect[
            "control_pseudobulk_cpm"
        ],

    "6h_cpm":
        g253_effect[
            "6h_pseudobulk_cpm"
        ],

    "control_pct":
        g253_effect[
            "control_pct_expressing"
        ],

    "6h_pct":
        g253_effect[
            "6h_pct_expressing"
        ],

    "delta_log2cpm1":
        g253_effect[
            "6h_delta_log2cpm1"
        ],

    "delta_pct":
        g253_effect[
            "6h_delta_pct_expressing"
        ]
})


set2_effect[
    "max_pair_pct"
] = (

    set2_effect[
        [
            "control_pct",
            "6h_pct"
        ]
    ]
    .max(
        axis=1
    )
)


set2_effect[
    "detected_ge_1pct"
] = (

    set2_effect[
        "max_pair_pct"
    ]
    >=
    0.01
)


set2_effect[
    "detected_ge_5pct"
] = (

    set2_effect[
        "max_pair_pct"
    ]
    >=
    0.05
)


all_6h_effects = pd.concat(

    [
        rep_6h_effects,
        set2_effect
    ],

    ignore_index=True
)


# ============================================================
# 12. REQUIRE RECEPTOR TO EXIST IN ALL THREE EXPERIMENTS
# ============================================================

experiment_counts = (

    all_6h_effects
    .groupby(
        "gene"
    )[
        "dataset"
    ]
    .nunique()
)


common_three_genes = (

    experiment_counts[
        experiment_counts
        ==
        3
    ]
    .index
)


all_6h_effects = (

    all_6h_effects[
        all_6h_effects[
            "gene"
        ]
        .isin(
            common_three_genes
        )
    ]
    .copy()
)


print(
    "\nReceptors represented in all three experiments:",
    len(common_three_genes)
)


# ============================================================
# 13. WIDE THREE-EXPERIMENT REPLICATION TABLE
# ============================================================

delta_wide = (

    all_6h_effects
    .pivot(
        index="gene",
        columns="dataset",
        values="delta_log2cpm1"
    )
)


pct_delta_wide = (

    all_6h_effects
    .pivot(
        index="gene",
        columns="dataset",
        values="delta_pct"
    )
)


detect1_wide = (

    all_6h_effects
    .pivot(
        index="gene",
        columns="dataset",
        values="detected_ge_1pct"
    )
)


detect5_wide = (

    all_6h_effects
    .pivot(
        index="gene",
        columns="dataset",
        values="detected_ge_5pct"
    )
)


expected_datasets = [
    "set1_GSE235394",
    "set2_GSE253749",
    "set3_GSE253751"
]


for dataset in expected_datasets:

    if dataset not in delta_wide.columns:

        raise RuntimeError(
            f"Missing experiment: {dataset}"
        )


replication_table = pd.DataFrame(
    index=delta_wide.index
)


# ============================================================
# 14. COPY INDIVIDUAL EXPERIMENT EFFECTS
# ============================================================

for dataset in expected_datasets:

    replication_table[
        f"{dataset}_delta_log2cpm1"
    ] = delta_wide[
        dataset
    ]


    replication_table[
        f"{dataset}_delta_pct"
    ] = pct_delta_wide[
        dataset
    ]


    replication_table[
        f"{dataset}_ge1pct"
    ] = detect1_wide[
        dataset
    ].astype(
        bool
    )


    replication_table[
        f"{dataset}_ge5pct"
    ] = detect5_wide[
        dataset
    ].astype(
        bool
    )


# ============================================================
# 15. DIRECTIONAL REPLICATION METRICS
# ============================================================

delta_matrix = delta_wide[
    expected_datasets
].to_numpy(
    dtype=float
)


replication_table[
    "n_positive"
] = (
    delta_matrix
    >
    0
).sum(
    axis=1
)


replication_table[
    "n_negative"
] = (
    delta_matrix
    <
    0
).sum(
    axis=1
)


replication_table[
    "all_three_same_direction"
] = (

    (
        replication_table[
            "n_positive"
        ]
        ==
        3
    )

    |

    (
        replication_table[
            "n_negative"
        ]
        ==
        3
    )
)


replication_table[
    "two_or_more_same_direction"
] = (

    replication_table[
        [
            "n_positive",
            "n_negative"
        ]
    ]
    .max(
        axis=1
    )
    >=
    2
)


replication_table[
    "median_6h_delta"
] = np.median(
    delta_matrix,
    axis=1
)


replication_table[
    "mean_6h_delta"
] = np.mean(
    delta_matrix,
    axis=1
)


replication_table[
    "min_abs_6h_delta"
] = np.min(
    np.abs(
        delta_matrix
    ),
    axis=1
)


replication_table[
    "max_abs_6h_delta"
] = np.max(
    np.abs(
        delta_matrix
    ),
    axis=1
)


replication_table[
    "expressed_ge1pct_all_three"
] = (

    detect1_wide[
        expected_datasets
    ]
    .all(
        axis=1
    )
)


replication_table[
    "expressed_ge5pct_all_three"
] = (

    detect5_wide[
        expected_datasets
    ]
    .all(
        axis=1
    )
)


# ============================================================
# 16. ROBUST DIRECTIONAL REPLICATION FLAG
#
# This is NOT a statistical-significance call.
#
# Requirements:
#
#   expression >=1% in every experiment
#   same sign in all 3 independent experiments
# ============================================================

replication_table[
    "directionally_replicated_ge1pct"
] = (

    replication_table[
        "expressed_ge1pct_all_three"
    ]

    &

    replication_table[
        "all_three_same_direction"
    ]
)


replication_table = (

    replication_table
    .reset_index()
)


# ============================================================
# 17. ADD PSN LIGAND CONTEXT
# ============================================================

lr_context = (

    lr_psn_detected[
        [
            "mouse_ligand",
            "mouse_receptor"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={

            "mouse_receptor":
                "gene"
        }
    )
)


ligand_context = (

    lr_context
    .groupby(
        "gene"
    )[
        "mouse_ligand"
    ]
    .apply(
        lambda x:
            ";".join(
                sorted(
                    set(
                        x.astype(str)
                    )
                )
            )
    )
    .reset_index(
        name="PSN_ligands"
    )
)


replication_table = (

    replication_table
    .merge(
        ligand_context,
        on="gene",
        how="left"
    )
)


# ============================================================
# 18. SAVE
# ============================================================

all_6h_effects.to_csv(

    OUT
    /
    "three_experiment_6h_receptor_effects_long.csv",

    index=False
)


replication_table.to_csv(

    OUT
    /
    "three_experiment_6h_receptor_directional_replication.csv",

    index=False
)


# ============================================================
# 19. SUMMARY
# ============================================================

print(
    "\n======================================"
)

print(
    "THREE-EXPERIMENT 6 h REPLICATION SUMMARY"
)

print(
    "======================================"
)


print(
    "Common receptors:",
    len(
        replication_table
    )
)


print(
    ">=1% expression in all three experiments:",
    int(
        replication_table[
            "expressed_ge1pct_all_three"
        ]
        .sum()
    )
)


print(
    "Same direction in all three:",
    int(
        replication_table[
            "all_three_same_direction"
        ]
        .sum()
    )
)


print(
    "Same direction + >=1% expression in all three:",
    int(
        replication_table[
            "directionally_replicated_ge1pct"
        ]
        .sum()
    )
)


# ============================================================
# 20. STRONGEST DIRECTIONALLY REPLICATED EFFECTS
#
# Sorted only for inspection.
# This is NOT the final biological ranking.
# ============================================================

print(
    "\n======================================"
)

print(
    "DIRECTIONALLY REPLICATED RECEPTORS"
)

print(
    "======================================"
)


replicated_view = (

    replication_table[
        replication_table[
            "directionally_replicated_ge1pct"
        ]
    ]
    .copy()
)


replicated_view[
    "abs_median_6h_delta"
] = (

    replicated_view[
        "median_6h_delta"
    ]
    .abs()
)


display(

    replicated_view
    .sort_values(
        [
            "abs_median_6h_delta",
            "min_abs_6h_delta"
        ],
        ascending=False
    )
    [
        [
            "gene",
            "PSN_ligands",

            "set1_GSE235394_delta_log2cpm1",
            "set2_GSE253749_delta_log2cpm1",
            "set3_GSE253751_delta_log2cpm1",

            "median_6h_delta",
            "min_abs_6h_delta",

            "set1_GSE235394_delta_pct",
            "set2_GSE253749_delta_pct",
            "set3_GSE253751_delta_pct",

            "expressed_ge5pct_all_three"
        ]
    ]
    .head(
        60
    )
)


# ============================================================
# 21. SELECTED NEUROVASCULAR RECEPTORS ACROSS ALL THREE
# ============================================================

selected_receptors = [

    "Calcrl",
    "Ramp1",
    "Ramp2",
    "Ramp3",

    "Ntrk2",

    "Kdr",
    "Nrp1",

    "Pdgfrb",

    "Vipr1",
    "Vipr2",

    "Galr1",
    "Galr2",

    "Sstr1",
    "Sstr2",

    "Tacr2"
]


print(
    "\n======================================"
)

print(
    "SELECTED RECEPTORS — THREE EXPERIMENTS"
)

print(
    "======================================"
)


selected_three = (

    replication_table[
        replication_table[
            "gene"
        ]
        .isin(
            selected_receptors
        )
    ]
    [
        [
            "gene",
            "PSN_ligands",

            "set1_GSE235394_delta_log2cpm1",
            "set2_GSE253749_delta_log2cpm1",
            "set3_GSE253751_delta_log2cpm1",

            "median_6h_delta",

            "set1_GSE235394_ge1pct",
            "set2_GSE253749_ge1pct",
            "set3_GSE253751_ge1pct",

            "all_three_same_direction",
            "directionally_replicated_ge1pct"
        ]
    ]
    .sort_values(
        "gene"
    )
)


display(
    selected_three
)


# ============================================================
# 22. SANITY CHECKS
# ============================================================

assert not replication_table[
    "gene"
].duplicated().any()


assert (
    all_6h_effects[
        "dataset"
    ]
    .nunique()
    ==
    3
)


# ============================================================
# 23. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE D6F THREE-EXPERIMENT 6 h REPLICATION COMPLETE"
)


print(
    "\nInterpretation rule:"
)


print(
    "Concordance across the three independent experiments "
    "is directional replication, not statistical significance."
)

In [ ]:
# ============================================================
# PHASE D7A — INVENTORY CURRENT PHASE-C AGING EVIDENCE
#
# Before merging metastatic replication with aging evidence,
# identify the EXACT current Phase-C objects/files in memory.
#
# This prevents accidentally using obsolete exploratory age
# tables from earlier notebook versions.
#
# NO scoring.
# NO ranking.
# NO biological filtering.
# ============================================================

import pandas as pd
from pathlib import Path


print("======================================")
print("PHASE D7A — CURRENT AGING-EVIDENCE INVENTORY")
print("======================================")


# ============================================================
# 1. INSPECT B4 PSN-LR TABLE SCHEMA
# ============================================================

print("\n======================================")
print("lr_psn_detected SCHEMA")
print("======================================")

print(
    "Shape:",
    lr_psn_detected.shape
)

print("Columns:")

for col in lr_psn_detected.columns:
    print(
        " -",
        col
    )


# ============================================================
# 2. FIND DATAFRAMES IN MEMORY THAT LOOK LIKE AGE ANALYSES
# ============================================================

candidate_age_objects = []

age_keywords = {
    "age",
    "aged",
    "young",
    "hedges",
    "welch",
    "fdr",
    "qvalue",
    "padj",
    "composition",
    "adjusted",
    "pseudobulk"
}


for name, obj in list(
    globals().items()
):

    if not isinstance(
        obj,
        pd.DataFrame
    ):
        continue

    columns_lower = [
        str(c).lower()
        for c in obj.columns
    ]

    text = (
        name.lower()
        +
        " "
        +
        " ".join(
            columns_lower
        )
    )

    hits = sorted(
        [
            keyword
            for keyword in age_keywords
            if keyword in text
        ]
    )

    # Require something age-related rather than merely FDR.
    if any(
        keyword in text
        for keyword in [
            "age",
            "aged",
            "young",
            "pseudobulk"
        ]
    ):

        candidate_age_objects.append({
            "object_name":
                name,

            "shape":
                str(
                    obj.shape
                ),

            "keyword_hits":
                ", ".join(
                    hits
                ),

            "columns":
                " | ".join(
                    map(
                        str,
                        obj.columns
                    )
                )
        })


age_object_inventory = pd.DataFrame(
    candidate_age_objects
)


print("\n======================================")
print("AGE-RELATED DATAFRAMES CURRENTLY IN MEMORY")
print("======================================")


if len(
    age_object_inventory
):

    display(
        age_object_inventory
        .sort_values(
            "object_name"
        )
    )

else:

    print(
        "No age-related DataFrames detected in memory."
    )


# ============================================================
# 3. PRINT SMALL PREVIEWS OF MOST RELEVANT OBJECTS
# ============================================================

print("\n======================================")
print("LIKELY RECEPTOR-AGING TABLE PREVIEWS")
print("======================================")


previewed = 0


for row in candidate_age_objects:

    name = row[
        "object_name"
    ]

    obj = globals()[
        name
    ]

    cols_lower = {
        str(c).lower()
        for c in obj.columns
    }

    has_gene = (
        "gene"
        in
        cols_lower
    )

    has_age_field = any(
        (
            "age"
            in
            str(c).lower()
        )
        or
        (
            "young"
            in
            str(c).lower()
        )
        or
        (
            "aged"
            in
            str(c).lower()
        )
        for c in obj.columns
    )


    if (
        has_gene
        and
        has_age_field
    ):

        print(
            "\n--------------------------------------"
        )

        print(
            "OBJECT:",
            name
        )

        print(
            "SHAPE:",
            obj.shape
        )

        print("COLUMNS:")

        for col in obj.columns:
            print(
                " ",
                col
            )

        print("\nHEAD:")

        display(
            obj.head(
                10
            )
        )

        previewed += 1

        if previewed >= 12:
            break


# ============================================================
# 4. INVENTORY SAVED PHASE-C OUTPUT FILES
# ============================================================

print("\n======================================")
print("GSE124872 / AGING FILES IN OUT")
print("======================================")


aging_files = sorted(
    [
        p
        for p in Path(
            OUT
        ).glob(
            "*.csv"
        )
        if (
            "124872"
            in
            p.name
        )
        or
        (
            "age"
            in
            p.name.lower()
        )
        or
        (
            "aging"
            in
            p.name.lower()
        )
        or
        (
            "composition"
            in
            p.name.lower()
        )
    ],
    key=lambda x:
        x.name
)


for path in aging_files:

    try:

        tmp = pd.read_csv(
            path,
            nrows=3
        )

        print(
            "\nFILE:",
            path.name
        )

        print(
            "Columns:",
            list(
                tmp.columns
            )
        )

    except Exception as exc:

        print(
            "\nFILE:",
            path.name,
            "| READ ERROR:",
            repr(
                exc
            )
        )


# ============================================================
# 5. CONFIRM D6F TABLE IS AVAILABLE
# ============================================================

print("\n======================================")
print("D6F REPLICATION TABLE")
print("======================================")


print(
    "Shape:",
    replication_table.shape
)


print(
    "Directionally replicated >=1%:",
    int(
        replication_table[
            "directionally_replicated_ge1pct"
        ]
        .sum()
    )
)


# ============================================================
# 6. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE D7A AGING-EVIDENCE INVENTORY COMPLETE"
)

In [ ]:
# ============================================================
# PHASE D7B — INTERSECT REPLICATED 6 h RESPONSE WITH AGING
#
# Starting universe:
#
#   receptors that:
#     - are PSN-linked
#     - are expressed in >=1% ECs in all 3 metastatic datasets
#     - change in the SAME direction at 6 h in all 3
#
# Aging evidence is kept compartment-specific:
#
#   1. pooled blood endothelial cells
#   2. composition-adjusted blood-EC sensitivity
#   3. smooth-muscle / mural-like receptor support
#   4. exploratory mural aging
#
# IMPORTANT:
#
# - pooled blood-EC age association != direct age regulation
# - composition-adjusted analysis is a sensitivity analysis
# - mural analysis is exploratory because replicate coverage is small
# - no integrated ranking yet
# ============================================================

import numpy as np
import pandas as pd


print(
    "======================================"
)

print(
    "PHASE D7B — METASTASIS × AGING CONVERGENCE"
)

print(
    "======================================"
)


# ============================================================
# 1. START ONLY FROM THREE-EXPERIMENT REPLICATED RECEPTORS
# ============================================================

convergence = (

    replication_table[
        replication_table[
            "directionally_replicated_ge1pct"
        ]
        .astype(bool)
    ]
    .copy()
)


print(
    "\nThree-experiment replicated metastatic receptors:",
    len(convergence)
)


if len(convergence) != 60:

    print(
        "WARNING: D6F previously reported 60."
    )


# ============================================================
# 2. PRIMARY BLOOD-EC AGE TABLE
#
# Prefer the current in-memory object identified by D7A.
# ============================================================

if (
    "age_effect_overall" in globals()
    and
    isinstance(age_effect_overall, pd.DataFrame)
    and
    len(age_effect_overall) > 0
):

    blood_age = (
        age_effect_overall.copy()
    )

else:

    blood_age = pd.read_csv(

        OUT
        /
        "GSE124872_vascular_receptor_age_effect_overall.csv"
    )


# Restrict explicitly to pooled blood EC if celltype exists.
if "celltype" in blood_age.columns:

    blood_age = (

        blood_age[
            blood_age[
                "celltype"
            ]
            .astype(str)
            ==
            "bloodEC_all"
        ]
        .copy()
    )


if blood_age[
    "gene"
].duplicated().any():

    raise RuntimeError(
        "Blood-EC age table contains duplicate genes."
    )


print(
    "Blood-EC age receptors:",
    len(blood_age)
)


# ============================================================
# 3. KEEP / RENAME PRIMARY AGE FIELDS
# ============================================================

blood_keep = [

    "gene",

    "n_young_animals",
    "n_aged_animals",

    "n_young_detected",
    "n_aged_detected",

    "young_mean_log2cpm",
    "aged_mean_log2cpm",

    "age_delta_log2cpm",

    "ci95_low",
    "ci95_high",

    "young_mean_pct_expressing",
    "aged_mean_pct_expressing",
    "age_delta_pct_expressing",

    "hedges_g",

    "p_value",
    "testable",
    "fdr_bh",

    "ci_excludes_zero"
]


blood_keep = [

    col
    for col in blood_keep
    if col in blood_age.columns

]


blood_age = (

    blood_age[
        blood_keep
    ]
    .copy()
)


blood_rename = {

    col:
        f"blood_age_{col}"

    for col in blood_age.columns

    if col != "gene"
}


blood_age = blood_age.rename(
    columns=blood_rename
)


# ============================================================
# 4. COMPOSITION-ADJUSTED BLOOD-EC SENSITIVITY
# ============================================================

composition_path = (

    OUT
    /
    "GSE124872_bloodEC_composition_adjusted_age_sensitivity.csv"
)


if not composition_path.exists():

    raise FileNotFoundError(
        composition_path
    )


composition = pd.read_csv(
    composition_path
)


if composition[
    "gene"
].duplicated().any():

    raise RuntimeError(
        "Composition-adjusted table contains duplicate genes."
    )


composition_keep = [

    "gene",

    "composition_adjusted_age_coef",
    "composition_adjusted_age_se_HC3",

    "composition_adjusted_ci95_low",
    "composition_adjusted_ci95_high",

    "composition_adjusted_p_value",
    "composition_adjusted_fdr_bh",

    "model_r2",
    "n_mice",

    "same_direction",
    "age_signal_class"
]


composition_keep = [

    col
    for col in composition_keep
    if col in composition.columns

]


composition = (

    composition[
        composition_keep
    ]
    .copy()
)


composition = composition.rename(
    columns={

        "composition_adjusted_age_coef":
            "comp_age_coef",

        "composition_adjusted_age_se_HC3":
            "comp_age_se_HC3",

        "composition_adjusted_ci95_low":
            "comp_age_ci95_low",

        "composition_adjusted_ci95_high":
            "comp_age_ci95_high",

        "composition_adjusted_p_value":
            "comp_age_p_value",

        "composition_adjusted_fdr_bh":
            "comp_age_fdr_bh",

        "model_r2":
            "comp_model_r2",

        "n_mice":
            "comp_n_mice",

        "same_direction":
            "comp_same_direction",

        "age_signal_class":
            "age_signal_class"
    }
)


print(
    "Composition-adjusted receptors:",
    len(composition)
)


# ============================================================
# 5. MURAL RECEPTOR SUPPORT
# ============================================================

mural_support_path = (

    OUT
    /
    "GSE124872_mural_receptor_support.csv"
)


if not mural_support_path.exists():

    raise FileNotFoundError(
        mural_support_path
    )


mural_support = pd.read_csv(
    mural_support_path
)


if mural_support[
    "gene"
].duplicated().any():

    raise RuntimeError(
        "Mural support table contains duplicate genes."
    )


mural_support = mural_support.rename(
    columns={

        col:
            f"mural_support_{col}"

        for col in mural_support.columns

        if col != "gene"
    }
)


# ============================================================
# 6. EXPLORATORY MURAL AGING
# ============================================================

mural_age_path = (

    OUT
    /
    "GSE124872_mural_mouselevel_exploratory_age_statistics.csv"
)


if not mural_age_path.exists():

    raise FileNotFoundError(
        mural_age_path
    )


mural_age = pd.read_csv(
    mural_age_path
)


if mural_age[
    "gene"
].duplicated().any():

    raise RuntimeError(
        "Mural age table contains duplicate genes."
    )


mural_age = mural_age.rename(
    columns={

        col:
            f"mural_age_{col}"

        for col in mural_age.columns

        if col != "gene"
    }
)


print(
    "Mural age receptors:",
    len(mural_age)
)


# ============================================================
# 7. MERGE ALL EVIDENCE
# ============================================================

convergence = (

    convergence

    .merge(
        blood_age,
        on="gene",
        how="left"
    )

    .merge(
        composition,
        on="gene",
        how="left"
    )

    .merge(
        mural_support,
        on="gene",
        how="left"
    )

    .merge(
        mural_age,
        on="gene",
        how="left"
    )
)


# ============================================================
# 8. BLOOD-EC AGE FLAGS
# ============================================================

convergence[
    "blood_age_available"
] = (

    convergence[
        "blood_age_age_delta_log2cpm"
    ]
    .notna()
)


if "blood_age_testable" in convergence.columns:

    convergence[
        "blood_age_testable"
    ] = (

        convergence[
            "blood_age_testable"
        ]
        .fillna(False)
        .astype(bool)
    )


convergence[
    "blood_age_fdr_lt_005"
] = (

    convergence[
        "blood_age_fdr_bh"
    ]
    .lt(
        0.05
    )
    .fillna(False)
)


convergence[
    "blood_age_fdr_lt_010"
] = (

    convergence[
        "blood_age_fdr_bh"
    ]
    .lt(
        0.10
    )
    .fillna(False)
)


convergence[
    "blood_age_metastasis_same_direction"
] = (

    convergence[
        "blood_age_age_delta_log2cpm"
    ]
    .notna()

    &

    (
        np.sign(
            convergence[
                "blood_age_age_delta_log2cpm"
            ]
        )

        ==

        np.sign(
            convergence[
                "median_6h_delta"
            ]
        )
    )
)


# ============================================================
# 9. COMPOSITION-ADJUSTED FLAGS
# ============================================================

convergence[
    "comp_age_available"
] = (

    convergence[
        "comp_age_coef"
    ]
    .notna()
)


convergence[
    "comp_age_fdr_lt_005"
] = (

    convergence[
        "comp_age_fdr_bh"
    ]
    .lt(
        0.05
    )
    .fillna(False)
)


convergence[
    "comp_age_fdr_lt_010"
] = (

    convergence[
        "comp_age_fdr_bh"
    ]
    .lt(
        0.10
    )
    .fillna(False)
)


convergence[
    "comp_age_metastasis_same_direction"
] = (

    convergence[
        "comp_age_coef"
    ]
    .notna()

    &

    (
        np.sign(
            convergence[
                "comp_age_coef"
            ]
        )

        ==

        np.sign(
            convergence[
                "median_6h_delta"
            ]
        )
    )
)


# ============================================================
# 10. MURAL FLAGS
# ============================================================

if (
    "mural_support_n_mice_detected"
    in
    convergence.columns
):

    convergence[
        "mural_receptor_detected"
    ] = (

        convergence[
            "mural_support_n_mice_detected"
        ]
        .fillna(0)
        >
        0
    )

else:

    convergence[
        "mural_receptor_detected"
    ] = False


if (
    "mural_age_testable"
    in
    convergence.columns
):

    convergence[
        "mural_age_testable"
    ] = (

        convergence[
            "mural_age_testable"
        ]
        .fillna(False)
        .astype(bool)
    )


convergence[
    "mural_age_fdr_lt_005"
] = (

    convergence[
        "mural_age_fdr_bh"
    ]
    .lt(
        0.05
    )
    .fillna(False)
)


convergence[
    "mural_age_fdr_lt_010"
] = (

    convergence[
        "mural_age_fdr_bh"
    ]
    .lt(
        0.10
    )
    .fillna(False)
)


convergence[
    "mural_age_metastasis_same_direction"
] = (

    convergence[
        "mural_age_age_delta_log2cpm"
    ]
    .notna()

    &

    (
        np.sign(
            convergence[
                "mural_age_age_delta_log2cpm"
            ]
        )

        ==

        np.sign(
            convergence[
                "median_6h_delta"
            ]
        )
    )
)


# ============================================================
# 11. SAVE COMPLETE CONVERGENCE TABLE
# ============================================================

convergence.to_csv(

    OUT
    /
    "PSN_receptor_metastasis_replication_with_aging_evidence.csv",

    index=False
)


# ============================================================
# 12. SUMMARY COUNTS
# ============================================================

print(
    "\n======================================"
)

print(
    "CONVERGENCE SUMMARY"
)

print(
    "======================================"
)


print(
    "Replicated metastatic receptors:",
    len(convergence)
)


print(
    "With blood-EC age record:",
    int(
        convergence[
            "blood_age_available"
        ]
        .sum()
    )
)


print(
    "Blood-EC age FDR <0.05:",
    int(
        convergence[
            "blood_age_fdr_lt_005"
        ]
        .sum()
    )
)


print(
    "Blood-EC age FDR <0.10:",
    int(
        convergence[
            "blood_age_fdr_lt_010"
        ]
        .sum()
    )
)


print(
    "Composition-adjusted age FDR <0.05:",
    int(
        convergence[
            "comp_age_fdr_lt_005"
        ]
        .sum()
    )
)


print(
    "Composition-adjusted age FDR <0.10:",
    int(
        convergence[
            "comp_age_fdr_lt_010"
        ]
        .sum()
    )
)


print(
    "Detected in mural compartment:",
    int(
        convergence[
            "mural_receptor_detected"
        ]
        .sum()
    )
)


print(
    "Mural age FDR <0.05:",
    int(
        convergence[
            "mural_age_fdr_lt_005"
        ]
        .sum()
    )
)


print(
    "Mural age FDR <0.10:",
    int(
        convergence[
            "mural_age_fdr_lt_010"
        ]
        .sum()
    )
)


# ============================================================
# 13. BLOOD-EC AGE INTERSECTION
#
# Inspection only — not a final ranking.
# ============================================================

print(
    "\n======================================"
)

print(
    "REPLICATED METASTASIS + BLOOD-EC AGE SIGNAL"
)

print(
    "======================================"
)


blood_view = (

    convergence[
        convergence[
            "blood_age_fdr_lt_010"
        ]
    ]
    .copy()
)


blood_columns = [

    "gene",
    "PSN_ligands",

    "set1_GSE235394_delta_log2cpm1",
    "set2_GSE253749_delta_log2cpm1",
    "set3_GSE253751_delta_log2cpm1",

    "median_6h_delta",

    "blood_age_age_delta_log2cpm",
    "blood_age_ci95_low",
    "blood_age_ci95_high",

    "blood_age_hedges_g",
    "blood_age_p_value",
    "blood_age_fdr_bh",

    "blood_age_metastasis_same_direction",

    "comp_age_coef",
    "comp_age_p_value",
    "comp_age_fdr_bh",

    "comp_age_metastasis_same_direction",

    "age_signal_class"
]


blood_columns = [

    c
    for c in blood_columns
    if c in blood_view.columns

]


display(

    blood_view[
        blood_columns
    ]
    .sort_values(
        [
            "blood_age_fdr_bh",
            "gene"
        ],
        na_position="last"
    )
)


# ============================================================
# 14. MURAL / PERIVASCULAR INTERSECTION
#
# Again: inspection only.
# ============================================================

print(
    "\n======================================"
)

print(
    "REPLICATED METASTASIS + MURAL SUPPORT"
)

print(
    "======================================"
)


mural_view = (

    convergence[
        convergence[
            "mural_receptor_detected"
        ]
    ]
    .copy()
)


mural_columns = [

    "gene",
    "PSN_ligands",

    "median_6h_delta",

    "mural_support_n_mice_total",
    "mural_support_n_mice_detected",
    "mural_support_n_mice_pct_ge_10",
    "mural_support_median_log2cpm",
    "mural_support_max_pct_expressing",

    "mural_age_n_young_animals",
    "mural_age_n_aged_animals",

    "mural_age_age_delta_log2cpm",
    "mural_age_hedges_g",

    "mural_age_permutation_p_value",
    "mural_age_fdr_bh",

    "mural_age_metastasis_same_direction"
]


mural_columns = [

    c
    for c in mural_columns
    if c in mural_view.columns

]


display(

    mural_view[
        mural_columns
    ]
    .sort_values(
        [
            "mural_age_fdr_bh",
            "gene"
        ],
        na_position="last"
    )
    .head(
        60
    )
)


# ============================================================
# 15. SPECIFIC AXES WE HAVE DISCUSSED
# ============================================================

selected_genes = [

    "Pdgfrb",
    "Ramp2",

    "Tnfrsf12a",
    "Plxnd1",
    "Mcam",

    "Cd74",
    "Cd36",
    "Itgb5",
    "Bmpr2",

    "Mertk",
    "Ednrb"
]


print(
    "\n======================================"
)

print(
    "SELECTED CONVERGENCE CHECK"
)

print(
    "======================================"
)


selected_columns = [

    "gene",
    "PSN_ligands",

    "median_6h_delta",

    "blood_age_age_delta_log2cpm",
    "blood_age_fdr_bh",
    "blood_age_metastasis_same_direction",

    "comp_age_coef",
    "comp_age_fdr_bh",
    "age_signal_class",

    "mural_support_n_mice_detected",
    "mural_support_max_pct_expressing",

    "mural_age_age_delta_log2cpm",
    "mural_age_fdr_bh"
]


selected_columns = [

    c
    for c in selected_columns
    if c in convergence.columns

]


display(

    convergence[
        convergence[
            "gene"
        ]
        .isin(
            selected_genes
        )
    ][
        selected_columns
    ]
    .sort_values(
        "gene"
    )
)


# ============================================================
# 16. SANITY CHECKS
# ============================================================

assert not convergence[
    "gene"
].duplicated().any()


assert len(
    convergence
) == int(
    replication_table[
        "directionally_replicated_ge1pct"
    ]
    .sum()
)


# ============================================================
# 17. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE D7B METASTASIS × AGING CONVERGENCE COMPLETE"
)

In [ ]:
# ============================================================
# PHASE D8A — HUMAN LUNG AGING VALIDATION TABLE AUDIT
#
# Study:
# De Man et al., Nature Communications, 2026
# Human Aging Lung Cell Atlas
# GEO: GSE281219
#
# We first use the authors' published age-DE tables rather
# than downloading ~1.5 GB raw 10x data.
#
# Supplementary Data 2 / 3:
#   GLMM differential-expression analyses
#
# Supplementary Data 4:
#   pseudobulk comparison / concordance
#
# THIS CELL:
#   - discovers official supplementary download links
#   - downloads Data 2, 3, 4
#   - prints workbook sheets, dimensions and schemas
#
# NO candidate filtering yet.
# NO ranking.
# ============================================================

import re
import requests
import pandas as pd

from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin


print(
    "======================================"
)

print(
    "PHASE D8A — HUMAN LUNG AGING TABLE AUDIT"
)

print(
    "======================================"
)


# ============================================================
# 1. OUTPUT DIRECTORY
# ============================================================

human_age_dir = (
    RAW
    /
    "GSE281219_publication_supplement"
)

human_age_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. OFFICIAL ARTICLE SOURCES
#
# Try Nature first, then PMC as fallback.
# ============================================================

article_urls = [

    "https://www.nature.com/articles/s41467-026-68810-9",

    "https://pmc.ncbi.nlm.nih.gov/articles/PMC12953888/"
]


headers = {

    "User-Agent":
        "Mozilla/5.0 KT-neurovascular-screen/1.0"
}


# ============================================================
# 3. DISCOVER SUPPLEMENTARY DATA LINKS
# ============================================================

target_labels = {

    "Supplementary Data 2":
        None,

    "Supplementary Data 3":
        None,

    "Supplementary Data 4":
        None
}


for article_url in article_urls:

    print(
        "\nChecking:",
        article_url
    )


    try:

        response = requests.get(
            article_url,
            headers=headers,
            timeout=60
        )

        response.raise_for_status()

    except Exception as exc:

        print(
            "Could not read:",
            repr(exc)
        )

        continue


    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )


    links = soup.find_all(
        "a",
        href=True
    )


    print(
        "Links found:",
        len(links)
    )


    for a in links:

        text = " ".join(
            a.get_text(
                " ",
                strip=True
            ).split()
        )


        for label in target_labels:

            if (
                target_labels[
                    label
                ]
                is None

                and

                label.lower()
                in
                text.lower()
            ):

                href = urljoin(
                    article_url,
                    a[
                        "href"
                    ]
                )


                target_labels[
                    label
                ] = href


    if all(
        v is not None
        for v in target_labels.values()
    ):

        break


print(
    "\n======================================"
)

print(
    "DISCOVERED SUPPLEMENTARY LINKS"
)

print(
    "======================================"
)


for label, url in target_labels.items():

    print(
        label,
        "=>",
        url
    )


missing_links = [

    label

    for label, url
    in target_labels.items()

    if url is None
]


if missing_links:

    raise RuntimeError(

        "Could not discover official links for: "
        +
        ", ".join(
            missing_links
        )
    )


# ============================================================
# 4. DOWNLOAD DATA 2 / 3 / 4
# ============================================================

human_age_supp_paths = {}


for label, url in target_labels.items():

    number = re.search(
        r"(\d+)$",
        label
    ).group(
        1
    )


    destination = (

        human_age_dir
        /
        f"DeMan2026_Supplementary_Data_{number}.xlsx"
    )


    human_age_supp_paths[
        label
    ] = destination


    if not destination.exists():

        print(
            "\nDownloading:",
            label
        )


        r = requests.get(
            url,
            headers=headers,
            timeout=120
        )

        r.raise_for_status()


        destination.write_bytes(
            r.content
        )

    else:

        print(
            "\nAlready present:",
            destination.name
        )


    print(
        label,
        "| MB:",
        round(
            destination.stat().st_size
            /
            1e6,
            3
        )
    )


    # XLSX files are ZIP containers.
    # Reject HTML/error pages masquerading as .xlsx.
    first_bytes = destination.read_bytes()[
        :4
    ]


    if first_bytes[:2] != b"PK":

        raise RuntimeError(

            f"{label}: downloaded object is not "
            "a valid XLSX/ZIP file."
        )


# ============================================================
# 5. AUDIT EVERY WORKBOOK
# ============================================================

human_age_workbook_inventory = []


for label, path in human_age_supp_paths.items():

    print(
        "\n======================================"
    )

    print(
        label
    )

    print(
        "======================================"
    )


    xls = pd.ExcelFile(
        path
    )


    print(
        "Sheets:",
        xls.sheet_names
    )


    for sheet in xls.sheet_names:

        df = pd.read_excel(
            path,
            sheet_name=sheet
        )


        human_age_workbook_inventory.append({

            "supplement":
                label,

            "sheet":
                sheet,

            "rows":
                df.shape[0],

            "columns_n":
                df.shape[1],

            "columns":
                " | ".join(
                    map(
                        str,
                        df.columns
                    )
                )
        })


        print(
            "\nSHEET:",
            sheet
        )


        print(
            "Shape:",
            df.shape
        )


        print(
            "Columns:"
        )


        for col in df.columns:

            print(
                " -",
                col
            )


        print(
            "First 3 rows:"
        )


        display(
            df.head(
                3
            )
        )


# ============================================================
# 6. COMPACT INVENTORY TABLE
# ============================================================

human_age_workbook_inventory = pd.DataFrame(
    human_age_workbook_inventory
)


print(
    "\n======================================"
)

print(
    "HUMAN AGING WORKBOOK INVENTORY"
)

print(
    "======================================"
)


display(
    human_age_workbook_inventory
)


human_age_workbook_inventory.to_csv(

    OUT
    /
    "DeMan2026_human_lung_aging_supplement_inventory.csv",

    index=False
)


# ============================================================
# 7. SEARCH SHEET NAMES FOR VASCULAR COMPARTMENTS
# ============================================================

print(
    "\n======================================"
)

print(
    "POTENTIAL ENDOTHELIAL / MURAL SHEETS"
)

print(
    "======================================"
)


vascular_terms = [

    "gcap",
    "aerocyte",
    "endothelial",
    "arterial",
    "venous",
    "lymphatic",
    "pericyte",
    "smooth",
    "smc"
]


for _, row in human_age_workbook_inventory.iterrows():

    sheet_lower = str(
        row[
            "sheet"
        ]
    ).lower()


    if any(
        term in sheet_lower
        for term in vascular_terms
    ):

        print(
            row[
                "supplement"
            ],
            "|",
            row[
                "sheet"
            ],
            "|",
            row[
                "rows"
            ],
            "rows"
        )


# ============================================================
# 8. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE D8A HUMAN LUNG AGING TABLE AUDIT COMPLETE"
)

In [ ]:
# ============================================================
# PHASE D8A — HUMAN LUNG AGING VALIDATION TABLE AUDIT
#
# Study:
# De Man et al., Nature Communications, 2026
# Human Aging Lung Cell Atlas
# GEO: GSE281219
#
# We first use the authors' published age-DE tables rather
# than downloading ~1.5 GB raw 10x data.
#
# Supplementary Data 2 / 3:
#   GLMM differential-expression analyses
#
# Supplementary Data 4:
#   pseudobulk comparison / concordance
#
# THIS CELL:
#   - discovers official supplementary download links
#   - downloads Data 2, 3, 4
#   - prints workbook sheets, dimensions and schemas
#
# NO candidate filtering yet.
# NO ranking.
# ============================================================

import re
import requests
import pandas as pd

from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin


print(
    "======================================"
)

print(
    "PHASE D8A — HUMAN LUNG AGING TABLE AUDIT"
)

print(
    "======================================"
)


# ============================================================
# 1. OUTPUT DIRECTORY
# ============================================================

human_age_dir = (
    RAW
    /
    "GSE281219_publication_supplement"
)

human_age_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. OFFICIAL ARTICLE SOURCES
#
# Try Nature first, then PMC as fallback.
# ============================================================

article_urls = [

    "https://www.nature.com/articles/s41467-026-68810-9",

    "https://pmc.ncbi.nlm.nih.gov/articles/PMC12953888/"
]


headers = {

    "User-Agent":
        "Mozilla/5.0 KT-neurovascular-screen/1.0"
}


# ============================================================
# 3. DISCOVER SUPPLEMENTARY DATA LINKS
# ============================================================

target_labels = {

    "Supplementary Data 2":
        None,

    "Supplementary Data 3":
        None,

    "Supplementary Data 4":
        None
}


for article_url in article_urls:

    print(
        "\nChecking:",
        article_url
    )


    try:

        response = requests.get(
            article_url,
            headers=headers,
            timeout=60
        )

        response.raise_for_status()

    except Exception as exc:

        print(
            "Could not read:",
            repr(exc)
        )

        continue


    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )


    links = soup.find_all(
        "a",
        href=True
    )


    print(
        "Links found:",
        len(links)
    )


    for a in links:

        text = " ".join(
            a.get_text(
                " ",
                strip=True
            ).split()
        )


        for label in target_labels:

            if (
                target_labels[
                    label
                ]
                is None

                and

                label.lower()
                in
                text.lower()
            ):

                href = urljoin(
                    article_url,
                    a[
                        "href"
                    ]
                )


                target_labels[
                    label
                ] = href


    if all(
        v is not None
        for v in target_labels.values()
    ):

        break


print(
    "\n======================================"
)

print(
    "DISCOVERED SUPPLEMENTARY LINKS"
)

print(
    "======================================"
)


for label, url in target_labels.items():

    print(
        label,
        "=>",
        url
    )


missing_links = [

    label

    for label, url
    in target_labels.items()

    if url is None
]


if missing_links:

    raise RuntimeError(

        "Could not discover official links for: "
        +
        ", ".join(
            missing_links
        )
    )


# ============================================================
# 4. DOWNLOAD DATA 2 / 3 / 4
# ============================================================

human_age_supp_paths = {}


for label, url in target_labels.items():

    number = re.search(
        r"(\d+)$",
        label
    ).group(
        1
    )


    destination = (

        human_age_dir
        /
        f"DeMan2026_Supplementary_Data_{number}.xlsx"
    )


    human_age_supp_paths[
        label
    ] = destination


    if not destination.exists():

        print(
            "\nDownloading:",
            label
        )


        r = requests.get(
            url,
            headers=headers,
            timeout=120
        )

        r.raise_for_status()


        destination.write_bytes(
            r.content
        )

    else:

        print(
            "\nAlready present:",
            destination.name
        )


    print(
        label,
        "| MB:",
        round(
            destination.stat().st_size
            /
            1e6,
            3
        )
    )


    # XLSX files are ZIP containers.
    # Reject HTML/error pages masquerading as .xlsx.
    first_bytes = destination.read_bytes()[
        :4
    ]


    if first_bytes[:2] != b"PK":

        raise RuntimeError(

            f"{label}: downloaded object is not "
            "a valid XLSX/ZIP file."
        )


# ============================================================
# 5. AUDIT EVERY WORKBOOK
# ============================================================

human_age_workbook_inventory = []


for label, path in human_age_supp_paths.items():

    print(
        "\n======================================"
    )

    print(
        label
    )

    print(
        "======================================"
    )


    xls = pd.ExcelFile(
        path
    )


    print(
        "Sheets:",
        xls.sheet_names
    )


    for sheet in xls.sheet_names:

        df = pd.read_excel(
            path,
            sheet_name=sheet
        )


        human_age_workbook_inventory.append({

            "supplement":
                label,

            "sheet":
                sheet,

            "rows":
                df.shape[0],

            "columns_n":
                df.shape[1],

            "columns":
                " | ".join(
                    map(
                        str,
                        df.columns
                    )
                )
        })


        print(
            "\nSHEET:",
            sheet
        )


        print(
            "Shape:",
            df.shape
        )


        print(
            "Columns:"
        )


        for col in df.columns:

            print(
                " -",
                col
            )


        print(
            "First 3 rows:"
        )


        display(
            df.head(
                3
            )
        )


# ============================================================
# 6. COMPACT INVENTORY TABLE
# ============================================================

human_age_workbook_inventory = pd.DataFrame(
    human_age_workbook_inventory
)


print(
    "\n======================================"
)

print(
    "HUMAN AGING WORKBOOK INVENTORY"
)

print(
    "======================================"
)


display(
    human_age_workbook_inventory
)


human_age_workbook_inventory.to_csv(

    OUT
    /
    "DeMan2026_human_lung_aging_supplement_inventory.csv",

    index=False
)


# ============================================================
# 7. SEARCH SHEET NAMES FOR VASCULAR COMPARTMENTS
# ============================================================

print(
    "\n======================================"
)

print(
    "POTENTIAL ENDOTHELIAL / MURAL SHEETS"
)

print(
    "======================================"
)


vascular_terms = [

    "gcap",
    "aerocyte",
    "endothelial",
    "arterial",
    "venous",
    "lymphatic",
    "pericyte",
    "smooth",
    "smc"
]


for _, row in human_age_workbook_inventory.iterrows():

    sheet_lower = str(
        row[
            "sheet"
        ]
    ).lower()


    if any(
        term in sheet_lower
        for term in vascular_terms
    ):

        print(
            row[
                "supplement"
            ],
            "|",
            row[
                "sheet"
            ],
            "|",
            row[
                "rows"
            ],
            "rows"
        )


# ============================================================
# 8. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE D8A HUMAN LUNG AGING TABLE AUDIT COMPLETE"
)

In [ ]:
# ============================================================
# PHASE D8B — HUMAN VASCULAR AGING VALIDATION OF THE
#             60 REPLICATED METASTATIC RECEPTORS
#
# Inputs:
#
#   D6F:
#       replication_table
#
#   D7B:
#       convergence
#
#   B4:
#       lr_psn_detected
#
#   D8A:
#       human_age_supp_paths
#
# Human aging study:
#
#   Supplementary Data 2:
#       GLMM
#       Gene, Beta1, p_val, p_val_adj
#
#   Supplementary Data 3:
#       independent pseudobulk edgeR LRT
#       gene, avg_logFC, old/young expression,
#       p_val, p_val_adj
#
# IMPORTANT:
#
# 1. Do NOT assume Beta1 orientation.
#    Determine it empirically by comparison with avg_logFC.
#
# 2. Mouse receptors are mapped back to their STRICT
#    human 1:1 ortholog already stored in lr_psn_detected.
#
# 3. Robust human age support requires:
#
#       GLMM FDR < 0.05
#       pseudobulk FDR < 0.05
#       same direction in both models
#
# 4. This is validation / convergence analysis.
#    It is NOT yet an integrated ranking.
# ============================================================

import numpy as np
import pandas as pd


print(
    "======================================"
)

print(
    "PHASE D8B — HUMAN VASCULAR AGING VALIDATION"
)

print(
    "======================================"
)


# ============================================================
# 1. CORE HUMAN VASCULAR COMPARTMENTS
# ============================================================

human_vascular_celltypes = [

    "Aerocyte",
    "gCap",
    "Arterial",
    "Venous",

    "Pericyte",
    "SMC"
]


human_endothelial_celltypes = {

    "Aerocyte",
    "gCap",
    "Arterial",
    "Venous"
}


human_mural_celltypes = {

    "Pericyte",
    "SMC"
}


# ============================================================
# 2. LOCATE SUPPLEMENTARY WORKBOOKS
# ============================================================

glmm_path = human_age_supp_paths[
    "Supplementary Data 2"
]


pb_path = human_age_supp_paths[
    "Supplementary Data 3"
]


print(
    "GLMM workbook:",
    glmm_path
)


print(
    "Pseudobulk workbook:",
    pb_path
)


# ============================================================
# 3. VERIFY REQUIRED SHEETS
# ============================================================

glmm_xls = pd.ExcelFile(
    glmm_path
)


pb_xls = pd.ExcelFile(
    pb_path
)


for ct in human_vascular_celltypes:

    if ct not in glmm_xls.sheet_names:

        raise RuntimeError(
            f"{ct} absent from GLMM workbook."
        )


    if ct not in pb_xls.sheet_names:

        raise RuntimeError(
            f"{ct} absent from pseudobulk workbook."
        )


print(
    "\nAll six vascular cell types present in both models."
)


# ============================================================
# 4. EMPIRICALLY DETERMINE Beta1 DIRECTION
#
# Compare Beta1 with explicit pseudobulk avg_logFC
# across all overlapping genes in the six cell types.
# ============================================================

orientation_rows = []

orientation_matches = []


for ct in human_vascular_celltypes:

    glmm = pd.read_excel(
        glmm_path,
        sheet_name=ct
    )


    pb = pd.read_excel(
        pb_path,
        sheet_name=ct
    )


    glmm_small = (

        glmm[
            [
                "Gene",
                "Beta1"
            ]
        ]
        .dropna()
        .copy()
    )


    glmm_small[
        "gene_key"
    ] = (

        glmm_small[
            "Gene"
        ]
        .astype(str)
        .str.upper()
    )


    pb_small = (

        pb[
            [
                "gene",
                "avg_logFC"
            ]
        ]
        .dropna()
        .copy()
    )


    pb_small[
        "gene_key"
    ] = (

        pb_small[
            "gene"
        ]
        .astype(str)
        .str.upper()
    )


    matched = (

        glmm_small[
            [
                "gene_key",
                "Beta1"
            ]
        ]
        .merge(

            pb_small[
                [
                    "gene_key",
                    "avg_logFC"
                ]
            ],

            on="gene_key",
            how="inner"
        )
    )


    matched = (

        matched[
            (
                matched[
                    "Beta1"
                ]
                != 0
            )

            &

            (
                matched[
                    "avg_logFC"
                ]
                != 0
            )
        ]
        .copy()
    )


    matched[
        "same_sign"
    ] = (

        np.sign(
            matched[
                "Beta1"
            ]
        )

        ==

        np.sign(
            matched[
                "avg_logFC"
            ]
        )
    )


    if len(
        matched
    ):

        same_fraction = float(
            matched[
                "same_sign"
            ]
            .mean()
        )

        corr = float(

            matched[
                [
                    "Beta1",
                    "avg_logFC"
                ]
            ]
            .corr(
                method="spearman"
            )
            .iloc[
                0,
                1
            ]
        )

    else:

        same_fraction = np.nan
        corr = np.nan


    orientation_rows.append({

        "celltype":
            ct,

        "n_overlap":
            len(
                matched
            ),

        "same_sign_fraction":
            same_fraction,

        "spearman_beta1_vs_logfc":
            corr
    })


    if len(
        matched
    ):

        matched[
            "celltype"
        ] = ct

        orientation_matches.append(
            matched
        )


orientation_audit = pd.DataFrame(
    orientation_rows
)


orientation_all = pd.concat(
    orientation_matches,
    ignore_index=True
)


overall_same_sign_fraction = float(

    orientation_all[
        "same_sign"
    ]
    .mean()
)


overall_spearman = float(

    orientation_all[
        [
            "Beta1",
            "avg_logFC"
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[
        0,
        1
    ]
)


print(
    "\n======================================"
)

print(
    "HUMAN AGE SIGN ORIENTATION AUDIT"
)

print(
    "======================================"
)


display(
    orientation_audit
)


print(
    "\nOverall matched genes:",
    len(
        orientation_all
    )
)


print(
    "Overall Beta1 / avg_logFC same-sign fraction:",
    round(
        overall_same_sign_fraction,
        4
    )
)


print(
    "Overall Spearman correlation:",
    round(
        overall_spearman,
        4
    )
)


# ------------------------------------------------------------
# Resolve orientation
# ------------------------------------------------------------

if overall_same_sign_fraction >= 0.75:

    beta1_multiplier = 1.0

    print(
        "\nResolved orientation:"
    )

    print(
        "Positive Beta1 corresponds to positive old-vs-young "
        "pseudobulk direction."
    )


elif overall_same_sign_fraction <= 0.25:

    beta1_multiplier = -1.0

    print(
        "\nResolved orientation:"
    )

    print(
        "Beta1 sign is opposite to old-vs-young "
        "pseudobulk direction; Beta1 will be inverted."
    )


else:

    raise RuntimeError(

        "Could not confidently resolve GLMM Beta1 orientation. "
        f"Same-sign fraction = {overall_same_sign_fraction:.3f}"
    )


# ============================================================
# 5. START FROM EXACTLY THE 60 D6F REPLICATED RECEPTORS
# ============================================================

replicated60 = (

    convergence[
        convergence[
            "directionally_replicated_ge1pct"
        ]
        .astype(bool)
    ]
    .copy()
)


if len(
    replicated60
) != 60:

    raise RuntimeError(

        "Expected 60 D6F replicated receptors; "
        f"found {len(replicated60)}."
    )


# ============================================================
# 6. STRICT MOUSE -> HUMAN RECEPTOR MAPPING
#
# Use our B3/B4 reciprocal orthology mapping,
# NOT capitalization as an orthology assumption.
# ============================================================

orthology = (

    lr_psn_detected[
        [
            "mouse_receptor",
            "human_receptor"
        ]
    ]
    .dropna()
    .drop_duplicates()
    .copy()
)


mapping_counts = (

    orthology
    .groupby(
        "mouse_receptor"
    )[
        "human_receptor"
    ]
    .nunique()
)


bad_mapping = (

    mapping_counts[
        mapping_counts
        !=
        1
    ]
)


if len(
    bad_mapping
):

    raise RuntimeError(

        "Non-1:1 receptor mappings detected:\n"
        +
        str(
            bad_mapping
        )
    )


orthology = (

    orthology
    .drop_duplicates(
        subset=[
            "mouse_receptor"
        ]
    )
    .rename(
        columns={

            "mouse_receptor":
                "gene",

            "human_receptor":
                "human_gene"
        }
    )
)


replicated60 = (

    replicated60
    .merge(
        orthology,
        on="gene",
        how="left",
        validate="one_to_one"
    )
)


missing_human_map = (

    replicated60[
        replicated60[
            "human_gene"
        ]
        .isna()
    ][
        "gene"
    ]
    .tolist()
)


print(
    "\nMissing strict human receptor mappings:",
    len(
        missing_human_map
    )
)


if missing_human_map:

    print(
        missing_human_map
    )


replicated60[
    "human_gene_key"
] = (

    replicated60[
        "human_gene"
    ]
    .astype(str)
    .str.upper()
)


# ============================================================
# 7. READ SIX HUMAN VASCULAR GLMM + PSEUDOBULK TABLES
# ============================================================

human_age_rows = []


for ct in human_vascular_celltypes:

    print(
        "\n--------------------------------------"
    )

    print(
        "Processing:",
        ct
    )


    # --------------------------------------------------------
    # GLMM
    # --------------------------------------------------------

    glmm = pd.read_excel(
        glmm_path,
        sheet_name=ct
    )


    glmm[
        "human_gene_key"
    ] = (

        glmm[
            "Gene"
        ]
        .astype(str)
        .str.upper()
    )


    # One row per gene expected
    if glmm[
        "human_gene_key"
    ].duplicated().any():

        raise RuntimeError(
            f"{ct}: duplicate genes in GLMM sheet."
        )


    glmm_lookup = (

        glmm
        .set_index(
            "human_gene_key"
        )
    )


    # --------------------------------------------------------
    # Pseudobulk
    # --------------------------------------------------------

    pb = pd.read_excel(
        pb_path,
        sheet_name=ct
    )


    pb[
        "human_gene_key"
    ] = (

        pb[
            "gene"
        ]
        .astype(str)
        .str.upper()
    )


    if pb[
        "human_gene_key"
    ].duplicated().any():

        raise RuntimeError(
            f"{ct}: duplicate genes in pseudobulk sheet."
        )


    pb_lookup = (

        pb
        .set_index(
            "human_gene_key"
        )
    )


    # --------------------------------------------------------
    # Candidate-by-candidate extraction
    # --------------------------------------------------------

    for _, candidate in replicated60.iterrows():

        mouse_gene = candidate[
            "gene"
        ]


        human_gene = candidate[
            "human_gene"
        ]


        human_key = candidate[
            "human_gene_key"
        ]


        metastasis_delta = candidate[
            "median_6h_delta"
        ]


        # ---------------- GLMM ----------------

        glmm_listed = (
            human_key
            in
            glmm_lookup.index
        )


        if glmm_listed:

            gr = glmm_lookup.loc[
                human_key
            ]


            glmm_beta1_raw = float(
                gr[
                    "Beta1"
                ]
            )


            glmm_beta1 = (

                beta1_multiplier
                *
                glmm_beta1_raw
            )


            glmm_p = float(
                gr[
                    "p_val"
                ]
            )


            glmm_fdr = float(
                gr[
                    "p_val_adj"
                ]
            )


        else:

            glmm_beta1_raw = np.nan
            glmm_beta1 = np.nan
            glmm_p = np.nan
            glmm_fdr = np.nan


        # ---------------- pseudobulk ----------------

        pb_tested = (
            human_key
            in
            pb_lookup.index
        )


        if pb_tested:

            pr = pb_lookup.loc[
                human_key
            ]


            pb_logfc = float(
                pr[
                    "avg_logFC"
                ]
            )


            pb_old_pct = float(
                pr[
                    "old.pct"
                ]
            )


            pb_young_pct = float(
                pr[
                    "young.pct"
                ]
            )


            pb_old_exp = float(
                pr[
                    "old.exp"
                ]
            )


            pb_young_exp = float(
                pr[
                    "young.exp"
                ]
            )


            pb_p = float(
                pr[
                    "p_val"
                ]
            )


            pb_fdr = float(
                pr[
                    "p_val_adj"
                ]
            )


        else:

            pb_logfc = np.nan
            pb_old_pct = np.nan
            pb_young_pct = np.nan
            pb_old_exp = np.nan
            pb_young_exp = np.nan
            pb_p = np.nan
            pb_fdr = np.nan


        # ====================================================
        # 8. CROSS-MODEL FLAGS
        # ====================================================

        both_models_available = (

            glmm_listed
            and
            pb_tested
        )


        if both_models_available:

            crossmodel_same_direction = (

                np.sign(
                    glmm_beta1
                )

                ==

                np.sign(
                    pb_logfc
                )
            )

        else:

            crossmodel_same_direction = False


        glmm_p_lt_005 = (

            glmm_listed
            and
            glmm_p < 0.05
        )


        glmm_fdr_lt_005 = (

            glmm_listed
            and
            glmm_fdr < 0.05
        )


        pb_p_lt_005 = (

            pb_tested
            and
            pb_p < 0.05
        )


        pb_fdr_lt_005 = (

            pb_tested
            and
            pb_fdr < 0.05
        )


        # Stringent cross-model human-aging support
        human_age_robust_fdr05 = (

            glmm_fdr_lt_005

            and

            pb_fdr_lt_005

            and

            crossmodel_same_direction
        )


        # Less stringent support retained for inspection only.
        human_age_both_p05 = (

            glmm_p_lt_005

            and

            pb_p_lt_005

            and

            crossmodel_same_direction
        )


        if (
            human_age_robust_fdr05
            and
            metastasis_delta != 0
        ):

            human_age_vs_metastasis_same_direction = (

                np.sign(
                    pb_logfc
                )

                ==

                np.sign(
                    metastasis_delta
                )
            )

        else:

            human_age_vs_metastasis_same_direction = False


        human_age_rows.append({

            "mouse_gene":
                mouse_gene,

            "human_gene":
                human_gene,

            "celltype":
                ct,

            "compartment":
                (
                    "endothelial"
                    if ct in human_endothelial_celltypes
                    else "mural"
                ),

            "metastasis_median_6h_delta":
                metastasis_delta,

            "glmm_listed":
                glmm_listed,

            "glmm_beta1_raw":
                glmm_beta1_raw,

            "glmm_age_effect_oriented":
                glmm_beta1,

            "glmm_p_value":
                glmm_p,

            "glmm_fdr":
                glmm_fdr,

            "glmm_p_lt_005":
                glmm_p_lt_005,

            "glmm_fdr_lt_005":
                glmm_fdr_lt_005,

            "pseudobulk_tested":
                pb_tested,

            "pseudobulk_old_vs_young_logFC":
                pb_logfc,

            "pseudobulk_old_pct":
                pb_old_pct,

            "pseudobulk_young_pct":
                pb_young_pct,

            "pseudobulk_old_exp":
                pb_old_exp,

            "pseudobulk_young_exp":
                pb_young_exp,

            "pseudobulk_p_value":
                pb_p,

            "pseudobulk_fdr":
                pb_fdr,

            "pseudobulk_p_lt_005":
                pb_p_lt_005,

            "pseudobulk_fdr_lt_005":
                pb_fdr_lt_005,

            "crossmodel_same_direction":
                crossmodel_same_direction,

            "human_age_both_p05":
                human_age_both_p05,

            "human_age_robust_fdr05":
                human_age_robust_fdr05,

            "human_age_vs_metastasis_same_direction":
                human_age_vs_metastasis_same_direction
        })


human_age_candidate_long = pd.DataFrame(
    human_age_rows
)


# ============================================================
# 9. SAVE LONG HUMAN VALIDATION TABLE
# ============================================================

human_age_candidate_long.to_csv(

    OUT
    /
    "DeMan2026_human_vascular_aging_validation_60_receptors_long.csv",

    index=False
)


# ============================================================
# 10. BUILD PER-RECEPTOR SUMMARY
# ============================================================

summary_rows = []


for mouse_gene, g in human_age_candidate_long.groupby(
    "mouse_gene"
):

    robust = g[
        g[
            "human_age_robust_fdr05"
        ]
    ]


    supportive = g[
        g[
            "human_age_both_p05"
        ]
    ]


    robust_ec = robust[
        robust[
            "compartment"
        ]
        ==
        "endothelial"
    ]


    robust_mural = robust[
        robust[
            "compartment"
        ]
        ==
        "mural"
    ]


    robust_same_met = robust[
        robust[
            "human_age_vs_metastasis_same_direction"
        ]
    ]


    robust_opposite_met = robust[
        ~robust[
            "human_age_vs_metastasis_same_direction"
        ]
    ]


    human_gene = (
        g[
            "human_gene"
        ]
        .iloc[
            0
        ]
    )


    metastasis_delta = float(
        g[
            "metastasis_median_6h_delta"
        ]
        .iloc[
            0
        ]
    )


    summary_rows.append({

        "gene":
            mouse_gene,

        "human_gene":
            human_gene,

        "median_6h_delta":
            metastasis_delta,

        "n_glmm_listed_celltypes":
            int(
                g[
                    "glmm_listed"
                ]
                .sum()
            ),

        "n_pseudobulk_tested_celltypes":
            int(
                g[
                    "pseudobulk_tested"
                ]
                .sum()
            ),

        "n_glmm_fdr05_celltypes":
            int(
                g[
                    "glmm_fdr_lt_005"
                ]
                .sum()
            ),

        "n_pseudobulk_fdr05_celltypes":
            int(
                g[
                    "pseudobulk_fdr_lt_005"
                ]
                .sum()
            ),

        "n_crossmodel_p05_celltypes":
            len(
                supportive
            ),

        "n_crossmodel_fdr05_celltypes":
            len(
                robust
            ),

        "n_crossmodel_fdr05_endothelial":
            len(
                robust_ec
            ),

        "n_crossmodel_fdr05_mural":
            len(
                robust_mural
            ),

        "robust_human_age_celltypes":
            ";".join(
                robust[
                    "celltype"
                ]
                .tolist()
            ),

        "robust_human_age_same_metastasis_celltypes":
            ";".join(
                robust_same_met[
                    "celltype"
                ]
                .tolist()
            ),

        "robust_human_age_opposite_metastasis_celltypes":
            ";".join(
                robust_opposite_met[
                    "celltype"
                ]
                .tolist()
            ),

        "any_robust_human_age":
            (
                len(
                    robust
                )
                >
                0
            ),

        "any_robust_human_endothelial_age":
            (
                len(
                    robust_ec
                )
                >
                0
            ),

        "any_robust_human_mural_age":
            (
                len(
                    robust_mural
                )
                >
                0
            ),

        "any_robust_human_age_same_metastasis":
            (
                len(
                    robust_same_met
                )
                >
                0
            )
    })


human_age_receptor_summary = pd.DataFrame(
    summary_rows
)


# ============================================================
# 11. MERGE BACK MOUSE AGING / PSN CONTEXT
# ============================================================

context_columns = [

    "gene",
    "PSN_ligands",

    "set1_GSE235394_delta_log2cpm1",
    "set2_GSE253749_delta_log2cpm1",
    "set3_GSE253751_delta_log2cpm1",

    "blood_age_age_delta_log2cpm",
    "blood_age_fdr_bh",

    "comp_age_coef",
    "comp_age_fdr_bh",

    "age_signal_class",

    "mural_support_n_mice_detected",
    "mural_support_max_pct_expressing",

    "mural_age_age_delta_log2cpm",
    "mural_age_fdr_bh"
]


context_columns = [

    col

    for col in context_columns

    if col in replicated60.columns
]


human_age_receptor_summary = (

    human_age_receptor_summary
    .merge(

        replicated60[
            context_columns
        ],

        on="gene",
        how="left",
        validate="one_to_one"
    )
)


human_age_receptor_summary.to_csv(

    OUT
    /
    "DeMan2026_human_vascular_aging_validation_60_receptors_summary.csv",

    index=False
)


# ============================================================
# 12. VALIDATION SUMMARY
# ============================================================

print(
    "\n======================================"
)

print(
    "HUMAN VASCULAR AGING VALIDATION SUMMARY"
)

print(
    "======================================"
)


print(
    "Replicated metastatic receptors tested:",
    len(
        human_age_receptor_summary
    )
)


print(
    "Any GLMM FDR<0.05 in six vascular cell types:",
    int(
        (
            human_age_receptor_summary[
                "n_glmm_fdr05_celltypes"
            ]
            >
            0
        )
        .sum()
    )
)


print(
    "Any pseudobulk FDR<0.05:",
    int(
        (
            human_age_receptor_summary[
                "n_pseudobulk_fdr05_celltypes"
            ]
            >
            0
        )
        .sum()
    )
)


print(
    "Cross-model robust FDR<0.05 + same direction:",
    int(
        human_age_receptor_summary[
            "any_robust_human_age"
        ]
        .sum()
    )
)


print(
    "Robust human ENDOTHELIAL aging:",
    int(
        human_age_receptor_summary[
            "any_robust_human_endothelial_age"
        ]
        .sum()
    )
)


print(
    "Robust human MURAL aging:",
    int(
        human_age_receptor_summary[
            "any_robust_human_mural_age"
        ]
        .sum()
    )
)


print(
    "Robust human aging AND same direction as 6 h metastasis:",
    int(
        human_age_receptor_summary[
            "any_robust_human_age_same_metastasis"
        ]
        .sum()
    )
)


# ============================================================
# 13. ROBUST HUMAN-AGING INTERSECTIONS
#
# Inspection only — NOT final ranking.
# ============================================================

print(
    "\n======================================"
)

print(
    "ROBUST HUMAN AGING INTERSECTIONS"
)

print(
    "======================================"
)


robust_human = (

    human_age_receptor_summary[
        human_age_receptor_summary[
            "any_robust_human_age"
        ]
    ]
    .copy()
)


robust_columns = [

    "gene",
    "human_gene",
    "PSN_ligands",

    "median_6h_delta",

    "n_crossmodel_fdr05_celltypes",
    "n_crossmodel_fdr05_endothelial",
    "n_crossmodel_fdr05_mural",

    "robust_human_age_celltypes",

    "robust_human_age_same_metastasis_celltypes",
    "robust_human_age_opposite_metastasis_celltypes",

    "blood_age_age_delta_log2cpm",
    "blood_age_fdr_bh",

    "comp_age_coef",
    "comp_age_fdr_bh",

    "age_signal_class"
]


robust_columns = [

    c
    for c in robust_columns
    if c in robust_human.columns
]


display(

    robust_human[
        robust_columns
    ]
    .sort_values(

        [
            "n_crossmodel_fdr05_celltypes",
            "n_crossmodel_fdr05_endothelial",
            "gene"
        ],

        ascending=[
            False,
            False,
            True
        ]
    )
)


# ============================================================
# 14. CELL-TYPE LEVEL DETAILS FOR ROBUST HITS
# ============================================================

print(
    "\n======================================"
)

print(
    "ROBUST HUMAN HIT DETAILS"
)

print(
    "======================================"
)


robust_details = (

    human_age_candidate_long[
        human_age_candidate_long[
            "human_age_robust_fdr05"
        ]
    ]
    .copy()
)


display(

    robust_details[
        [
            "mouse_gene",
            "human_gene",
            "celltype",
            "compartment",

            "metastasis_median_6h_delta",

            "glmm_age_effect_oriented",
            "glmm_fdr",

            "pseudobulk_old_vs_young_logFC",
            "pseudobulk_fdr",

            "pseudobulk_old_pct",
            "pseudobulk_young_pct",

            "crossmodel_same_direction",

            "human_age_vs_metastasis_same_direction"
        ]
    ]
    .sort_values(
        [
            "mouse_gene",
            "celltype"
        ]
    )
)


# ============================================================
# 15. SELECTED AXES WE HAVE DISCUSSED
# ============================================================

selected_axes = [

    "Bmpr2",
    "Cd36",
    "Cd74",
    "Itgb5",

    "Pdgfrb",
    "Ramp2",

    "Tnfrsf12a",
    "Plxnd1",
    "Mcam",

    "Mertk",
    "Ednrb"
]


print(
    "\n======================================"
)

print(
    "SELECTED AXES — HUMAN AGING CHECK"
)

print(
    "======================================"
)


selected_summary = (

    human_age_receptor_summary[
        human_age_receptor_summary[
            "gene"
        ]
        .isin(
            selected_axes
        )
    ]
    .copy()
)


display(

    selected_summary[
        [
            "gene",
            "human_gene",
            "PSN_ligands",

            "median_6h_delta",

            "n_crossmodel_fdr05_celltypes",
            "n_crossmodel_fdr05_endothelial",
            "n_crossmodel_fdr05_mural",

            "robust_human_age_celltypes",

            "robust_human_age_same_metastasis_celltypes",
            "robust_human_age_opposite_metastasis_celltypes",

            "blood_age_age_delta_log2cpm",
            "blood_age_fdr_bh",

            "comp_age_coef",
            "comp_age_fdr_bh"
        ]
    ]
    .sort_values(
        "gene"
    )
)


# ============================================================
# 16. SELECTED AXES — CELL-TYPE DETAIL
# ============================================================

print(
    "\n======================================"
)

print(
    "SELECTED AXES — HUMAN CELL-TYPE DETAIL"
)

print(
    "======================================"
)


selected_detail = (

    human_age_candidate_long[
        human_age_candidate_long[
            "mouse_gene"
        ]
        .isin(
            selected_axes
        )
    ]
    .copy()
)


display(

    selected_detail[
        [
            "mouse_gene",
            "human_gene",
            "celltype",

            "glmm_listed",
            "glmm_age_effect_oriented",
            "glmm_fdr",

            "pseudobulk_tested",
            "pseudobulk_old_vs_young_logFC",
            "pseudobulk_fdr",

            "crossmodel_same_direction",
            "human_age_robust_fdr05",

            "human_age_vs_metastasis_same_direction"
        ]
    ]
    .sort_values(
        [
            "mouse_gene",
            "celltype"
        ]
    )
)


# ============================================================
# 17. SANITY CHECKS
# ============================================================

assert len(
    human_age_receptor_summary
) == 60


assert not human_age_receptor_summary[
    "gene"
].duplicated().any()


assert len(
    human_age_candidate_long
) == (

    60
    *
    len(
        human_vascular_celltypes
    )
)


# ============================================================
# 18. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE D8B HUMAN VASCULAR AGING VALIDATION COMPLETE"
)

In [ ]:
# ============================================================
# PHASE D8C — AUTHOR-ALIGNED HUMAN AGING CONCORDANCE AUDIT
#
# D8B used an intentionally stringent rule:
#
#   GLMM FDR < 0.05
#   AND pseudobulk FDR < 0.05
#   AND same direction
#
# That yielded zero hits.
#
# However, the De Man et al. paper defines age-associated
# differential expression primarily using nominal p < 0.05
# after cell-type-specific expression filtering, and then uses
# an independent pseudobulk LRT to assess reproducibility.
#
# THIS CELL DOES NOT RELAX OR REWRITE P-VALUES.
#
# It reports, separately:
#
#   A. published GLMM FDR<0.05 signals
#   B. published pseudobulk FDR<0.05 signals
#   C. cross-model nominal p<0.05 + same direction
#   D. whether those nominally replicated human-aging effects
#      agree with the replicated 6 h metastatic direction
#
# NO integrated ranking.
# NO new multiple-testing correction.
# NO claim that nominal p<0.05 = FDR significance.
# ============================================================

import numpy as np
import pandas as pd


print(
    "======================================"
)

print(
    "PHASE D8C — AUTHOR-ALIGNED HUMAN AGING CONCORDANCE"
)

print(
    "======================================"
)


# ============================================================
# 1. RESTORE D8B LONG TABLE
# ============================================================

# ============================================================
# 1. RESTORE D8B LONG TABLE
#
# Prefer in-memory object if available.
# Otherwise restore the exact table saved by D8B.
# ============================================================
# ============================================================
# RESTORE PROJECT PATHS AFTER COLAB RUNTIME RESET
# ============================================================

from pathlib import Path
import os


# Mount Google Drive only if the current runtime has lost it.
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")


ROOT = Path(
    "/content/drive/MyDrive/KT_Neurovascular_Screen"
)

RAW = ROOT / "raw"
EXTRACTED = ROOT / "extracted"
CACHE = ROOT / "cache"
OUT = ROOT / "outputs"
FIG = OUT / "figures"


# Confirm the project and output directory really exist.
if not ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found: {ROOT}"
    )

if not OUT.exists():
    raise FileNotFoundError(
        f"Output directory not found: {OUT}"
    )


print(
    "Restored project root:",
    ROOT
)

print(
    "Restored output directory:",
    OUT
)
d8b_long_path = (
    OUT
    /
    "DeMan2026_human_vascular_aging_validation_60_receptors_long.csv"
)


if (
    "human_age_candidate_long" in globals()
    and
    isinstance(
        human_age_candidate_long,
        pd.DataFrame
    )
    and
    len(
        human_age_candidate_long
    ) > 0
):

    d8c = (
        human_age_candidate_long
        .copy()
    )

    print(
        "Restored D8B long table from memory:",
        d8c.shape
    )

else:

    if not d8b_long_path.exists():

        raise FileNotFoundError(
            "D8B saved long table not found: "
            f"{d8b_long_path}"
        )

    human_age_candidate_long = pd.read_csv(
        d8b_long_path
    )

    d8c = (
        human_age_candidate_long
        .copy()
    )

    print(
        "Restored D8B long table from saved CSV:",
        d8c.shape
    )


# ------------------------------------------------------------
# Also restore D7B convergence table if runtime memory was lost.
# D8C needs it later for mouse-aging / PSN context.
# ------------------------------------------------------------

if not (
    "convergence" in globals()
    and
    isinstance(
        convergence,
        pd.DataFrame
    )
    and
    len(
        convergence
    ) > 0
):

    convergence_path = (
        OUT
        /
        "PSN_receptor_metastasis_replication_with_aging_evidence.csv"
    )

    if not convergence_path.exists():

        raise FileNotFoundError(
            "D7B convergence table not found: "
            f"{convergence_path}"
        )

    convergence = pd.read_csv(
        convergence_path
    )

    print(
        "Restored D7B convergence table from saved CSV:",
        convergence.shape
    )


required_columns = [

    "mouse_gene",
    "human_gene",
    "celltype",
    "compartment",

    "metastasis_median_6h_delta",

    "glmm_listed",
    "glmm_age_effect_oriented",
    "glmm_p_value",
    "glmm_fdr",

    "pseudobulk_tested",
    "pseudobulk_old_vs_young_logFC",
    "pseudobulk_p_value",
    "pseudobulk_fdr",

    "crossmodel_same_direction"
]


missing = [

    c
    for c in required_columns
    if c not in d8c.columns

]


if missing:

    raise RuntimeError(
        f"D8B table missing columns: {missing}"
    )


print(
    "Rows:",
    len(d8c)
)


print(
    "Unique receptors:",
    d8c[
        "mouse_gene"
    ].nunique()
)


print(
    "Cell types:",
    sorted(
        d8c[
            "celltype"
        ].unique()
    )
)


# ============================================================
# 2. REBUILD EVIDENCE FLAGS EXPLICITLY
# ============================================================

d8c[
    "glmm_nominal_p05"
] = (

    d8c[
        "glmm_listed"
    ].astype(bool)

    &

    d8c[
        "glmm_p_value"
    ].lt(
        0.05
    )
)


d8c[
    "pb_nominal_p05"
] = (

    d8c[
        "pseudobulk_tested"
    ].astype(bool)

    &

    d8c[
        "pseudobulk_p_value"
    ].lt(
        0.05
    )
)


d8c[
    "glmm_fdr05"
] = (

    d8c[
        "glmm_listed"
    ].astype(bool)

    &

    d8c[
        "glmm_fdr"
    ].lt(
        0.05
    )
)


d8c[
    "pb_fdr05"
] = (

    d8c[
        "pseudobulk_tested"
    ].astype(bool)

    &

    d8c[
        "pseudobulk_fdr"
    ].lt(
        0.05
    )
)


# Author-aligned nominal overlap:
# significant at p<0.05 in BOTH independent analyses
# and directionally concordant.
d8c[
    "crossmodel_nominal_p05_same_direction"
] = (

    d8c[
        "glmm_nominal_p05"
    ]

    &

    d8c[
        "pb_nominal_p05"
    ]

    &

    d8c[
        "crossmodel_same_direction"
    ].astype(bool)
)


# Keep the stringent D8B criterion separately.
d8c[
    "crossmodel_fdr05_same_direction"
] = (

    d8c[
        "glmm_fdr05"
    ]

    &

    d8c[
        "pb_fdr05"
    ]

    &

    d8c[
        "crossmodel_same_direction"
    ].astype(bool)
)


# ============================================================
# 3. HUMAN AGE VS METASTASIS DIRECTION
# ============================================================

d8c[
    "nominal_human_age_same_metastasis_direction"
] = (

    d8c[
        "crossmodel_nominal_p05_same_direction"
    ]

    &

    (
        np.sign(
            d8c[
                "pseudobulk_old_vs_young_logFC"
            ]
        )

        ==

        np.sign(
            d8c[
                "metastasis_median_6h_delta"
            ]
        )
    )
)


d8c[
    "nominal_human_age_opposite_metastasis_direction"
] = (

    d8c[
        "crossmodel_nominal_p05_same_direction"
    ]

    &

    (
        np.sign(
            d8c[
                "pseudobulk_old_vs_young_logFC"
            ]
        )

        !=

        np.sign(
            d8c[
                "metastasis_median_6h_delta"
            ]
        )
    )
)


# ============================================================
# 4. SUMMARY COUNTS
# ============================================================

print(
    "\n======================================"
)

print(
    "HUMAN AGING EVIDENCE LAYERS"
)

print(
    "======================================"
)


print(
    "Unique receptors with GLMM nominal p<0.05:",
    d8c.loc[
        d8c[
            "glmm_nominal_p05"
        ],
        "mouse_gene"
    ].nunique()
)


print(
    "Unique receptors with pseudobulk nominal p<0.05:",
    d8c.loc[
        d8c[
            "pb_nominal_p05"
        ],
        "mouse_gene"
    ].nunique()
)


print(
    "Unique receptors with GLMM FDR<0.05:",
    d8c.loc[
        d8c[
            "glmm_fdr05"
        ],
        "mouse_gene"
    ].nunique()
)


print(
    "Unique receptors with pseudobulk FDR<0.05:",
    d8c.loc[
        d8c[
            "pb_fdr05"
        ],
        "mouse_gene"
    ].nunique()
)


print(
    "Unique receptors with BOTH nominal p<0.05 + same direction:",
    d8c.loc[
        d8c[
            "crossmodel_nominal_p05_same_direction"
        ],
        "mouse_gene"
    ].nunique()
)


print(
    "Unique receptors with BOTH FDR<0.05 + same direction:",
    d8c.loc[
        d8c[
            "crossmodel_fdr05_same_direction"
        ],
        "mouse_gene"
    ].nunique()
)


print(
    "Nominal cross-model receptors with at least one "
    "same-direction metastasis cell type:",
    d8c.loc[
        d8c[
            "nominal_human_age_same_metastasis_direction"
        ],
        "mouse_gene"
    ].nunique()
)


print(
    "Nominal cross-model receptors with at least one "
    "opposite-direction metastasis cell type:",
    d8c.loc[
        d8c[
            "nominal_human_age_opposite_metastasis_direction"
        ],
        "mouse_gene"
    ].nunique()
)


# ============================================================
# 5. EXACT SINGLE-MODEL FDR<0.05 SIGNALS
#
# We need to know which receptors generated the D8B counts:
#
#   2 GLMM
#   1 pseudobulk
# ============================================================

print(
    "\n======================================"
)

print(
    "PUBLISHED FDR<0.05 SIGNALS AMONG THE 60"
)

print(
    "======================================"
)


single_model_fdr = (

    d8c[
        d8c[
            "glmm_fdr05"
        ]

        |

        d8c[
            "pb_fdr05"
        ]
    ]
    .copy()
)


display(

    single_model_fdr[
        [
            "mouse_gene",
            "human_gene",

            "celltype",
            "compartment",

            "metastasis_median_6h_delta",

            "glmm_age_effect_oriented",
            "glmm_p_value",
            "glmm_fdr",

            "pseudobulk_old_vs_young_logFC",
            "pseudobulk_p_value",
            "pseudobulk_fdr",

            "crossmodel_same_direction"
        ]
    ]
    .sort_values(
        [
            "mouse_gene",
            "celltype"
        ]
    )
)


# ============================================================
# 6. AUTHOR-ALIGNED NOMINAL CROSS-MODEL OVERLAP
# ============================================================

print(
    "\n======================================"
)

print(
    "NOMINAL CROSS-MODEL HUMAN AGING OVERLAP"
)

print(
    "======================================"
)


nominal_overlap = (

    d8c[
        d8c[
            "crossmodel_nominal_p05_same_direction"
        ]
    ]
    .copy()
)


if len(
    nominal_overlap
) == 0:

    print(
        "No receptor/cell-type pair has p<0.05 in both "
        "human-aging models with concordant direction."
    )

else:

    display(

        nominal_overlap[
            [
                "mouse_gene",
                "human_gene",

                "celltype",
                "compartment",

                "metastasis_median_6h_delta",

                "glmm_age_effect_oriented",
                "glmm_p_value",
                "glmm_fdr",

                "pseudobulk_old_vs_young_logFC",
                "pseudobulk_p_value",
                "pseudobulk_fdr",

                "nominal_human_age_same_metastasis_direction",
                "nominal_human_age_opposite_metastasis_direction"
            ]
        ]
        .sort_values(
            [
                "mouse_gene",
                "celltype"
            ]
        )
    )


# ============================================================
# 7. PER-RECEPTOR NOMINAL HUMAN-AGING SUMMARY
# ============================================================

receptor_rows = []


for mouse_gene, g in d8c.groupby(
    "mouse_gene"
):

    nominal = (

        g[
            g[
                "crossmodel_nominal_p05_same_direction"
            ]
        ]
        .copy()
    )


    nominal_same_met = (

        nominal[
            nominal[
                "nominal_human_age_same_metastasis_direction"
            ]
        ]
    )


    nominal_opp_met = (

        nominal[
            nominal[
                "nominal_human_age_opposite_metastasis_direction"
            ]
        ]
    )


    receptor_rows.append({

        "gene":
            mouse_gene,

        "human_gene":
            g[
                "human_gene"
            ].iloc[0],

        "median_6h_delta":
            float(
                g[
                    "metastasis_median_6h_delta"
                ].iloc[0]
            ),

        "n_nominal_crossmodel_celltypes":
            len(
                nominal
            ),

        "n_nominal_crossmodel_endothelial":
            int(
                (
                    nominal[
                        "compartment"
                    ]
                    ==
                    "endothelial"
                ).sum()
            ),

        "n_nominal_crossmodel_mural":
            int(
                (
                    nominal[
                        "compartment"
                    ]
                    ==
                    "mural"
                ).sum()
            ),

        "nominal_human_age_celltypes":
            ";".join(
                nominal[
                    "celltype"
                ].tolist()
            ),

        "nominal_same_metastasis_celltypes":
            ";".join(
                nominal_same_met[
                    "celltype"
                ].tolist()
            ),

        "nominal_opposite_metastasis_celltypes":
            ";".join(
                nominal_opp_met[
                    "celltype"
                ].tolist()
            ),

        "any_nominal_human_age":
            (
                len(
                    nominal
                )
                >
                0
            ),

        "any_nominal_same_metastasis":
            (
                len(
                    nominal_same_met
                )
                >
                0
            ),

        "any_nominal_opposite_metastasis":
            (
                len(
                    nominal_opp_met
                )
                >
                0
            )
    })


d8c_receptor_summary = pd.DataFrame(
    receptor_rows
)


# ============================================================
# 8. MERGE PSN + MOUSE-AGING CONTEXT
# ============================================================

context_cols = [

    "gene",
    "PSN_ligands",

    "set1_GSE235394_delta_log2cpm1",
    "set2_GSE253749_delta_log2cpm1",
    "set3_GSE253751_delta_log2cpm1",

    "blood_age_age_delta_log2cpm",
    "blood_age_fdr_bh",

    "comp_age_coef",
    "comp_age_fdr_bh",

    "age_signal_class",

    "mural_support_n_mice_detected",
    "mural_support_max_pct_expressing",

    "mural_age_age_delta_log2cpm",
    "mural_age_fdr_bh"
]


context_cols = [

    c
    for c in context_cols
    if c in convergence.columns

]


d8c_receptor_summary = (

    d8c_receptor_summary
    .merge(

        convergence[
            context_cols
        ],

        on="gene",
        how="left",
        validate="one_to_one"
    )
)


# ============================================================
# 9. SHOW ONLY RECEPTORS WITH NOMINAL CROSS-MODEL HUMAN SUPPORT
#
# This is NOT a ranking.
# ============================================================

print(
    "\n======================================"
)

print(
    "RECEPTORS WITH NOMINAL HUMAN CROSS-MODEL SUPPORT"
)

print(
    "======================================"
)


nominal_receptor_view = (

    d8c_receptor_summary[
        d8c_receptor_summary[
            "any_nominal_human_age"
        ]
    ]
    .copy()
)


display(

    nominal_receptor_view[
        [
            "gene",
            "human_gene",
            "PSN_ligands",

            "median_6h_delta",

            "n_nominal_crossmodel_celltypes",
            "n_nominal_crossmodel_endothelial",
            "n_nominal_crossmodel_mural",

            "nominal_human_age_celltypes",

            "nominal_same_metastasis_celltypes",
            "nominal_opposite_metastasis_celltypes",

            "blood_age_age_delta_log2cpm",
            "blood_age_fdr_bh",

            "comp_age_coef",
            "comp_age_fdr_bh",

            "age_signal_class"
        ]
    ]
    .sort_values(
        [
            "n_nominal_crossmodel_celltypes",
            "gene"
        ],
        ascending=[
            False,
            True
        ]
    )
)


# ============================================================
# 10. SELECTED BIOLOGICAL AXES
# ============================================================

selected_axes = [

    "Bmpr2",
    "Cd36",
    "Cd74",
    "Itgb5",

    "Pdgfrb",
    "Ramp2",

    "Tnfrsf12a",
    "Plxnd1",
    "Mcam",

    "Mertk",
    "Ednrb"
]


print(
    "\n======================================"
)

print(
    "SELECTED AXES — AUTHOR-ALIGNED HUMAN SUPPORT"
)

print(
    "======================================"
)


display(

    d8c_receptor_summary[
        d8c_receptor_summary[
            "gene"
        ]
        .isin(
            selected_axes
        )
    ][
        [
            "gene",
            "human_gene",
            "PSN_ligands",

            "median_6h_delta",

            "n_nominal_crossmodel_celltypes",
            "n_nominal_crossmodel_endothelial",
            "n_nominal_crossmodel_mural",

            "nominal_human_age_celltypes",

            "nominal_same_metastasis_celltypes",
            "nominal_opposite_metastasis_celltypes",

            "blood_age_age_delta_log2cpm",
            "blood_age_fdr_bh",

            "comp_age_coef",
            "comp_age_fdr_bh"
        ]
    ]
    .sort_values(
        "gene"
    )
)


# ============================================================
# 11. SAVE
# ============================================================

d8c.to_csv(

    OUT
    /
    "DeMan2026_60receptors_author_aligned_human_age_long.csv",

    index=False
)


d8c_receptor_summary.to_csv(

    OUT
    /
    "DeMan2026_60receptors_author_aligned_human_age_summary.csv",

    index=False
)


# ============================================================
# 12. SANITY CHECKS
# ============================================================

assert d8c[
    "mouse_gene"
].nunique() == 60


assert len(
    d8c_receptor_summary
) == 60


assert not d8c_receptor_summary[
    "gene"
].duplicated().any()


# D8B stringent result must remain zero.
assert int(

    d8c[
        "crossmodel_fdr05_same_direction"
    ].sum()

) == 0


# ============================================================
# 13. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE D8C AUTHOR-ALIGNED HUMAN AGING AUDIT COMPLETE"
)

In [ ]:
# ============================================================
# PHASE D8D — EVIDENCE MATRIX FOR THE 11 HUMAN-AGING
#             CROSS-MODEL CANDIDATES
#
# Starting point:
#
#   11 receptors with:
#     - replicated 6 h metastatic response in all 3 experiments
#     - nominal p < 0.05 in BOTH human-aging models
#     - concordant direction between the 2 human-aging models
#
# Goals:
#
#   1. Separate age/metastasis direction classes:
#        same
#        opposite
#        mixed
#
#   2. Quantify robustness of the 3-experiment metastatic
#      effect without ranking receptors.
#
#   3. Recover exact PSN ligand evidence from B4:
#        support tier
#        mean TPM
#        max subtype mean
#        top subtype
#        subtype cell number
#        small-subtype warning
#
#   4. Carry forward mouse-aging and mural evidence.
#
# NO biological winner.
# NO novelty claim.
# NO integrated score.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


print(
    "======================================"
)

print(
    "PHASE D8D — 11-CANDIDATE EVIDENCE MATRIX"
)

print(
    "======================================"
)


# ============================================================
# 1. RESTORE PROJECT PATHS IF RUNTIME WAS RESET
# ============================================================

if "OUT" not in globals():

    if not Path(
        "/content/drive/MyDrive"
    ).exists():

        from google.colab import drive

        drive.mount(
            "/content/drive"
        )


    ROOT = Path(
        "/content/drive/MyDrive/KT_Neurovascular_Screen"
    )

    RAW = ROOT / "raw"
    EXTRACTED = ROOT / "extracted"
    CACHE = ROOT / "cache"
    OUT = ROOT / "outputs"
    FIG = OUT / "figures"


print(
    "OUT:",
    OUT
)


# ============================================================
# 2. RESTORE D8C LONG HUMAN-AGING TABLE
# ============================================================

d8c_path = (

    OUT
    /
    "DeMan2026_60receptors_author_aligned_human_age_long.csv"
)


if not (
    "d8c" in globals()
    and
    isinstance(
        d8c,
        pd.DataFrame
    )
    and
    len(
        d8c
    ) > 0
):

    if not d8c_path.exists():

        raise FileNotFoundError(
            d8c_path
        )


    d8c = pd.read_csv(
        d8c_path
    )


    print(
        "Restored D8C long table:",
        d8c.shape
    )


# ============================================================
# 3. RESTORE D7B CONVERGENCE TABLE
# ============================================================

convergence_path = (

    OUT
    /
    "PSN_receptor_metastasis_replication_with_aging_evidence.csv"
)


if not (
    "convergence" in globals()
    and
    isinstance(
        convergence,
        pd.DataFrame
    )
    and
    len(
        convergence
    ) > 0
):

    if not convergence_path.exists():

        raise FileNotFoundError(
            convergence_path
        )


    convergence = pd.read_csv(
        convergence_path
    )


    print(
        "Restored convergence table:",
        convergence.shape
    )


# ============================================================
# 4. RESTORE B4 PSN LIGAND-RECEPTOR TABLE
# ============================================================

psn_lr_path = (

    OUT
    /
    "PSN_detected_ligand_receptor_pairs.csv"
)


if not psn_lr_path.exists():

    raise FileNotFoundError(
        psn_lr_path
    )


psn_lr = pd.read_csv(
    psn_lr_path
)


required_psn_columns = [

    "mouse_ligand",
    "mouse_receptor",

    "psn_support_tier",
    "psn_expression_score",

    "psn_mean_tpm",
    "psn_max_subtype_mean",
    "psn_max_subtype_pct_tpm_ge_1",

    "psn_top_subtype",
    "psn_top_subtype_n_cells",
    "psn_small_top_subtype_flag"
]


missing_psn = [

    col

    for col in required_psn_columns

    if col not in psn_lr.columns
]


if missing_psn:

    raise RuntimeError(

        "B4 PSN table missing expected columns: "
        +
        str(
            missing_psn
        )
    )


print(
    "PSN LR table:",
    psn_lr.shape
)


# ============================================================
# 5. IDENTIFY THE EXACT 11 D8C RECEPTORS
# ============================================================

candidate_long = (

    d8c[
        d8c[
            "crossmodel_nominal_p05_same_direction"
        ]
        .astype(bool)
    ]
    .copy()
)


candidate_genes = sorted(

    candidate_long[
        "mouse_gene"
    ]
    .unique()
)


print(
    "\n======================================"
)

print(
    "D8C CANDIDATE UNIVERSE"
)

print(
    "======================================"
)


print(
    "Candidate receptors:",
    len(
        candidate_genes
    )
)


print(
    candidate_genes
)


if len(
    candidate_genes
) != 11:

    raise RuntimeError(

        "Expected 11 D8C candidate receptors; "
        f"found {len(candidate_genes)}."
    )


# ============================================================
# 6. BUILD HUMAN-AGING DIRECTION SUMMARY
# ============================================================

human_summary_rows = []


for gene in candidate_genes:

    g = (

        candidate_long[
            candidate_long[
                "mouse_gene"
            ]
            ==
            gene
        ]
        .copy()
    )


    same_met = (

        g[
            g[
                "nominal_human_age_same_metastasis_direction"
            ]
            .astype(bool)
        ]
    )


    opposite_met = (

        g[
            g[
                "nominal_human_age_opposite_metastasis_direction"
            ]
            .astype(bool)
        ]
    )


    if (
        len(
            same_met
        ) > 0
        and
        len(
            opposite_met
        ) == 0
    ):

        direction_class = (
            "same_as_metastasis"
        )


    elif (
        len(
            opposite_met
        ) > 0
        and
        len(
            same_met
        ) == 0
    ):

        direction_class = (
            "opposite_to_metastasis"
        )


    else:

        direction_class = (
            "mixed_across_celltypes"
        )


    human_summary_rows.append({

        "gene":
            gene,

        "human_gene":
            g[
                "human_gene"
            ]
            .iloc[
                0
            ],

        "human_age_direction_class":
            direction_class,

        "n_human_age_celltypes":
            len(
                g
            ),

        "n_human_age_endothelial":
            int(
                (
                    g[
                        "compartment"
                    ]
                    ==
                    "endothelial"
                )
                .sum()
            ),

        "n_human_age_mural":
            int(
                (
                    g[
                        "compartment"
                    ]
                    ==
                    "mural"
                )
                .sum()
            ),

        "human_age_celltypes":
            ";".join(
                g[
                    "celltype"
                ]
                .tolist()
            ),

        "same_metastasis_celltypes":
            ";".join(
                same_met[
                    "celltype"
                ]
                .tolist()
            ),

        "opposite_metastasis_celltypes":
            ";".join(
                opposite_met[
                    "celltype"
                ]
                .tolist()
            ),

        "best_glmm_p":
            g[
                "glmm_p_value"
            ]
            .min(),

        "best_glmm_fdr":
            g[
                "glmm_fdr"
            ]
            .min(),

        "best_pseudobulk_p":
            g[
                "pseudobulk_p_value"
            ]
            .min(),

        "best_pseudobulk_fdr":
            g[
                "pseudobulk_fdr"
            ]
            .min()
    })


human_summary = pd.DataFrame(
    human_summary_rows
)


# ============================================================
# 7. EXTRACT 3-EXPERIMENT METASTATIC RESPONSE
# ============================================================

metastasis_columns = [

    "set1_GSE235394_delta_log2cpm1",
    "set2_GSE253749_delta_log2cpm1",
    "set3_GSE253751_delta_log2cpm1"
]


missing_metastasis = [

    c

    for c in metastasis_columns

    if c not in convergence.columns
]


if missing_metastasis:

    raise RuntimeError(

        "Missing metastatic replication columns: "
        +
        str(
            missing_metastasis
        )
    )


base = (

    convergence[
        convergence[
            "gene"
        ]
        .isin(
            candidate_genes
        )
    ]
    .copy()
)


if len(
    base
) != 11:

    raise RuntimeError(

        "Expected 11 candidates in convergence table; "
        f"found {len(base)}."
    )


base[
    "min_abs_6h_delta"
] = (

    base[
        metastasis_columns
    ]
    .abs()
    .min(
        axis=1
    )
)


base[
    "max_abs_6h_delta"
] = (

    base[
        metastasis_columns
    ]
    .abs()
    .max(
        axis=1
    )
)


base[
    "mean_abs_6h_delta"
] = (

    base[
        metastasis_columns
    ]
    .abs()
    .mean(
        axis=1
    )
)


# ============================================================
# 8. EXACT PSN LIGAND EVIDENCE FOR THE 11 RECEPTORS
# ============================================================

candidate_psn_lr = (

    psn_lr[
        psn_lr[
            "mouse_receptor"
        ]
        .isin(
            candidate_genes
        )
    ][
        required_psn_columns
    ]
    .drop_duplicates()
    .copy()
)


candidate_psn_lr[
    "psn_mean_tpm"
] = pd.to_numeric(

    candidate_psn_lr[
        "psn_mean_tpm"
    ],

    errors="coerce"
)


candidate_psn_lr[
    "psn_expression_score"
] = pd.to_numeric(

    candidate_psn_lr[
        "psn_expression_score"
    ],

    errors="coerce"
)


print(
    "\nCandidate PSN LR rows:",
    len(
        candidate_psn_lr
    )
)


missing_lr_candidates = sorted(

    set(
        candidate_genes
    )
    -
    set(
        candidate_psn_lr[
            "mouse_receptor"
        ]
        .unique()
    )
)


if missing_lr_candidates:

    raise RuntimeError(

        "D8C candidates missing from B4 PSN LR table: "
        +
        str(
            missing_lr_candidates
        )
    )


# ============================================================
# 9. PER-RECEPTOR PSN SOURCE SUMMARY
# ============================================================

psn_summary_rows = []


for receptor, g in candidate_psn_lr.groupby(
    "mouse_receptor"
):

    g = (
        g
        .copy()
        .sort_values(
            "psn_mean_tpm",
            ascending=False
        )
    )


    top = g.iloc[
        0
    ]


    strong_ligands = sorted(

        g.loc[
            g[
                "psn_support_tier"
            ]
            .astype(str)
            .str.lower()
            ==
            "strong",

            "mouse_ligand"
        ]
        .astype(str)
        .unique()
    )


    supported_ligands = sorted(

        g[
            "mouse_ligand"
        ]
        .astype(str)
        .unique()
    )


    psn_summary_rows.append({

        "gene":
            receptor,

        "n_psn_ligands":
            len(
                supported_ligands
            ),

        "psn_ligands_exact":
            ";".join(
                supported_ligands
            ),

        "n_strong_psn_ligands":
            len(
                strong_ligands
            ),

        "strong_psn_ligands":
            ";".join(
                strong_ligands
            ),

        "top_psn_ligand":
            str(
                top[
                    "mouse_ligand"
                ]
            ),

        "top_psn_ligand_support_tier":
            str(
                top[
                    "psn_support_tier"
                ]
            ),

        "top_psn_ligand_mean_tpm":
            float(
                top[
                    "psn_mean_tpm"
                ]
            ),

        "top_psn_ligand_expression_score":
            float(
                top[
                    "psn_expression_score"
                ]
            ),

        "top_psn_subtype":
            str(
                top[
                    "psn_top_subtype"
                ]
            ),

        "top_psn_subtype_n_cells":
            int(
                top[
                    "psn_top_subtype_n_cells"
                ]
            ),

        "top_psn_small_subtype_flag":
            bool(
                top[
                    "psn_small_top_subtype_flag"
                ]
            )
    })


psn_summary = pd.DataFrame(
    psn_summary_rows
)


# ============================================================
# 10. BUILD COMPLETE 11-CANDIDATE MATRIX
# ============================================================

candidate_matrix = (

    base

    .merge(
        human_summary,
        on="gene",
        how="left",
        validate="one_to_one"
    )

    .merge(
        psn_summary,
        on="gene",
        how="left",
        validate="one_to_one"
    )
)


# ============================================================
# 11. DISPLAY COMPLETE CANDIDATE MATRIX
#
# Sorted only by biological direction class + gene.
# NOT by score.
# ============================================================

display_columns = [

    "gene",
    "human_gene",

    "PSN_ligands",

    "n_psn_ligands",
    "n_strong_psn_ligands",
    "strong_psn_ligands",

    "top_psn_ligand",
    "top_psn_ligand_support_tier",
    "top_psn_ligand_mean_tpm",
    "top_psn_subtype",
    "top_psn_subtype_n_cells",
    "top_psn_small_subtype_flag",

    "set1_GSE235394_delta_log2cpm1",
    "set2_GSE253749_delta_log2cpm1",
    "set3_GSE253751_delta_log2cpm1",

    "median_6h_delta",
    "min_abs_6h_delta",
    "mean_abs_6h_delta",

    "human_age_direction_class",

    "n_human_age_celltypes",
    "n_human_age_endothelial",
    "n_human_age_mural",

    "human_age_celltypes",

    "same_metastasis_celltypes",
    "opposite_metastasis_celltypes",

    "best_glmm_p",
    "best_glmm_fdr",

    "best_pseudobulk_p",
    "best_pseudobulk_fdr",

    "blood_age_age_delta_log2cpm",
    "blood_age_fdr_bh",

    "comp_age_coef",
    "comp_age_fdr_bh",

    "age_signal_class",

    "mural_support_n_mice_detected",
    "mural_support_max_pct_expressing",

    "mural_age_age_delta_log2cpm",
    "mural_age_fdr_bh"
]


display_columns = [

    c

    for c in display_columns

    if c in candidate_matrix.columns
]


print(
    "\n======================================"
)

print(
    "11-CANDIDATE INTEGRATED EVIDENCE MATRIX"
)

print(
    "======================================"
)


display(

    candidate_matrix[
        display_columns
    ]
    .sort_values(
        [
            "human_age_direction_class",
            "gene"
        ]
    )
)


# ============================================================
# 12. DIRECTION-CLASS COUNTS
# ============================================================

print(
    "\n======================================"
)

print(
    "AGE × METASTASIS DIRECTION CLASSES"
)

print(
    "======================================"
)


display(

    human_summary[
        "human_age_direction_class"
    ]
    .value_counts()
    .rename_axis(
        "class"
    )
    .reset_index(
        name="n_receptors"
    )
)


# ============================================================
# 13. SAME-DIRECTION FOUR
# ============================================================

print(
    "\n======================================"
)

print(
    "SAME-DIRECTION HUMAN-AGING CANDIDATES"
)

print(
    "======================================"
)


same_direction_view = (

    candidate_matrix[
        candidate_matrix[
            "human_age_direction_class"
        ]
        ==
        "same_as_metastasis"
    ]
    .copy()
)


display(

    same_direction_view[
        display_columns
    ]
    .sort_values(
        "gene"
    )
)


# ============================================================
# 14. OPPOSITE-DIRECTION SEVEN
# ============================================================

print(
    "\n======================================"
)

print(
    "OPPOSITE-DIRECTION HUMAN-AGING CANDIDATES"
)

print(
    "======================================"
)


opposite_direction_view = (

    candidate_matrix[
        candidate_matrix[
            "human_age_direction_class"
        ]
        ==
        "opposite_to_metastasis"
    ]
    .copy()
)


display(

    opposite_direction_view[
        display_columns
    ]
    .sort_values(
        "gene"
    )
)


# ============================================================
# 15. LIGAND-LEVEL PSN SOURCE AUDIT
# ============================================================

print(
    "\n======================================"
)

print(
    "LIGAND-LEVEL PSN SOURCE EVIDENCE"
)

print(
    "======================================"
)


display(

    candidate_psn_lr[
        [
            "mouse_receptor",
            "mouse_ligand",

            "psn_support_tier",
            "psn_expression_score",
            "psn_mean_tpm",

            "psn_max_subtype_mean",
            "psn_max_subtype_pct_tpm_ge_1",

            "psn_top_subtype",
            "psn_top_subtype_n_cells",
            "psn_small_top_subtype_flag"
        ]
    ]
    .sort_values(
        [
            "mouse_receptor",
            "psn_mean_tpm"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# ============================================================
# 16. SAVE
# ============================================================

candidate_matrix.to_csv(

    OUT
    /
    "D8D_11candidate_integrated_evidence_matrix.csv",

    index=False
)


candidate_psn_lr.to_csv(

    OUT
    /
    "D8D_11candidate_PSN_ligand_source_evidence.csv",

    index=False
)


# ============================================================
# 17. SANITY CHECKS
# ============================================================

assert len(
    candidate_matrix
) == 11


assert not candidate_matrix[
    "gene"
].duplicated().any()


assert set(
    candidate_matrix[
        "gene"
    ]
) == set(
    candidate_genes
)


# ============================================================
# 18. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE D8D 11-CANDIDATE EVIDENCE MATRIX COMPLETE"
)

In [ ]:
# ============================================================
# PHASE D9A — AUDIT GSE275770 VAGAL-SENSORY DATA STRUCTURE
#
# Dataset:
#   Wei et al., Nature 2026
#   GSE275770
#
# Biological use:
#   Independent neuronal-side validation of whether candidate
#   PSN ligands are altered in vagal sensory neurons by lung
#   tumour presence.
#
# IMPORTANT:
#   This is PRIMARY lung adenocarcinoma, not metastatic seeding.
#
# THIS CELL:
#   - downloads only barcode + feature files
#   - inspects whether replicate identity survives in barcodes
#   - confirms candidate ligands are in the matrix
#   - audits the authors' GitHub repository for processed
#     metadata / RData / RDS objects and relevant scripts
#
# NO matrix download yet.
# NO differential expression.
# ============================================================

from pathlib import Path
from collections import Counter

import gzip
import json
import re
import requests
import pandas as pd


print("======================================")
print("PHASE D9A — GSE275770 STRUCTURE AUDIT")
print("======================================")


# ============================================================
# 1. RESTORE PROJECT PATHS IF NEEDED
# ============================================================

if "RAW" not in globals():

    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")

    ROOT = Path(
        "/content/drive/MyDrive/KT_Neurovascular_Screen"
    )

    RAW = ROOT / "raw"
    OUT = ROOT / "outputs"


g275_dir = RAW / "GSE275770"

g275_dir.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "Directory:",
    g275_dir
)


# ============================================================
# 2. DOWNLOAD SMALL BARCODE / FEATURE FILES ONLY
# ============================================================

geo_base = (
    "https://ftp.ncbi.nlm.nih.gov/geo/"
    "series/GSE275nnn/GSE275770/suppl/"
)


small_files = [

    "GSE275770_kp3_barcodes.tsv.gz",
    "GSE275770_kp3_features.tsv.gz",

    "GSE275770_wt_barcodes.tsv.gz",
    "GSE275770_wt_features.tsv.gz"
]


headers = {
    "User-Agent":
        "Mozilla/5.0 KT-neurovascular-screen/1.0"
}


for filename in small_files:

    path = g275_dir / filename

    if not path.exists():

        print(
            "Downloading:",
            filename
        )

        r = requests.get(
            geo_base + filename,
            headers=headers,
            timeout=120
        )

        r.raise_for_status()

        path.write_bytes(
            r.content
        )

    print(
        filename,
        "| MB:",
        round(
            path.stat().st_size / 1e6,
            4
        )
    )


# ============================================================
# 3. READ BARCODES
# ============================================================

def read_gz_lines(path):

    with gzip.open(
        path,
        "rt"
    ) as f:

        return [
            line.strip()
            for line in f
            if line.strip()
        ]


barcodes = {}


for condition in [
    "kp3",
    "wt"
]:

    path = (
        g275_dir
        /
        f"GSE275770_{condition}_barcodes.tsv.gz"
    )

    vals = read_gz_lines(
        path
    )

    barcodes[
        condition
    ] = vals


    print(
        "\n======================================"
    )

    print(
        f"{condition.upper()} BARCODES"
    )

    print(
        "======================================"
    )


    print(
        "N barcodes:",
        len(vals)
    )


    print(
        "First 20:"
    )

    print(
        vals[:20]
    )


    print(
        "Last 20:"
    )

    print(
        vals[-20:]
    )


# ============================================================
# 4. AUDIT BARCODE SUFFIXES FOR REPLICATE IDENTITY
#
# Cell Ranger aggregation commonly preserves sample identity
# as barcode suffixes such as -1, -2, -3, -4.
# We DO NOT assume this — we inspect it.
# ============================================================

barcode_suffix_summary = []


for condition, vals in barcodes.items():

    suffixes = []

    for barcode in vals:

        m = re.search(
            r"-([0-9]+)$",
            barcode
        )

        suffixes.append(
            m.group(1)
            if m
            else "NO_NUMERIC_SUFFIX"
        )


    counts = Counter(
        suffixes
    )


    print(
        "\n======================================"
    )

    print(
        f"{condition.upper()} BARCODE SUFFIX COUNTS"
    )

    print(
        "======================================"
    )


    for suffix, n in sorted(
        counts.items()
    ):

        print(
            suffix,
            "=>",
            n
        )


        barcode_suffix_summary.append({

            "condition":
                condition,

            "suffix":
                suffix,

            "n_cells":
                n
        })


barcode_suffix_summary = pd.DataFrame(
    barcode_suffix_summary
)


# ============================================================
# 5. READ FEATURES AND VERIFY GENE SYMBOL COLUMN
# ============================================================

features = {}


for condition in [
    "kp3",
    "wt"
]:

    path = (
        g275_dir
        /
        f"GSE275770_{condition}_features.tsv.gz"
    )


    feat = pd.read_csv(
        path,
        sep="\t",
        header=None
    )


    features[
        condition
    ] = feat


    print(
        "\n======================================"
    )

    print(
        f"{condition.upper()} FEATURES"
    )

    print(
        "======================================"
    )


    print(
        "Shape:",
        feat.shape
    )


    print(
        "First 10 rows:"
    )


    display(
        feat.head(
            10
        )
    )


# ============================================================
# 6. DETERMINE LIKELY GENE SYMBOL COLUMN
# ============================================================

candidate_symbols = {

    # Current literature-priority axes
    "Flrt1",
    "Flrt2",
    "Flrt3",

    "Edn1",
    "Edn2",
    "Edn3",

    "Ndp",
    "Wnt5a",

    "Sema7a",

    # Remaining D8D ligands
    "Bmp2",
    "Bmp7",
    "Gdf9",

    "Wnt4",

    "Chad",

    "Calr",
    "Lama4",
    "Lamb1",
    "Lamb3",
    "Lamc1",
    "Lamc2",
    "Thbs1",

    "Angpt1",
    "L1cam",
    "Spp1",

    "Ltbp3",

    "Pgf",
    "Sema3a",
    "Sema3b",
    "Sema3c",
    "Sema3f",
    "Vegfa",
    "Vegfc"
}


feature_audit_rows = []


for condition, feat in features.items():

    best_col = None
    best_overlap = -1
    best_found = set()


    for col in feat.columns:

        values = set(
            feat[
                col
            ]
            .astype(str)
        )


        overlap = (
            candidate_symbols
            &
            values
        )


        if len(overlap) > best_overlap:

            best_overlap = len(
                overlap
            )

            best_col = col

            best_found = overlap


    print(
        "\nCondition:",
        condition
    )

    print(
        "Likely gene-symbol column:",
        best_col
    )

    print(
        "Candidate ligands found:",
        len(
            best_found
        ),
        "/",
        len(
            candidate_symbols
        )
    )


    print(
        "Found:"
    )

    print(
        sorted(
            best_found
        )
    )


    print(
        "Missing:"
    )

    print(
        sorted(
            candidate_symbols
            -
            best_found
        )
    )


    for gene in sorted(
        candidate_symbols
    ):

        feature_audit_rows.append({

            "condition":
                condition,

            "gene":
                gene,

            "present":
                gene in best_found
        })


feature_audit = pd.DataFrame(
    feature_audit_rows
)


# ============================================================
# 7. AUDIT AUTHOR GITHUB REPOSITORY
#
# The paper states that analysis scripts are public at:
# KWhh666/Manuscript_2024
#
# We inspect the repository tree rather than assuming which
# processed objects or metadata files exist.
# ============================================================

print(
    "\n======================================"
)

print(
    "AUTHOR GITHUB REPOSITORY AUDIT"
)

print(
    "======================================"
)


repo_tree_url = (
    "https://api.github.com/repos/"
    "KWhh666/Manuscript_2024/git/trees/main?recursive=1"
)


repo_response = requests.get(
    repo_tree_url,
    headers=headers,
    timeout=60
)


repo_response.raise_for_status()


repo_json = repo_response.json()


tree = repo_json.get(
    "tree",
    []
)


repo_paths = [

    item.get(
        "path",
        ""
    )

    for item in tree
]


print(
    "Repository entries:",
    len(
        repo_paths
    )
)


interesting_terms = [

    "rds",
    "rdata",
    "metadata",
    "meta",
    "cluster",
    "marker",
    "single",
    "seurat",
    "fig.1"
]


interesting_paths = sorted(

    {
        path

        for path in repo_paths

        if any(
            term in path.lower()
            for term in interesting_terms
        )
    }
)


print(
    "\nPotentially useful repository files:"
)


for path in interesting_paths:

    print(
        " -",
        path
    )


# ============================================================
# 8. AUDIT RELEASE ASSETS
#
# README refers to RData/RDS resources; some large files may
# be attached to GitHub Releases rather than normal Git tree.
# ============================================================

print(
    "\n======================================"
)

print(
    "GITHUB RELEASE ASSETS"
)

print(
    "======================================"
)


release_url = (
    "https://api.github.com/repos/"
    "KWhh666/Manuscript_2024/releases"
)


release_response = requests.get(
    release_url,
    headers=headers,
    timeout=60
)


release_response.raise_for_status()


releases = release_response.json()


release_rows = []


for release in releases:

    print(
        "\nRelease:",
        release.get(
            "name"
        ),
        "| tag:",
        release.get(
            "tag_name"
        )
    )


    assets = release.get(
        "assets",
        []
    )


    if not assets:

        print(
            "  No assets."
        )


    for asset in assets:

        print(
            "  ",
            asset.get(
                "name"
            ),
            "| MB:",
            round(
                asset.get(
                    "size",
                    0
                )
                /
                1e6,
                3
            )
        )


        release_rows.append({

            "release":
                release.get(
                    "name"
                ),

            "tag":
                release.get(
                    "tag_name"
                ),

            "asset":
                asset.get(
                    "name"
                ),

            "size_mb":
                asset.get(
                    "size",
                    0
                )
                /
                1e6,

            "download_url":
                asset.get(
                    "browser_download_url"
                )
        })


release_assets = pd.DataFrame(
    release_rows
)


# ============================================================
# 9. SAVE AUDIT TABLES
# ============================================================

barcode_suffix_summary.to_csv(

    OUT
    /
    "GSE275770_barcode_suffix_audit.csv",

    index=False
)


feature_audit.to_csv(

    OUT
    /
    "GSE275770_candidate_ligand_feature_audit.csv",

    index=False
)


if len(
    release_assets
):

    release_assets.to_csv(

        OUT
        /
        "GSE275770_GitHub_release_asset_audit.csv",

        index=False
    )


# ============================================================
# 10. FINAL CHECKPOINT
# ============================================================

print(
    "\n======================================"
)

print(
    "GSE275770 STRUCTURE DECISION"
)

print(
    "======================================"
)


print(
    "KP3 unique numeric barcode suffixes:",
    sorted(
        barcode_suffix_summary.loc[
            barcode_suffix_summary[
                "condition"
            ]
            ==
            "kp3",
            "suffix"
        ]
        .astype(str)
        .tolist()
    )
)


print(
    "WT unique numeric barcode suffixes:",
    sorted(
        barcode_suffix_summary.loc[
            barcode_suffix_summary[
                "condition"
            ]
            ==
            "wt",
            "suffix"
        ]
        .astype(str)
        .tolist()
    )
)


print(
    "\nPHASE D9A GSE275770 STRUCTURE AUDIT COMPLETE"
)

In [ ]:
# ============================================================
# PHASE D9B — GSE275770 SAMPLE-PROVENANCE + AUTHOR-ANALYSIS AUDIT
#
# D9A showed:
#
#   - condition-level processed matrices
#   - all barcodes end in -1
#   - therefore processed matrices do NOT preserve sample ID
#
# But GEO contains 8 raw SRA/GSM records:
#
#   KP tumour:
#       GSM8484333
#       GSM8484334
#       GSM8484335
#       GSM8484336
#
#   WT:
#       GSM8484337
#       GSM8484338
#       GSM8484339
#       GSM8484340
#
# THIS CELL:
#
#   1. audits exact GEO sample metadata
#   2. audits author Fig.1 / ED.Fig.1 R scripts
#   3. checks whether scripts preserve sample/orig.ident
#   4. audits Raw_Txt resources
#   5. checks Zenodo for missing RDS/RData files
#   6. downloads only the small official Source Data
#      Fig.1 and Extended Data Fig.1 workbooks
#   7. searches those source-data workbooks for our
#      32 candidate neuronal ligands
#
# NO count matrix download.
# NO SRA FASTQ download.
# NO differential expression.
# ============================================================

from pathlib import Path

import base64
import io
import re
import requests
import pandas as pd


print(
    "======================================"
)

print(
    "PHASE D9B — GSE275770 PROVENANCE AUDIT"
)

print(
    "======================================"
)


# ============================================================
# 1. RESTORE PROJECT PATHS
# ============================================================

if "RAW" not in globals():

    if not Path(
        "/content/drive/MyDrive"
    ).exists():

        from google.colab import drive

        drive.mount(
            "/content/drive"
        )


    ROOT = Path(
        "/content/drive/MyDrive/KT_Neurovascular_Screen"
    )

    RAW = ROOT / "raw"
    OUT = ROOT / "outputs"


g275_dir = (
    RAW
    /
    "GSE275770"
)


g275_dir.mkdir(
    parents=True,
    exist_ok=True
)


headers = {

    "User-Agent":
        "Mozilla/5.0 KT-neurovascular-screen/1.0"
}


# ============================================================
# 2. EXACT GEO SAMPLE INVENTORY
# ============================================================

gsm_design = {

    "GSM8484333": {
        "condition": "KP",
        "submitted_label": "I1"
    },

    "GSM8484334": {
        "condition": "KP",
        "submitted_label": "R1"
    },

    "GSM8484335": {
        "condition": "KP",
        "submitted_label": "R2"
    },

    "GSM8484336": {
        "condition": "KP",
        "submitted_label": "R3"
    },

    "GSM8484337": {
        "condition": "WT",
        "submitted_label": "I1"
    },

    "GSM8484338": {
        "condition": "WT",
        "submitted_label": "I2"
    },

    "GSM8484339": {
        "condition": "WT",
        "submitted_label": "I3"
    },

    "GSM8484340": {
        "condition": "WT",
        "submitted_label": "I4"
    }
}


def extract_geo_field(
    text,
    field
):

    pattern = re.compile(
        rf"^!Sample_{re.escape(field)}\s*=\s*(.*)$",
        flags=re.MULTILINE
    )

    match = pattern.search(
        text
    )

    if match:

        return match.group(
            1
        ).strip()

    return None


def extract_all_geo_fields(
    text,
    field
):

    pattern = re.compile(
        rf"^!Sample_{re.escape(field)}\s*=\s*(.*)$",
        flags=re.MULTILINE
    )

    return [

        m.strip()

        for m in pattern.findall(
            text
        )
    ]


geo_sample_rows = []


print(
    "\n======================================"
)

print(
    "GEO SAMPLE-LEVEL METADATA"
)

print(
    "======================================"
)


for gsm, design in gsm_design.items():

    geo_url = (

        "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi"
        f"?acc={gsm}"
        "&targ=self"
        "&form=text"
        "&view=full"
    )


    r = requests.get(
        geo_url,
        headers=headers,
        timeout=60
    )

    r.raise_for_status()


    text = r.text


    title = extract_geo_field(
        text,
        "title"
    )


    source_name = extract_geo_field(
        text,
        "source_name_ch1"
    )


    characteristics = extract_all_geo_fields(
        text,
        "characteristics_ch1"
    )


    descriptions = extract_all_geo_fields(
        text,
        "description"
    )


    relations = extract_all_geo_fields(
        text,
        "relation"
    )


    srx = None
    biosample = None


    for relation in relations:

        m_srx = re.search(
            r"\b(SRX[0-9]+)\b",
            relation
        )

        if m_srx:

            srx = m_srx.group(
                1
            )


        m_bio = re.search(
            r"\b(SAMN[0-9]+)\b",
            relation
        )

        if m_bio:

            biosample = m_bio.group(
                1
            )


    geo_sample_rows.append({

        "gsm":
            gsm,

        "condition":
            design[
                "condition"
            ],

        "submitted_label":
            design[
                "submitted_label"
            ],

        "geo_title":
            title,

        "source_name":
            source_name,

        "characteristics":
            " | ".join(
                characteristics
            ),

        "description":
            " | ".join(
                descriptions
            ),

        "biosample":
            biosample,

        "srx":
            srx
    })


    print(
        "\n",
        gsm,
        "|",
        design[
            "condition"
        ],
        "|",
        design[
            "submitted_label"
        ]
    )


    print(
        "Title:",
        title
    )


    print(
        "Characteristics:",
        characteristics
    )


    print(
        "Descriptions:",
        descriptions
    )


    print(
        "BioSample:",
        biosample
    )


    print(
        "SRA experiment:",
        srx
    )


geo_sample_inventory = pd.DataFrame(
    geo_sample_rows
)


display(
    geo_sample_inventory
)


geo_sample_inventory.to_csv(

    OUT
    /
    "GSE275770_GSM_sample_inventory.csv",

    index=False
)


# ============================================================
# 3. CHECK WHETHER GEO PROVIDES AN ANIMAL / MOUSE IDENTIFIER
# ============================================================

animal_terms = [

    "mouse",
    "animal",
    "donor",
    "replicate",
    "biological",
    "pool",
    "pooled",
    "channel"
]


print(
    "\n======================================"
)

print(
    "GEO BIOLOGICAL-REPLICATE METADATA SEARCH"
)

print(
    "======================================"
)


for _, row in geo_sample_inventory.iterrows():

    metadata_text = (

        str(
            row[
                "characteristics"
            ]
        )
        +
        " "
        +
        str(
            row[
                "description"
            ]
        )
        +
        " "
        +
        str(
            row[
                "geo_title"
            ]
        )
    ).lower()


    hits = [

        term

        for term in animal_terms

        if term in metadata_text
    ]


    print(
        row[
            "gsm"
        ],
        "| hits:",
        hits
    )


# ============================================================
# 4. AUTHOR GITHUB TREE
# ============================================================

print(
    "\n======================================"
)

print(
    "AUTHOR SCRIPT PROVENANCE AUDIT"
)

print(
    "======================================"
)


tree_url = (

    "https://api.github.com/repos/"
    "KWhh666/Manuscript_2024/"
    "git/trees/main?recursive=1"
)


tree_response = requests.get(
    tree_url,
    headers=headers,
    timeout=60
)


tree_response.raise_for_status()


tree = tree_response.json().get(
    "tree",
    []
)


tree_by_path = {

    item.get(
        "path"
    ):
        item

    for item in tree
}


print(
    "Repository entries:",
    len(
        tree_by_path
    )
)


# ============================================================
# 5. EXACT FIGURE-1 SCRIPT SET
# ============================================================

script_paths = [

    "Source/Fig.1f.R",
    "Source/Fig.1g.R",
    "Source/Fig.1h.R",
    "Source/Fig.1i.R",

    "Source/ED.Fig.1g.R",
    "Source/ED.Fig.1h.R",
    "Source/ED.Fig.1i.R",
    "Source/ED.Fig.1j.R",
    "Source/ED.Fig.1k.R",
    "Source/ED.Fig.1l.R"
]


script_keywords = [

    "FindMarkers",
    "FindAllMarkers",

    "orig.ident",
    "sample",
    "replicate",

    "kp3",
    "wt",

    "Kcng1",
    "Phox2b",

    "seurat_cluster",
    "seurat_clusters",

    "subset",
    "WhichCells",

    "readRDS",
    "saveRDS",
    "load(",

    "RData",
    ".rds",

    "group.by",
    "ident.1",
    "ident.2"
]


script_audit_rows = []

script_texts = {}


def fetch_github_blob_text(
    item
):

    blob_url = item.get(
        "url"
    )


    if not blob_url:

        return None


    r = requests.get(
        blob_url,
        headers=headers,
        timeout=60
    )


    r.raise_for_status()


    payload = r.json()


    encoded = payload.get(
        "content",
        ""
    )


    return base64.b64decode(
        encoded
    ).decode(
        "utf-8",
        errors="replace"
    )


for path in script_paths:

    print(
        "\n--------------------------------------"
    )

    print(
        "SCRIPT:",
        path
    )


    item = tree_by_path.get(
        path
    )


    if item is None:

        print(
            "NOT FOUND"
        )

        continue


    text = fetch_github_blob_text(
        item
    )


    script_texts[
        path
    ] = text


    lines = text.splitlines()


    print(
        "Lines:",
        len(
            lines
        )
    )


    matched_lines = []


    for i, line in enumerate(
        lines,
        start=1
    ):

        if any(

            keyword.lower()
            in
            line.lower()

            for keyword
            in script_keywords
        ):

            matched_lines.append(
                (
                    i,
                    line
                )
            )


    if not matched_lines:

        print(
            "No audit keywords found."
        )

    else:

        for line_no, line in matched_lines:

            print(
                f"{line_no:4d}:",
                line
            )


            script_audit_rows.append({

                "script":
                    path,

                "line":
                    line_no,

                "text":
                    line
            })


script_audit = pd.DataFrame(
    script_audit_rows
)


if len(
    script_audit
):

    script_audit.to_csv(

        OUT
        /
        "GSE275770_author_Fig1_script_audit.csv",

        index=False
    )


# ============================================================
# 6. EXTRACT FILE REFERENCES FROM AUTHOR SCRIPTS
#
# Identify exactly which RDS/RData/text objects their code
# expects, even if those objects are absent from GitHub.
# ============================================================

print(
    "\n======================================"
)

print(
    "AUTHOR SCRIPT FILE REFERENCES"
)

print(
    "======================================"
)


file_reference_rows = []


file_pattern = re.compile(

    r"""(?i)
    ["']
    ([^"']*
    \.(?:rds|rdata|rda|txt|csv|tsv|xlsx))
    ["']
    """,

    flags=re.VERBOSE
)


for path, text in script_texts.items():

    refs = sorted(
        set(
            file_pattern.findall(
                text
            )
        )
    )


    print(
        "\n",
        path
    )


    if refs:

        for ref in refs:

            print(
                " -",
                ref
            )


            file_reference_rows.append({

                "script":
                    path,

                "file_reference":
                    ref
            })

    else:

        print(
            " No explicit data-file references detected."
        )


file_reference_audit = pd.DataFrame(
    file_reference_rows
)


if len(
    file_reference_audit
):

    file_reference_audit.to_csv(

        OUT
        /
        "GSE275770_author_script_file_references.csv",

        index=False
    )


# ============================================================
# 7. CURRENT Raw_Txt CONTENTS IN REPOSITORY
# ============================================================

print(
    "\n======================================"
)

print(
    "GITHUB Raw_Txt FILES"
)

print(
    "======================================"
)


raw_txt_items = [

    item

    for item in tree

    if str(
        item.get(
            "path",
            ""
        )
    ).startswith(
        "Raw_Txt/"
    )

    and

    item.get(
        "type"
    )
    ==
    "blob"
]


for item in raw_txt_items:

    print(
        item.get(
            "path"
        ),
        "| bytes:",
        item.get(
            "size"
        )
    )


# ============================================================
# 8. SEARCH SMALL Raw_Txt FILES FOR OUR 32 LIGANDS
# ============================================================

candidate_ligands = {

    "Flrt1",
    "Flrt2",
    "Flrt3",

    "Edn1",
    "Edn2",
    "Edn3",

    "Ndp",
    "Wnt5a",

    "Sema7a",

    "Bmp2",
    "Bmp7",
    "Gdf9",

    "Wnt4",

    "Chad",

    "Calr",
    "Lama4",
    "Lamb1",
    "Lamb3",
    "Lamc1",
    "Lamc2",
    "Thbs1",

    "Angpt1",
    "L1cam",
    "Spp1",

    "Ltbp3",

    "Pgf",
    "Sema3a",
    "Sema3b",
    "Sema3c",
    "Sema3f",
    "Vegfa",
    "Vegfc"
}


raw_txt_gene_hits = []


for item in raw_txt_items:

    size = item.get(
        "size",
        0
    ) or 0


    # Avoid downloading unexpectedly large files.
    if size > 2_000_000:

        continue


    try:

        text = fetch_github_blob_text(
            item
        )

    except Exception:

        continue


    if text is None:

        continue


    for gene in sorted(
        candidate_ligands
    ):

        if re.search(
            rf"(?<![A-Za-z0-9]){re.escape(gene)}(?![A-Za-z0-9])",
            text,
            flags=re.IGNORECASE
        ):

            raw_txt_gene_hits.append({

                "file":
                    item.get(
                        "path"
                    ),

                "gene":
                    gene
            })


raw_txt_gene_hits = pd.DataFrame(
    raw_txt_gene_hits
)


print(
    "\nCandidate-gene hits in Raw_Txt:"
)


if len(
    raw_txt_gene_hits
):

    display(

        raw_txt_gene_hits
        .sort_values(
            [
                "file",
                "gene"
            ]
        )
    )

else:

    print(
        "No candidate ligand names detected."
    )


# ============================================================
# 9. ZENODO RECORD AUDIT
#
# The published paper cites Zenodo record 17841956.
# Query metadata only; do NOT download large files here.
# ============================================================

print(
    "\n======================================"
)

print(
    "ZENODO RECORD AUDIT"
)

print(
    "======================================"
)


zenodo_url = (
    "https://zenodo.org/api/records/17841956"
)


zenodo_response = requests.get(
    zenodo_url,
    headers=headers,
    timeout=60
)


print(
    "HTTP status:",
    zenodo_response.status_code
)


zenodo_files = []


if zenodo_response.ok:

    zenodo_json = zenodo_response.json()


    print(
        "Title:",
        zenodo_json
        .get(
            "metadata",
            {}
        )
        .get(
            "title"
        )
    )


    files_meta = zenodo_json.get(
        "files",
        []
    )


    print(
        "Files:",
        len(
            files_meta
        )
    )


    for file_meta in files_meta:

        key = file_meta.get(
            "key"
        )


        size = file_meta.get(
            "size",
            0
        )


        print(
            " -",
            key,
            "| MB:",
            round(
                size / 1e6,
                3
            )
        )


        zenodo_files.append({

            "file":
                key,

            "size_mb":
                size / 1e6,

            "download_url":
                (
                    file_meta
                    .get(
                        "links",
                        {}
                    )
                    .get(
                        "content"
                    )
                )
        })


else:

    print(
        zenodo_response.text[
            :500
        ]
    )


zenodo_file_inventory = pd.DataFrame(
    zenodo_files
)


if len(
    zenodo_file_inventory
):

    zenodo_file_inventory.to_csv(

        OUT
        /
        "GSE275770_Zenodo_file_inventory.csv",

        index=False
    )


# ============================================================
# 10. OFFICIAL NATURE SOURCE-DATA WORKBOOKS
#
# Fig.1 contains the main scRNA-seq panels.
# Extended Data Fig.1 contains supporting scRNA-seq panels.
#
# Only small XLSX files are downloaded.
# ============================================================

source_data_dir = (

    g275_dir
    /
    "publication_source_data"
)


source_data_dir.mkdir(
    parents=True,
    exist_ok=True
)


source_workbooks = {

    "Source_Data_Fig1":

        (
            "https://media.springernature.com/original/"
            "springer-static/esm/"
            "art%3A10.1038%2Fs41586-025-10028-8/"
            "MediaObjects/41586_2025_10028_MOESM5_ESM.xlsx"
        ),

    "Source_Data_ED_Fig1":

        (
            "https://media.springernature.com/original/"
            "springer-static/esm/"
            "art%3A10.1038%2Fs41586-025-10028-8/"
            "MediaObjects/41586_2025_10028_MOESM10_ESM.xlsx"
        )
}


publication_gene_hits = []


print(
    "\n======================================"
)

print(
    "OFFICIAL FIGURE SOURCE-DATA AUDIT"
)

print(
    "======================================"
)


for label, url in source_workbooks.items():

    path = (
        source_data_dir
        /
        f"{label}.xlsx"
    )


    if not path.exists():

        print(
            "\nDownloading:",
            label
        )


        r = requests.get(
            url,
            headers=headers,
            timeout=120
        )


        r.raise_for_status()


        path.write_bytes(
            r.content
        )


    print(
        "\n",
        label,
        "| MB:",
        round(
            path.stat().st_size
            /
            1e6,
            4
        )
    )


    if path.read_bytes()[
        :2
    ] != b"PK":

        raise RuntimeError(

            f"{label} is not a valid XLSX file."
        )


    xls = pd.ExcelFile(
        path
    )


    print(
        "Sheets:",
        xls.sheet_names
    )


    for sheet in xls.sheet_names:

        df = pd.read_excel(
            path,
            sheet_name=sheet,
            header=None
        )


        print(
            "  ",
            sheet,
            "=>",
            df.shape
        )


        # Convert every cell to text and search exact gene symbols.
        text_df = df.fillna(
            ""
        ).astype(
            str
        )


        for gene in sorted(
            candidate_ligands
        ):

            gene_regex = re.compile(

                rf"(?<![A-Za-z0-9])"
                rf"{re.escape(gene)}"
                rf"(?![A-Za-z0-9])",

                flags=re.IGNORECASE
            )


            row_hits = []


            for row_idx in range(
                len(
                    text_df
                )
            ):

                row_text = " | ".join(

                    text_df
                    .iloc[
                        row_idx
                    ]
                    .tolist()
                )


                if gene_regex.search(
                    row_text
                ):

                    row_hits.append(
                        row_idx
                    )


            for row_idx in row_hits:

                row_values = (

                    text_df
                    .iloc[
                        row_idx
                    ]
                    .tolist()
                )


                row_values = [

                    value

                    for value in row_values

                    if value
                    not in {
                        "",
                        "nan",
                        "None"
                    }
                ]


                publication_gene_hits.append({

                    "workbook":
                        label,

                    "sheet":
                        sheet,

                    "gene":
                        gene,

                    "excel_row_1based":
                        row_idx + 1,

                    "row_text":
                        " | ".join(
                            row_values
                        )[
                            :1500
                        ]
                })


publication_gene_hits = pd.DataFrame(
    publication_gene_hits
)


print(
    "\n======================================"
)

print(
    "CANDIDATE LIGANDS FOUND IN PUBLISHED SOURCE DATA"
)

print(
    "======================================"
)


if len(
    publication_gene_hits
):

    display(

        publication_gene_hits
        .sort_values(
            [
                "workbook",
                "sheet",
                "gene",
                "excel_row_1based"
            ]
        )
    )

else:

    print(
        "No candidate ligand symbols detected "
        "in Fig.1 / Extended Data Fig.1 source workbooks."
    )


publication_gene_hits.to_csv(

    OUT
    /
    "GSE275770_candidate_ligands_in_publication_source_data.csv",

    index=False
)


# ============================================================
# 11. DECISION CHECKPOINT
# ============================================================

print(
    "\n======================================"
)

print(
    "D9B REPLICATE-PROVENANCE DECISION"
)

print(
    "======================================"
)


print(
    "GEO sample records:",
    len(
        geo_sample_inventory
    )
)


print(
    "KP GSM records:",
    int(
        (
            geo_sample_inventory[
                "condition"
            ]
            ==
            "KP"
        )
        .sum()
    )
)


print(
    "WT GSM records:",
    int(
        (
            geo_sample_inventory[
                "condition"
            ]
            ==
            "WT"
        )
        .sum()
    )
)


print(
    "Distinct BioSamples:",
    geo_sample_inventory[
        "biosample"
    ]
    .nunique(
        dropna=True
    )
)


print(
    "Distinct SRA experiments:",
    geo_sample_inventory[
        "srx"
    ]
    .nunique(
        dropna=True
    )
)


print(
    "\nIMPORTANT:"
)

print(
    "Do NOT label the four GSM/SRA records per condition "
    "as independent animal replicates unless the metadata "
    "or author files explicitly map them to independent mice."
)


print(
    "\nPHASE D9B GSE275770 PROVENANCE AUDIT COMPLETE"
)

In [ ]:
# ============================================================
# PHASE D9C — AUTHOR-DERIVED LUNG-VSN RESPONSE OF
#             D8D CANDIDATE LIGANDS
#
# Purpose:
#
# Interrogate the authors' own cluster-6 KP3-vs-WT
# FindMarkers output:
#
#   Raw_Txt/Lung_KP3vsWT_DEG.csv
#
# and compare descriptively with:
#
#   Raw_Txt/Kcng1-_group_KP3_vs_group_WT.csv
#
# We also inspect:
#
#   Raw_Txt/DEG_heatmap_key_All_Kcng1_padj0.05_FC1.csv
#
# IMPORTANT:
#
# - Cluster 6 = lung-innervating Kcng1+ VSN compartment
#   according to the paper / author analysis.
#
# - Author comparison:
#       ident.1 = KP3
#       ident.2 = WT
#
#   therefore positive logFC = higher in KP3 tumour condition.
#
# - Paper DEG rule:
#       adjusted p < 0.05
#       |log2FC| > 1
#       expression >=10% in at least one group
#
# - This is AUTHOR CELL-LEVEL DE evidence.
#   It is NOT animal-level pseudobulk replication.
#
# - Presence in a CSV alone != differential expression.
#
# NO candidate ranking.
# NO novelty claim.
# ============================================================

from pathlib import Path

import base64
import io
import re
import requests
import numpy as np
import pandas as pd


print(
    "======================================"
)

print(
    "PHASE D9C — AUTHOR LUNG-VSN LIGAND RESPONSE"
)

print(
    "======================================"
)


# ============================================================
# 1. RESTORE PROJECT PATHS
# ============================================================

if "RAW" not in globals():

    if not Path(
        "/content/drive/MyDrive"
    ).exists():

        from google.colab import drive

        drive.mount(
            "/content/drive"
        )

    ROOT = Path(
        "/content/drive/MyDrive/KT_Neurovascular_Screen"
    )

    RAW = ROOT / "raw"
    OUT = ROOT / "outputs"


g275_dir = (
    RAW
    /
    "GSE275770"
)


author_txt_dir = (
    g275_dir
    /
    "author_Raw_Txt"
)


author_txt_dir.mkdir(
    parents=True,
    exist_ok=True
)


headers = {

    "User-Agent":
        "Mozilla/5.0 KT-neurovascular-screen/1.0"
}


# ============================================================
# 2. RESTORE D8D EXACT LIGAND-RECEPTOR SET
# ============================================================

d8d_lr_path = (

    OUT
    /
    "D8D_11candidate_PSN_ligand_source_evidence.csv"
)


if not d8d_lr_path.exists():

    raise FileNotFoundError(
        d8d_lr_path
    )


candidate_psn_lr = pd.read_csv(
    d8d_lr_path
)


required_d8d = {

    "mouse_receptor",
    "mouse_ligand",

    "psn_support_tier",
    "psn_expression_score",
    "psn_mean_tpm",

    "psn_top_subtype",
    "psn_top_subtype_n_cells",
    "psn_small_top_subtype_flag"
}


missing = (

    required_d8d
    -
    set(
        candidate_psn_lr.columns
    )
)


if missing:

    raise RuntimeError(

        "D8D ligand table missing columns: "
        +
        str(
            sorted(
                missing
            )
        )
    )


candidate_ligands = sorted(

    candidate_psn_lr[
        "mouse_ligand"
    ]
    .dropna()
    .astype(str)
    .unique()
)


print(
    "D8D LR rows:",
    len(
        candidate_psn_lr
    )
)


print(
    "Unique candidate ligands:",
    len(
        candidate_ligands
    )
)


print(
    candidate_ligands
)


# ============================================================
# 3. FETCH AUTHOR Raw_Txt FILES FROM GITHUB
# ============================================================

repo_tree_url = (

    "https://api.github.com/repos/"
    "KWhh666/Manuscript_2024/"
    "git/trees/main?recursive=1"
)


tree_response = requests.get(
    repo_tree_url,
    headers=headers,
    timeout=60
)


tree_response.raise_for_status()


tree = tree_response.json().get(
    "tree",
    []
)


tree_lookup = {

    item.get(
        "path"
    ):
        item

    for item in tree
}


target_files = {

    "lung":

        "Raw_Txt/Lung_KP3vsWT_DEG.csv",

    "nonlung":

        "Raw_Txt/Kcng1-_group_KP3_vs_group_WT.csv",

    "heatmap_key":

        "Raw_Txt/DEG_heatmap_key_All_Kcng1_padj0.05_FC1.csv"
}


def fetch_github_blob(
    item
):

    r = requests.get(
        item[
            "url"
        ],
        headers=headers,
        timeout=60
    )

    r.raise_for_status()

    payload = r.json()

    return base64.b64decode(
        payload[
            "content"
        ]
    )


local_paths = {}


print(
    "\n======================================"
)

print(
    "AUTHOR Raw_Txt DOWNLOAD"
)

print(
    "======================================"
)


for key, repo_path in target_files.items():

    if repo_path not in tree_lookup:

        raise FileNotFoundError(

            f"Repository file not found: {repo_path}"
        )


    local_path = (

        author_txt_dir
        /
        Path(
            repo_path
        ).name
    )


    local_paths[
        key
    ] = local_path


    if not local_path.exists():

        content = fetch_github_blob(
            tree_lookup[
                repo_path
            ]
        )

        local_path.write_bytes(
            content
        )


    print(
        key,
        "=>",
        local_path.name,
        "| bytes:",
        local_path.stat().st_size
    )


# ============================================================
# 4. ROBUST TABLE READER
#
# Some author ".csv" files are actually tab-delimited.
# Let pandas infer separator.
# ============================================================

def read_author_table(
    path
):

    text = path.read_text(
        encoding="utf-8",
        errors="replace"
    )


    df = pd.read_csv(

        io.StringIO(
            text
        ),

        sep=None,
        engine="python"
    )


    return df


author_tables = {}


for key, path in local_paths.items():

    df = read_author_table(
        path
    )


    author_tables[
        key
    ] = df


    print(
        "\n======================================"
    )

    print(
        key.upper(),
        "AUTHOR TABLE"
    )

    print(
        "======================================"
    )


    print(
        "Shape:",
        df.shape
    )


    print(
        "Columns:"
    )


    for col in df.columns:

        print(
            " -",
            repr(
                col
            )
        )


    print(
        "\nFirst 5 rows:"
    )


    display(
        df.head(
            5
        )
    )


# ============================================================
# 5. IDENTIFY GENE COLUMN
# ============================================================

def infer_gene_column(
    df,
    expected_genes
):

    expected_upper = {

        str(g).upper()

        for g in expected_genes
    }


    best_col = None
    best_overlap = -1


    for col in df.columns:

        vals = (

            df[
                col
            ]
            .dropna()
            .astype(str)
            .str.upper()
        )


        overlap = len(

            set(
                vals
            )
            &
            expected_upper
        )


        if overlap > best_overlap:

            best_overlap = overlap
            best_col = col


    return (
        best_col,
        best_overlap
    )


gene_columns = {}


for key, df in author_tables.items():

    gene_col, overlap = infer_gene_column(

        df,
        candidate_ligands
    )


    gene_columns[
        key
    ] = gene_col


    print(
        "\n",
        key,
        "| inferred gene column:",
        repr(
            gene_col
        ),
        "| candidate overlap:",
        overlap
    )


# ============================================================
# 6. STANDARDIZE AUTHOR DE TABLE
# ============================================================

def first_existing_column(
    df,
    names
):

    lower_map = {

        str(c).lower():
            c

        for c in df.columns
    }


    for name in names:

        if name.lower() in lower_map:

            return lower_map[
                name.lower()
            ]


    return None


def standardize_de_table(
    df,
    gene_col,
    label
):

    out = df.copy()


    out[
        "gene"
    ] = (

        out[
            gene_col
        ]
        .astype(str)
        .str.strip()
    )


    logfc_col = first_existing_column(

        out,

        [
            "avg_log2FC",
            "avg_logFC",
            "avg_log2fc",
            "avg_logfc",
            "log2FC",
            "logFC"
        ]
    )


    p_col = first_existing_column(

        out,

        [
            "p_val",
            "pvalue",
            "p_value",
            "p"
        ]
    )


    padj_col = first_existing_column(

        out,

        [
            "p_val_adj",
            "padj",
            "p_adj",
            "adj.P.Val",
            "FDR"
        ]
    )


    pct1_col = first_existing_column(

        out,

        [
            "pct.1",
            "pct_1",
            "pct1"
        ]
    )


    pct2_col = first_existing_column(

        out,

        [
            "pct.2",
            "pct_2",
            "pct2"
        ]
    )


    print(
        "\n",
        label,
        "resolved fields:"
    )

    print(
        "  gene:",
        repr(
            gene_col
        )
    )

    print(
        "  logFC:",
        repr(
            logfc_col
        )
    )

    print(
        "  p:",
        repr(
            p_col
        )
    )

    print(
        "  adjusted p:",
        repr(
            padj_col
        )
    )

    print(
        "  pct.1:",
        repr(
            pct1_col
        )
    )

    print(
        "  pct.2:",
        repr(
            pct2_col
        )
    )


    if logfc_col is None:

        raise RuntimeError(

            f"{label}: could not identify logFC column."
        )


    if padj_col is None:

        raise RuntimeError(

            f"{label}: could not identify adjusted-p column."
        )


    out[
        "author_log2fc"
    ] = pd.to_numeric(

        out[
            logfc_col
        ],

        errors="coerce"
    )


    if p_col is not None:

        out[
            "author_p"
        ] = pd.to_numeric(

            out[
                p_col
            ],

            errors="coerce"
        )

    else:

        out[
            "author_p"
        ] = np.nan


    out[
        "author_padj"
    ] = pd.to_numeric(

        out[
            padj_col
        ],

        errors="coerce"
    )


    if pct1_col is not None:

        out[
            "pct_kp3"
        ] = pd.to_numeric(

            out[
                pct1_col
            ],

            errors="coerce"
        )

    else:

        out[
            "pct_kp3"
        ] = np.nan


    if pct2_col is not None:

        out[
            "pct_wt"
        ] = pd.to_numeric(

            out[
                pct2_col
            ],

            errors="coerce"
        )

    else:

        out[
            "pct_wt"
        ] = np.nan


    # --------------------------------------------------------
    # Author comparison:
    #
    # ident.1 = KP3
    # ident.2 = WT
    #
    # therefore:
    #
    #   positive FC = higher in tumour-bearing VSNs
    #   negative FC = lower in tumour-bearing VSNs
    # --------------------------------------------------------

    out[
        "direction"
    ] = np.select(

        [
            out[
                "author_log2fc"
            ] > 0,

            out[
                "author_log2fc"
            ] < 0
        ],

        [
            "KP3_up",
            "KP3_down"
        ],

        default="zero"
    )


    # --------------------------------------------------------
    # Paper-reported DEG rule:
    #
    #   adjusted P < 0.05
    #   |log2FC| > 1
    #   >=10% expression in at least one group
    #
    # If pct columns are absent, do NOT silently pretend the
    # expression criterion was verified.
    # --------------------------------------------------------

    has_pct = (

        pct1_col is not None
        and
        pct2_col is not None
    )


    if has_pct:

        out[
            "max_pct"
        ] = (

            out[
                [
                    "pct_kp3",
                    "pct_wt"
                ]
            ]
            .max(
                axis=1
            )
        )


        out[
            "meets_paper_deg_rule"
        ] = (

            out[
                "author_padj"
            ]
            .lt(
                0.05
            )

            &

            out[
                "author_log2fc"
            ]
            .abs()
            .gt(
                1.0
            )

            &

            out[
                "max_pct"
            ]
            .gt(
                0.10
            )
        )


    else:

        out[
            "max_pct"
        ] = np.nan


        out[
            "meets_paper_deg_rule"
        ] = False


    return out


lung_de = standardize_de_table(

    author_tables[
        "lung"
    ],

    gene_columns[
        "lung"
    ],

    "LUNG CLUSTER-6 KP3 vs WT"
)


nonlung_de = standardize_de_table(

    author_tables[
        "nonlung"
    ],

    gene_columns[
        "nonlung"
    ],

    "Kcng1- KP3 vs WT"
)


# ============================================================
# 7. CHECK WHETHER GENE IDENTIFIERS ARE UNIQUE
# ============================================================

print(
    "\n======================================"
)

print(
    "GENE UNIQUENESS CHECK"
)

print(
    "======================================"
)


print(
    "Lung duplicate gene rows:",
    int(
        lung_de[
            "gene"
        ]
        .duplicated()
        .sum()
    )
)


print(
    "Non-lung duplicate gene rows:",
    int(
        nonlung_de[
            "gene"
        ]
        .duplicated()
        .sum()
    )
)


# ============================================================
# 8. EXTRACT THE D8D CANDIDATE LIGANDS
# ============================================================

candidate_upper = {

    gene.upper():
        gene

    for gene in candidate_ligands
}


def extract_candidate_rows(
    df
):

    temp = df.copy()


    temp[
        "gene_upper"
    ] = (

        temp[
            "gene"
        ]
        .astype(str)
        .str.upper()
    )


    return (

        temp[
            temp[
                "gene_upper"
            ]
            .isin(
                candidate_upper.keys()
            )
        ]
        .copy()
    )


lung_candidates = extract_candidate_rows(
    lung_de
)


nonlung_candidates = extract_candidate_rows(
    nonlung_de
)


print(
    "\n======================================"
)

print(
    "CANDIDATE LIGAND COVERAGE"
)

print(
    "======================================"
)


print(
    "Candidate ligands:",
    len(
        candidate_ligands
    )
)


print(
    "Listed in lung cluster-6 DE table:",
    lung_candidates[
        "gene_upper"
    ].nunique()
)


print(
    "Listed in Kcng1- DE table:",
    nonlung_candidates[
        "gene_upper"
    ].nunique()
)


print(
    "Meeting paper DEG rule in lung cluster 6:",
    int(
        lung_candidates[
            "meets_paper_deg_rule"
        ]
        .sum()
    )
)


print(
    "Meeting same operational rule in Kcng1- comparator:",
    int(
        nonlung_candidates[
            "meets_paper_deg_rule"
        ]
        .sum()
    )
)


# ============================================================
# 9. LUNG-INNERVATING VSN CANDIDATE LIGAND RESULTS
# ============================================================

print(
    "\n======================================"
)

print(
    "D8D LIGANDS IN LUNG-INNERVATING VSNs"
)

print(
    "======================================"
)


lung_display = [

    "gene",
    "author_log2fc",
    "direction",

    "pct_kp3",
    "pct_wt",

    "author_p",
    "author_padj",

    "meets_paper_deg_rule"
]


display(

    lung_candidates[
        lung_display
    ]
    .sort_values(

        [
            "meets_paper_deg_rule",
            "author_padj",
            "gene"
        ],

        ascending=[
            False,
            True,
            True
        ]
    )
)


# ============================================================
# 10. SIGNIFICANT CANDIDATE LIGANDS UNDER PAPER RULE
# ============================================================

print(
    "\n======================================"
)

print(
    "CANDIDATE LIGANDS MEETING PAPER DEG RULE"
)

print(
    "======================================"
)


lung_sig_candidates = (

    lung_candidates[
        lung_candidates[
            "meets_paper_deg_rule"
        ]
    ]
    .copy()
)


if len(
    lung_sig_candidates
):

    display(

        lung_sig_candidates[
            lung_display
        ]
        .sort_values(

            "author_log2fc",

            ascending=False
        )
    )

else:

    print(
        "No D8D candidate ligand meets the full "
        "paper DEG rule in cluster 6."
    )


# ============================================================
# 11. IMPORTANT AXES — EXACT SOURCE-SIDE RESPONSE
# ============================================================

priority_ligands = [

    # ADGRL3
    "Flrt1",
    "Flrt2",
    "Flrt3",

    # EDNRB
    "Edn1",
    "Edn2",
    "Edn3",

    # FZD4
    "Ndp",
    "Wnt5a",

    # PLXNC1
    "Sema7a",

    # NRP2 / semaphorin comparison
    "Sema3a",
    "Sema3b",
    "Sema3c",
    "Sema3f",

    "Pgf",
    "Vegfa",
    "Vegfc"
]


print(
    "\n======================================"
)

print(
    "PRIORITY AXES — LUNG-VSN SOURCE RESPONSE"
)

print(
    "======================================"
)


priority_lung = (

    lung_candidates[
        lung_candidates[
            "gene"
        ]
        .isin(
            priority_ligands
        )
    ]
    .copy()
)


display(

    priority_lung[
        lung_display
    ]
    .sort_values(
        "gene"
    )
)


# ============================================================
# 12. Kcng1- COMPARATOR
#
# Descriptive only.
#
# This is NOT a formal interaction test between lung and
# non-lung neurons.
# ============================================================

print(
    "\n======================================"
)

print(
    "Kcng1- COMPARATOR — CANDIDATE LIGANDS"
)

print(
    "======================================"
)


nonlung_display = [

    "gene",
    "author_log2fc",
    "direction",

    "pct_kp3",
    "pct_wt",

    "author_p",
    "author_padj",

    "meets_paper_deg_rule"
]


display(

    nonlung_candidates[
        nonlung_display
    ]
    .sort_values(

        [
            "meets_paper_deg_rule",
            "author_padj",
            "gene"
        ],

        ascending=[
            False,
            True,
            True
        ]
    )
)


# ============================================================
# 13. MERGE LUNG + NON-LUNG DESCRIPTIVE RESPONSE
# ============================================================

lung_small = (

    lung_candidates[
        [
            "gene",
            "author_log2fc",
            "direction",
            "pct_kp3",
            "pct_wt",
            "author_p",
            "author_padj",
            "meets_paper_deg_rule"
        ]
    ]
    .copy()
    .rename(
        columns={

            "author_log2fc":
                "lung_log2fc",

            "direction":
                "lung_direction",

            "pct_kp3":
                "lung_pct_kp3",

            "pct_wt":
                "lung_pct_wt",

            "author_p":
                "lung_p",

            "author_padj":
                "lung_padj",

            "meets_paper_deg_rule":
                "lung_meets_paper_deg_rule"
        }
    )
)


nonlung_small = (

    nonlung_candidates[
        [
            "gene",
            "author_log2fc",
            "direction",
            "pct_kp3",
            "pct_wt",
            "author_p",
            "author_padj",
            "meets_paper_deg_rule"
        ]
    ]
    .copy()
    .rename(
        columns={

            "author_log2fc":
                "nonlung_log2fc",

            "direction":
                "nonlung_direction",

            "pct_kp3":
                "nonlung_pct_kp3",

            "pct_wt":
                "nonlung_pct_wt",

            "author_p":
                "nonlung_p",

            "author_padj":
                "nonlung_padj",

            "meets_paper_deg_rule":
                "nonlung_meets_paper_deg_rule"
        }
    )
)


ligand_response = (

    pd.DataFrame({

        "mouse_ligand":
            candidate_ligands
    })

    .merge(

        lung_small.rename(
            columns={
                "gene":
                    "mouse_ligand"
            }
        ),

        on="mouse_ligand",
        how="left"
    )

    .merge(

        nonlung_small.rename(
            columns={
                "gene":
                    "mouse_ligand"
            }
        ),

        on="mouse_ligand",
        how="left"
    )
)


ligand_response[
    "lung_threshold_only"
] = (

    ligand_response[
        "lung_meets_paper_deg_rule"
    ]
    .fillna(False)

    &

    ~ligand_response[
        "nonlung_meets_paper_deg_rule"
    ]
    .fillna(False)
)


ligand_response[
    "both_compartments_threshold"
] = (

    ligand_response[
        "lung_meets_paper_deg_rule"
    ]
    .fillna(False)

    &

    ligand_response[
        "nonlung_meets_paper_deg_rule"
    ]
    .fillna(False)
)


# ============================================================
# 14. MERGE BACK RECEPTOR + PSN SOURCE EVIDENCE
# ============================================================

axis_response = (

    candidate_psn_lr

    .merge(

        ligand_response,

        on="mouse_ligand",
        how="left"
    )
)


print(
    "\n======================================"
)

print(
    "RECEPTOR–LIGAND NEURONAL RESPONSE MATRIX"
)

print(
    "======================================"
)


axis_columns = [

    "mouse_receptor",
    "mouse_ligand",

    "psn_support_tier",
    "psn_expression_score",
    "psn_mean_tpm",

    "psn_top_subtype",
    "psn_top_subtype_n_cells",
    "psn_small_top_subtype_flag",

    "lung_log2fc",
    "lung_direction",
    "lung_pct_kp3",
    "lung_pct_wt",
    "lung_p",
    "lung_padj",
    "lung_meets_paper_deg_rule",

    "nonlung_log2fc",
    "nonlung_direction",
    "nonlung_padj",
    "nonlung_meets_paper_deg_rule",

    "lung_threshold_only",
    "both_compartments_threshold"
]


display(

    axis_response[
        axis_columns
    ]
    .sort_values(

        [
            "lung_meets_paper_deg_rule",
            "mouse_receptor",
            "mouse_ligand"
        ],

        ascending=[
            False,
            True,
            True
        ]
    )
)


# ============================================================
# 15. AUTHOR HEATMAP KEY
#
# Determine whether any candidate ligand was selected into
# the authors' published cluster-6 heatmap key.
# ============================================================

heatmap_df = author_tables[
    "heatmap_key"
].copy()


heatmap_gene_col = gene_columns[
    "heatmap_key"
]


heatmap_genes = set(

    heatmap_df[
        heatmap_gene_col
    ]
    .dropna()
    .astype(str)
    .str.upper()
)


heatmap_candidate_hits = [

    gene

    for gene in candidate_ligands

    if gene.upper()
    in
    heatmap_genes
]


print(
    "\n======================================"
)

print(
    "AUTHOR HEATMAP-KEY CANDIDATE HITS"
)

print(
    "======================================"
)


print(
    "Candidate ligands appearing in author heatmap key:",
    len(
        heatmap_candidate_hits
    )
)


print(
    sorted(
        heatmap_candidate_hits
    )
)


# ============================================================
# 16. SUMMARY
# ============================================================

print(
    "\n======================================"
)

print(
    "D9C NEURONAL-SOURCE SUMMARY"
)

print(
    "======================================"
)


print(
    "D8D candidate ligands:",
    len(
        candidate_ligands
    )
)


print(
    "Listed in lung cluster-6 author DE table:",
    int(
        ligand_response[
            "lung_log2fc"
        ]
        .notna()
        .sum()
    )
)


print(
    "Meet paper DEG rule in lung cluster 6:",
    int(
        ligand_response[
            "lung_meets_paper_deg_rule"
        ]
        .fillna(False)
        .sum()
    )
)


print(
    "Meet same threshold in Kcng1- comparator:",
    int(
        ligand_response[
            "nonlung_meets_paper_deg_rule"
        ]
        .fillna(False)
        .sum()
    )
)


print(
    "Meet threshold in lung cluster 6 but not Kcng1-:",
    int(
        ligand_response[
            "lung_threshold_only"
        ]
        .sum()
    )
)


print(
    "Candidate ligands in published author heatmap key:",
    len(
        heatmap_candidate_hits
    )
)


# ============================================================
# 17. SAVE
# ============================================================

ligand_response.to_csv(

    OUT
    /
    "GSE275770_D9C_candidate_ligand_author_DE_response.csv",

    index=False
)


axis_response.to_csv(

    OUT
    /
    "GSE275770_D9C_receptor_ligand_neuronal_response_matrix.csv",

    index=False
)


# ============================================================
# 18. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE D9C AUTHOR LUNG-VSN LIGAND RESPONSE COMPLETE"
)

In [ ]:
# ============================================================
# PHASE D9D — SEMA3C RECEPTOR-COMPLEX AUDIT
#
# Biological question:
#
# Is the tumour-responsive neuronal ligand Sema3c connected
# to a plausible vascular receptor complex in our metastatic
# and aging datasets?
#
# Core complex components:
#
#   Nrp2
#   Plxnd1
#   Plxna1
#
# Rationale:
#
# SEMA3C signaling generally requires neuropilin + plexin
# context; NRP2 alone is not sufficient evidence of a
# functional SEMA3C pathway.
#
# THIS CELL:
#
#   1. restores D9C neuronal Sema3c response
#   2. extracts replicated 6 h metastatic evidence
#      for Nrp2 / Plxnd1 where available
#   3. checks mouse blood-EC aging for all 3 components
#   4. checks composition-adjusted mouse aging
#   5. interrogates human GLMM + pseudobulk aging
#      in six vascular compartments
#
# NO integrated ranking.
# NO causal claim.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


print(
    "======================================"
)

print(
    "PHASE D9D — SEMA3C RECEPTOR-COMPLEX AUDIT"
)

print(
    "======================================"
)


# ============================================================
# 1. RESTORE PATHS
# ============================================================

if "OUT" not in globals():

    if not Path(
        "/content/drive/MyDrive"
    ).exists():

        from google.colab import drive

        drive.mount(
            "/content/drive"
        )


    ROOT = Path(
        "/content/drive/MyDrive/KT_Neurovascular_Screen"
    )

    RAW = ROOT / "raw"
    OUT = ROOT / "outputs"


# ============================================================
# 2. DEFINE SEMA3C RECEPTOR COMPLEX
# ============================================================

mouse_complex = [

    "Nrp2",
    "Plxnd1",
    "Plxna1"
]


human_complex = {

    "Nrp2":
        "NRP2",

    "Plxnd1":
        "PLXND1",

    "Plxna1":
        "PLXNA1"
}


print(
    "Mouse components:",
    mouse_complex
)


# ============================================================
# 3. RESTORE D7B METASTASIS × AGING TABLE
# ============================================================

convergence_path = (

    OUT
    /
    "PSN_receptor_metastasis_replication_with_aging_evidence.csv"
)


if not convergence_path.exists():

    raise FileNotFoundError(
        convergence_path
    )


convergence = pd.read_csv(
    convergence_path
)


# ============================================================
# 4. RESTORE D9C NEURONAL LIGAND RESPONSE
# ============================================================

d9c_path = (

    OUT
    /
    "GSE275770_D9C_candidate_ligand_author_DE_response.csv"
)


if not d9c_path.exists():

    raise FileNotFoundError(
        d9c_path
    )


d9c_ligands = pd.read_csv(
    d9c_path
)


sema3c_source = (

    d9c_ligands[
        d9c_ligands[
            "mouse_ligand"
        ]
        ==
        "Sema3c"
    ]
    .copy()
)


print(
    "\n======================================"
)

print(
    "NEURONAL SEMA3C SOURCE RESPONSE"
)

print(
    "======================================"
)


display(
    sema3c_source
)


# ============================================================
# 5. EARLY METASTATIC EC RESPONSE
#
# convergence contains only the 60 directionally replicated
# receptors. Therefore absence here means "not among the 60",
# NOT "not expressed".
# ============================================================

print(
    "\n======================================"
)

print(
    "METASTATIC 6 h RECEPTOR-COMPLEX RESPONSE"
)

print(
    "======================================"
)


met_cols = [

    "gene",
    "PSN_ligands",

    "set1_GSE235394_delta_log2cpm1",
    "set2_GSE253749_delta_log2cpm1",
    "set3_GSE253751_delta_log2cpm1",

    "median_6h_delta"
]


met_cols = [

    c
    for c in met_cols
    if c in convergence.columns
]


metastasis_complex = (

    convergence[
        convergence[
            "gene"
        ]
        .isin(
            mouse_complex
        )
    ][
        met_cols
    ]
    .copy()
)


display(
    metastasis_complex
    .sort_values(
        "gene"
    )
)


for gene in mouse_complex:

    if gene not in set(
        metastasis_complex[
            "gene"
        ]
    ):

        print(
            gene,
            "=> NOT among the 60 replicated metastatic receptors."
        )


# ============================================================
# 6. MOUSE BLOOD-EC AGING
# ============================================================

mouse_age_path = (

    OUT
    /
    "GSE124872_bloodEC_mouselevel_age_statistics.csv"
)


if not mouse_age_path.exists():

    raise FileNotFoundError(
        mouse_age_path
    )


mouse_age = pd.read_csv(
    mouse_age_path
)


mouse_age_complex = (

    mouse_age[
        mouse_age[
            "gene"
        ]
        .isin(
            mouse_complex
        )
    ]
    .copy()
)


print(
    "\n======================================"
)

print(
    "MOUSE BLOOD-EC AGING — RECEPTOR COMPLEX"
)

print(
    "======================================"
)


mouse_age_cols = [

    "gene",

    "n_young_animals",
    "n_aged_animals",

    "n_young_detected",
    "n_aged_detected",

    "young_mean_log2cpm",
    "aged_mean_log2cpm",

    "age_delta_log2cpm",

    "ci95_low",
    "ci95_high",

    "hedges_g",

    "p_value",
    "fdr_bh"
]


mouse_age_cols = [

    c
    for c in mouse_age_cols
    if c in mouse_age_complex.columns
]


display(

    mouse_age_complex[
        mouse_age_cols
    ]
    .sort_values(
        "gene"
    )
)


# ============================================================
# 7. COMPOSITION-ADJUSTED MOUSE AGING
# ============================================================

comp_path = (

    OUT
    /
    "GSE124872_bloodEC_composition_adjusted_age_sensitivity.csv"
)


if not comp_path.exists():

    raise FileNotFoundError(
        comp_path
    )


comp_age = pd.read_csv(
    comp_path
)


comp_complex = (

    comp_age[
        comp_age[
            "gene"
        ]
        .isin(
            mouse_complex
        )
    ]
    .copy()
)


print(
    "\n======================================"
)

print(
    "COMPOSITION-ADJUSTED MOUSE AGING"
)

print(
    "======================================"
)


comp_cols = [

    "gene",

    "age_delta_log2cpm",
    "fdr_bh",

    "composition_adjusted_age_coef",
    "composition_adjusted_age_se_HC3",

    "composition_adjusted_ci95_low",
    "composition_adjusted_ci95_high",

    "composition_adjusted_p_value",
    "composition_adjusted_fdr_bh",

    "age_signal_class"
]


comp_cols = [

    c
    for c in comp_cols
    if c in comp_complex.columns
]


display(

    comp_complex[
        comp_cols
    ]
    .sort_values(
        "gene"
    )
)


# ============================================================
# 8. HUMAN AGING WORKBOOKS
# ============================================================

human_dir = (

    RAW
    /
    "GSE281219_publication_supplement"
)


glmm_path = (

    human_dir
    /
    "DeMan2026_Supplementary_Data_2.xlsx"
)


pb_path = (

    human_dir
    /
    "DeMan2026_Supplementary_Data_3.xlsx"
)


if not glmm_path.exists():

    raise FileNotFoundError(
        glmm_path
    )


if not pb_path.exists():

    raise FileNotFoundError(
        pb_path
    )


human_celltypes = [

    "Aerocyte",
    "gCap",
    "Arterial",
    "Venous",
    "Pericyte",
    "SMC"
]


human_rows = []


for ct in human_celltypes:

    glmm = pd.read_excel(
        glmm_path,
        sheet_name=ct
    )


    pb = pd.read_excel(
        pb_path,
        sheet_name=ct
    )


    glmm[
        "gene_key"
    ] = (

        glmm[
            "Gene"
        ]
        .astype(str)
        .str.upper()
    )


    pb[
        "gene_key"
    ] = (

        pb[
            "gene"
        ]
        .astype(str)
        .str.upper()
    )


    glmm_lookup = (

        glmm
        .set_index(
            "gene_key"
        )
    )


    pb_lookup = (

        pb
        .set_index(
            "gene_key"
        )
    )


    for mouse_gene in mouse_complex:

        human_gene = human_complex[
            mouse_gene
        ]


        # ---------------- GLMM ----------------

        if human_gene in glmm_lookup.index:

            gr = glmm_lookup.loc[
                human_gene
            ]


            # D8B established:
            # Beta1 orientation is opposite to old-vs-young.
            glmm_age_effect = (

                -1.0
                *
                float(
                    gr[
                        "Beta1"
                    ]
                )
            )


            glmm_p = float(
                gr[
                    "p_val"
                ]
            )


            glmm_fdr = float(
                gr[
                    "p_val_adj"
                ]
            )


        else:

            glmm_age_effect = np.nan
            glmm_p = np.nan
            glmm_fdr = np.nan


        # ---------------- pseudobulk ----------------

        if human_gene in pb_lookup.index:

            pr = pb_lookup.loc[
                human_gene
            ]


            pb_logfc = float(
                pr[
                    "avg_logFC"
                ]
            )


            pb_p = float(
                pr[
                    "p_val"
                ]
            )


            pb_fdr = float(
                pr[
                    "p_val_adj"
                ]
            )


            old_pct = float(
                pr[
                    "old.pct"
                ]
            )


            young_pct = float(
                pr[
                    "young.pct"
                ]
            )


        else:

            pb_logfc = np.nan
            pb_p = np.nan
            pb_fdr = np.nan
            old_pct = np.nan
            young_pct = np.nan


        both_available = (

            pd.notna(
                glmm_age_effect
            )
            and
            pd.notna(
                pb_logfc
            )
        )


        same_direction = (

            both_available

            and

            np.sign(
                glmm_age_effect
            )
            ==
            np.sign(
                pb_logfc
            )
        )


        nominal_both = (

            same_direction

            and

            glmm_p < 0.05

            and

            pb_p < 0.05
        )


        fdr_both = (

            same_direction

            and

            glmm_fdr < 0.05

            and

            pb_fdr < 0.05
        )


        human_rows.append({

            "mouse_gene":
                mouse_gene,

            "human_gene":
                human_gene,

            "celltype":
                ct,

            "glmm_age_effect_old_vs_young":
                glmm_age_effect,

            "glmm_p":
                glmm_p,

            "glmm_fdr":
                glmm_fdr,

            "pseudobulk_old_vs_young_logFC":
                pb_logfc,

            "pseudobulk_p":
                pb_p,

            "pseudobulk_fdr":
                pb_fdr,

            "old_pct":
                old_pct,

            "young_pct":
                young_pct,

            "crossmodel_same_direction":
                same_direction,

            "crossmodel_nominal_p05":
                nominal_both,

            "crossmodel_fdr05":
                fdr_both
        })


human_complex_age = pd.DataFrame(
    human_rows
)


print(
    "\n======================================"
)

print(
    "HUMAN VASCULAR AGING — RECEPTOR COMPLEX"
)

print(
    "======================================"
)


display(

    human_complex_age
    .sort_values(
        [
            "mouse_gene",
            "celltype"
        ]
    )
)


# ============================================================
# 9. HUMAN NOMINAL CROSS-MODEL HITS ONLY
# ============================================================

print(
    "\n======================================"
)

print(
    "HUMAN NOMINAL AGE SIGNALS — COMPLEX"
)

print(
    "======================================"
)


human_nominal = (

    human_complex_age[
        human_complex_age[
            "crossmodel_nominal_p05"
        ]
    ]
    .copy()
)


if len(
    human_nominal
):

    display(

        human_nominal
        .sort_values(
            [
                "mouse_gene",
                "celltype"
            ]
        )
    )

else:

    print(
        "No complex component has nominal p<0.05 "
        "in both human-aging models with concordant direction."
    )


# ============================================================
# 10. COMPACT COMPLEX SUMMARY
# ============================================================

summary_rows = []


for gene in mouse_complex:

    met = (

        metastasis_complex[
            metastasis_complex[
                "gene"
            ]
            ==
            gene
        ]
    )


    ma = (

        mouse_age_complex[
            mouse_age_complex[
                "gene"
            ]
            ==
            gene
        ]
    )


    ca = (

        comp_complex[
            comp_complex[
                "gene"
            ]
            ==
            gene
        ]
    )


    ha = (

        human_complex_age[
            human_complex_age[
                "mouse_gene"
            ]
            ==
            gene
        ]
    )


    hn = (

        ha[
            ha[
                "crossmodel_nominal_p05"
            ]
        ]
    )


    summary_rows.append({

        "gene":
            gene,

        "replicated_metastasis_60":
            len(
                met
            ) > 0,

        "median_6h_delta":
            (
                float(
                    met[
                        "median_6h_delta"
                    ]
                    .iloc[
                        0
                    ]
                )
                if len(
                    met
                )
                else np.nan
            ),

        "mouse_age_delta":
            (
                float(
                    ma[
                        "age_delta_log2cpm"
                    ]
                    .iloc[
                        0
                    ]
                )
                if len(
                    ma
                )
                else np.nan
            ),

        "mouse_age_fdr":
            (
                float(
                    ma[
                        "fdr_bh"
                    ]
                    .iloc[
                        0
                    ]
                )
                if len(
                    ma
                )
                else np.nan
            ),

        "composition_adjusted_age_coef":
            (
                float(
                    ca[
                        "composition_adjusted_age_coef"
                    ]
                    .iloc[
                        0
                    ]
                )
                if len(
                    ca
                )
                else np.nan
            ),

        "composition_adjusted_fdr":
            (
                float(
                    ca[
                        "composition_adjusted_fdr_bh"
                    ]
                    .iloc[
                        0
                    ]
                )
                if len(
                    ca
                )
                else np.nan
            ),

        "human_nominal_age_celltypes":
            ";".join(
                hn[
                    "celltype"
                ]
                .tolist()
            ),

        "n_human_nominal_age_celltypes":
            len(
                hn
            ),

        "any_human_dual_fdr05":
            bool(
                ha[
                    "crossmodel_fdr05"
                ]
                .any()
            )
    })


sema3c_complex_summary = pd.DataFrame(
    summary_rows
)


print(
    "\n======================================"
)

print(
    "SEMA3C COMPLEX SUMMARY"
)

print(
    "======================================"
)


display(
    sema3c_complex_summary
)


# ============================================================
# 11. SAVE
# ============================================================

human_complex_age.to_csv(

    OUT
    /
    "D9D_SEMA3C_NRP2_plexin_human_aging_detail.csv",

    index=False
)


sema3c_complex_summary.to_csv(

    OUT
    /
    "D9D_SEMA3C_NRP2_plexin_complex_summary.csv",

    index=False
)


# ============================================================
# 12. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE D9D SEMA3C RECEPTOR-COMPLEX AUDIT COMPLETE"
)

In [ ]:
# ============================================================
# PHASE D9E — HEAD-TO-HEAD AXIS EVIDENCE AUDIT
#
# Compare:
#
#   AXIS A:
#       Sema3c -> Nrp2 / Plxnd1 / Plxna1 complex
#
#   AXIS B:
#       Flrt3 -> Adgrl3
#
# Purpose:
#
#   - correct D9D mouse-aging extraction by explicitly using
#     celltype == "bloodEC_all"
#
#   - compare neuronal source response
#   - compare 3-experiment metastatic receptor response
#   - compare mouse aging
#   - compare composition-adjusted mouse aging
#   - compare human vascular aging
#
# NO numeric integrated score.
# NO winner assigned by code.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


print(
    "======================================"
)

print(
    "PHASE D9E — TWO-AXIS EVIDENCE AUDIT"
)

print(
    "======================================"
)


# ============================================================
# 1. RESTORE PROJECT PATHS
# ============================================================

if "OUT" not in globals():

    if not Path(
        "/content/drive/MyDrive"
    ).exists():

        from google.colab import drive

        drive.mount(
            "/content/drive"
        )

    ROOT = Path(
        "/content/drive/MyDrive/KT_Neurovascular_Screen"
    )

    RAW = ROOT / "raw"
    OUT = ROOT / "outputs"


# ============================================================
# 2. AXIS DEFINITIONS
# ============================================================

axis_definitions = [

    {
        "axis":
            "SEMA3C_NRP2_PLEXIN",

        "ligand":
            "Sema3c",

        "receptor":
            "Nrp2",

        "component_role":
            "neuropilin_receptor"
    },

    {
        "axis":
            "SEMA3C_NRP2_PLEXIN",

        "ligand":
            "Sema3c",

        "receptor":
            "Plxnd1",

        "component_role":
            "plexin_coreceptor"
    },

    {
        "axis":
            "SEMA3C_NRP2_PLEXIN",

        "ligand":
            "Sema3c",

        "receptor":
            "Plxna1",

        "component_role":
            "plexin_coreceptor"
    },

    {
        "axis":
            "FLRT3_ADGRL3",

        "ligand":
            "Flrt3",

        "receptor":
            "Adgrl3",

        "component_role":
            "adhesion_GPCR_receptor"
    }
]


axis = pd.DataFrame(
    axis_definitions
)


display(
    axis
)


# ============================================================
# 3. RESTORE D9C LUNG-VSN LIGAND RESPONSE
# ============================================================

d9c_path = (

    OUT
    /
    "GSE275770_D9C_candidate_ligand_author_DE_response.csv"
)


if not d9c_path.exists():

    raise FileNotFoundError(
        d9c_path
    )


d9c = pd.read_csv(
    d9c_path
)


source_cols = [

    "mouse_ligand",

    "lung_log2fc",
    "lung_direction",
    "lung_pct_kp3",
    "lung_pct_wt",
    "lung_p",
    "lung_padj",
    "lung_meets_paper_deg_rule",

    "nonlung_log2fc",
    "nonlung_direction",
    "nonlung_padj",
    "nonlung_meets_paper_deg_rule"
]


source_cols = [

    c
    for c in source_cols
    if c in d9c.columns
]


source = (

    d9c[
        d9c[
            "mouse_ligand"
        ]
        .isin(
            [
                "Sema3c",
                "Flrt3"
            ]
        )
    ][
        source_cols
    ]
    .copy()
    .rename(
        columns={
            "mouse_ligand":
                "ligand"
        }
    )
)


print(
    "\n======================================"
)

print(
    "NEURONAL-SOURCE COMPARISON"
)

print(
    "======================================"
)


display(
    source
    .sort_values(
        "ligand"
    )
)


# ============================================================
# 4. RESTORE BASELINE PSN SOURCE EXPRESSION
# ============================================================

psn_path = (

    OUT
    /
    "D8D_11candidate_PSN_ligand_source_evidence.csv"
)


if not psn_path.exists():

    raise FileNotFoundError(
        psn_path
    )


psn = pd.read_csv(
    psn_path
)


psn_source = (

    psn[
        psn[
            "mouse_ligand"
        ]
        .isin(
            [
                "Sema3c",
                "Flrt3"
            ]
        )
    ]
    .copy()
)


# One ligand may occur with multiple receptors.
# Aggregate source properties at ligand level deterministically.

psn_source_summary = (

    psn_source

    .sort_values(
        "psn_mean_tpm",
        ascending=False
    )

    .groupby(
        "mouse_ligand",
        as_index=False
    )

    .first()

    .rename(
        columns={
            "mouse_ligand":
                "ligand"
        }
    )
)


psn_keep = [

    "ligand",

    "psn_support_tier",
    "psn_expression_score",
    "psn_mean_tpm",

    "psn_max_subtype_mean",
    "psn_max_subtype_pct_tpm_ge_1",

    "psn_top_subtype",
    "psn_top_subtype_n_cells",
    "psn_small_top_subtype_flag"
]


psn_keep = [

    c
    for c in psn_keep
    if c in psn_source_summary.columns
]


psn_source_summary = (

    psn_source_summary[
        psn_keep
    ]
)


print(
    "\n======================================"
)

print(
    "BASELINE PSN SOURCE EXPRESSION"
)

print(
    "======================================"
)


display(
    psn_source_summary
)


# ============================================================
# 5. RESTORE THREE-EXPERIMENT METASTATIC RESPONSE
# ============================================================

conv_path = (

    OUT
    /
    "PSN_receptor_metastasis_replication_with_aging_evidence.csv"
)


if not conv_path.exists():

    raise FileNotFoundError(
        conv_path
    )


conv = pd.read_csv(
    conv_path
)


receptors = sorted(
    axis[
        "receptor"
    ]
    .unique()
)


metastasis_cols = [

    "gene",

    "set1_GSE235394_delta_log2cpm1",
    "set2_GSE253749_delta_log2cpm1",
    "set3_GSE253751_delta_log2cpm1",

    "median_6h_delta",
    "min_abs_6h_delta",

    "directionally_replicated_ge1pct",
    "expressed_ge5pct_all_three"
]


metastasis_cols = [

    c
    for c in metastasis_cols
    if c in conv.columns
]


metastasis = (

    conv[
        conv[
            "gene"
        ]
        .isin(
            receptors
        )
    ][
        metastasis_cols
    ]
    .copy()
    .rename(
        columns={
            "gene":
                "receptor"
        }
    )
)


# Explicit rows for components absent from the replicated-60 table.

met_missing = (

    sorted(
        set(
            receptors
        )
        -
        set(
            metastasis[
                "receptor"
            ]
        )
    )
)


if met_missing:

    missing_rows = pd.DataFrame({

        "receptor":
            met_missing
    })


    metastasis = pd.concat(

        [
            metastasis,
            missing_rows
        ],

        ignore_index=True
    )


metastasis[
    "replicated_metastasis_60"
] = (

    metastasis[
        "receptor"
    ]
    .isin(
        set(
            conv[
                "gene"
            ]
        )
    )
)


print(
    "\n======================================"
)

print(
    "EARLY METASTATIC RECEPTOR RESPONSE"
)

print(
    "======================================"
)


display(

    metastasis
    .sort_values(
        "receptor"
    )
)


# ============================================================
# 6. CORRECT MOUSE AGING EXTRACTION
#
# IMPORTANT:
# D9D accidentally displayed all four blood-EC rows per gene.
#
# Here we explicitly require:
#
#       celltype == "bloodEC_all"
#
# ============================================================

mouse_age_path = (

    OUT
    /
    "GSE124872_bloodEC_mouselevel_age_statistics.csv"
)


mouse_age = pd.read_csv(
    mouse_age_path
)


if "celltype" not in mouse_age.columns:

    raise RuntimeError(
        "Mouse aging table lacks celltype column."
    )


mouse_age_all = (

    mouse_age[
        (
            mouse_age[
                "celltype"
            ]
            ==
            "bloodEC_all"
        )

        &

        mouse_age[
            "gene"
        ]
        .isin(
            receptors
        )
    ]
    .copy()
)


if mouse_age_all[
    "gene"
].duplicated().any():

    raise RuntimeError(
        "Duplicate bloodEC_all receptor rows detected."
    )


mouse_age_all = (

    mouse_age_all
    .rename(
        columns={
            "gene":
                "receptor",

            "age_delta_log2cpm":
                "mouse_bloodEC_age_delta",

            "p_value":
                "mouse_bloodEC_age_p",

            "fdr_bh":
                "mouse_bloodEC_age_fdr",

            "n_young_detected":
                "mouse_bloodEC_n_young_detected",

            "n_aged_detected":
                "mouse_bloodEC_n_aged_detected"
        }
    )
)


mouse_age_keep = [

    "receptor",

    "mouse_bloodEC_n_young_detected",
    "mouse_bloodEC_n_aged_detected",

    "mouse_bloodEC_age_delta",
    "mouse_bloodEC_age_p",
    "mouse_bloodEC_age_fdr"
]


mouse_age_keep = [

    c
    for c in mouse_age_keep
    if c in mouse_age_all.columns
]


mouse_age_all = (

    mouse_age_all[
        mouse_age_keep
    ]
)


print(
    "\n======================================"
)

print(
    "CORRECTED POOLED MOUSE BLOOD-EC AGING"
)

print(
    "======================================"
)


display(
    mouse_age_all
    .sort_values(
        "receptor"
    )
)


# ============================================================
# 7. COMPOSITION-ADJUSTED MOUSE AGING
# ============================================================

comp_path = (

    OUT
    /
    "GSE124872_bloodEC_composition_adjusted_age_sensitivity.csv"
)


comp = pd.read_csv(
    comp_path
)


comp_axis = (

    comp[
        comp[
            "gene"
        ]
        .isin(
            receptors
        )
    ]
    .copy()
    .rename(
        columns={
            "gene":
                "receptor",

            "composition_adjusted_age_coef":
                "mouse_comp_age_coef",

            "composition_adjusted_p_value":
                "mouse_comp_age_p",

            "composition_adjusted_fdr_bh":
                "mouse_comp_age_fdr"
        }
    )
)


comp_keep = [

    "receptor",

    "mouse_comp_age_coef",
    "mouse_comp_age_p",
    "mouse_comp_age_fdr",

    "age_signal_class"
]


comp_keep = [

    c
    for c in comp_keep
    if c in comp_axis.columns
]


comp_axis = (

    comp_axis[
        comp_keep
    ]
)


# ============================================================
# 8. HUMAN AGING — ADGRL3 FROM D8C
# ============================================================

d8c_path = (

    OUT
    /
    "DeMan2026_60receptors_author_aligned_human_age_long.csv"
)


if not d8c_path.exists():

    raise FileNotFoundError(
        d8c_path
    )


d8c = pd.read_csv(
    d8c_path
)


adgrl3_human = (

    d8c[
        d8c[
            "mouse_gene"
        ]
        ==
        "Adgrl3"
    ]
    .copy()
)


adgrl3_human[
    "crossmodel_nominal"
] = (

    adgrl3_human[
        "crossmodel_nominal_p05_same_direction"
    ]
    .astype(bool)
)


adgrl3_human = (

    adgrl3_human[
        adgrl3_human[
            "crossmodel_nominal"
        ]
    ]
    .copy()
)


adgrl3_human[
    "receptor"
] = "Adgrl3"


adgrl3_human[
    "human_age_direction"
] = np.where(

    adgrl3_human[
        "pseudobulk_old_vs_young_logFC"
    ]
    > 0,

    "higher_with_age",

    "lower_with_age"
)


# ============================================================
# 9. HUMAN AGING — SEMA3C COMPLEX FROM D9D
# ============================================================

d9d_human_path = (

    OUT
    /
    "D9D_SEMA3C_NRP2_plexin_human_aging_detail.csv"
)


if not d9d_human_path.exists():

    raise FileNotFoundError(
        d9d_human_path
    )


d9d_human = pd.read_csv(
    d9d_human_path
)


sema_human = (

    d9d_human[
        d9d_human[
            "crossmodel_nominal_p05"
        ]
        .astype(bool)
    ]
    .copy()
    .rename(
        columns={
            "mouse_gene":
                "receptor"
        }
    )
)


sema_human[
    "human_age_direction"
] = np.where(

    sema_human[
        "pseudobulk_old_vs_young_logFC"
    ]
    > 0,

    "higher_with_age",

    "lower_with_age"
)


# ============================================================
# 10. STANDARDIZE HUMAN NOMINAL SIGNALS
# ============================================================

adgrl3_standard = pd.DataFrame({

    "receptor":
        adgrl3_human[
            "receptor"
        ],

    "celltype":
        adgrl3_human[
            "celltype"
        ],

    "compartment":
        adgrl3_human[
            "compartment"
        ],

    "human_age_direction":
        adgrl3_human[
            "human_age_direction"
        ],

    "human_glmm_effect":
        adgrl3_human[
            "glmm_age_effect_oriented"
        ],

    "human_glmm_p":
        adgrl3_human[
            "glmm_p_value"
        ],

    "human_glmm_fdr":
        adgrl3_human[
            "glmm_fdr"
        ],

    "human_pseudobulk_logfc":
        adgrl3_human[
            "pseudobulk_old_vs_young_logFC"
        ],

    "human_pseudobulk_p":
        adgrl3_human[
            "pseudobulk_p_value"
        ],

    "human_pseudobulk_fdr":
        adgrl3_human[
            "pseudobulk_fdr"
        ]
})


sema_standard = pd.DataFrame({

    "receptor":
        sema_human[
            "receptor"
        ],

    "celltype":
        sema_human[
            "celltype"
        ],

    "compartment":
        np.where(
            sema_human[
                "celltype"
            ]
            .isin(
                [
                    "Aerocyte",
                    "gCap",
                    "Arterial",
                    "Venous"
                ]
            ),
            "endothelial",
            "mural"
        ),

    "human_age_direction":
        sema_human[
            "human_age_direction"
        ],

    "human_glmm_effect":
        sema_human[
            "glmm_age_effect_old_vs_young"
        ],

    "human_glmm_p":
        sema_human[
            "glmm_p"
        ],

    "human_glmm_fdr":
        sema_human[
            "glmm_fdr"
        ],

    "human_pseudobulk_logfc":
        sema_human[
            "pseudobulk_old_vs_young_logFC"
        ],

    "human_pseudobulk_p":
        sema_human[
            "pseudobulk_p"
        ],

    "human_pseudobulk_fdr":
        sema_human[
            "pseudobulk_fdr"
        ]
})


human_nominal = pd.concat(

    [
        adgrl3_standard,
        sema_standard
    ],

    ignore_index=True
)


print(
    "\n======================================"
)

print(
    "HUMAN CROSS-MODEL NOMINAL AGE SIGNALS"
)

print(
    "======================================"
)


display(

    human_nominal
    .sort_values(
        [
            "receptor",
            "celltype"
        ]
    )
)


# ============================================================
# 11. PER-RECEPTOR HUMAN SUMMARY
# ============================================================

human_summary_rows = []


for receptor in receptors:

    g = (

        human_nominal[
            human_nominal[
                "receptor"
            ]
            ==
            receptor
        ]
    )


    human_summary_rows.append({

        "receptor":
            receptor,

        "n_human_nominal_age_celltypes":
            len(
                g
            ),

        "n_human_nominal_endothelial":
            int(
                (
                    g[
                        "compartment"
                    ]
                    ==
                    "endothelial"
                )
                .sum()
            )
            if len(g)
            else 0,

        "n_human_nominal_mural":
            int(
                (
                    g[
                        "compartment"
                    ]
                    ==
                    "mural"
                )
                .sum()
            )
            if len(g)
            else 0,

        "human_nominal_age_celltypes":
            ";".join(
                g[
                    "celltype"
                ]
                .astype(str)
                .tolist()
            ),

        "human_age_directions":
            ";".join(
                g[
                    "human_age_direction"
                ]
                .astype(str)
                .tolist()
            ),

        "best_human_glmm_fdr":
            (
                g[
                    "human_glmm_fdr"
                ]
                .min()
                if len(g)
                else np.nan
            ),

        "best_human_pseudobulk_fdr":
            (
                g[
                    "human_pseudobulk_fdr"
                ]
                .min()
                if len(g)
                else np.nan
            )
    })


human_summary = pd.DataFrame(
    human_summary_rows
)


# ============================================================
# 12. BUILD COMPLETE AXIS-COMPONENT MATRIX
# ============================================================

comparison = (

    axis

    .merge(
        source,
        on="ligand",
        how="left"
    )

    .merge(
        psn_source_summary,
        on="ligand",
        how="left"
    )

    .merge(
        metastasis,
        on="receptor",
        how="left"
    )

    .merge(
        mouse_age_all,
        on="receptor",
        how="left"
    )

    .merge(
        comp_axis,
        on="receptor",
        how="left"
    )

    .merge(
        human_summary,
        on="receptor",
        how="left"
    )
)


# ============================================================
# 13. DIRECTION RELATION FLAGS
# ============================================================

comparison[
    "ligand_tumour_direction"
] = np.select(

    [
        comparison[
            "lung_log2fc"
        ] > 0,

        comparison[
            "lung_log2fc"
        ] < 0
    ],

    [
        "up",
        "down"
    ],

    default="not_available"
)


comparison[
    "receptor_metastasis_direction"
] = np.select(

    [
        comparison[
            "median_6h_delta"
        ] > 0,

        comparison[
            "median_6h_delta"
        ] < 0
    ],

    [
        "up",
        "down"
    ],

    default="not_replicated"
)


comparison[
    "mouse_age_direction"
] = np.select(

    [
        comparison[
            "mouse_bloodEC_age_delta"
        ] > 0,

        comparison[
            "mouse_bloodEC_age_delta"
        ] < 0
    ],

    [
        "higher_with_age",
        "lower_with_age"
    ],

    default="not_available"
)


# ============================================================
# 14. DISPLAY HEAD-TO-HEAD MATRIX
# ============================================================

display_cols = [

    "axis",
    "ligand",
    "receptor",
    "component_role",

    "psn_support_tier",
    "psn_mean_tpm",
    "psn_top_subtype",
    "psn_top_subtype_n_cells",

    "lung_log2fc",
    "lung_padj",
    "lung_meets_paper_deg_rule",
    "ligand_tumour_direction",

    "set1_GSE235394_delta_log2cpm1",
    "set2_GSE253749_delta_log2cpm1",
    "set3_GSE253751_delta_log2cpm1",
    "median_6h_delta",
    "replicated_metastasis_60",
    "receptor_metastasis_direction",

    "mouse_bloodEC_age_delta",
    "mouse_bloodEC_age_p",
    "mouse_bloodEC_age_fdr",
    "mouse_age_direction",

    "mouse_comp_age_coef",
    "mouse_comp_age_p",
    "mouse_comp_age_fdr",

    "n_human_nominal_age_celltypes",
    "n_human_nominal_endothelial",
    "n_human_nominal_mural",
    "human_nominal_age_celltypes",
    "human_age_directions",

    "best_human_glmm_fdr",
    "best_human_pseudobulk_fdr"
]


display_cols = [

    c
    for c in display_cols
    if c in comparison.columns
]


print(
    "\n======================================"
)

print(
    "HEAD-TO-HEAD AXIS MATRIX"
)

print(
    "======================================"
)


display(

    comparison[
        display_cols
    ]
    .sort_values(
        [
            "axis",
            "receptor"
        ]
    )
)


# ============================================================
# 15. AXIS-LEVEL DESCRIPTIVE SUMMARY
#
# This is evidence bookkeeping only.
# It does NOT calculate a winner or score.
# ============================================================

axis_summary_rows = []


for axis_name, g in comparison.groupby(
    "axis"
):

    axis_summary_rows.append({

        "axis":
            axis_name,

        "ligand":
            ";".join(
                sorted(
                    g[
                        "ligand"
                    ]
                    .dropna()
                    .unique()
                )
            ),

        "ligand_meets_author_DEG_rule":
            bool(
                g[
                    "lung_meets_paper_deg_rule"
                ]
                .fillna(False)
                .any()
            ),

        "ligand_best_padj":
            g[
                "lung_padj"
            ]
            .min(),

        "ligand_max_abs_log2fc":
            g[
                "lung_log2fc"
            ]
            .abs()
            .max(),

        "n_receptor_components":
            len(
                g
            ),

        "n_replicated_metastasis_components":
            int(
                g[
                    "replicated_metastasis_60"
                ]
                .fillna(False)
                .sum()
            ),

        "n_mouse_age_fdr05_components":
            int(
                g[
                    "mouse_bloodEC_age_fdr"
                ]
                .lt(
                    0.05
                )
                .sum()
            ),

        "n_composition_adjusted_fdr05_components":
            int(
                g[
                    "mouse_comp_age_fdr"
                ]
                .lt(
                    0.05
                )
                .sum()
            ),

        "n_human_nominal_endothelial_signals":
            int(
                g[
                    "n_human_nominal_endothelial"
                ]
                .fillna(
                    0
                )
                .sum()
            ),

        "n_human_nominal_mural_signals":
            int(
                g[
                    "n_human_nominal_mural"
                ]
                .fillna(
                    0
                )
                .sum()
            ),

        "human_nominal_celltypes":
            ";".join(

                sorted(

                    {

                        ct

                        for value in g[
                            "human_nominal_age_celltypes"
                        ]
                        .dropna()
                        .astype(str)

                        for ct in value.split(
                            ";"
                        )

                        if ct
                    }
                )
            )
    })


axis_summary = pd.DataFrame(
    axis_summary_rows
)


print(
    "\n======================================"
)

print(
    "AXIS-LEVEL EVIDENCE SUMMARY"
)

print(
    "======================================"
)


display(
    axis_summary
)


# ============================================================
# 16. EXPLICIT EVIDENCE GAPS
# ============================================================

print(
    "\n======================================"
)

print(
    "EVIDENCE GAPS"
)

print(
    "======================================"
)


print(
    "SEMA3C axis:"
)

print(
    "- neuronal tumour-response evidence is strong"
)

print(
    "- Nrp2 and Plxnd1 replicate during 6 h metastatic seeding"
)

print(
    "- Plxna1 does not satisfy the replicated-60 criterion"
)

print(
    "- no mouse complex component is age FDR<0.05 after "
    "composition adjustment"
)

print(
    "- human cross-model aging is currently limited to "
    "NRP2 in SMC"
)


print(
    "\nFLRT3–ADGRL3 axis:"
)

print(
    "- Flrt3 tumour response is FDR-significant but below "
    "the author's |log2FC| > 1 DEG cutoff"
)

print(
    "- Adgrl3 replicates during 6 h metastatic seeding"
)

print(
    "- human ADGRL3 aging occurs in Aerocyte and gCap"
)

print(
    "- mouse aging support may be absent / weak and must "
    "not be inferred from the human result"
)


# ============================================================
# 17. SAVE
# ============================================================

comparison.to_csv(

    OUT
    /
    "D9E_SEMA3C_vs_FLRT3_axis_component_matrix.csv",

    index=False
)


axis_summary.to_csv(

    OUT
    /
    "D9E_SEMA3C_vs_FLRT3_axis_summary.csv",

    index=False
)


human_nominal.to_csv(

    OUT
    /
    "D9E_SEMA3C_vs_FLRT3_human_age_detail.csv",

    index=False
)


# ============================================================
# 18. SANITY CHECKS
# ============================================================

assert set(
    comparison[
        "receptor"
    ]
) == {

    "Nrp2",
    "Plxnd1",
    "Plxna1",
    "Adgrl3"
}


assert not mouse_age_all[
    "receptor"
].duplicated().any()


assert all(

    mouse_age[
        mouse_age[
            "gene"
        ]
        .isin(
            receptors
        )
    ][
        "celltype"
    ]
    .notna()
)


# ============================================================
# 19. FINAL CHECKPOINT
# ============================================================

print(
    "\nPHASE D9E TWO-AXIS EVIDENCE AUDIT COMPLETE"
)

## Optional deep GSE253749 analysis

Set `DEEP_GSE253749 = True` later if we need endothelial subclusters/UMAP after the first ranking.

In [ ]:
if DEEP_GSE253749 and DOWNLOAD_GSE253749:
    adatas = []
    for prefix, fg in groups253.items():
        features = pd.read_csv(fg["features"], sep="\t", header=None, compression="gzip")
        genes_ = features.iloc[:,1].astype(str).tolist() if features.shape[1] >= 2 else features.iloc[:,0].astype(str).tolist()
        barcodes_ = pd.read_csv(fg["barcodes"], sep="\t", header=None, compression="gzip").iloc[:,0].astype(str).tolist()
        with gzip.open(fg["matrix"], "rb") as fh:
            M = mmread(fh).tocsr()
        if M.shape[0] == len(genes_):
            M = M.T.tocsr()
        a = ad.AnnData(
            X=M,
            obs=pd.DataFrame(index=[f"{prefix}{x}" for x in barcodes_]),
            var=pd.DataFrame(index=pd.Index(genes_, name="gene"))
        )
        a.var_names_make_unique()
        a.obs["condition"] = COND_253[prefix]
        a.obs["sample"] = prefix
        adatas.append(a)

    deep253 = ad.concat(adatas, join="outer", fill_value=0)
    sc.pp.filter_genes(deep253, min_cells=5)
    sc.pp.filter_cells(deep253, min_genes=1000)
    deep253.var["mt"] = deep253.var_names.str.startswith("mt-")
    deep253.var["ribo"] = deep253.var_names.str.startswith(("Rpl","Rps"))
    sc.pp.calculate_qc_metrics(deep253, qc_vars=["mt","ribo"], inplace=True)
    deep253 = deep253[
        (deep253.obs["n_genes_by_counts"] <= 5000) &
        (deep253.obs["pct_counts_mt"] < 5) &
        (deep253.obs["pct_counts_ribo"] < 20)
    ].copy()
    deep253.layers["counts"] = deep253.X.copy()
    sc.pp.normalize_total(deep253, target_sum=1e4)
    sc.pp.log1p(deep253)
    sc.pp.highly_variable_genes(deep253, n_top_genes=3000, batch_key="sample")
    sc.pp.scale(deep253, max_value=10)
    sc.tl.pca(deep253, use_highly_variable=True)
    sc.pp.neighbors(deep253, n_neighbors=15, n_pcs=30)
    sc.tl.umap(deep253)
    sc.tl.leiden(deep253, resolution=0.6)
    deep253.write_h5ad(CACHE / "GSE253749_deep_scanpy.h5ad")
    sc.pl.umap(deep253, color=["condition","leiden"], show=False)
    plt.savefig(FIG / "GSE253749_umap_condition_leiden.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("Deep GSE253749 analysis is OFF.")

# Phase E — pericyte and independent vascular support

In [ ]:
def read_count_table(path):
    df = pd.read_csv(path, sep=None, engine="python", compression="infer")
    first = df.columns[0]
    df[first] = df[first].astype(str)
    df = df.set_index(first)
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(axis=1, how="all").fillna(0)
    return df.groupby(df.index).sum()

def cpm_log1p(count_df):
    lib = count_df.sum(axis=0).replace(0, np.nan)
    cpm = count_df.divide(lib, axis=1) * 1e6
    return np.log1p(cpm.fillna(0))

def summarize_bulk_receptors(count_df, receptor_genes, dataset_name):
    x = cpm_log1p(count_df)
    genes = sorted(set(receptor_genes) & set(x.index))
    if not genes:
        return pd.DataFrame()
    sub = x.loc[genes]
    return pd.DataFrame({
        "gene": sub.index,
        "mean_log1p_cpm": sub.mean(axis=1).values,
        "max_log1p_cpm": sub.max(axis=1).values,
        "pct_samples_detected": (count_df.loc[genes] > 0).mean(axis=1).values,
        "dataset": dataset_name
    })

small_files = {
    "GSE137363_pericyte": RAW / "GSE137363" / "GSE137363_processed_data_counts.txt.gz",
    "GSE210291_lung_EC": RAW / "GSE210291" / "GSE210291_Count_matrix.tsv.gz",
    "GSE278974_B16F10_vascular_stromal": RAW / "GSE278974" / "GSE278974_B16F10_counts_matrix.csv.gz",
}

loaded_bulk = {}
small_bulk_outputs = []
for name, path in small_files.items():
    try:
        df = read_count_table(path)
        loaded_bulk[name] = df
        print(name, df.shape)
        small_bulk_outputs.append(summarize_bulk_receptors(df, candidate_receptors, name))
    except Exception as e:
        print(name, "loader issue:", e)

bulk_receptor_summary = pd.concat(small_bulk_outputs, ignore_index=True) if small_bulk_outputs else pd.DataFrame()
bulk_receptor_summary.to_csv(OUT / "small_bulk_receptor_support.csv", index=False)
display(bulk_receptor_summary.head(50))

In [ ]:
for name, df in loaded_bulk.items():
    print("\n", name)
    print(list(df.columns))

In [ ]:
pericyte_response = pd.DataFrame()

if "GSE137363_pericyte" in loaded_bulk:
    df = loaded_bulk["GSE137363_pericyte"]
    x = cpm_log1p(df)
    genes = sorted(set(candidate_receptors) & set(x.index))
    med_cols = [c for c in x.columns if re.search(r"Med", c, flags=re.I)]
    inj_cols = [c for c in x.columns if re.search(r"Inj", c, flags=re.I)]
    un_cols = [c for c in x.columns if re.search(r"Un", c, flags=re.I)]

    if med_cols and inj_cols:
        pericyte_response = pd.DataFrame(index=genes)
        pericyte_response["media_mean"] = x.loc[genes, med_cols].mean(axis=1)
        pericyte_response["injured_BALF_mean"] = x.loc[genes, inj_cols].mean(axis=1)
        pericyte_response["injured_minus_media"] = pericyte_response["injured_BALF_mean"] - pericyte_response["media_mean"]
        if un_cols:
            pericyte_response["uninjured_BALF_mean"] = x.loc[genes, un_cols].mean(axis=1)
            pericyte_response["uninjured_minus_media"] = pericyte_response["uninjured_BALF_mean"] - pericyte_response["media_mean"]
        pericyte_response = pericyte_response.reset_index().rename(columns={"index":"gene"})

pericyte_response.to_csv(OUT / "GSE137363_pericyte_receptor_responsiveness.csv", index=False)
display(pericyte_response.head(60))

# Phase F — convergent ranking

The score is for **prioritization**, not a claim of biological truth, novelty, or causality.

In [ ]:
age_rank = age_effect_overall.copy()
if len(age_rank):
    age_rank["age_effect_abs"] = age_rank["age_delta_mean"].abs()
    age_rank["age_effect_pctile"] = age_rank["age_effect_abs"].rank(pct=True)
else:
    age_rank = pd.DataFrame(columns=["gene","age_effect_pctile"])

met_rank = g253_effect.copy()
if len(met_rank):
    met_rank["met_effect_pctile"] = met_rank["max_abs_metastatic_delta"].rank(pct=True)
else:
    met_rank = pd.DataFrame(columns=["gene","met_effect_pctile"])

if len(pericyte_response):
    peri_rank = pericyte_response[["gene","injured_minus_media"]].copy()
    peri_rank["pericyte_response_abs"] = peri_rank["injured_minus_media"].abs()
    peri_rank["pericyte_response_pctile"] = peri_rank["pericyte_response_abs"].rank(pct=True)
else:
    peri_rank = pd.DataFrame(columns=["gene","pericyte_response_pctile"])

if len(bulk_receptor_summary):
    bulk_support = bulk_receptor_summary.groupby("gene").agg(
        independent_datasets_detected=("dataset","nunique"),
        independent_max_expression=("max_log1p_cpm","max")
    ).reset_index()
    bulk_support["independent_support_pctile"] = bulk_support["independent_datasets_detected"].rank(pct=True)
else:
    bulk_support = pd.DataFrame(columns=["gene","independent_support_pctile"])

In [ ]:
ranked = lr_psn_detected.copy()

ranked = ranked.merge(
    age_rank[["gene","age_effect_pctile","age_delta_mean","age_delta_pct"]].rename(columns={"gene":"mouse_receptor"}),
    on="mouse_receptor", how="left"
)

if len(met_rank):
    keep = ["gene","met_effect_pctile"] + [c for c in ["6h_delta_mean","30h_delta_mean","6h_delta_pct","30h_delta_pct"] if c in met_rank.columns]
    ranked = ranked.merge(met_rank[keep].rename(columns={"gene":"mouse_receptor"}), on="mouse_receptor", how="left")
else:
    ranked["met_effect_pctile"] = np.nan

if len(peri_rank):
    ranked = ranked.merge(
        peri_rank[["gene","pericyte_response_pctile","injured_minus_media"]].rename(columns={"gene":"mouse_receptor"}),
        on="mouse_receptor", how="left"
    )
else:
    ranked["pericyte_response_pctile"] = np.nan

if len(bulk_support):
    ranked = ranked.merge(
        bulk_support[["gene","independent_support_pctile","independent_datasets_detected"]].rename(columns={"gene":"mouse_receptor"}),
        on="mouse_receptor", how="left"
    )
else:
    ranked["independent_support_pctile"] = np.nan

ranked["orthology_confidence"] = 1 / np.sqrt(
    ranked["ligand_orthology_n"].clip(lower=1) *
    ranked["receptor_orthology_n"].clip(lower=1)
)

score_components = {
    "psn_expression_score": 0.30,
    "age_effect_pctile": 0.25,
    "met_effect_pctile": 0.30,
    "pericyte_response_pctile": 0.05,
    "independent_support_pctile": 0.05,
    "orthology_confidence": 0.05,
}

weighted_sum = np.zeros(len(ranked), dtype=float)
weight_used = np.zeros(len(ranked), dtype=float)

for col, w in score_components.items():
    vals = pd.to_numeric(ranked[col], errors="coerce")
    ok = vals.notna().values
    weighted_sum[ok] += w * vals[ok].values
    weight_used[ok] += w

ranked["evidence_score"] = np.divide(
    weighted_sum, weight_used,
    out=np.full_like(weighted_sum, np.nan),
    where=weight_used > 0
)
ranked["evidence_weight_observed"] = weight_used

ranked = ranked.sort_values("evidence_score", ascending=False).drop_duplicates(
    ["mouse_ligand","mouse_receptor"], keep="first"
)

ranked.to_csv(OUT / "ranked_neurovascular_axes_pre_novelty.csv", index=False)

display_cols = [
    "mouse_ligand","mouse_receptor","evidence_score",
    "psn_expression_score","psn_top_subtype",
    "age_effect_pctile","age_delta_mean",
    "met_effect_pctile","pericyte_response_pctile",
    "independent_support_pctile","ligand_orthology_n","receptor_orthology_n"
]
display(ranked[[c for c in display_cols if c in ranked.columns]].head(100))

# Phase G — literature collision / novelty audit

Novelty is **not** inferred from expression. The initial collision flags identify areas we already know are crowded and need especially strict literature review.

In [ ]:
collision = pd.DataFrame([
    {"ligand_pattern":r"^Tac1$|^TAC1$", "receptor_pattern":r"^Tacr1$|^TACR1$", "collision_level":"high", "reason":"Substance P–TACR1 sensory-neuron metastasis literature"},
    {"ligand_pattern":r"^Calca$|^Calcb$|^CGRP", "receptor_pattern":r"^Calcrl$|^Ramp1$", "collision_level":"high", "reason":"CGRP lung-cancer neuroimmune literature"},
    {"ligand_pattern":r"catechol|norepine|epine|^Dbh$|^Th$", "receptor_pattern":r"^Adrb", "collision_level":"high", "reason":"Adrenergic metastasis literature"},
    {"ligand_pattern":r"^Apln$|^Apela$", "receptor_pattern":r"^Aplnr$", "collision_level":"high", "reason":"Ashik vascular-aging/metastasis axis"},
])

def collision_flag(row):
    hits = []
    for _, c in collision.iterrows():
        if (
            re.search(c["ligand_pattern"], str(row["mouse_ligand"]), flags=re.I) and
            re.search(c["receptor_pattern"], str(row["mouse_receptor"]), flags=re.I)
        ):
            hits.append((c["collision_level"], c["reason"]))
    if not hits:
        return pd.Series(["not_preflagged",""])
    return pd.Series([hits[0][0], " | ".join(x[1] for x in hits)])

ranked[["collision_level","collision_reason"]] = ranked.apply(collision_flag, axis=1)
ranked.to_csv(OUT / "ranked_neurovascular_axes_with_collision_flags.csv", index=False)

top30 = ranked.head(30).copy()
review_cols = [
    "mouse_ligand","mouse_receptor","evidence_score",
    "psn_expression_score","psn_top_subtype","psn_pct_nonzero",
    "age_effect_pctile","age_delta_mean","age_delta_pct",
    "met_effect_pctile","pericyte_response_pctile",
    "independent_support_pctile","collision_level","collision_reason"
]
review_cols += [c for c in ["6h_delta_mean","30h_delta_mean","6h_delta_pct","30h_delta_pct"] if c in top30.columns]
review_cols = [c for c in review_cols if c in top30.columns]

top30_review = top30[review_cols]
top30_review.to_csv(OUT / "TOP30_neurovascular_axes_for_manual_review.csv", index=False)
display(top30_review)

In [ ]:
plot_df = ranked.head(25).copy()
plot_df["axis"] = plot_df["mouse_ligand"] + " → " + plot_df["mouse_receptor"]
plot_df = plot_df.iloc[::-1]

plt.figure(figsize=(9,9))
plt.barh(plot_df["axis"], plot_df["evidence_score"])
plt.xlabel("Integrated evidence score")
plt.ylabel("")
plt.title("Top pulmonary neural → vascular candidate axes")
plt.tight_layout()
plt.savefig(FIG / "top25_neurovascular_axes.png", dpi=220, bbox_inches="tight")
plt.show()

# Phase H — optional replication/validation

After the top candidates are known, select only datasets that answer a specific question:

- **GSE235394** — replicate early B16F10 endothelial response.
- **GSE281219** — human lung aging validation.
- **GSE275770** — tumour-induced changes in vagal sensory-neuron ligands.
- **GSE275763** — adrenergic benchmark/collision.
- **GSE233052 / GSE216592** — liver vascular aging/generalization.

Do not download these blindly before the first ranking.

In [ ]:
def download_geo_supplementary(accession, filename, force=False):
    return download_file(
        geo_suppl_url(accession, filename),
        RAW / accession / filename,
        force=force
    )

if RUN_OPTIONAL_VALIDATION:
    for acc in OPTIONAL_ACCESSIONS:
        try:
            base, files = list_geo_supplementary(acc)
            print(f"\n### {acc}")
            for f in files:
                print(f)
        except Exception as e:
            print(acc, e)
else:
    print("Optional validation is OFF until top candidates are known.")

# Final checkpoint before proposal drafting

Do not draft until:

1. At least one axis has convergent pulmonary-neuron, vascular receptor, aging, and metastatic-seeding support.
2. The direction of the age effect comes from data rather than assumption.
3. Top candidates have been checked individually against cancer-neuroscience, vascular-metastasis, pericyte, and Ashik/Augustin literature.
4. The wet-lab plan retains a true **screening/discovery** structure.
5. The project can be reduced to one discovery screen + causal validation + focused in-vivo test.

In [ ]:
import zipfile

bundle = OUT / "KT_neurovascular_screen_results.zip"
include_names = [
    "download_manifest.csv",
    "file_integrity_audit.csv",
    "software_versions.csv",
    "GSE186180_PSN_gene_expression_scores.csv",
    "GSE186180_PSN_subtype_mean_expression.csv",
    "omnipath_mouse_ligand_receptor_pairs.csv",
    "PSN_detected_ligand_receptor_pairs.csv",
    "GSE124872_vascular_receptor_expression_by_age.csv",
    "GSE124872_vascular_receptor_age_effect_overall.csv",
    "GSE124872_vascular_receptor_age_effect_by_celltype.csv",
    "GSE124872_pseudobulk_receptor_age_effects.csv",
    "GSE253749_receptor_expression_by_condition.csv",
    "GSE253749_receptor_early_metastasis_effects.csv",
    "GSE137363_pericyte_receptor_responsiveness.csv",
    "small_bulk_receptor_support.csv",
    "ranked_neurovascular_axes_pre_novelty.csv",
    "ranked_neurovascular_axes_with_collision_flags.csv",
    "TOP30_neurovascular_axes_for_manual_review.csv",
]

with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for name in include_names:
        p = OUT / name
        if p.exists():
            z.write(p, arcname=name)
    for p in FIG.glob("*.png"):
        z.write(p, arcname=f"figures/{p.name}")

print("Bundle:", bundle)
print("Size MB:", bundle.stat().st_size/1e6)
print("Upload ONLY this ZIP back to ChatGPT.")

## Interpretation discipline

- Expression ≠ functional signalling.
- Age-associated expression change ≠ causal aging mechanism.
- Individual cells are not biological replicates.
- Low RNA abundance does not automatically exclude receptor function.
- Multi-subunit receptors require all necessary components.
- The integrated score is a **screening priority**, not a novelty or causality score.
- The top computational hits still require functional screening and targeted literature validation.